In [1]:
# ============================================================
# 0. Validation Notebook Setup
# ============================================================

# import duckdb
import joblib
import numpy as np
import pandas as pd

from config import (
    DB_PATH,
    MODEL_PATH,
    EXCLUDE_SEASON_ID,
)

from feature_engineering import build_features
from prediction_model.inference import score_dataframe


# ============================================================
# 0A. Database Connection
# ============================================================
DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\data\duckdb\nhl_data.duckdb"
con = duckdb.connect(DB_PATH) 
print("Connected:", DB_PATH)


# ============================================================
# 0B. Load Current Production Model
# ============================================================

bundle = joblib.load(MODEL_PATH)

FEATURE_COLS = bundle["features"]

print(f"Model: {MODEL_PATH}")
print(f"Model version: {bundle.get('version')}")
print(f"Model type: {bundle.get('model_type')}")
print(f"Model features: {len(FEATURE_COLS)}")


# ============================================================
# 0C. Validation Configuration
# ============================================================

print(f"Excluded season: {EXCLUDE_SEASON_ID}")



pd.options.mode.chained_assignment = None

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

NameError: name 'duckdb' is not defined

In [ ]:
# 1. Load rebuilt historical training data
team_daily_stats = con.execute("""
    SELECT *
    FROM team_daily_stats
    WHERE season_id != ?
    ORDER BY season_id, game_date, team_id
""", [EXCLUDE_SEASON_ID]).df()

print(f"Rows: {len(team_daily_stats):,}")
print(
    f"Seasons: "
    f"{team_daily_stats['season_id'].min()} → "
    f"{team_daily_stats['season_id'].max()}"
)

team_daily_stats.groupby("season_id").agg(
    rows=("game_id", "size"),
    games=("game_id", "nunique"),
    teams=("team_id", "nunique"),
)

In [ ]:
# # ============================================================
# # Restore made_playoffs from all_clinch_history
# # ============================================================

# con.execute("""
#     UPDATE team_daily_stats AS s
#     SET made_playoffs =
#         CASE
#             WHEN c.clinch_type IN ('x', 'y') THEN 1
#             WHEN c.clinch_type = 'e' THEN 0
#             ELSE NULL
#         END
#     FROM all_clinch_history AS c
#     WHERE s.season_id = c.season_id
#       AND s.tri_code = c.tri_code
# """)

In [ ]:
con.execute("""
    SELECT
        made_playoffs,
        COUNT(*) AS rows
    FROM team_daily_stats
    GROUP BY made_playoffs
    ORDER BY made_playoffs
""").df()

In [ ]:
# ============================================================
# 2. Rebuild features
# ============================================================

from feature_engineering import build_features


df_features = build_features(
    team_daily_stats.copy()
)

print(f"Feature rows: {len(df_features):,}")
print(f"Available columns: {len(df_features.columns)}")

In [ ]:
# ============================================================
# 3. Validate team-game keys
# ============================================================

key_cols = [
    "season_id",
    "game_id",
    "team_id",
]

# Duplicate team-game rows
duplicates = team_daily_stats[
    team_daily_stats.duplicated(
        subset=key_cols,
        keep=False,
    )
]

print(f"Duplicate team-game rows: {len(duplicates):,}")

# Null keys
null_keys = team_daily_stats[
    team_daily_stats[key_cols].isna().any(axis=1)
]

print(f"Rows with null keys: {len(null_keys):,}")

# Every game should have exactly 2 teams
teams_per_game = (
    team_daily_stats
    .groupby(["season_id", "game_id"])["team_id"]
    .nunique()
)

bad_games = teams_per_game[teams_per_game != 2]

print(f"Games without exactly 2 teams: {len(bad_games):,}")

if not bad_games.empty:
    display(bad_games.head(20))

In [ ]:
# ============================================================
# 4. Validate model feature contract
# ============================================================

import joblib
from config import MODEL_PATH

# Load current production model bundle
bundle = joblib.load(MODEL_PATH)

FEATURE_COLS = bundle["features"]

# ------------------------------------------------------------
# Check required features
# ------------------------------------------------------------

missing_features = [
    col
    for col in FEATURE_COLS
    if col not in df_features.columns
]

extra_features = [
    col
    for col in df_features.columns
    if col not in FEATURE_COLS
]

print(f"Model features: {len(FEATURE_COLS)}")
print(f"Available columns: {len(df_features.columns)}")

print("\nMissing required features:")
print(missing_features)

print(
    f"\nExtra columns not used by model: "
    f"{len(extra_features)}"
)

# ------------------------------------------------------------
# Fail if model contract is broken
# ------------------------------------------------------------

assert not missing_features, (
    "❌ Required model features are missing: "
    + ", ".join(missing_features)
)

print("\n✅ Feature contract validated successfully.")

In [ ]:
# ============================================================
# 5. Validate season progression
# ============================================================

check = (
    df_features
    .sort_values(
        ["season_id", "team_id", "games_played", "game_id"]
    )
    .copy()
)


# ------------------------------------------------------------
# 1. games_played should advance exactly 1 per team game
# ------------------------------------------------------------

check["prev_games_played"] = (
    check
    .groupby(["season_id", "team_id"])["games_played"]
    .shift(1)
)

bad_games_played = check[
    check["prev_games_played"].notna()
    & (
        check["games_played"]
        != check["prev_games_played"] + 1
    )
].copy()

print(
    "Rows with invalid games_played progression:",
    len(bad_games_played)
)


# ------------------------------------------------------------
# 2. games_before should equal games_played - 1
#
# This is the canonical Game N timing check:
# after Game N, games_played = N and games_before = N - 1
# ------------------------------------------------------------

bad_games_before = check[
    check["games_before"]
    != check["games_played"] - 1
].copy()

print(
    "Rows where games_before != games_played - 1:",
    len(bad_games_before)
)


# ------------------------------------------------------------
# 3. Inspect games_remaining progression
#
# This is diagnostic only.
#
# games_remaining comes from the standings snapshot and can
# occasionally remain unchanged at season-end boundaries.
# Therefore we inspect exceptions but do not fail validation
# solely because games_remaining did not decrease by exactly 1.
# ------------------------------------------------------------

check["prev_games_remaining"] = (
    check
    .groupby(["season_id", "team_id"])["games_remaining"]
    .shift(1)
)

bad_remaining = check[
    check["prev_games_remaining"].notna()
    & (
        check["games_remaining"]
        != check["prev_games_remaining"] - 1
    )
].copy()

print(
    "Rows where games_remaining did not decrease by 1:",
    len(bad_remaining)
)

if not bad_remaining.empty:

    bad_remaining["remaining_change"] = (
        bad_remaining["games_remaining"]
        - bad_remaining["prev_games_remaining"]
    )

    display(
        bad_remaining[
            [
                "season_id",
                "game_date",
                "game_id",
                "team_id",
                "tri_code",
                "games_played",
                "prev_games_remaining",
                "games_remaining",
                "remaining_change",
            ]
        ]
        .sort_values(
            ["season_id", "team_id", "games_played"]
        )
    )


# ------------------------------------------------------------
# 4. First game should be Game 1 / games_before = 0
# ------------------------------------------------------------

first_games = (
    check
    .groupby(
        ["season_id", "team_id"],
        as_index=False,
    )
    .first()
)

bad_first_games = first_games[
    (first_games["games_played"] != 1)
    | (first_games["games_before"] != 0)
].copy()

print(
    "Team-seasons not starting at Game 1:",
    len(bad_first_games)
)


# ------------------------------------------------------------
# 5. Fail only on canonical progression problems
# ------------------------------------------------------------

assert bad_games_played.empty, (
    f"❌ {len(bad_games_played):,} rows have invalid "
    "games_played progression."
)

assert bad_games_before.empty, (
    f"❌ {len(bad_games_before):,} rows have incorrect "
    "games_before."
)

assert bad_first_games.empty, (
    f"❌ {len(bad_first_games):,} team-seasons do not "
    "start at Game 1 / games_before = 0."
)


print("\n✅ Canonical team-season progression is correct.")

if bad_remaining.empty:
    print("✅ games_remaining also decreases by 1 on every row.")
else:
    print(
        f"ℹ️ {len(bad_remaining):,} games_remaining exceptions "
        "found and displayed above for review."
    )

In [ ]:
# ============================================================
# 6. Score Full Historical Dataset
# ============================================================

from prediction_model.inference import score_dataframe


# ------------------------------------------------------------
# Score corrected historical features using current model
#
# NOTE:
# This is a pipeline / compatibility check only.
# The production model was trained on the previous dataset
# and will be retrained after historical validation is complete.
# ------------------------------------------------------------

df_scored = score_dataframe(
    df_features.copy()
)

print(f"Rows scored: {len(df_scored):,}")


# ------------------------------------------------------------
# Probability columns
# ------------------------------------------------------------

prob_cols = [
    "playoff_prob_xgb",
    "playoff_prob_rf",
    "playoff_prob",
]


# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

display(
    df_scored[
        prob_cols
    ]
    .describe()
    .round(4)
)


# ------------------------------------------------------------
# Null probabilities
# ------------------------------------------------------------

print("\nNull probabilities:")

print(
    df_scored[
        prob_cols
    ].isna().sum()
)


# ------------------------------------------------------------
# Out-of-range probabilities
# ------------------------------------------------------------

print("\nOut-of-range probabilities:")

for col in prob_cols:

    bad = (
        ~df_scored[col].between(0, 1)
    ).sum()

    print(
        f"{col}: {bad:,}"
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert df_scored[prob_cols].notna().all().all(), (
    "❌ Null playoff probabilities found."
)

assert all(
    df_scored[col].between(0, 1).all()
    for col in prob_cols
), (
    "❌ Playoff probabilities outside [0, 1] found."
)


print(
    "\n✅ Full historical dataset scored successfully "
    "with the current model."
)

In [ ]:
# ============================================================
# 7. Validate Target / Season Structure
# ============================================================

print("Target values across all team-game rows:")

print(
    df_scored["made_playoffs"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 1. Validate target values
# ------------------------------------------------------------

null_targets = (
    df_scored["made_playoffs"]
    .isna()
    .sum()
)

invalid_targets = (
    ~df_scored["made_playoffs"].isin([0, 1])
).sum()

print(f"\nNull targets: {null_targets:,}")
print(f"Invalid target values: {invalid_targets:,}")


# ------------------------------------------------------------
# 2. Create one record per team-season
#
# made_playoffs is constant throughout each team-season,
# so the final row can represent the team's season outcome.
# ------------------------------------------------------------

team_seasons = (
    df_scored
    .sort_values(
        ["season_id", "team_id", "game_date", "game_id"]
    )
    .groupby(
        ["season_id", "team_id"],
        as_index=False,
    )
    .last()
)


# ------------------------------------------------------------
# 3. Count rows and games by season
# ------------------------------------------------------------

season_games = (
    df_scored
    .groupby("season_id")
    .agg(
        rows=("team_id", "size"),
        games=("game_id", "nunique"),
    )
)


# ------------------------------------------------------------
# 4. Count teams and outcomes by season
# ------------------------------------------------------------

season_teams = (
    team_seasons
    .groupby("season_id")
    .agg(
        teams=("team_id", "nunique"),

        playoff_teams=(
            "made_playoffs",
            lambda x: (x == 1).sum(),
        ),

        non_playoff_teams=(
            "made_playoffs",
            lambda x: (x == 0).sum(),
        ),
    )
)


# ------------------------------------------------------------
# 5. Combine season summaries
# ------------------------------------------------------------

season_summary = (
    season_games
    .join(season_teams)
)

print("\nSeason structure:")

display(season_summary)


# ------------------------------------------------------------
# 6. Validate two team rows per game
# ------------------------------------------------------------

bad_row_counts = season_summary[
    season_summary["rows"]
    != season_summary["games"] * 2
]

print(
    "\nSeasons without exactly 2 team rows per game:",
    len(bad_row_counts)
)

if not bad_row_counts.empty:
    display(bad_row_counts)


# ------------------------------------------------------------
# 7. Validate every team has an outcome
# ------------------------------------------------------------

bad_team_outcomes = season_summary[
    (
        season_summary["playoff_teams"]
        + season_summary["non_playoff_teams"]
    )
    != season_summary["teams"]
]

print(
    "Seasons with incomplete team outcomes:",
    len(bad_team_outcomes)
)

if not bad_team_outcomes.empty:
    display(bad_team_outcomes)


# ------------------------------------------------------------
# 8. Validate playoff-team count
#
# All seasons in this historical dataset should contain
# exactly 16 playoff teams.
# ------------------------------------------------------------

bad_playoff_counts = season_summary[
    season_summary["playoff_teams"] != 16
]

print(
    "Seasons without exactly 16 playoff teams:",
    len(bad_playoff_counts)
)

if not bad_playoff_counts.empty:
    display(bad_playoff_counts)


# ------------------------------------------------------------
# 9. Assertions
# ------------------------------------------------------------

assert null_targets == 0, (
    f"❌ {null_targets:,} rows have null made_playoffs."
)

assert invalid_targets == 0, (
    f"❌ {invalid_targets:,} rows have invalid made_playoffs."
)

assert bad_row_counts.empty, (
    "❌ At least one season does not have exactly "
    "2 team rows per game."
)

assert bad_team_outcomes.empty, (
    "❌ At least one season has teams without a valid "
    "playoff/non-playoff outcome."
)

assert bad_playoff_counts.empty, (
    "❌ At least one season does not have exactly "
    "16 playoff teams."
)


print(
    "\n✅ Target and season structure validated successfully."
)

In [ ]:
# ============================================================
# 8. Validate Target Consistency
# ============================================================

target_consistency = (
    df_scored
    .groupby(
        ["season_id", "team_id"]
    )["made_playoffs"]
    .nunique()
)

bad_targets = target_consistency[
    target_consistency != 1
]

print(
    "Team-seasons with inconsistent target:",
    len(bad_targets)
)


# ------------------------------------------------------------
# Diagnose any failures
# ------------------------------------------------------------

if bad_targets.empty:

    print(
        "✅ made_playoffs is consistent "
        "within every team-season."
    )

else:

    bad_team_seasons = (
        bad_targets
        .reset_index()[
            ["season_id", "team_id"]
        ]
    )

    bad_target_rows = (
        df_scored
        .merge(
            bad_team_seasons,
            on=["season_id", "team_id"],
            how="inner",
        )
        .sort_values(
            [
                "season_id",
                "team_id",
                "games_played",
                "game_id",
            ]
        )
    )

    target_issue_summary = (
        bad_target_rows
        .groupby(
            ["season_id", "team_id"]
        )
        .agg(
            rows=("game_id", "size"),

            min_games_played=(
                "games_played",
                "min",
            ),

            max_games_played=(
                "games_played",
                "max",
            ),

            target_values=(
                "made_playoffs",
                lambda x: sorted(
                    x.dropna()
                    .unique()
                    .tolist()
                ),
            ),

            first_target=(
                "made_playoffs",
                "first",
            ),

            last_target=(
                "made_playoffs",
                "last",
            ),
        )
        .reset_index()
    )

    display(target_issue_summary)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert bad_targets.empty, (
    f"❌ {len(bad_targets):,} team-seasons have "
    "inconsistent made_playoffs values."
)

print(
    "\n✅ Target consistency validated successfully."
)

In [ ]:
# ============================================================
# 9. Validate Team-Season Progression
# ============================================================

progression = (
    df_scored
    .sort_values(
        ["season_id", "team_id", "games_played", "game_id"]
    )
    .groupby(
        ["season_id", "team_id"]
    )
    .agg(
        rows=("game_id", "size"),

        min_games_played=(
            "games_played",
            "min",
        ),

        max_games_played=(
            "games_played",
            "max",
        ),

        unique_games_played=(
            "games_played",
            "nunique",
        ),
    )
)


# ------------------------------------------------------------
# 1. Duplicate games_played values
# ------------------------------------------------------------

progression["duplicate_game_counts"] = (
    progression["rows"]
    - progression["unique_games_played"]
)


print("Team-season progression summary:")

display(
    progression.describe()
)


bad_progression = progression[
    progression["duplicate_game_counts"] != 0
]

print(
    "\nTeam-seasons with duplicate games_played values:",
    len(bad_progression)
)

if not bad_progression.empty:
    display(bad_progression)


# ------------------------------------------------------------
# 2. Every team-season should begin at Game 1
# ------------------------------------------------------------

bad_starts = progression[
    progression["min_games_played"] != 1
]

print(
    "Team-seasons not starting at Game 1:",
    len(bad_starts)
)

if not bad_starts.empty:
    display(bad_starts)


# ------------------------------------------------------------
# 3. Number of rows should equal final games_played
#
# If a team has rows for Games 1 through N exactly once,
# then:
#
#     rows == max_games_played
#
# ------------------------------------------------------------

bad_row_progression = progression[
    progression["rows"]
    != progression["max_games_played"]
]

print(
    "Team-seasons with missing/extra game rows:",
    len(bad_row_progression)
)

if not bad_row_progression.empty:
    display(bad_row_progression)


# ------------------------------------------------------------
# 4. Assertions
# ------------------------------------------------------------

assert bad_progression.empty, (
    f"❌ {len(bad_progression):,} team-seasons have "
    "duplicate games_played values."
)

assert bad_starts.empty, (
    f"❌ {len(bad_starts):,} team-seasons do not "
    "start at Game 1."
)

assert bad_row_progression.empty, (
    f"❌ {len(bad_row_progression):,} team-seasons "
    "have missing or extra game rows."
)


print(
    "\n✅ Team-season progression validated successfully."
)

In [ ]:
# ============================================================
# 10. Validate Postgame Standings State
# ============================================================

check = (
    df_scored
    .sort_values(
        ["season_id", "team_id", "games_played", "game_id"]
    )
    .copy()
)


# ------------------------------------------------------------
# 1. Calculate expected cumulative wins from game results
#
# Because each row represents the state AFTER Game N,
# wins on that row must include the result of Game N.
# ------------------------------------------------------------

check["expected_wins"] = (
    check
    .groupby(["season_id", "team_id"])["game_win"]
    .cumsum()
)


bad_wins = check[
    check["wins"] != check["expected_wins"]
].copy()

print(
    "Rows with incorrect postgame wins:",
    len(bad_wins)
)


if not bad_wins.empty:

    display(
        bad_wins[
            [
                "season_id",
                "team_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "game_win",
                "wins",
                "expected_wins",
            ]
        ].head(20)
    )


# ------------------------------------------------------------
# 2. First-game validation
#
# After Game 1:
#
# game_win = 1 -> wins = 1
# game_win = 0 -> wins = 0
# ------------------------------------------------------------

first_games = check[
    check["games_played"] == 1
].copy()

bad_first_game = first_games[
    first_games["wins"]
    != first_games["game_win"]
].copy()

print(
    "First-game rows with incorrect postgame wins:",
    len(bad_first_game)
)


if not bad_first_game.empty:

    display(
        bad_first_game[
            [
                "season_id",
                "team_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "game_win",
                "wins",
            ]
        ].head(20)
    )


# ------------------------------------------------------------
# 3. Validate row-to-row win progression
#
# wins[N] - wins[N-1] must equal the result of Game N.
# ------------------------------------------------------------

check["wins_change"] = (
    check
    .groupby(["season_id", "team_id"])["wins"]
    .diff()
)


bad_win_progression = check[
    (check["games_played"] > 1)
    & (
        check["wins_change"]
        != check["game_win"]
    )
].copy()

print(
    "Rows with incorrect postgame win progression:",
    len(bad_win_progression)
)


if not bad_win_progression.empty:

    display(
        bad_win_progression[
            [
                "season_id",
                "team_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "game_win",
                "wins",
                "wins_change",
            ]
        ].head(20)
    )


# ------------------------------------------------------------
# 4. Assertions
# ------------------------------------------------------------

assert bad_wins.empty, (
    f"❌ {len(bad_wins):,} rows have incorrect "
    "postgame cumulative wins."
)

assert bad_first_game.empty, (
    f"❌ {len(bad_first_game):,} Game 1 rows have "
    "incorrect postgame wins."
)

assert bad_win_progression.empty, (
    f"❌ {len(bad_win_progression):,} rows have "
    "incorrect postgame win progression."
)


print(
    "\n✅ Postgame wins progression is correct."
)

In [ ]:
# ============================================================
# 11. Validate Postgame W/L/OTL Progression
# ============================================================

check = (
    df_scored
    .sort_values(
        ["season_id", "team_id", "games_played", "game_id"]
    )
    .copy()
)

group = check.groupby(
    ["season_id", "team_id"],
    sort=False,
)

# ------------------------------------------------------------
# Changes between consecutive POSTGAME standings states
# ------------------------------------------------------------

for col in ["wins", "losses", "ot_losses"]:
    check[f"{col}_change"] = group[col].diff()


# ------------------------------------------------------------
# 1. W + L + OTL must equal games_played
#
# Each row represents the state AFTER Game N.
# ------------------------------------------------------------

check["result_total"] = (
    check["wins"]
    + check["losses"]
    + check["ot_losses"]
)

bad_result_total = check[
    check["result_total"] != check["games_played"]
].copy()

print(
    f"Rows where W + L + OTL != games_played: "
    f"{len(bad_result_total):,}"
)

if not bad_result_total.empty:
    display(
        bad_result_total[
            [
                "season_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "wins",
                "losses",
                "ot_losses",
                "result_total",
            ]
        ].head(20)
    )


# ------------------------------------------------------------
# 2. First game must contain exactly one result
#
# After Game 1, one of:
#
# 1-0-0
# 0-1-0
# 0-0-1
# ------------------------------------------------------------

first_games = check[
    check["games_played"] == 1
].copy()

bad_first = first_games[
    (
        first_games["wins"]
        + first_games["losses"]
        + first_games["ot_losses"]
    ) != 1
].copy()

print(
    f"First-game rows without exactly one result: "
    f"{len(bad_first):,}"
)

if not bad_first.empty:
    display(
        bad_first[
            [
                "season_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "game_win",
                "wins",
                "losses",
                "ot_losses",
            ]
        ].head(20)
    )


# ------------------------------------------------------------
# 3. For Game 2+, exactly ONE result counter must increase
#
# From Game N-1 to Game N:
#
# win             -> wins +1
# regulation loss -> losses +1
# OT/SO loss      -> ot_losses +1
#
# Nothing may decrease.
# Nothing may increase by more than 1.
# ------------------------------------------------------------

later = check["games_played"] > 1

check["result_counter_change"] = (
    check["wins_change"]
    + check["losses_change"]
    + check["ot_losses_change"]
)

bad_result_progression = check[
    later
    & (
        (check["wins_change"] < 0)
        | (check["losses_change"] < 0)
        | (check["ot_losses_change"] < 0)
        | (check["wins_change"] > 1)
        | (check["losses_change"] > 1)
        | (check["ot_losses_change"] > 1)
        | (check["result_counter_change"] != 1)
    )
].copy()

print(
    f"Rows with invalid postgame W/L/OTL progression: "
    f"{len(bad_result_progression):,}"
)

if not bad_result_progression.empty:
    display(
        bad_result_progression[
            [
                "season_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "game_win",
                "wins",
                "losses",
                "ot_losses",
                "wins_change",
                "losses_change",
                "ot_losses_change",
                "result_counter_change",
            ]
        ].head(20)
    )


# ------------------------------------------------------------
# 4. Win result must agree with standings change
#
# If game_win == 1, wins must increase by 1.
# If game_win == 0, wins must not increase.
#
# Handle Game 1 separately because diff() is NaN.
# ------------------------------------------------------------

bad_first_win = first_games[
    first_games["wins"] != first_games["game_win"]
].copy()

bad_later_win = check[
    later
    & (
        check["wins_change"]
        != check["game_win"]
    )
].copy()

print(
    f"Game 1 rows where wins disagree with game_win: "
    f"{len(bad_first_win):,}"
)

print(
    f"Later rows where wins change disagrees with game_win: "
    f"{len(bad_later_win):,}"
)


# ------------------------------------------------------------
# 5. Assertions
# ------------------------------------------------------------

assert bad_result_total.empty, (
    f"❌ {len(bad_result_total):,} rows have "
    "W + L + OTL != games_played."
)

assert bad_first.empty, (
    f"❌ {len(bad_first):,} Game 1 rows do not contain "
    "exactly one result."
)

assert bad_result_progression.empty, (
    f"❌ {len(bad_result_progression):,} rows have invalid "
    "postgame W/L/OTL progression."
)

assert bad_first_win.empty, (
    f"❌ {len(bad_first_win):,} Game 1 rows have wins "
    "inconsistent with game_win."
)

assert bad_later_win.empty, (
    f"❌ {len(bad_later_win):,} rows have win progression "
    "inconsistent with game_win."
)


print(
    "\n✅ Postgame W/L/OTL progression is correct."
)

In [ ]:
# ============================================================
# 12. Validate Games Remaining Progression
# ============================================================

check = (
    df_scored
    .sort_values(
        ["season_id", "team_id", "games_played", "game_id"]
    )
    .copy()
)

group = check.groupby(
    ["season_id", "team_id"],
    sort=False,
)

# Change from one game row to the next
check["games_remaining_change"] = (
    group["games_remaining"].diff()
)

# ------------------------------------------------------------
# Every subsequent game should reduce games_remaining by 1
# ------------------------------------------------------------

bad_progression = check[
    (check["games_played"] > 1)
    & (check["games_remaining_change"] != -1)
]

print(
    f"Rows with invalid games_remaining progression: "
    f"{len(bad_progression):,}"
)

# ------------------------------------------------------------
# Last game of each team-season should have 0 remaining
# ------------------------------------------------------------

last_games = (
    check
    .groupby(["season_id", "team_id"], as_index=False)
    .last()
)

bad_last_game = last_games[
    (last_games["season_id"] != 20192020)
    & (last_games["games_remaining"] != 0)
]

print(
    f"Team-seasons not ending at 0 games_remaining: "
    f"{len(bad_last_game):,}"
)

if not bad_progression.empty:
    display(
        bad_progression[
            [
                "season_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "games_remaining",
                "games_remaining_change",
            ]
        ].head(20)
    )

if not bad_last_game.empty:
    display(
        bad_last_game[
            [
                "season_id",
                "tri_code",
                "games_played",
                "games_remaining",
            ]
        ].head(20)
    )

assert bad_progression.empty
assert bad_last_game.empty

print("✅ games_remaining progression is correct.")

In [ ]:
# ============================================================
# 13. Validate Postgame Points
# ============================================================

check = df_scored.copy()

# After Game N:
#   win        = 2 points
#   OT/SO loss = 1 point
#   regulation loss = 0 points
check["expected_points"] = (
    (check["wins"] * 2)
    + check["ot_losses"]
)

bad_points = check[
    check["points"] != check["expected_points"]
].copy()

print(
    f"Rows with incorrect postgame points: "
    f"{len(bad_points):,}"
)

if not bad_points.empty:
    display(
        bad_points[
            [
                "season_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "wins",
                "losses",
                "ot_losses",
                "points",
                "expected_points",
            ]
        ].head(20)
    )

assert bad_points.empty, (
    f"❌ {len(bad_points):,} rows have incorrect postgame points."
)

print("✅ Postgame points are correct.")

In [ ]:
# ============================================================
# 14. Validate Postgame Points Percentage
# ============================================================

check = df_scored.copy()

# ------------------------------------------------------------
# Each row represents the state AFTER Game N.
#
# points_pct = points earned / maximum possible points
#            = points / (2 * games_played)
# ------------------------------------------------------------

check["expected_points_pct"] = (
    check["points"]
    / (2 * check["games_played"])
)

# ------------------------------------------------------------
# Compare stored vs expected
# Allow tiny floating-point differences
# ------------------------------------------------------------

bad_points_pct = check[
    ~np.isclose(
        check["points_pct"].astype(float),
        check["expected_points_pct"].astype(float),
        atol=1e-6,
    )
].copy()

print(
    f"Rows with incorrect postgame points_pct: "
    f"{len(bad_points_pct):,}"
)

if not bad_points_pct.empty:
    display(
        bad_points_pct[
            [
                "season_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "points",
                "points_pct",
                "expected_points_pct",
            ]
        ].head(20)
    )

assert bad_points_pct.empty, (
    f"❌ {len(bad_points_pct):,} rows have "
    "incorrect postgame points_pct."
)

print("✅ Postgame points_pct is correct.")

In [ ]:
# ============================================================
# 15. Probability Behavior After Game N
# ============================================================

prob_by_gp = (
    df_scored
    .groupby("games_played")
    .agg(
        rows=("playoff_prob", "size"),
        avg_prob=("playoff_prob", "mean"),
        min_prob=("playoff_prob", "min"),
        max_prob=("playoff_prob", "max"),
        avg_actual=("made_playoffs", "mean"),
    )
    .reset_index()
)

display(prob_by_gp.head())

In [ ]:
prob_by_outcome = (
    df_scored
    .groupby(["games_played", "made_playoffs"])
    .agg(
        rows=("team_id", "size"),
        avg_prob=("playoff_prob", "mean"),
        median_prob=("playoff_prob", "median"),
    )
    .reset_index()
)

display(prob_by_outcome.head())

In [ ]:
# ============================================================
# 16. Model Performance After Game N
# ============================================================

from sklearn.metrics import roc_auc_score, brier_score_loss

checkpoints = [
    1,
    5,
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
]

results = []

for gp in checkpoints:

    subset = df_scored[
        df_scored["games_played"] == gp
    ].copy()

    # Need both outcome classes for AUC
    if subset["made_playoffs"].nunique() < 2:
        continue

    results.append({
        "games_played": gp,
        "rows": len(subset),

        "roc_auc": roc_auc_score(
            subset["made_playoffs"],
            subset["playoff_prob"],
        ),

        "brier_score": brier_score_loss(
            subset["made_playoffs"],
            subset["playoff_prob"],
        ),

        "avg_prob_playoff": subset.loc[
            subset["made_playoffs"] == 1,
            "playoff_prob",
        ].mean(),

        "avg_prob_non_playoff": subset.loc[
            subset["made_playoffs"] == 0,
            "playoff_prob",
        ].mean(),
    })


performance_by_gp = pd.DataFrame(results)

display(
    performance_by_gp.round(4)
)

In [ ]:
# ============================================================
# 17. Validate Probability Calibration
# ============================================================

calibration = df_scored[
    ["playoff_prob", "made_playoffs"]
].copy()

# ------------------------------------------------------------
# Create 10 probability buckets:
#
# 0.0-0.1
# 0.1-0.2
# ...
# 0.9-1.0
# ------------------------------------------------------------

calibration["prob_bucket"] = pd.cut(
    calibration["playoff_prob"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True,
)

calibration_summary = (
    calibration
    .groupby(
        "prob_bucket",
        observed=True,
    )
    .agg(
        rows=("made_playoffs", "size"),
        avg_predicted=("playoff_prob", "mean"),
        actual_playoff_rate=("made_playoffs", "mean"),
    )
    .reset_index()
)

calibration_summary["calibration_error"] = (
    calibration_summary["avg_predicted"]
    - calibration_summary["actual_playoff_rate"]
)

calibration_summary["abs_calibration_error"] = (
    calibration_summary["calibration_error"].abs()
)

display(
    calibration_summary.round(4)
)


# ------------------------------------------------------------
# Overall weighted calibration error
# ------------------------------------------------------------

weighted_calibration_error = (
    calibration_summary["abs_calibration_error"]
    * calibration_summary["rows"]
).sum() / calibration_summary["rows"].sum()

print(
    f"\nWeighted mean absolute calibration error: "
    f"{weighted_calibration_error:.4f}"
)

In [ ]:
# ============================================================
# 18. Calibration by Season Stage
# ============================================================

calibration = df_scored[
    [
        "games_played",
        "playoff_prob",
        "made_playoffs",
    ]
].copy()

# ------------------------------------------------------------
# Season stages
# ------------------------------------------------------------

calibration["season_stage"] = pd.cut(
    calibration["games_played"],
    bins=[0, 10, 20, 40, 60, 82],
    labels=[
        "Games 1-10",
        "Games 11-20",
        "Games 21-40",
        "Games 41-60",
        "Games 61-82",
    ],
    include_lowest=True,
)

# ------------------------------------------------------------
# Probability buckets
# ------------------------------------------------------------

calibration["prob_bucket"] = pd.cut(
    calibration["playoff_prob"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True,
)

# ------------------------------------------------------------
# Calibration within each season stage
# ------------------------------------------------------------

calibration_by_stage = (
    calibration
    .groupby(
        ["season_stage", "prob_bucket"],
        observed=True,
    )
    .agg(
        rows=("made_playoffs", "size"),
        avg_predicted=("playoff_prob", "mean"),
        actual_playoff_rate=("made_playoffs", "mean"),
    )
    .reset_index()
)

calibration_by_stage["calibration_error"] = (
    calibration_by_stage["avg_predicted"]
    - calibration_by_stage["actual_playoff_rate"]
)

calibration_by_stage["abs_calibration_error"] = (
    calibration_by_stage["calibration_error"].abs()
)

display(
    calibration_by_stage.round(4)
)


# ------------------------------------------------------------
# Weighted calibration error by season stage
# ------------------------------------------------------------

stage_summary = (
    calibration_by_stage
    .groupby("season_stage", observed=True)
    .apply(
        lambda x: pd.Series({
            "rows": x["rows"].sum(),
            "weighted_calibration_error": (
                x["abs_calibration_error"] * x["rows"]
            ).sum() / x["rows"].sum(),
        }),
        include_groups=False,
    )
    .reset_index()
)

display(
    stage_summary.round(4)
)

In [ ]:
print(bundle.keys())

In [ ]:
for key in bundle.keys():
    if key not in ["xgb_model", "rf_model", "model"]:
        value = bundle[key]

        if isinstance(value, (str, int, float, bool, list, tuple)):
            print(f"{key}: {value}")
        else:
            print(f"{key}: {type(value)}")

In [ ]:
# ============================================================
# 19. Inspect Model Training Metadata
# ============================================================

metadata_keys = [
    "trained_on",
    "model_type",
    "snapshot",
    "version",
    "source_table",
    "target",
    "metrics",
]

for key in metadata_keys:
    print(f"\n{key}:")
    print(bundle.get(key))
    
print("\nXGB pipeline:")
print(type(bundle["xgb_pipeline"]))

print("\nRF pipeline:")
print(type(bundle["rf_pipeline"]))

In [ ]:
# ============================================================
# 20. Rolling Historical Season Backtest
# ============================================================


import shap

from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
)

# Import the SAME model-building functions used by training.
# Change "prediction_model.train_model" below only if your
# training script has a different module filename.
from prediction_model.training.train_new_model import (
    _build_feature_matrix,
    _make_xgb_pipeline,
    _make_rf_pipeline,
    select_features_from_shap,
)

# ============================================================
# 1. Seasons available for rolling evaluation
# ============================================================

seasons = sorted(
    df_features["season_id"]
    .dropna()
    .astype(int)
    .unique()
)

print("Available seasons:")
print(seasons)


# We need at least one prior season to train on,
# so the first season cannot itself be evaluated.
test_seasons = seasons[1:]

print("\nRolling holdout seasons:")
print(test_seasons)


# ============================================================
# 2. Build feature matrix ONCE
# ============================================================

X_all, initial_features = _build_feature_matrix(
    df_features
)

y_all = (
    df_features["made_playoffs"]
    .astype(int)
)

print(
    f"\nInitial candidate features: "
    f"{len(initial_features)}"
)


# ============================================================
# 3. Results containers
# ============================================================

season_results = []
stage_results = []

# NEW:
# Keep every genuine out-of-sample prediction so later
# calibration/stability analyses do NOT require retraining.
rolling_predictions = []


# ============================================================
# 4. Rolling season backtest
#
# For each season:
#
#     TRAIN = every season before it
#     TEST  = that season only
#
# SHAP feature selection is rerun using ONLY the historical
# training data available before that holdout season.
# ============================================================

for test_season in test_seasons:

    print("\n" + "=" * 70)
    print(f"🏒 HOLDOUT SEASON: {test_season}")
    print("=" * 70)


    # --------------------------------------------------------
    # Train / test masks
    # --------------------------------------------------------

    train_mask = (
        df_features["season_id"] < test_season
    )

    test_mask = (
        df_features["season_id"] == test_season
    )

    train_rows = int(train_mask.sum())
    test_rows = int(test_mask.sum())


    if train_rows == 0 or test_rows == 0:
        print("⚠️ Skipping — insufficient rows.")
        continue


    # --------------------------------------------------------
    # Build this holdout's train/test data
    # --------------------------------------------------------

    X_train = (
        X_all.loc[train_mask]
        .copy()
    )

    X_test = (
        X_all.loc[test_mask]
        .copy()
    )

    y_train = (
        y_all.loc[train_mask]
        .copy()
    )

    y_test = (
        y_all.loc[test_mask]
        .copy()
    )


    # --------------------------------------------------------
    # Sanity checks
    # --------------------------------------------------------

    if y_train.nunique() < 2:

        print(
            "⚠️ Skipping — training data "
            "contains only one target class."
        )

        continue


    if y_test.nunique() < 2:

        print(
            "⚠️ Skipping — test season "
            "contains only one target class."
        )

        continue


    train_start_season = int(
        df_features.loc[
            train_mask,
            "season_id",
        ].min()
    )

    train_end_season = int(
        df_features.loc[
            train_mask,
            "season_id",
        ].max()
    )


    print(
        f"Train seasons: "
        f"{train_start_season} "
        f"→ "
        f"{train_end_season}"
    )

    print(
        f"Training rows: "
        f"{len(X_train):,}"
    )

    print(
        f"Test rows: "
        f"{len(X_test):,}"
    )

    print(
        f"Training positive rate: "
        f"{y_train.mean():.4f}"
    )

    print(
        f"Test positive rate: "
        f"{y_test.mean():.4f}"
    )


    # ========================================================
    # 5. Initial XGBoost
    # ========================================================

    print("\n🏋️ Initial XGBoost")

    initial_xgb = _make_xgb_pipeline()

    initial_xgb.fit(
        X_train,
        y_train,
    )


    # ========================================================
    # 6. SHAP feature selection
    # ========================================================

    print("🔍 SHAP feature selection")

    X_train_scaled = (
        initial_xgb
        .named_steps["scaler"]
        .transform(X_train)
    )


    explainer = shap.TreeExplainer(
        initial_xgb.named_steps["model"]
    )


    shap_values = (
        explainer(
            X_train_scaled
        )
        .values
    )


    selected_features = select_features_from_shap(
        X_train,
        shap_values,
        min_usage=0.03,
        min_impact=0.0005,
    )


    if not selected_features:

        print(
            "⚠️ Skipping — SHAP selected "
            "zero features."
        )

        continue


    X_train_selected = (
        X_train[
            selected_features
        ]
        .copy()
    )

    X_test_selected = (
        X_test[
            selected_features
        ]
        .copy()
    )


    print(
        f"Selected features: "
        f"{len(selected_features)}"
    )


    # ========================================================
    # 7. Final XGBoost for this holdout
    # ========================================================

    print("🔁 Retraining XGBoost")

    xgb_pipeline = (
        _make_xgb_pipeline()
    )

    xgb_pipeline.fit(
        X_train_selected,
        y_train,
    )


    xgb_prob = (
        xgb_pipeline
        .predict_proba(
            X_test_selected
        )[:, 1]
    )


    # ========================================================
    # 8. Random Forest
    # ========================================================

    print("🌲 Training Random Forest")

    rf_pipeline = (
        _make_rf_pipeline()
    )

    rf_pipeline.fit(
        X_train_selected,
        y_train,
    )


    rf_prob = (
        rf_pipeline
        .predict_proba(
            X_test_selected
        )[:, 1]
    )


    # ========================================================
    # 9. Ensemble
    # ========================================================

    ensemble_prob = (
        xgb_prob
        + rf_prob
    ) / 2


    # ========================================================
    # 10. Overall holdout-season metrics
    # ========================================================

    xgb_brier = (
        brier_score_loss(
            y_test,
            xgb_prob,
        )
    )


    rf_brier = (
        brier_score_loss(
            y_test,
            rf_prob,
        )
    )


    ensemble_brier = (
        brier_score_loss(
            y_test,
            ensemble_prob,
        )
    )


    ensemble_auc = (
        roc_auc_score(
            y_test,
            ensemble_prob,
        )
    )


    ensemble_accuracy = (
        (
            (ensemble_prob >= 0.50)
            .astype(int)
            ==
            y_test.to_numpy()
        )
        .mean()
    )


    season_results.append({

        "test_season":
            test_season,

        "train_start_season":
            train_start_season,

        "train_end_season":
            train_end_season,

        "train_rows":
            len(X_train),

        "test_rows":
            len(X_test),

        "selected_features":
            len(selected_features),

        "xgb_brier":
            xgb_brier,

        "rf_brier":
            rf_brier,

        "ensemble_brier":
            ensemble_brier,

        "ensemble_auc":
            ensemble_auc,

        "ensemble_accuracy":
            ensemble_accuracy,

        "mean_predicted":
            ensemble_prob.mean(),

        "actual_playoff_rate":
            y_test.mean(),
    })


    # ========================================================
    # 11. Build holdout prediction DataFrame
    # ========================================================

    holdout = (
        df_features.loc[
            test_mask,
            [
                "season_id",
                "team_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "made_playoffs",
            ],
        ]
        .copy()
    )


    # Because holdout was selected with the same test_mask,
    # these arrays are aligned with its rows.

    holdout["playoff_prob_xgb"] = (
        xgb_prob
    )

    holdout["playoff_prob_rf"] = (
        rf_prob
    )

    holdout["playoff_prob"] = (
        ensemble_prob
    )


    # --------------------------------------------------------
    # Season-stage bucket
    # --------------------------------------------------------

    holdout["season_bucket"] = pd.cut(
        holdout["games_played"],
        bins=[
            0,
            20,
            50,
            float("inf"),
        ],
        labels=[
            "early",
            "mid",
            "late",
        ],
        include_lowest=True,
    )


    # ========================================================
    # 12. SAVE EVERY OUT-OF-SAMPLE PREDICTION
    # ========================================================

    rolling_predictions.append(

        holdout[
            [
                "season_id",
                "team_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "made_playoffs",
                "playoff_prob_xgb",
                "playoff_prob_rf",
                "playoff_prob",
                "season_bucket",
            ]
        ].copy()

    )


    # ========================================================
    # 13. Season-stage performance
    # ========================================================

    for bucket in [
        "early",
        "mid",
        "late",
    ]:

        bucket_df = (
            holdout[
                holdout[
                    "season_bucket"
                ] == bucket
            ]
            .copy()
        )


        if bucket_df.empty:
            continue


        y_bucket = (
            bucket_df[
                "made_playoffs"
            ]
            .astype(int)
        )


        prob_bucket = (
            bucket_df[
                "playoff_prob"
            ]
        )


        if y_bucket.nunique() >= 2:

            bucket_auc = (
                roc_auc_score(
                    y_bucket,
                    prob_bucket,
                )
            )

        else:

            bucket_auc = np.nan


        stage_results.append({

            "test_season":
                test_season,

            "season_bucket":
                bucket,

            "rows":
                len(bucket_df),

            "brier":
                brier_score_loss(
                    y_bucket,
                    prob_bucket,
                ),

            "roc_auc":
                bucket_auc,

            "mean_predicted":
                prob_bucket.mean(),

            "actual_playoff_rate":
                y_bucket.mean(),
        })


    # ========================================================
    # 14. Holdout-season completion output
    # ========================================================

    print(
        f"\n✅ {test_season} complete"
    )

    print(
        f"Ensemble Brier: "
        f"{ensemble_brier:.4f}"
    )

    print(
        f"Ensemble AUC: "
        f"{ensemble_auc:.4f}"
    )


# ============================================================
# 15. Combine results
# ============================================================

rolling_backtest = pd.DataFrame(
    season_results
)


rolling_stage_backtest = pd.DataFrame(
    stage_results
)


rolling_predictions = pd.concat(
    rolling_predictions,
    ignore_index=True,
)


# ============================================================
# 16. Validate saved OOS predictions
# ============================================================

print("\n" + "=" * 70)
print("🔎 OUT-OF-SAMPLE PREDICTION CHECK")
print("=" * 70)


print(
    f"Out-of-sample prediction rows: "
    f"{len(rolling_predictions):,}"
)


print(
    f"Holdout seasons saved: "
    f"{rolling_predictions['season_id'].nunique():,}"
)


print("\nRows by holdout season:")

print(
    rolling_predictions[
        "season_id"
    ]
    .value_counts()
    .sort_index()
)


# Every saved prediction should have a target and probability

assert (
    rolling_predictions[
        "made_playoffs"
    ]
    .notna()
    .all()
)

assert (
    rolling_predictions[
        [
            "playoff_prob_xgb",
            "playoff_prob_rf",
            "playoff_prob",
        ]
    ]
    .notna()
    .all()
    .all()
)


# Probabilities must all be between 0 and 1

assert (
    rolling_predictions[
        [
            "playoff_prob_xgb",
            "playoff_prob_rf",
            "playoff_prob",
        ]
    ]
    .apply(
        lambda x: x.between(
            0,
            1,
        )
        .all()
    )
    .all()
)


print(
    "\n✅ All rolling predictions "
    "are valid out-of-sample predictions."
)


# ============================================================
# 17. Display rolling season results
# ============================================================

print("\n" + "=" * 70)
print("📊 ROLLING HISTORICAL BACKTEST")
print("=" * 70)


display(
    rolling_backtest.round(4)
)


# ============================================================
# 18. Display stage results
# ============================================================

print("\n" + "=" * 70)
print("📊 PERFORMANCE BY SEASON STAGE")
print("=" * 70)


display(
    rolling_stage_backtest.round(4)
)


# ============================================================
# 19. Preview saved OOS predictions
# ============================================================

print("\n" + "=" * 70)
print("📋 SAVED OUT-OF-SAMPLE PREDICTIONS")
print("=" * 70)


display(
    rolling_predictions.head(20)
)

In [ ]:
# ============================================================
# 21. Out-of-Sample Probability Calibration
# ============================================================


# ============================================================
# 1. Overall OOS calibration
# ============================================================

calibration = rolling_predictions[
    [
        "season_id",
        "games_played",
        "playoff_prob",
        "made_playoffs",
    ]
].copy()


# ------------------------------------------------------------
# Create probability buckets
# ------------------------------------------------------------

prob_bins = np.linspace(0, 1, 11)

calibration["prob_bucket"] = pd.cut(
    calibration["playoff_prob"],
    bins=prob_bins,
    include_lowest=True,
)


# ------------------------------------------------------------
# Calibration summary
# ------------------------------------------------------------

oos_calibration = (
    calibration
    .groupby(
        "prob_bucket",
        observed=True,
    )
    .agg(
        rows=("made_playoffs", "size"),
        avg_predicted=("playoff_prob", "mean"),
        actual_playoff_rate=("made_playoffs", "mean"),
    )
    .reset_index()
)


oos_calibration["calibration_error"] = (
    oos_calibration["avg_predicted"]
    - oos_calibration["actual_playoff_rate"]
)


oos_calibration["abs_calibration_error"] = (
    oos_calibration["calibration_error"].abs()
)


print("=" * 70)
print("📊 OVERALL OUT-OF-SAMPLE CALIBRATION")
print("=" * 70)

display(
    oos_calibration.round(4)
)


# ============================================================
# 2. Overall weighted calibration error
# ============================================================

weighted_calibration_error = (
    (
        oos_calibration["abs_calibration_error"]
        * oos_calibration["rows"]
    ).sum()
    / oos_calibration["rows"].sum()
)


print(
    "\nOverall weighted mean absolute "
    f"calibration error: "
    f"{weighted_calibration_error:.4f}"
)


# ============================================================
# 3. Calibration by season stage
#
# Use the same stages as Block 20:
#
# early = Games 1-20
# mid   = Games 21-50
# late  = Games 51+
# ============================================================

stage_calibration = (
    rolling_predictions[
        [
            "season_id",
            "games_played",
            "season_bucket",
            "playoff_prob",
            "made_playoffs",
        ]
    ]
    .copy()
)


stage_calibration["prob_bucket"] = pd.cut(
    stage_calibration["playoff_prob"],
    bins=prob_bins,
    include_lowest=True,
)


stage_calibration_summary = (
    stage_calibration
    .groupby(
        [
            "season_bucket",
            "prob_bucket",
        ],
        observed=True,
    )
    .agg(
        rows=("made_playoffs", "size"),
        avg_predicted=("playoff_prob", "mean"),
        actual_playoff_rate=("made_playoffs", "mean"),
    )
    .reset_index()
)


stage_calibration_summary["calibration_error"] = (
    stage_calibration_summary["avg_predicted"]
    - stage_calibration_summary["actual_playoff_rate"]
)


stage_calibration_summary["abs_calibration_error"] = (
    stage_calibration_summary[
        "calibration_error"
    ].abs()
)


print("\n" + "=" * 70)
print("📊 OUT-OF-SAMPLE CALIBRATION BY SEASON STAGE")
print("=" * 70)

display(
    stage_calibration_summary.round(4)
)


# ============================================================
# 4. Weighted calibration error by season stage
# ============================================================

stage_weighted_error = (
    stage_calibration_summary
    .assign(
        weighted_error=lambda x:
            x["abs_calibration_error"]
            * x["rows"]
    )
    .groupby(
        "season_bucket",
        observed=True,
    )
    .agg(
        rows=("rows", "sum"),
        total_weighted_error=(
            "weighted_error",
            "sum",
        ),
    )
    .reset_index()
)


stage_weighted_error[
    "weighted_calibration_error"
] = (
    stage_weighted_error[
        "total_weighted_error"
    ]
    / stage_weighted_error["rows"]
)


stage_weighted_error = (
    stage_weighted_error[
        [
            "season_bucket",
            "rows",
            "weighted_calibration_error",
        ]
    ]
)


print("\n" + "=" * 70)
print("📊 WEIGHTED CALIBRATION ERROR BY SEASON STAGE")
print("=" * 70)

display(
    stage_weighted_error.round(4)
)


# ============================================================
# 5. Calibration by holdout season
# ============================================================

season_calibration = (
    rolling_predictions
    .groupby("season_id")
    .apply(
        lambda x: pd.Series({
            "rows": len(x),

            "mean_predicted":
                x["playoff_prob"].mean(),

            "actual_playoff_rate":
                x["made_playoffs"].mean(),

            "mean_error":
                (
                    x["playoff_prob"].mean()
                    - x["made_playoffs"].mean()
                ),

            "mean_abs_probability_error":
                np.mean(
                    np.abs(
                        x["playoff_prob"]
                        - x["made_playoffs"]
                    )
                ),
        }),
        include_groups=False,
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("📊 CALIBRATION BY HOLDOUT SEASON")
print("=" * 70)

display(
    season_calibration.round(4)
)


# ============================================================
# 6. Basic interpretation helpers
# ============================================================

worst_bucket = (
    oos_calibration
    .sort_values(
        "abs_calibration_error",
        ascending=False,
    )
    .iloc[0]
)


worst_stage = (
    stage_weighted_error
    .sort_values(
        "weighted_calibration_error",
        ascending=False,
    )
    .iloc[0]
)


print("\n" + "=" * 70)
print("🔎 CALIBRATION SUMMARY")
print("=" * 70)


print(
    f"Overall weighted calibration error: "
    f"{weighted_calibration_error:.4f}"
)


print(
    f"Worst probability bucket: "
    f"{worst_bucket['prob_bucket']}"
)


print(
    f"  Avg predicted: "
    f"{worst_bucket['avg_predicted']:.4f}"
)


print(
    f"  Actual rate: "
    f"{worst_bucket['actual_playoff_rate']:.4f}"
)


print(
    f"  Absolute error: "
    f"{worst_bucket['abs_calibration_error']:.4f}"
)


print(
    f"\nWorst-calibrated season stage: "
    f"{worst_stage['season_bucket']}"
)


print(
    f"  Weighted calibration error: "
    f"{worst_stage['weighted_calibration_error']:.4f}"
)

In [ ]:
# ============================================================
# 22. Out-of-Sample Performance & Calibration by Season
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
)


# ============================================================
# 1. Function to calculate bucketed ECE
# ============================================================

def calculate_ece(df, prob_col="playoff_prob", target_col="made_playoffs"):

    temp = df[
        [prob_col, target_col]
    ].copy()

    temp["prob_bucket"] = pd.cut(
        temp[prob_col],
        bins=np.linspace(0, 1, 11),
        include_lowest=True,
    )

    bucket_summary = (
        temp
        .groupby(
            "prob_bucket",
            observed=True,
        )
        .agg(
            rows=(target_col, "size"),
            avg_predicted=(prob_col, "mean"),
            actual_rate=(target_col, "mean"),
        )
        .reset_index()
    )

    bucket_summary["abs_error"] = (
        bucket_summary["avg_predicted"]
        - bucket_summary["actual_rate"]
    ).abs()

    ece = (
        (
            bucket_summary["abs_error"]
            * bucket_summary["rows"]
        ).sum()
        / bucket_summary["rows"].sum()
    )

    return ece


# ============================================================
# 2. Calculate metrics for every holdout season
# ============================================================

season_metrics = []

for season_id, season_df in rolling_predictions.groupby("season_id"):

    y_true = (
        season_df["made_playoffs"]
        .astype(int)
    )

    y_prob = (
        season_df["playoff_prob"]
        .astype(float)
    )

    season_metrics.append({

        "season_id":
            int(season_id),

        "rows":
            len(season_df),

        "roc_auc":
            roc_auc_score(
                y_true,
                y_prob,
            ),

        "brier_score":
            brier_score_loss(
                y_true,
                y_prob,
            ),

        "ece":
            calculate_ece(
                season_df
            ),

        "mean_predicted":
            y_prob.mean(),

        "actual_playoff_rate":
            y_true.mean(),

        "mean_bias":
            y_prob.mean()
            - y_true.mean(),
    })


season_metrics = (
    pd.DataFrame(season_metrics)
    .sort_values("season_id")
    .reset_index(drop=True)
)


print("=" * 70)
print("📊 OUT-OF-SAMPLE PERFORMANCE BY SEASON")
print("=" * 70)

display(
    season_metrics.round(4)
)


# ============================================================
# 3. Add change from previous season
# ============================================================

season_metrics["auc_change"] = (
    season_metrics["roc_auc"].diff()
)

season_metrics["brier_change"] = (
    season_metrics["brier_score"].diff()
)

season_metrics["ece_change"] = (
    season_metrics["ece"].diff()
)


print("\n" + "=" * 70)
print("📈 YEAR-OVER-YEAR CHANGE")
print("=" * 70)

display(
    season_metrics[
        [
            "season_id",
            "roc_auc",
            "auc_change",
            "brier_score",
            "brier_change",
            "ece",
            "ece_change",
        ]
    ].round(4)
)


# ============================================================
# 4. Recent vs historical performance
# ============================================================

# Most recent three completed seasons in the backtest
recent_seasons = (
    season_metrics["season_id"]
    .tail(3)
    .tolist()
)

season_metrics["period"] = np.where(
    season_metrics["season_id"].isin(recent_seasons),
    "Recent 3 Seasons",
    "Earlier Seasons",
)


period_summary = (
    season_metrics
    .groupby("period")
    .agg(
        seasons=("season_id", "size"),
        avg_auc=("roc_auc", "mean"),
        avg_brier=("brier_score", "mean"),
        avg_ece=("ece", "mean"),
        avg_bias=("mean_bias", "mean"),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("📊 RECENT VS HISTORICAL")
print("=" * 70)

display(
    period_summary.round(4)
)


# ============================================================
# 5. Performance by season stage AND holdout season
# ============================================================

stage_season_metrics = []

for (
    season_id,
    season_bucket
), subset in rolling_predictions.groupby(
    ["season_id", "season_bucket"],
    observed=True,
):

    if subset["made_playoffs"].nunique() < 2:
        continue

    y_true = (
        subset["made_playoffs"]
        .astype(int)
    )

    y_prob = (
        subset["playoff_prob"]
        .astype(float)
    )

    stage_season_metrics.append({

        "season_id":
            int(season_id),

        "season_bucket":
            str(season_bucket),

        "rows":
            len(subset),

        "roc_auc":
            roc_auc_score(
                y_true,
                y_prob,
            ),

        "brier_score":
            brier_score_loss(
                y_true,
                y_prob,
            ),

        "ece":
            calculate_ece(
                subset
            ),
    })


stage_season_metrics = (
    pd.DataFrame(stage_season_metrics)
    .sort_values(
        [
            "season_id",
            "season_bucket",
        ]
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("📊 PERFORMANCE BY SEASON + STAGE")
print("=" * 70)

display(
    stage_season_metrics.round(4)
)


# ============================================================
# 6. Recent stage comparison
# ============================================================

recent_stage_summary = (
    stage_season_metrics[
        stage_season_metrics[
            "season_id"
        ].isin(recent_seasons)
    ]
    .groupby("season_bucket")
    .agg(
        rows=("rows", "sum"),
        avg_auc=("roc_auc", "mean"),
        avg_brier=("brier_score", "mean"),
        avg_ece=("ece", "mean"),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("📊 RECENT 3-SEASON PERFORMANCE BY STAGE")
print("=" * 70)

display(
    recent_stage_summary.round(4)
)

In [ ]:
# ============================================================
# 23. 2025-26 Team-Level Error Analysis
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Isolate 2025-26 out-of-sample predictions
# ============================================================

TEST_SEASON = 20252026

season_2526 = (
    rolling_predictions[
        rolling_predictions["season_id"] == TEST_SEASON
    ]
    .copy()
)


print("=" * 70)
print("🏒 2025-26 TEAM-LEVEL ERROR ANALYSIS")
print("=" * 70)

print(f"Rows: {len(season_2526):,}")
print(
    f"Teams: "
    f"{season_2526['team_id'].nunique():,}"
)


# ============================================================
# 2. Check which team identifier columns are available
# ============================================================

team_label_col = None

for col in [
    "tri_code",
    "team_abbrev",
    "team_name",
]:
    if col in season_2526.columns:
        team_label_col = col
        break


if team_label_col is None:
    team_label_col = "team_id"


print(
    f"Team label column: {team_label_col}"
)


# ============================================================
# 3. Probability error for every team-game
#
# Positive error:
#   model probability was above final outcome
#
# Negative error:
#   model probability was below final outcome
#
# Example:
#
# made playoffs = 0
# probability   = .80
# error         = +.80
#
# made playoffs = 1
# probability   = .20
# error         = -.80
# ============================================================

season_2526["probability_error"] = (
    season_2526["playoff_prob"]
    - season_2526["made_playoffs"]
)

season_2526["absolute_error"] = (
    season_2526["probability_error"].abs()
)

season_2526["squared_error"] = (
    season_2526["probability_error"] ** 2
)


# ============================================================
# 4. Overall team-level error
# ============================================================

team_error = (
    season_2526
    .groupby(
        [
            "team_id",
            team_label_col,
        ],
        as_index=False,
    )
    .agg(
        games=("games_played", "size"),
        made_playoffs=("made_playoffs", "first"),
        avg_probability=("playoff_prob", "mean"),
        median_probability=("playoff_prob", "median"),
        min_probability=("playoff_prob", "min"),
        max_probability=("playoff_prob", "max"),
        mean_error=("probability_error", "mean"),
        mean_absolute_error=("absolute_error", "mean"),
        mean_squared_error=("squared_error", "mean"),
    )
)


team_error = (
    team_error
    .sort_values(
        "mean_squared_error",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("📊 TEAMS WITH LARGEST SEASON-LONG ERRORS")
print("=" * 70)

display(
    team_error.round(4)
)


# ============================================================
# 5. Checkpoint probabilities
# ============================================================

checkpoints = [
    10,
    20,
    40,
    60,
    80,
]


checkpoint_rows = (
    season_2526[
        season_2526["games_played"].isin(checkpoints)
    ]
    [
        [
            "team_id",
            team_label_col,
            "games_played",
            "playoff_prob",
            "made_playoffs",
        ]
    ]
    .copy()
)


checkpoint_table = (
    checkpoint_rows
    .pivot_table(
        index=[
            "team_id",
            team_label_col,
            "made_playoffs",
        ],
        columns="games_played",
        values="playoff_prob",
        aggfunc="first",
    )
    .reset_index()
)


checkpoint_table.columns.name = None


# Rename probability columns
checkpoint_table = checkpoint_table.rename(
    columns={
        10: "prob_game_10",
        20: "prob_game_20",
        40: "prob_game_40",
        60: "prob_game_60",
        80: "prob_game_80",
    }
)


# ============================================================
# 6. Add checkpoint error
# ============================================================

for gp in checkpoints:

    prob_col = f"prob_game_{gp}"
    error_col = f"error_game_{gp}"

    if prob_col in checkpoint_table.columns:

        checkpoint_table[error_col] = (
            checkpoint_table[prob_col]
            - checkpoint_table["made_playoffs"]
        )


print("\n" + "=" * 70)
print("📊 TEAM PROBABILITIES AT KEY CHECKPOINTS")
print("=" * 70)

display(
    checkpoint_table.round(4)
)


# ============================================================
# 7. Largest misses at Game 20
# ============================================================

if "error_game_20" in checkpoint_table.columns:

    game20_misses = (
        checkpoint_table
        .assign(
            abs_error_game_20=lambda x:
                x["error_game_20"].abs()
        )
        .sort_values(
            "abs_error_game_20",
            ascending=False,
        )
    )

    print("\n" + "=" * 70)
    print("🚨 LARGEST MISSES AT GAME 20")
    print("=" * 70)

    display(
        game20_misses[
            [
                "team_id",
                team_label_col,
                "made_playoffs",
                "prob_game_20",
                "error_game_20",
                "abs_error_game_20",
            ]
        ]
        .head(15)
        .round(4)
    )


# ============================================================
# 8. Largest misses at Game 40
# ============================================================

if "error_game_40" in checkpoint_table.columns:

    game40_misses = (
        checkpoint_table
        .assign(
            abs_error_game_40=lambda x:
                x["error_game_40"].abs()
        )
        .sort_values(
            "abs_error_game_40",
            ascending=False,
        )
    )

    print("\n" + "=" * 70)
    print("🚨 LARGEST MISSES AT GAME 40")
    print("=" * 70)

    display(
        game40_misses[
            [
                "team_id",
                team_label_col,
                "made_playoffs",
                "prob_game_40",
                "error_game_40",
                "abs_error_game_40",
            ]
        ]
        .head(15)
        .round(4)
    )


# ============================================================
# 9. Largest misses at Game 60
# ============================================================

if "error_game_60" in checkpoint_table.columns:

    game60_misses = (
        checkpoint_table
        .assign(
            abs_error_game_60=lambda x:
                x["error_game_60"].abs()
        )
        .sort_values(
            "abs_error_game_60",
            ascending=False,
        )
    )

    print("\n" + "=" * 70)
    print("🚨 LARGEST MISSES AT GAME 60")
    print("=" * 70)

    display(
        game60_misses[
            [
                "team_id",
                team_label_col,
                "made_playoffs",
                "prob_game_60",
                "error_game_60",
                "abs_error_game_60",
            ]
        ]
        .head(15)
        .round(4)
    )


# ============================================================
# 10. Identify false optimism / false pessimism
# ============================================================

# Non-playoff teams that received high probabilities
false_optimism = (
    checkpoint_table[
        checkpoint_table["made_playoffs"] == 0
    ]
    .copy()
)

# Playoff teams that received low probabilities
false_pessimism = (
    checkpoint_table[
        checkpoint_table["made_playoffs"] == 1
    ]
    .copy()
)


if "prob_game_40" in checkpoint_table.columns:

    false_optimism = (
        false_optimism
        .sort_values(
            "prob_game_40",
            ascending=False,
        )
    )

    false_pessimism = (
        false_pessimism
        .sort_values(
            "prob_game_40",
            ascending=True,
        )
    )


print("\n" + "=" * 70)
print("⬆️ FALSE OPTIMISM — NON-PLAYOFF TEAMS")
print("=" * 70)

display(
    false_optimism.round(4)
)


print("\n" + "=" * 70)
print("⬇️ FALSE PESSIMISM — PLAYOFF TEAMS")
print("=" * 70)

display(
    false_pessimism.round(4)
)


# ============================================================
# 11. Probability movement by team
# ============================================================

movement = (
    checkpoint_table.copy()
)


if (
    "prob_game_10" in movement.columns
    and "prob_game_40" in movement.columns
):

    movement["change_10_to_40"] = (
        movement["prob_game_40"]
        - movement["prob_game_10"]
    )


if (
    "prob_game_20" in movement.columns
    and "prob_game_60" in movement.columns
):

    movement["change_20_to_60"] = (
        movement["prob_game_60"]
        - movement["prob_game_20"]
    )


if (
    "prob_game_40" in movement.columns
    and "prob_game_80" in movement.columns
):

    movement["change_40_to_80"] = (
        movement["prob_game_80"]
        - movement["prob_game_40"]
    )


print("\n" + "=" * 70)
print("📈 PROBABILITY MOVEMENT BY TEAM")
print("=" * 70)

display(
    movement.round(4)
)


# ============================================================
# 12. Summary
# ============================================================

print("\n" + "=" * 70)
print("🔎 2025-26 ERROR SUMMARY")
print("=" * 70)

print(
    f"Average team-game absolute error: "
    f"{season_2526['absolute_error'].mean():.4f}"
)

print(
    f"Average team-game squared error: "
    f"{season_2526['squared_error'].mean():.4f}"
)

print(
    "\nTop 5 teams contributing the most "
    "season-long squared error:"
)

display(
    team_error[
        [
            "team_id",
            team_label_col,
            "made_playoffs",
            "avg_probability",
            "mean_squared_error",
        ]
    ]
    .head(5)
    .round(4)
)

In [ ]:
# ============================================================
# 24. Major-Miss Feature Diagnosis
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Define major-miss teams and checkpoints
# ============================================================

MISS_SEASON = 20252026

MISS_TEAMS = [
    "DET",
    "LAK",
    "PHI",
    "NYI",
    "BUF",
]

CHECKPOINTS = [
    10,
    20,
    40,
    60,
    80,
]

JOIN_KEYS = [
    "season_id",
    "team_id",
    "game_id",
]


# ============================================================
# 2. Verify join keys are unique
# ============================================================

pred_duplicates = (
    rolling_predictions
    .duplicated(JOIN_KEYS)
    .sum()
)

feature_duplicates = (
    df_scored
    .duplicated(JOIN_KEYS)
    .sum()
)

print("=" * 70)
print("🔎 MAJOR-MISS FEATURE DIAGNOSIS")
print("=" * 70)

print(
    f"Duplicate prediction keys: "
    f"{pred_duplicates:,}"
)

print(
    f"Duplicate feature keys: "
    f"{feature_duplicates:,}"
)

assert pred_duplicates == 0, (
    "❌ rolling_predictions contains duplicate join keys."
)

assert feature_duplicates == 0, (
    "❌ df_scored contains duplicate join keys."
)


# ============================================================
# 3. Determine model features available in df_scored
# ============================================================

model_features = bundle["features"]

available_features = [
    col
    for col in model_features
    if col in df_scored.columns
]

missing_features = [
    col
    for col in model_features
    if col not in df_scored.columns
]


print(
    f"\nModel features: "
    f"{len(model_features)}"
)

print(
    f"Available in df_scored: "
    f"{len(available_features)}"
)

print(
    f"Missing from df_scored: "
    f"{len(missing_features)}"
)

if missing_features:

    print("\nMissing features:")

    print(missing_features)


assert len(available_features) > 0, (
    "❌ No model features were found in df_scored."
)


# ============================================================
# 4. Build feature lookup
#
# Keep only the join keys + model features.
# This avoids duplicate tri_code, games_played, etc.
# ============================================================

feature_lookup = (
    df_scored[
        JOIN_KEYS
        + available_features
    ]
    .copy()
)


# ============================================================
# 5. Merge OOS predictions with their original features
# ============================================================

rolling_with_features = (
    rolling_predictions
    .merge(
        feature_lookup,
        on=JOIN_KEYS,
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)


merge_counts = (
    rolling_with_features["_merge"]
    .value_counts(dropna=False)
)

print("\nPrediction → feature merge:")
print(merge_counts)


unmatched_predictions = (
    rolling_with_features[
        rolling_with_features["_merge"] != "both"
    ]
)


print(
    f"\nPredictions without feature rows: "
    f"{len(unmatched_predictions):,}"
)


if not unmatched_predictions.empty:

    display(
        unmatched_predictions[
            [
                "season_id",
                "team_id",
                "tri_code",
                "game_id",
                "game_date",
                "games_played",
                "_merge",
            ]
        ].head(20)
    )


assert unmatched_predictions.empty, (
    "❌ Some rolling OOS predictions could not be "
    "matched back to df_scored."
)


rolling_with_features = (
    rolling_with_features
    .drop(columns="_merge")
)


# ============================================================
# 6. Confirm feature completeness after merge
# ============================================================

feature_null_summary = (
    rolling_with_features[
        available_features
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
)


features_with_nulls = (
    feature_null_summary[
        feature_null_summary > 0
    ]
)


print(
    f"\nFeatures containing nulls after merge: "
    f"{len(features_with_nulls)}"
)


if not features_with_nulls.empty:

    display(
        features_with_nulls
        .rename("null_rows")
        .to_frame()
        .head(30)
    )


# ============================================================
# 7. Pull major-miss teams
# ============================================================

miss_rows = (
    rolling_with_features[
        (
            rolling_with_features["season_id"]
            == MISS_SEASON
        )
        & (
            rolling_with_features["tri_code"]
            .isin(MISS_TEAMS)
        )
    ]
    .copy()
)


print("\n" + "=" * 70)
print("🏒 MAJOR-MISS TEAMS")
print("=" * 70)

print(
    f"Rows: "
    f"{len(miss_rows):,}"
)

print(
    f"Teams: "
    f"{miss_rows['tri_code'].nunique():,}"
)

print(
    "Team list:",
    sorted(
        miss_rows["tri_code"]
        .dropna()
        .unique()
    ),
)


assert (
    miss_rows["tri_code"].nunique()
    == len(MISS_TEAMS)
), (
    "❌ Not all major-miss teams were found."
)


# ============================================================
# 8. Prediction errors
# ============================================================

miss_rows["prediction_error"] = (
    miss_rows["playoff_prob"]
    - miss_rows["made_playoffs"]
)

miss_rows["abs_prediction_error"] = (
    miss_rows["prediction_error"]
    .abs()
)

miss_rows["squared_error"] = (
    miss_rows["prediction_error"]
    ** 2
)


# ============================================================
# 9. Pull checkpoint rows
# ============================================================

checkpoint_rows = (
    miss_rows[
        miss_rows["games_played"]
        .isin(CHECKPOINTS)
    ]
    .copy()
)


checkpoint_summary = (
    checkpoint_rows[
        [
            "tri_code",
            "games_played",
            "made_playoffs",
            "playoff_prob",
            "playoff_prob_xgb",
            "playoff_prob_rf",
            "prediction_error",
            "abs_prediction_error",
        ]
    ]
    .sort_values(
        [
            "games_played",
            "abs_prediction_error",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("📊 MAJOR-MISS PROBABILITIES AT CHECKPOINTS")
print("=" * 70)

display(
    checkpoint_summary.round(4)
)


# ============================================================
# 10. Compare feature values to league at SAME checkpoint
#
# Important:
# Compare each team only with other teams in 2025-26
# at exactly the same games_played value.
# ============================================================

feature_comparisons = []


for _, row in checkpoint_rows.iterrows():

    gp = row["games_played"]

    league_same_point = (
        rolling_with_features[
            (
                rolling_with_features["season_id"]
                == MISS_SEASON
            )
            & (
                rolling_with_features["games_played"]
                == gp
            )
        ]
    )


    for feature in available_features:

        team_value = pd.to_numeric(
            pd.Series([row[feature]]),
            errors="coerce",
        ).iloc[0]

        league_values = pd.to_numeric(
            league_same_point[feature],
            errors="coerce",
        )


        if (
            pd.isna(team_value)
            or league_values.notna().sum() < 2
        ):
            continue


        league_mean = (
            league_values.mean()
        )

        league_median = (
            league_values.median()
        )

        league_std = (
            league_values.std()
        )


        if (
            pd.isna(league_std)
            or league_std == 0
        ):
            z_score = np.nan

        else:

            z_score = (
                team_value
                - league_mean
            ) / league_std


        percentile = (
            league_values
            .rank(
                pct=True,
                method="average",
            )
        )

        # Percentile of the team value relative
        # to all teams at the same checkpoint
        percentile_rank = (
            (league_values <= team_value)
            .mean()
        )


        feature_comparisons.append(
            {
                "tri_code":
                    row["tri_code"],

                "games_played":
                    gp,

                "made_playoffs":
                    row["made_playoffs"],

                "playoff_prob":
                    row["playoff_prob"],

                "feature":
                    feature,

                "feature_value":
                    team_value,

                "league_mean":
                    league_mean,

                "league_median":
                    league_median,

                "league_std":
                    league_std,

                "z_score":
                    z_score,

                "abs_z_score":
                    (
                        abs(z_score)
                        if pd.notna(z_score)
                        else np.nan
                    ),

                "percentile":
                    percentile_rank,
            }
        )


# Explicit columns prevent another empty-frame
# KeyError if no rows are generated.
comparison_columns = [
    "tri_code",
    "games_played",
    "made_playoffs",
    "playoff_prob",
    "feature",
    "feature_value",
    "league_mean",
    "league_median",
    "league_std",
    "z_score",
    "abs_z_score",
    "percentile",
]


feature_comparisons = pd.DataFrame(
    feature_comparisons,
    columns=comparison_columns,
)


print(
    f"\nFeature comparison rows created: "
    f"{len(feature_comparisons):,}"
)


assert not feature_comparisons.empty, (
    "❌ No feature comparisons were generated."
)


# ============================================================
# 11. Most unusual features for every team/checkpoint
# ============================================================

most_unusual_features = (
    feature_comparisons
    .dropna(
        subset=["abs_z_score"]
    )
    .sort_values(
        [
            "tri_code",
            "games_played",
            "abs_z_score",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .groupby(
        [
            "tri_code",
            "games_played",
        ],
        group_keys=False,
    )
    .head(15)
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("📊 MOST UNUSUAL FEATURES VS LEAGUE")
print("=" * 70)

display(
    most_unusual_features[
        [
            "tri_code",
            "games_played",
            "made_playoffs",
            "playoff_prob",
            "feature",
            "feature_value",
            "league_mean",
            "z_score",
            "percentile",
        ]
    ]
    .round(4)
)


# ============================================================
# 12. Focus on Games 40 and 60
# ============================================================

late_miss_features = (
    most_unusual_features[
        most_unusual_features[
            "games_played"
        ].isin(
            [
                40,
                60,
            ]
        )
    ]
    .copy()
)


print("\n" + "=" * 70)
print("🔬 MAJOR-MISS FEATURES — GAMES 40 & 60")
print("=" * 70)

display(
    late_miss_features[
        [
            "tri_code",
            "games_played",
            "made_playoffs",
            "playoff_prob",
            "feature",
            "feature_value",
            "league_mean",
            "z_score",
            "percentile",
        ]
    ]
    .round(4)
)


# ============================================================
# 13. Features repeatedly extreme across miss teams
#
# abs(z) >= 1 means at least one standard deviation
# away from the league mean at that checkpoint.
# ============================================================

extreme_features = (
    feature_comparisons[
        feature_comparisons[
            "abs_z_score"
        ] >= 1.0
    ]
    .groupby(
        "feature",
        as_index=False,
    )
    .agg(
        extreme_occurrences=(
            "feature",
            "size",
        ),
        teams=(
            "tri_code",
            "nunique",
        ),
        checkpoints=(
            "games_played",
            "nunique",
        ),
        avg_abs_z=(
            "abs_z_score",
            "mean",
        ),
        max_abs_z=(
            "abs_z_score",
            "max",
        ),
    )
    .sort_values(
        [
            "teams",
            "extreme_occurrences",
            "avg_abs_z",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("🔎 FEATURES REPEATEDLY EXTREME IN MAJOR MISSES")
print("=" * 70)

display(
    extreme_features
    .head(30)
    .round(4)
)


# ============================================================
# 14. XGB vs RF disagreement on major misses
# ============================================================

checkpoint_rows[
    "model_disagreement"
] = (
    checkpoint_rows[
        "playoff_prob_xgb"
    ]
    - checkpoint_rows[
        "playoff_prob_rf"
    ]
).abs()


model_disagreement = (
    checkpoint_rows[
        [
            "tri_code",
            "games_played",
            "made_playoffs",
            "playoff_prob_xgb",
            "playoff_prob_rf",
            "playoff_prob",
            "model_disagreement",
            "abs_prediction_error",
        ]
    ]
    .sort_values(
        "model_disagreement",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("⚖️ XGB VS RF DISAGREEMENT")
print("=" * 70)

display(
    model_disagreement.round(4)
)


# ============================================================
# 15. Final diagnostic summary
# ============================================================

print("\n" + "=" * 70)
print("🔎 BLOCK 24 SUMMARY")
print("=" * 70)

print(
    f"Major-miss teams analyzed: "
    f"{miss_rows['tri_code'].nunique()}"
)

print(
    f"Checkpoint rows analyzed: "
    f"{len(checkpoint_rows)}"
)

print(
    f"Model features analyzed: "
    f"{len(available_features)}"
)

print(
    f"Feature comparisons generated: "
    f"{len(feature_comparisons):,}"
)

print(
    f"Features extreme at least once: "
    f"{len(extreme_features):,}"
)

In [ ]:
# ============================================================
# 25. XGB vs RF Model Disagreement Analysis
# ============================================================


# ============================================================
# 1. Build disagreement dataset
# ============================================================

disagreement_df = rolling_predictions.copy()

disagreement_df["model_disagreement"] = (
    disagreement_df["playoff_prob_xgb"]
    - disagreement_df["playoff_prob_rf"]
).abs()

disagreement_df["xgb_error"] = (
    disagreement_df["playoff_prob_xgb"]
    - disagreement_df["made_playoffs"]
).abs()

disagreement_df["rf_error"] = (
    disagreement_df["playoff_prob_rf"]
    - disagreement_df["made_playoffs"]
).abs()

disagreement_df["ensemble_error"] = (
    disagreement_df["playoff_prob"]
    - disagreement_df["made_playoffs"]
).abs()


# ============================================================
# 2. Overall disagreement distribution
# ============================================================

print("=" * 70)
print("🤝 XGB vs RF DISAGREEMENT")
print("=" * 70)

print(
    disagreement_df["model_disagreement"]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .round(4)
)


# ============================================================
# 3. Create disagreement buckets
# ============================================================

disagreement_df["disagreement_bucket"] = pd.cut(
    disagreement_df["model_disagreement"],
    bins=[
        -0.001,
        0.05,
        0.10,
        0.20,
        0.30,
        1.00,
    ],
    labels=[
        "0–5%",
        "5–10%",
        "10–20%",
        "20–30%",
        "30%+",
    ],
)


# ============================================================
# 4. Performance by disagreement level
# ============================================================

disagreement_summary = (
    disagreement_df
    .groupby(
        "disagreement_bucket",
        observed=True,
    )
    .agg(
        rows=("made_playoffs", "size"),
        avg_disagreement=("model_disagreement", "mean"),
        xgb_error=("xgb_error", "mean"),
        rf_error=("rf_error", "mean"),
        ensemble_error=("ensemble_error", "mean"),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("📊 ERROR BY MODEL DISAGREEMENT")
print("=" * 70)

display(
    disagreement_summary.round(4)
)


# ============================================================
# 5. Which model wins when they disagree?
# ============================================================

disagreement_df["better_model"] = np.select(
    [
        disagreement_df["xgb_error"]
        < disagreement_df["rf_error"],

        disagreement_df["rf_error"]
        < disagreement_df["xgb_error"],
    ],
    [
        "XGB",
        "RF",
    ],
    default="Tie",
)


winner_summary = (
    disagreement_df
    .groupby(
        "disagreement_bucket",
        observed=True,
    )["better_model"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)


print("\n" + "=" * 70)
print("🏆 BETTER MODEL BY DISAGREEMENT LEVEL")
print("=" * 70)

display(winner_summary)


# ============================================================
# 6. Disagreement by season stage
# ============================================================

stage_disagreement = (
    disagreement_df
    .groupby(
        "season_bucket",
        observed=True,
    )
    .agg(
        rows=("made_playoffs", "size"),
        avg_disagreement=("model_disagreement", "mean"),
        median_disagreement=("model_disagreement", "median"),
        xgb_error=("xgb_error", "mean"),
        rf_error=("rf_error", "mean"),
        ensemble_error=("ensemble_error", "mean"),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("📊 MODEL DISAGREEMENT BY SEASON STAGE")
print("=" * 70)

display(
    stage_disagreement.round(4)
)


# ============================================================
# 7. Disagreement by holdout season
# ============================================================

season_disagreement = (
    disagreement_df
    .groupby("season_id")
    .agg(
        rows=("made_playoffs", "size"),
        avg_disagreement=("model_disagreement", "mean"),
        median_disagreement=("model_disagreement", "median"),
        xgb_error=("xgb_error", "mean"),
        rf_error=("rf_error", "mean"),
        ensemble_error=("ensemble_error", "mean"),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("📅 MODEL DISAGREEMENT BY HOLDOUT SEASON")
print("=" * 70)

display(
    season_disagreement.round(4)
)


# ============================================================
# 8. Largest disagreements
# ============================================================

largest_disagreements = (
    disagreement_df
    .sort_values(
        "model_disagreement",
        ascending=False,
    )
    [
        [
            "season_id",
            "tri_code",
            "games_played",
            "made_playoffs",
            "playoff_prob_xgb",
            "playoff_prob_rf",
            "playoff_prob",
            "model_disagreement",
            "xgb_error",
            "rf_error",
            "ensemble_error",
        ]
    ]
    .head(50)
)


print("\n" + "=" * 70)
print("🔎 50 LARGEST XGB / RF DISAGREEMENTS")
print("=" * 70)

display(
    largest_disagreements.round(4)
)


# ============================================================
# 9. Does disagreement identify bad predictions?
# ============================================================

disagreement_df["major_miss"] = (
    disagreement_df["ensemble_error"] >= 0.50
)

miss_disagreement = (
    disagreement_df
    .groupby("major_miss")
    .agg(
        rows=("made_playoffs", "size"),
        avg_disagreement=("model_disagreement", "mean"),
        median_disagreement=("model_disagreement", "median"),
        p90_disagreement=(
            "model_disagreement",
            lambda x: x.quantile(0.90),
        ),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("🚨 DISAGREEMENT: NORMAL PREDICTIONS VS MAJOR MISSES")
print("=" * 70)

display(
    miss_disagreement.round(4)
)

In [ ]:
# ============================================================
# 26. Feature Drift / Temporal Stability
# ============================================================


# ============================================================
# 1. Use model features available in validated data
# ============================================================

model_features = [
    col
    for col in bundle["features"]
    if col in df_scored.columns
]


# ============================================================
# 2. Compare each season's feature distribution
# ============================================================

feature_drift_rows = []

seasons = sorted(
    df_scored["season_id"]
    .dropna()
    .unique()
)


for feature in model_features:

    values = pd.to_numeric(
        df_scored[feature],
        errors="coerce",
    )

    overall_mean = values.mean()
    overall_std = values.std()

    if (
        pd.isna(overall_std)
        or overall_std == 0
    ):
        continue

    for season_id in seasons:

        season_mask = (
            df_scored["season_id"]
            == season_id
        )

        season_values = pd.to_numeric(
            df_scored.loc[
                season_mask,
                feature,
            ],
            errors="coerce",
        )

        valid = season_values.dropna()

        if valid.empty:
            continue

        season_mean = valid.mean()
        season_median = valid.median()

        standardized_mean_shift = (
            season_mean - overall_mean
        ) / overall_std

        feature_drift_rows.append(
            {
                "season_id": int(season_id),
                "feature": feature,
                "rows": len(valid),
                "season_mean": season_mean,
                "season_median": season_median,
                "overall_mean": overall_mean,
                "overall_std": overall_std,
                "mean_shift_z":
                    standardized_mean_shift,
                "abs_mean_shift_z":
                    abs(standardized_mean_shift),
            }
        )


feature_drift = pd.DataFrame(
    feature_drift_rows
)


# ============================================================
# 3. Overall drift by season
# ============================================================

season_drift = (
    feature_drift
    .groupby("season_id")
    .agg(
        features=("feature", "nunique"),
        avg_abs_shift=(
            "abs_mean_shift_z",
            "mean",
        ),
        median_abs_shift=(
            "abs_mean_shift_z",
            "median",
        ),
        max_abs_shift=(
            "abs_mean_shift_z",
            "max",
        ),
        features_over_05=(
            "abs_mean_shift_z",
            lambda x: (x >= 0.5).sum(),
        ),
        features_over_10=(
            "abs_mean_shift_z",
            lambda x: (x >= 1.0).sum(),
        ),
    )
    .reset_index()
)


print("=" * 70)
print("📊 FEATURE DRIFT BY SEASON")
print("=" * 70)

display(
    season_drift.round(4)
)


# ============================================================
# 4. Most shifted features in recent season
# ============================================================

latest_season = max(seasons)

latest_drift = (
    feature_drift[
        feature_drift["season_id"]
        == latest_season
    ]
    .sort_values(
        "abs_mean_shift_z",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print(
    f"🔎 MOST SHIFTED FEATURES — {latest_season}"
)
print("=" * 70)

display(
    latest_drift[
        [
            "feature",
            "rows",
            "season_mean",
            "overall_mean",
            "mean_shift_z",
            "abs_mean_shift_z",
        ]
    ]
    .head(30)
    .round(4)
)


# ============================================================
# 5. Compare recent seasons directly
# ============================================================

recent_seasons = seasons[-5:]

recent_drift = (
    feature_drift[
        feature_drift["season_id"]
        .isin(recent_seasons)
    ]
    .pivot(
        index="feature",
        columns="season_id",
        values="mean_shift_z",
    )
)


# Maximum recent shift for each feature
recent_drift["max_recent_abs_shift"] = (
    recent_drift[
        recent_seasons
    ]
    .abs()
    .max(axis=1)
)


recent_drift = (
    recent_drift
    .sort_values(
        "max_recent_abs_shift",
        ascending=False,
    )
)


print("\n" + "=" * 70)
print("📊 FEATURE DRIFT — RECENT 5 SEASONS")
print("=" * 70)

display(
    recent_drift
    .head(30)
    .round(4)
)


# ============================================================
# 6. Persistent recent drift
#
# Count features shifted >= 0.5 SD in each of the
# most recent three seasons.
# ============================================================

last_three = seasons[-3:]

persistent_recent_drift = (
    feature_drift[
        feature_drift["season_id"]
        .isin(last_three)
    ]
    .assign(
        materially_shifted=lambda x:
            x["abs_mean_shift_z"] >= 0.5
    )
    .groupby("feature")
    .agg(
        seasons_shifted=(
            "materially_shifted",
            "sum",
        ),
        avg_abs_shift=(
            "abs_mean_shift_z",
            "mean",
        ),
        max_abs_shift=(
            "abs_mean_shift_z",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "seasons_shifted",
            "avg_abs_shift",
        ],
        ascending=[
            False,
            False,
        ],
    )
)


print("\n" + "=" * 70)
print("🔎 PERSISTENT RECENT FEATURE DRIFT")
print("=" * 70)

display(
    persistent_recent_drift
    .head(30)
    .round(4)
)


# ============================================================
# 7. Compare feature drift with OOS model performance
# ============================================================

drift_vs_performance = (
    season_metrics[
        [
            "season_id",
            "roc_auc",
            "brier_score",
            "ece",
        ]
    ]
    .merge(
        season_drift,
        on="season_id",
        how="left",
    )
)


print("\n" + "=" * 70)
print("📊 FEATURE DRIFT VS MODEL PERFORMANCE")
print("=" * 70)

display(
    drift_vs_performance.round(4)
)


# ============================================================
# 8. Correlations
# ============================================================

drift_correlations = (
    drift_vs_performance[
        [
            "roc_auc",
            "brier_score",
            "ece",
            "avg_abs_shift",
            "median_abs_shift",
            "max_abs_shift",
            "features_over_05",
            "features_over_10",
        ]
    ]
    .corr()
)


print("\n" + "=" * 70)
print("🔗 DRIFT / PERFORMANCE CORRELATIONS")
print("=" * 70)

display(
    drift_correlations.round(3)
)


# ============================================================
# 9. Summary
# ============================================================

print("\n" + "=" * 70)
print("🔎 BLOCK 26 SUMMARY")
print("=" * 70)

print(
    f"Seasons analyzed: "
    f"{len(seasons)}"
)

print(
    f"Model features analyzed: "
    f"{feature_drift['feature'].nunique()}"
)

print(
    f"Latest season: "
    f"{latest_season}"
)

latest_summary = (
    season_drift[
        season_drift["season_id"]
        == latest_season
    ]
    .iloc[0]
)

print(
    f"Latest-season average absolute shift: "
    f"{latest_summary['avg_abs_shift']:.4f}"
)

print(
    f"Latest-season features shifted >= 0.5 SD: "
    f"{int(latest_summary['features_over_05'])}"
)

print(
    f"Latest-season features shifted >= 1.0 SD: "
    f"{int(latest_summary['features_over_10'])}"
)

In [ ]:
# ============================================================
# 27. Prediction Stability / Game-to-Game Movement
# ============================================================


# ============================================================
# 1. Sort predictions chronologically by team-season
# ============================================================

stability = (
    rolling_predictions
    .sort_values(
        [
            "season_id",
            "team_id",
            "games_played",
        ]
    )
    .copy()
)


# ============================================================
# 2. Previous prediction
# ============================================================

stability["previous_prob"] = (
    stability
    .groupby(
        [
            "season_id",
            "team_id",
        ]
    )["playoff_prob"]
    .shift(1)
)


stability["prob_change"] = (
    stability["playoff_prob"]
    - stability["previous_prob"]
)

stability["abs_prob_change"] = (
    stability["prob_change"]
    .abs()
)


# Remove first prediction for each team-season
movement = (
    stability[
        stability["previous_prob"].notna()
    ]
    .copy()
)


# ============================================================
# 3. Overall movement distribution
# ============================================================

print("=" * 70)
print("📈 GAME-TO-GAME PREDICTION MOVEMENT")
print("=" * 70)

print(
    movement["abs_prob_change"]
    .describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .round(4)
)


# ============================================================
# 4. Large movement thresholds
# ============================================================

movement_thresholds = []

for threshold in [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
]:

    movement_thresholds.append(
        {
            "threshold":
                threshold,

            "rows":
                (
                    movement[
                        "abs_prob_change"
                    ] >= threshold
                ).sum(),

            "pct_predictions":
                (
                    movement[
                        "abs_prob_change"
                    ] >= threshold
                ).mean(),
        }
    )


movement_thresholds = pd.DataFrame(
    movement_thresholds
)


print("\n" + "=" * 70)
print("📊 LARGE PREDICTION MOVEMENTS")
print("=" * 70)

display(
    movement_thresholds.round(4)
)


# ============================================================
# 5. Movement by season stage
# ============================================================

movement_by_stage = (
    movement
    .groupby(
        "season_bucket",
        observed=True,
    )
    .agg(
        rows=("abs_prob_change", "size"),
        avg_change=("abs_prob_change", "mean"),
        median_change=("abs_prob_change", "median"),
        p90_change=(
            "abs_prob_change",
            lambda x: x.quantile(.90),
        ),
        p95_change=(
            "abs_prob_change",
            lambda x: x.quantile(.95),
        ),
        pct_over_10=(
            "abs_prob_change",
            lambda x: (x >= .10).mean(),
        ),
        pct_over_20=(
            "abs_prob_change",
            lambda x: (x >= .20).mean(),
        ),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("📊 PREDICTION MOVEMENT BY SEASON STAGE")
print("=" * 70)

display(
    movement_by_stage.round(4)
)


# ============================================================
# 6. Movement by holdout season
# ============================================================

movement_by_season = (
    movement
    .groupby("season_id")
    .agg(
        rows=("abs_prob_change", "size"),
        avg_change=("abs_prob_change", "mean"),
        median_change=("abs_prob_change", "median"),
        p95_change=(
            "abs_prob_change",
            lambda x: x.quantile(.95),
        ),
        pct_over_10=(
            "abs_prob_change",
            lambda x: (x >= .10).mean(),
        ),
        pct_over_20=(
            "abs_prob_change",
            lambda x: (x >= .20).mean(),
        ),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("📅 PREDICTION MOVEMENT BY HOLDOUT SEASON")
print("=" * 70)

display(
    movement_by_season.round(4)
)


# ============================================================
# 7. Largest individual jumps
# ============================================================

largest_jumps = (
    movement
    .sort_values(
        "abs_prob_change",
        ascending=False,
    )
    [
        [
            "season_id",
            "tri_code",
            "games_played",
            "made_playoffs",
            "previous_prob",
            "playoff_prob",
            "prob_change",
            "abs_prob_change",
        ]
    ]
    .head(50)
)


print("\n" + "=" * 70)
print("🚨 50 LARGEST GAME-TO-GAME JUMPS")
print("=" * 70)

display(
    largest_jumps.round(4)
)


# ============================================================
# 8. Direction reversals
#
# Did the prediction increase after decreasing,
# or decrease after increasing?
# ============================================================

movement["previous_change"] = (
    movement
    .groupby(
        [
            "season_id",
            "team_id",
        ]
    )["prob_change"]
    .shift(1)
)


movement["direction_reversal"] = (
    (
        movement["prob_change"]
        * movement["previous_change"]
    ) < 0
)


reversal_summary = (
    movement[
        movement["previous_change"].notna()
    ]
    .groupby(
        "season_bucket",
        observed=True,
    )
    .agg(
        rows=("direction_reversal", "size"),
        reversals=("direction_reversal", "sum"),
        reversal_rate=("direction_reversal", "mean"),
        avg_change=("abs_prob_change", "mean"),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("🔄 DIRECTION REVERSALS BY SEASON STAGE")
print("=" * 70)

display(
    reversal_summary.round(4)
)


# ============================================================
# 9. Team-season stability
# ============================================================

team_stability = (
    movement
    .groupby(
        [
            "season_id",
            "team_id",
            "tri_code",
            "made_playoffs",
        ]
    )
    .agg(
        games=("abs_prob_change", "size"),
        avg_change=("abs_prob_change", "mean"),
        median_change=("abs_prob_change", "median"),
        max_change=("abs_prob_change", "max"),
        volatility=("playoff_prob", "std"),
        pct_over_10=(
            "abs_prob_change",
            lambda x: (x >= .10).mean(),
        ),
        pct_over_20=(
            "abs_prob_change",
            lambda x: (x >= .20).mean(),
        ),
    )
    .reset_index()
)


most_unstable_teams = (
    team_stability
    .sort_values(
        [
            "avg_change",
            "max_change",
        ],
        ascending=False,
    )
    .head(30)
)


print("\n" + "=" * 70)
print("🚨 MOST UNSTABLE TEAM-SEASONS")
print("=" * 70)

display(
    most_unstable_teams.round(4)
)


# ============================================================
# 10. Stability vs major misses
# ============================================================

movement["major_miss"] = (
    (
        (
            movement["made_playoffs"] == 1
        )
        & (
            movement["playoff_prob"] <= .25
        )
    )
    |
    (
        (
            movement["made_playoffs"] == 0
        )
        & (
            movement["playoff_prob"] >= .75
        )
    )
)


stability_vs_miss = (
    movement
    .groupby("major_miss")
    .agg(
        rows=("abs_prob_change", "size"),
        avg_change=("abs_prob_change", "mean"),
        median_change=("abs_prob_change", "median"),
        p90_change=(
            "abs_prob_change",
            lambda x: x.quantile(.90),
        ),
        pct_over_10=(
            "abs_prob_change",
            lambda x: (x >= .10).mean(),
        ),
        pct_over_20=(
            "abs_prob_change",
            lambda x: (x >= .20).mean(),
        ),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("🚨 STABILITY: NORMAL VS MAJOR MISSES")
print("=" * 70)

display(
    stability_vs_miss.round(4)
)


# ============================================================
# 11. Summary
# ============================================================

print("\n" + "=" * 70)
print("🔎 BLOCK 27 SUMMARY")
print("=" * 70)

print(
    f"Prediction transitions analyzed: "
    f"{len(movement):,}"
)

print(
    f"Average game-to-game movement: "
    f"{movement['abs_prob_change'].mean():.4f}"
)

print(
    f"Median game-to-game movement: "
    f"{movement['abs_prob_change'].median():.4f}"
)

print(
    f"95th percentile movement: "
    f"{movement['abs_prob_change'].quantile(.95):.4f}"
)

print(
    f"Predictions moving >= 10 points: "
    f"{(movement['abs_prob_change'] >= .10).mean():.2%}"
)

print(
    f"Predictions moving >= 20 points: "
    f"{(movement['abs_prob_change'] >= .20).mean():.2%}"
)

In [ ]:
# ============================================================
# BLOCK 28 — TEMPORAL INTEGRITY / LEAKAGE DIAGNOSIS
# ============================================================

print("=" * 70)
print("🔎 BLOCK 28 — TEMPORAL INTEGRITY / LEAKAGE DIAGNOSIS")
print("=" * 70)

features = bundle["features"]

df = df_scored.copy()

# ------------------------------------------------------------
# 1. Basic ordering / uniqueness
# ------------------------------------------------------------

df = df.sort_values(
    ["season_id", "team_id", "games_played", "game_date", "game_id"]
).copy()

key_cols = ["season_id", "team_id", "game_id"]

print("\n📋 ROW / KEY CHECKS")
print("-" * 70)

print(f"Rows: {len(df):,}")
print(
    "Duplicate season/team/game keys:",
    df.duplicated(key_cols).sum()
)

# games_played should increase monotonically within team-season
gp_check = (
    df.groupby(["season_id", "team_id"])["games_played"]
      .apply(lambda x: x.is_monotonic_increasing)
)

print(
    "Team-seasons with non-monotonic games_played:",
    (~gp_check).sum()
)


# ------------------------------------------------------------
# 2. Find suspicious feature names
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🚨 SUSPICIOUS FEATURE-NAME SCAN")
print("=" * 70)

leak_terms = [
    "made_playoffs",
    "playoff_prob",
    "clinched",
    "eliminated",
    "final",
    "season_end",
    "end_of_season",
    "future_result",
    "next_game",
    "next_",
    "outcome",
    "result",
    "target",
]

suspicious_features = [
    f for f in features
    if any(term in f.lower() for term in leak_terms)
]

print(f"Model features: {len(features)}")
print(f"Suspicious names found: {len(suspicious_features)}")

if suspicious_features:
    for f in suspicious_features:
        print("  ⚠️", f)
else:
    print("  ✅ None")


# ------------------------------------------------------------
# 3. Features suspiciously constant within a season
#
# A feature that contains an eventual season result can often
# appear constant for every row of a team-season.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("📊 WITHIN TEAM-SEASON FEATURE VARIATION")
print("=" * 70)

variation_rows = []

for feature in features:

    n_unique = (
        df.groupby(["season_id", "team_id"])[feature]
          .nunique(dropna=True)
    )

    variation_rows.append({
        "feature": feature,
        "team_seasons": len(n_unique),
        "constant_team_seasons": (n_unique <= 1).sum(),
        "pct_constant": (n_unique <= 1).mean()
    })

feature_variation = (
    pd.DataFrame(variation_rows)
      .sort_values("pct_constant", ascending=False)
)

print("\nFeatures most often constant within team-season:")
display(feature_variation.head(25))


# ------------------------------------------------------------
# 4. Compare early values with end-of-season values
#
# We are looking for features that somehow already equal their
# final value very early in the season.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("⏳ EARLY VALUE == FINAL VALUE")
print("=" * 70)

early_final_rows = []

for feature in features:

    temp = df[
        ["season_id", "team_id", "games_played", feature]
    ].dropna(subset=[feature]).copy()

    if temp.empty:
        continue

    first_rows = (
        temp.sort_values("games_played")
            .groupby(["season_id", "team_id"])
            .first()
            .reset_index()
    )

    last_rows = (
        temp.sort_values("games_played")
            .groupby(["season_id", "team_id"])
            .last()
            .reset_index()
    )

    comp = first_rows.merge(
        last_rows,
        on=["season_id", "team_id"],
        suffixes=("_first", "_last")
    )

    if len(comp) == 0:
        continue

    same = (
        comp[f"{feature}_first"]
        == comp[f"{feature}_last"]
    )

    early_final_rows.append({
        "feature": feature,
        "team_seasons": len(comp),
        "same_first_last": same.sum(),
        "pct_same_first_last": same.mean()
    })

early_final_check = (
    pd.DataFrame(early_final_rows)
      .sort_values("pct_same_first_last", ascending=False)
)

display(early_final_check.head(30))


# ------------------------------------------------------------
# 5. Perfect / near-perfect target relationships
#
# A feature being extremely predictive from Game 1-10 is worth
# inspecting for leakage.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎯 EARLY-SEASON FEATURE/TARGET RELATIONSHIPS")
print("=" * 70)

early = df[df["games_played"] <= 10].copy()

target_corrs = []

for feature in features:

    x = pd.to_numeric(early[feature], errors="coerce")
    y = early["made_playoffs"]

    valid = x.notna() & y.notna()

    if valid.sum() < 50:
        continue

    if x[valid].nunique() <= 1:
        continue

    corr = x[valid].corr(y[valid])

    target_corrs.append({
        "feature": feature,
        "rows": valid.sum(),
        "correlation": corr,
        "abs_correlation": abs(corr)
    })

early_target_corr = (
    pd.DataFrame(target_corrs)
      .sort_values("abs_correlation", ascending=False)
)

print("\nStrongest Game 1-10 target relationships:")
display(early_target_corr.head(30))


# ------------------------------------------------------------
# 6. Features that look suspiciously perfect
# ------------------------------------------------------------

near_perfect = early_target_corr[
    early_target_corr["abs_correlation"] >= 0.90
].copy()

print("\n" + "=" * 70)
print("🚨 NEAR-PERFECT EARLY TARGET RELATIONSHIPS")
print("=" * 70)

if len(near_perfect):
    display(near_perfect)
else:
    print("✅ No early-season feature has |correlation| >= 0.90")


# ------------------------------------------------------------
# 7. Missingness by games played
#
# This helps detect features populated using information that
# only becomes available later.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🕳️ FEATURE AVAILABILITY — GAME 1 VS LATER")
print("=" * 70)

availability_rows = []

game1 = df[df["games_played"] == 1]
later = df[df["games_played"] >= 10]

for feature in features:

    availability_rows.append({
        "feature": feature,
        "game1_null_pct": game1[feature].isna().mean(),
        "later_null_pct": later[feature].isna().mean(),
        "difference": (
            game1[feature].isna().mean()
            - later[feature].isna().mean()
        )
    })

availability = (
    pd.DataFrame(availability_rows)
      .sort_values("difference", ascending=False)
)

display(availability.head(30))


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🔎 BLOCK 28 SUMMARY")
print("=" * 70)

print(f"Rows checked: {len(df):,}")
print(f"Model features checked: {len(features)}")
print(f"Suspicious feature names: {len(suspicious_features)}")
print(f"Near-perfect early target relationships: {len(near_perfect)}")

print(
    "Features >=90% constant within team-season:",
    (feature_variation["pct_constant"] >= 0.90).sum()
)

print(
    "Features where first == final in >=90% of team-seasons:",
    (early_final_check["pct_same_first_last"] >= 0.90).sum()
)

In [ ]:
# ============================================================
# BLOCK 29 — FINAL MODEL VALIDATION SUMMARY
# ============================================================

print("=" * 70)
print("🏁 BLOCK 29 — FINAL MODEL VALIDATION SUMMARY")
print("=" * 70)


# ============================================================
# 1. DATA INTEGRITY
# ============================================================

print("\n📋 DATA INTEGRITY")
print("-" * 70)

data_checks = {
    "Rows scored": len(df_scored),

    "Duplicate team-game rows":
        df_scored.duplicated(
            ["season_id", "team_id", "game_id"]
        ).sum(),

    "Null targets":
        df_scored["made_playoffs"].isna().sum(),

    "Invalid targets":
        (~df_scored["made_playoffs"].isin([0, 1])).sum(),

    "Model features":
        len(bundle["features"]),

    "Missing model features":
        len(
            set(bundle["features"])
            - set(df_scored.columns)
        ),
}

for check, value in data_checks.items():
    print(f"{check}: {value:,}")


# ============================================================
# 2. OUT-OF-SAMPLE PERFORMANCE
# ============================================================

print("\n" + "=" * 70)
print("🎯 OUT-OF-SAMPLE PERFORMANCE")
print("=" * 70)

print(
    f"Holdout seasons evaluated: "
    f"{season_metrics['season_id'].nunique()}"
)

print(
    f"Mean ROC AUC: "
    f"{season_metrics['roc_auc'].mean():.4f}"
)

print(
    f"Median ROC AUC: "
    f"{season_metrics['roc_auc'].median():.4f}"
)

print(
    f"Mean Brier score: "
    f"{season_metrics['brier_score'].mean():.4f}"
)

print(
    f"Median Brier score: "
    f"{season_metrics['brier_score'].median():.4f}"
)

print(
    f"Mean ECE: "
    f"{season_metrics['ece'].mean():.4f}"
)


# ============================================================
# 3. BEST / WORST HOLDOUT SEASONS
# ============================================================

print("\n" + "=" * 70)
print("📅 HOLDOUT-SEASON RANGE")
print("=" * 70)

best_auc = season_metrics.loc[
    season_metrics["roc_auc"].idxmax()
]

worst_auc = season_metrics.loc[
    season_metrics["roc_auc"].idxmin()
]

best_brier = season_metrics.loc[
    season_metrics["brier_score"].idxmin()
]

worst_brier = season_metrics.loc[
    season_metrics["brier_score"].idxmax()
]

print(
    f"Highest AUC: "
    f"{int(best_auc['season_id'])} "
    f"({best_auc['roc_auc']:.4f})"
)

print(
    f"Lowest AUC: "
    f"{int(worst_auc['season_id'])} "
    f"({worst_auc['roc_auc']:.4f})"
)

print(
    f"Lowest Brier: "
    f"{int(best_brier['season_id'])} "
    f"({best_brier['brier_score']:.4f})"
)

print(
    f"Highest Brier: "
    f"{int(worst_brier['season_id'])} "
    f"({worst_brier['brier_score']:.4f})"
)


# ============================================================
# 4. PERFORMANCE BY SEASON STAGE
# ============================================================

print("\n" + "=" * 70)
print("📈 PERFORMANCE BY SEASON STAGE")
print("=" * 70)

overall_stage_summary = (
    stage_season_metrics
    .groupby("season_bucket")
    .agg(
        seasons=("season_id", "nunique"),
        rows=("rows", "sum"),
        avg_auc=("roc_auc", "mean"),
        avg_brier=("brier_score", "mean"),
        avg_ece=("ece", "mean"),
    )
    .reset_index()
)

stage_order = {
    "early": 0,
    "mid": 1,
    "late": 2,
}

overall_stage_summary["_order"] = (
    overall_stage_summary["season_bucket"]
    .map(stage_order)
)

overall_stage_summary = (
    overall_stage_summary
    .sort_values("_order")
    .drop(columns="_order")
)

display(overall_stage_summary.round(4))


# ============================================================
# 5. RECENT PERFORMANCE
# ============================================================

print("\n" + "=" * 70)
print("🕒 RECENT PERFORMANCE")
print("=" * 70)

display(period_summary.round(4))

print("\nRecent 3 seasons by stage:")
display(recent_stage_summary.round(4))


# ============================================================
# 6. CALIBRATION
# ============================================================

print("\n" + "=" * 70)
print("🎯 CALIBRATION")
print("=" * 70)

overall_ece = calculate_ece(
    rolling_predictions
)

print(
    f"Overall out-of-sample ECE: "
    f"{overall_ece:.4f}"
)

print(
    f"Average holdout-season ECE: "
    f"{season_metrics['ece'].mean():.4f}"
)

print(
    f"Maximum holdout-season ECE: "
    f"{season_metrics['ece'].max():.4f}"
)


# ============================================================
# 7. PREDICTION STABILITY
# ============================================================

print("\n" + "=" * 70)
print("📉 PREDICTION STABILITY")
print("=" * 70)

stability = (
    rolling_predictions
    .sort_values(
        ["season_id", "team_id", "games_played"]
    )
    .copy()
)

stability["prob_change"] = (
    stability
    .groupby(["season_id", "team_id"])["playoff_prob"]
    .diff()
    .abs()
)

moves = stability["prob_change"].dropna()

print(
    f"Transitions analyzed: "
    f"{len(moves):,}"
)

print(
    f"Average game-to-game movement: "
    f"{moves.mean():.4f}"
)

print(
    f"Median game-to-game movement: "
    f"{moves.median():.4f}"
)

print(
    f"95th percentile movement: "
    f"{moves.quantile(.95):.4f}"
)

print(
    f"Moves >= 10 points: "
    f"{(moves >= .10).mean():.2%}"
)

print(
    f"Moves >= 20 points: "
    f"{(moves >= .20).mean():.2%}"
)


# ============================================================
# 8. LEAKAGE / TEMPORAL INTEGRITY
# ============================================================

print("\n" + "=" * 70)
print("🔒 TEMPORAL INTEGRITY")
print("=" * 70)

print(
    f"Suspicious feature names: "
    f"{len(suspicious_features)}"
)

print(
    f"Near-perfect early target relationships: "
    f"{len(near_perfect)}"
)

print(
    "Features >=90% constant within team-season:",
    (
        feature_variation["pct_constant"] >= .90
    ).sum()
)

print(
    "Features first == final in >=90% of team-seasons:",
    (
        early_final_check["pct_same_first_last"] >= .90
    ).sum()
)


# ============================================================
# 9. VALIDATION FLAGS
# ============================================================

print("\n" + "=" * 70)
print("🚦 VALIDATION FLAGS")
print("=" * 70)

flags = []

# ----- Critical data checks

if data_checks["Duplicate team-game rows"] == 0:
    flags.append(("PASS", "No duplicate team-game rows"))
else:
    flags.append(("FIX", "Duplicate team-game rows exist"))

if data_checks["Null targets"] == 0:
    flags.append(("PASS", "No null targets"))
else:
    flags.append(("FIX", "Null targets exist"))

if data_checks["Invalid targets"] == 0:
    flags.append(("PASS", "Targets contain only 0/1"))
else:
    flags.append(("FIX", "Invalid target values exist"))

if data_checks["Missing model features"] == 0:
    flags.append(("PASS", "All model features available"))
else:
    flags.append(("FIX", "Model features are missing"))


# ----- Leakage

if len(suspicious_features) == 0:
    flags.append((
        "PASS",
        "No suspicious feature names"
    ))
else:
    flags.append((
        "WATCH",
        "Potentially suspicious feature names"
    ))

if len(near_perfect) == 0:
    flags.append((
        "PASS",
        "No near-perfect early target relationships"
    ))
else:
    flags.append((
        "FIX",
        "Near-perfect early target relationship detected"
    ))


# ----- Calibration

if overall_ece <= .03:
    flags.append((
        "PASS",
        f"Strong overall calibration (ECE {overall_ece:.4f})"
    ))
elif overall_ece <= .05:
    flags.append((
        "WATCH",
        f"Moderate calibration error (ECE {overall_ece:.4f})"
    ))
else:
    flags.append((
        "FIX",
        f"High calibration error (ECE {overall_ece:.4f})"
    ))


# ----- Recent degradation

recent_auc = (
    season_metrics
    .tail(3)["roc_auc"]
    .mean()
)

historical_auc = (
    season_metrics
    .iloc[:-3]["roc_auc"]
    .mean()
)

if recent_auc >= historical_auc - .03:
    flags.append((
        "PASS",
        "Recent AUC is consistent with historical performance"
    ))
else:
    flags.append((
        "WATCH",
        "Recent AUC is below historical performance"
    ))


# ----- Latest season

latest = season_metrics.iloc[-1]

if latest["roc_auc"] >= .80:
    flags.append((
        "PASS",
        f"Latest-season AUC remains useful "
        f"({latest['roc_auc']:.4f})"
    ))
else:
    flags.append((
        "WATCH",
        f"Latest-season AUC is weak "
        f"({latest['roc_auc']:.4f})"
    ))


for status, message in flags:

    icon = {
        "PASS": "✅",
        "WATCH": "⚠️",
        "FIX": "❌",
    }[status]

    print(
        f"{icon} {status:<5} — {message}"
    )


# ============================================================
# 10. FINAL COUNTS
# ============================================================

pass_count = sum(
    status == "PASS"
    for status, _ in flags
)

watch_count = sum(
    status == "WATCH"
    for status, _ in flags
)

fix_count = sum(
    status == "FIX"
    for status, _ in flags
)

print("\n" + "=" * 70)
print("🏁 FINAL VALIDATION RESULT")
print("=" * 70)

print(f"PASS:  {pass_count}")
print(f"WATCH: {watch_count}")
print(f"FIX:   {fix_count}")

if fix_count:
    print(
        "\n❌ MODEL HAS ISSUES THAT SHOULD BE FIXED "
        "BEFORE PRODUCTION USE."
    )

elif watch_count:
    print(
        "\n⚠️ MODEL PASSES CORE VALIDATION, "
        "WITH ITEMS TO MONITOR."
    )

else:
    print(
        "\n✅ MODEL PASSES ALL CORE VALIDATION CHECKS."
    )

In [ ]:
# ============================================================
# BLOCK 30 — FEATURE SIGNAL BY SEASON STAGE
# ============================================================

print("=" * 70)
print("🔎 BLOCK 30 — FEATURE SIGNAL BY SEASON STAGE")
print("=" * 70)


# ============================================================
# 1. Merge OOS prediction rows back to model features
# ============================================================

features = bundle["features"]

key_cols = [
    "season_id",
    "team_id",
    "game_id",
]

feature_source = (
    df_scored[
        key_cols
        + features
    ]
    .drop_duplicates(key_cols)
    .copy()
)

stage_features = (
    rolling_predictions
    .merge(
        feature_source,
        on=key_cols,
        how="left",
        validate="one_to_one",
    )
)

print(f"\nOOS rows: {len(stage_features):,}")
print(f"Model features: {len(features)}")

missing_feature_rows = (
    stage_features[features]
    .isna()
    .all(axis=1)
    .sum()
)

print(
    "Rows missing ALL model features:",
    f"{missing_feature_rows:,}"
)


# ============================================================
# 2. Calculate feature/target relationship by stage
# ============================================================

signal_rows = []

for stage, subset in stage_features.groupby(
    "season_bucket",
    observed=True,
):

    for feature in features:

        x = pd.to_numeric(
            subset[feature],
            errors="coerce",
        )

        y = pd.to_numeric(
            subset["made_playoffs"],
            errors="coerce",
        )

        valid = (
            x.notna()
            & y.notna()
        )

        if valid.sum() < 50:
            continue

        if x[valid].nunique() <= 1:
            continue

        corr = x[valid].corr(
            y[valid]
        )

        signal_rows.append({
            "season_bucket": str(stage),
            "feature": feature,
            "rows": valid.sum(),
            "correlation": corr,
            "abs_correlation": abs(corr),
        })


feature_signal = pd.DataFrame(signal_rows)


# ============================================================
# 3. Strongest features in each season stage
# ============================================================

stage_order = [
    "early",
    "mid",
    "late",
]

for stage in stage_order:

    print("\n" + "=" * 70)
    print(
        f"🎯 STRONGEST {stage.upper()}-SEASON FEATURES"
    )
    print("=" * 70)

    stage_table = (
        feature_signal[
            feature_signal["season_bucket"] == stage
        ]
        .sort_values(
            "abs_correlation",
            ascending=False,
        )
        .head(25)
    )

    display(
        stage_table[
            [
                "feature",
                "rows",
                "correlation",
                "abs_correlation",
            ]
        ].round(4)
    )


# ============================================================
# 4. Put early / mid / late signal side-by-side
# ============================================================

signal_pivot = (
    feature_signal
    .pivot(
        index="feature",
        columns="season_bucket",
        values="abs_correlation",
    )
    .reset_index()
)

for stage in stage_order:

    if stage not in signal_pivot.columns:
        signal_pivot[stage] = np.nan


# ============================================================
# 5. Calculate change in signal over the season
# ============================================================

signal_pivot["early_to_mid"] = (
    signal_pivot["mid"]
    - signal_pivot["early"]
)

signal_pivot["mid_to_late"] = (
    signal_pivot["late"]
    - signal_pivot["mid"]
)

signal_pivot["early_to_late"] = (
    signal_pivot["late"]
    - signal_pivot["early"]
)


print("\n" + "=" * 70)
print("📈 FEATURES THAT GAIN THE MOST SIGNAL")
print("=" * 70)

gaining_signal = (
    signal_pivot
    .sort_values(
        "early_to_late",
        ascending=False,
    )
    .head(25)
)

display(
    gaining_signal[
        [
            "feature",
            "early",
            "mid",
            "late",
            "early_to_late",
        ]
    ].round(4)
)


print("\n" + "=" * 70)
print("📉 FEATURES THAT LOSE THE MOST SIGNAL")
print("=" * 70)

losing_signal = (
    signal_pivot
    .sort_values(
        "early_to_late",
        ascending=True,
    )
    .head(25)
)

display(
    losing_signal[
        [
            "feature",
            "early",
            "mid",
            "late",
            "early_to_late",
        ]
    ].round(4)
)


# ============================================================
# 6. Features strongest specifically EARLY
# ============================================================

print("\n" + "=" * 70)
print("🌱 STRONGEST EARLY-SEASON SIGNALS")
print("=" * 70)

early_strength = (
    signal_pivot
    .sort_values(
        "early",
        ascending=False,
    )
    .head(30)
)

display(
    early_strength[
        [
            "feature",
            "early",
            "mid",
            "late",
            "early_to_late",
        ]
    ].round(4)
)


# ============================================================
# 7. Features weak early but strong late
# ============================================================

weak_early_strong_late = (
    signal_pivot[
        (signal_pivot["early"] < 0.15)
        & (signal_pivot["late"] >= 0.40)
    ]
    .sort_values(
        "late",
        ascending=False,
    )
)


print("\n" + "=" * 70)
print("⏳ WEAK EARLY → STRONG LATE")
print("=" * 70)

if len(weak_early_strong_late):

    display(
        weak_early_strong_late[
            [
                "feature",
                "early",
                "mid",
                "late",
                "early_to_late",
            ]
        ].round(4)
    )

else:

    print(
        "No features met the "
        "early < .15 / late >= .40 threshold."
    )


# ============================================================
# 8. Feature availability by season stage
# ============================================================

availability_rows = []

for stage, subset in stage_features.groupby(
    "season_bucket",
    observed=True,
):

    for feature in features:

        availability_rows.append({
            "season_bucket": str(stage),
            "feature": feature,
            "null_pct": subset[feature].isna().mean(),
        })


feature_availability = pd.DataFrame(
    availability_rows
)

availability_pivot = (
    feature_availability
    .pivot(
        index="feature",
        columns="season_bucket",
        values="null_pct",
    )
    .reset_index()
)

for stage in stage_order:

    if stage not in availability_pivot.columns:
        availability_pivot[stage] = np.nan


availability_pivot[
    "early_minus_late_null"
] = (
    availability_pivot["early"]
    - availability_pivot["late"]
)


print("\n" + "=" * 70)
print("🕳️ FEATURES LEAST AVAILABLE EARLY")
print("=" * 70)

display(
    availability_pivot
    .sort_values(
        "early",
        ascending=False,
    )
    .head(25)
    [
        [
            "feature",
            "early",
            "mid",
            "late",
            "early_minus_late_null",
        ]
    ]
    .round(4)
)


# ============================================================
# 9. Overall stage signal summary
# ============================================================

stage_signal_summary = (
    feature_signal
    .groupby("season_bucket")
    .agg(
        features=("feature", "nunique"),
        mean_abs_signal=(
            "abs_correlation",
            "mean",
        ),
        median_abs_signal=(
            "abs_correlation",
            "median",
        ),
        max_abs_signal=(
            "abs_correlation",
            "max",
        ),
    )
    .reset_index()
)

stage_signal_summary["_order"] = (
    stage_signal_summary["season_bucket"]
    .map({
        "early": 0,
        "mid": 1,
        "late": 2,
    })
)

stage_signal_summary = (
    stage_signal_summary
    .sort_values("_order")
    .drop(columns="_order")
)


print("\n" + "=" * 70)
print("📊 OVERALL FEATURE SIGNAL BY STAGE")
print("=" * 70)

display(
    stage_signal_summary.round(4)
)


# ============================================================
# 10. Summary
# ============================================================

strongest_early = (
    signal_pivot
    .sort_values(
        "early",
        ascending=False,
    )
    .iloc[0]
)

biggest_gainer = (
    signal_pivot
    .sort_values(
        "early_to_late",
        ascending=False,
    )
    .iloc[0]
)

biggest_loser = (
    signal_pivot
    .sort_values(
        "early_to_late",
        ascending=True,
    )
    .iloc[0]
)


print("\n" + "=" * 70)
print("🔎 BLOCK 30 SUMMARY")
print("=" * 70)

print(
    "Strongest early feature:",
    strongest_early["feature"],
    f"({strongest_early['early']:.4f})",
)

print(
    "Biggest signal gainer:",
    biggest_gainer["feature"],
    f"({biggest_gainer['early_to_late']:+.4f})",
)

print(
    "Biggest signal loser:",
    biggest_loser["feature"],
    f"({biggest_loser['early_to_late']:+.4f})",
)

print(
    "Weak-early / strong-late features:",
    len(weak_early_strong_late),
)

In [ ]:
# ============================================================
# 31. Diagnose 2025-26 Holdout Performance
# ============================================================

from sklearn.metrics import roc_auc_score, brier_score_loss

print("=" * 70)
print("🔎 BLOCK 31 — 2025-26 HOLDOUT DIAGNOSIS")
print("=" * 70)


# ============================================================
# 1. Isolate 2025-26
# ============================================================

season_2526 = (
    rolling_predictions[
        rolling_predictions["season_id"] == 20252026
    ]
    .copy()
)

season_2526["squared_error"] = (
    season_2526["playoff_prob"]
    - season_2526["made_playoffs"]
) ** 2

season_2526["abs_error"] = (
    season_2526["playoff_prob"]
    - season_2526["made_playoffs"]
).abs()


print(f"Rows: {len(season_2526):,}")
print(
    f"AUC: "
    f"{roc_auc_score(season_2526['made_playoffs'], season_2526['playoff_prob']):.4f}"
)
print(
    f"Brier: "
    f"{brier_score_loss(season_2526['made_playoffs'], season_2526['playoff_prob']):.4f}"
)


# ============================================================
# 2. Error by team
# ============================================================

team_errors = (
    season_2526
    .groupby(
        ["team_id", "tri_code", "made_playoffs"],
        as_index=False,
    )
    .agg(
        rows=("game_id", "size"),
        avg_prob=("playoff_prob", "mean"),
        median_prob=("playoff_prob", "median"),
        min_prob=("playoff_prob", "min"),
        max_prob=("playoff_prob", "max"),
        mean_abs_error=("abs_error", "mean"),
        brier=("squared_error", "mean"),
    )
    .sort_values(
        "brier",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("🚨 TEAMS WITH LARGEST PREDICTION ERROR")
print("=" * 70)

display(
    team_errors.head(15).round(4)
)


# ============================================================
# 3. How much of total Brier error comes from each team?
# ============================================================

team_error_contribution = (
    season_2526
    .groupby(
        ["team_id", "tri_code", "made_playoffs"],
        as_index=False,
    )
    .agg(
        total_squared_error=("squared_error", "sum"),
        rows=("game_id", "size"),
    )
)

total_error = (
    team_error_contribution["total_squared_error"].sum()
)

team_error_contribution["error_share"] = (
    team_error_contribution["total_squared_error"]
    / total_error
)

team_error_contribution = (
    team_error_contribution
    .sort_values(
        "error_share",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("📊 SHARE OF TOTAL 2025-26 ERROR BY TEAM")
print("=" * 70)

display(
    team_error_contribution.head(15).round(4)
)


# ============================================================
# 4. Error by season stage
# ============================================================

stage_errors = (
    season_2526
    .groupby(
        "season_bucket",
        observed=True,
    )
    .agg(
        rows=("game_id", "size"),
        avg_prob=("playoff_prob", "mean"),
        actual_rate=("made_playoffs", "mean"),
        mean_abs_error=("abs_error", "mean"),
        brier=("squared_error", "mean"),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("📅 2025-26 ERROR BY SEASON STAGE")
print("=" * 70)

display(
    stage_errors.round(4)
)


# ============================================================
# 5. Error by playoff outcome + season stage
# ============================================================

outcome_stage = (
    season_2526
    .groupby(
        ["season_bucket", "made_playoffs"],
        observed=True,
    )
    .agg(
        rows=("game_id", "size"),
        avg_prob=("playoff_prob", "mean"),
        median_prob=("playoff_prob", "median"),
        mean_abs_error=("abs_error", "mean"),
        brier=("squared_error", "mean"),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("🏒 ERROR BY STAGE + FINAL PLAYOFF OUTCOME")
print("=" * 70)

display(
    outcome_stage.round(4)
)


# ============================================================
# 6. Team probabilities at useful checkpoints
# ============================================================

checkpoints = [5, 10, 20, 30, 40, 50, 60, 70, 80]

checkpoint_probs = (
    season_2526[
        season_2526["games_played"].isin(checkpoints)
    ]
    .pivot_table(
        index=[
            "tri_code",
            "made_playoffs",
        ],
        columns="games_played",
        values="playoff_prob",
        aggfunc="first",
    )
    .reset_index()
)

checkpoint_probs.columns.name = None


print("\n" + "=" * 70)
print("📈 TEAM PROBABILITY TRAJECTORIES")
print("=" * 70)

display(
    checkpoint_probs.round(4)
)


# ============================================================
# 7. Most confidently wrong individual predictions
# ============================================================

season_2526["wrong_confidence"] = np.where(
    season_2526["made_playoffs"] == 1,
    1 - season_2526["playoff_prob"],
    season_2526["playoff_prob"],
)

most_wrong = (
    season_2526
    .sort_values(
        "wrong_confidence",
        ascending=False,
    )
    [
        [
            "tri_code",
            "game_id",
            "game_date",
            "games_played",
            "season_bucket",
            "made_playoffs",
            "playoff_prob",
            "wrong_confidence",
        ]
    ]
    .head(30)
)


print("\n" + "=" * 70)
print("❌ MOST CONFIDENTLY WRONG PREDICTIONS")
print("=" * 70)

display(
    most_wrong.round(4)
)


# ============================================================
# 8. Summary
# ============================================================

worst_team = team_errors.iloc[0]

top_5_error_share = (
    team_error_contribution
    .head(5)["error_share"]
    .sum()
)

worst_stage = (
    stage_errors
    .sort_values("brier", ascending=False)
    .iloc[0]
)


print("\n" + "=" * 70)
print("🔎 BLOCK 31 SUMMARY")
print("=" * 70)

print(
    f"Worst-predicted team: "
    f"{worst_team['tri_code']} "
    f"(Brier {worst_team['brier']:.4f})"
)

print(
    f"Top 5 teams account for "
    f"{top_5_error_share:.2%} "
    f"of total squared error."
)

print(
    f"Worst season stage: "
    f"{worst_stage['season_bucket']} "
    f"(Brier {worst_stage['brier']:.4f})"
)

In [ ]:
# ============================================================
# 32. Compare XGB vs RF vs Ensemble
# ============================================================

from sklearn.metrics import roc_auc_score, brier_score_loss

print("=" * 70)
print("🤖 BLOCK 32 — MODEL COMPONENT COMPARISON")
print("=" * 70)


# ============================================================
# 1. Overall out-of-sample performance
# ============================================================

model_cols = {
    "XGBoost": "playoff_prob_xgb",
    "Random Forest": "playoff_prob_rf",
    "Ensemble": "playoff_prob",
}

model_results = []

for model_name, prob_col in model_cols.items():

    y_true = rolling_predictions["made_playoffs"].astype(int)
    y_prob = rolling_predictions[prob_col].astype(float)

    model_results.append({
        "model": model_name,
        "rows": len(rolling_predictions),
        "roc_auc": roc_auc_score(
            y_true,
            y_prob,
        ),
        "brier_score": brier_score_loss(
            y_true,
            y_prob,
        ),
        "ece": calculate_ece(
            rolling_predictions,
            prob_col=prob_col,
        ),
        "mean_predicted": y_prob.mean(),
        "actual_rate": y_true.mean(),
    })


model_comparison = (
    pd.DataFrame(model_results)
    .sort_values("brier_score")
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("📊 OVERALL OUT-OF-SAMPLE PERFORMANCE")
print("=" * 70)

display(
    model_comparison.round(4)
)


# ============================================================
# 2. Compare by holdout season
# ============================================================

season_model_results = []

for season_id, season_df in rolling_predictions.groupby("season_id"):

    y_true = season_df["made_playoffs"].astype(int)

    for model_name, prob_col in model_cols.items():

        y_prob = season_df[prob_col].astype(float)

        season_model_results.append({
            "season_id": int(season_id),
            "model": model_name,
            "rows": len(season_df),

            "roc_auc": roc_auc_score(
                y_true,
                y_prob,
            ),

            "brier_score": brier_score_loss(
                y_true,
                y_prob,
            ),

            "ece": calculate_ece(
                season_df,
                prob_col=prob_col,
            ),
        })


season_model_comparison = pd.DataFrame(
    season_model_results
)


print("\n" + "=" * 70)
print("📅 PERFORMANCE BY HOLDOUT SEASON")
print("=" * 70)

display(
    season_model_comparison
    .sort_values(
        ["season_id", "model"]
    )
    .round(4)
)


# ============================================================
# 3. Determine season-level winner
#
# Lower Brier = better
# ============================================================

season_brier = (
    season_model_comparison
    .pivot(
        index="season_id",
        columns="model",
        values="brier_score",
    )
    .reset_index()
)

season_brier["best_model"] = (
    season_brier[
        [
            "XGBoost",
            "Random Forest",
            "Ensemble",
        ]
    ]
    .idxmin(axis=1)
)


print("\n" + "=" * 70)
print("🏆 LOWEST BRIER BY HOLDOUT SEASON")
print("=" * 70)

display(
    season_brier.round(4)
)


# ============================================================
# 4. Count wins
# ============================================================

model_wins = (
    season_brier["best_model"]
    .value_counts()
    .rename_axis("model")
    .reset_index(name="season_wins")
)


print("\n" + "=" * 70)
print("📊 SEASON-LEVEL WINS")
print("=" * 70)

display(model_wins)


# ============================================================
# 5. Compare by season stage
# ============================================================

stage_model_results = []

for season_bucket, subset in rolling_predictions.groupby(
    "season_bucket",
    observed=True,
):

    y_true = subset["made_playoffs"].astype(int)

    for model_name, prob_col in model_cols.items():

        y_prob = subset[prob_col].astype(float)

        stage_model_results.append({
            "season_bucket": str(season_bucket),
            "model": model_name,
            "rows": len(subset),

            "roc_auc": roc_auc_score(
                y_true,
                y_prob,
            ),

            "brier_score": brier_score_loss(
                y_true,
                y_prob,
            ),

            "ece": calculate_ece(
                subset,
                prob_col=prob_col,
            ),
        })


stage_model_comparison = pd.DataFrame(
    stage_model_results
)


print("\n" + "=" * 70)
print("📈 PERFORMANCE BY SEASON STAGE")
print("=" * 70)

display(
    stage_model_comparison
    .sort_values(
        ["season_bucket", "model"]
    )
    .round(4)
)


# ============================================================
# 6. How often does ensemble beat BOTH individual models?
# ============================================================

season_compare = (
    season_model_comparison
    .pivot(
        index="season_id",
        columns="model",
        values="brier_score",
    )
)

season_compare["ensemble_beats_both"] = (
    (season_compare["Ensemble"] < season_compare["XGBoost"])
    &
    (season_compare["Ensemble"] < season_compare["Random Forest"])
)

beats_both = (
    season_compare["ensemble_beats_both"].sum()
)

total_seasons = len(season_compare)


print("\n" + "=" * 70)
print("🔎 BLOCK 32 SUMMARY")
print("=" * 70)

print(
    f"Ensemble beats BOTH component models in "
    f"{beats_both} of {total_seasons} holdout seasons "
    f"({beats_both / total_seasons:.1%})."
)

print("\nSeason-level Brier winners:")
print(
    season_brier["best_model"]
    .value_counts()
)

In [ ]:
# ============================================================
# 33. Finalize V6 Validation Artifact
# ============================================================

from datetime import datetime, timezone
import joblib


print("=" * 70)
print("📦 BLOCK 33 — FINALIZE V6 VALIDATION ARTIFACT")
print("=" * 70)


# ============================================================
# 1. Overall rolling out-of-sample metrics
# ============================================================

y_true = rolling_predictions["made_playoffs"].astype(int)

overall_model_metrics = {}

model_cols = {
    "xgb": "playoff_prob_xgb",
    "rf": "playoff_prob_rf",
    "ensemble": "playoff_prob",
}

for model_name, prob_col in model_cols.items():

    y_prob = rolling_predictions[prob_col].astype(float)

    overall_model_metrics[model_name] = {
        "roc_auc": float(
            roc_auc_score(
                y_true,
                y_prob,
            )
        ),
        "brier_score": float(
            brier_score_loss(
                y_true,
                y_prob,
            )
        ),
        "ece": float(
            calculate_ece(
                rolling_predictions,
                prob_col=prob_col,
            )
        ),
    }


# ============================================================
# 2. Validation coverage
# ============================================================

holdout_seasons = (
    rolling_predictions["season_id"]
    .drop_duplicates()
    .sort_values()
    .astype(int)
    .tolist()
)

validation_coverage = {
    "first_holdout_season": holdout_seasons[0],
    "last_holdout_season": holdout_seasons[-1],
    "holdout_seasons": holdout_seasons,
    "n_holdout_seasons": len(holdout_seasons),
    "oos_rows": len(rolling_predictions),
}


# ============================================================
# 3. Model-component comparison
# ============================================================

season_winners = (
    season_brier[
        [
            "season_id",
            "best_model",
        ]
    ]
    .copy()
)

winner_counts = (
    season_winners["best_model"]
    .value_counts()
    .to_dict()
)

ensemble_beats_both = int(
    (
        (season_brier["Ensemble"] < season_brier["XGBoost"])
        &
        (season_brier["Ensemble"] < season_brier["Random Forest"])
    ).sum()
)


component_comparison = {
    "season_brier_wins": {
        str(k): int(v)
        for k, v in winner_counts.items()
    },

    "ensemble_beats_both_seasons":
        ensemble_beats_both,

    "ensemble_beats_both_rate":
        float(
            ensemble_beats_both
            / len(season_brier)
        ),

    "production_probability":
        "ensemble",
}


# ============================================================
# 4. Stage-level performance
# ============================================================

stage_validation = (
    stage_model_comparison[
        stage_model_comparison["model"] == "Ensemble"
    ]
    [
        [
            "season_bucket",
            "rows",
            "roc_auc",
            "brier_score",
            "ece",
        ]
    ]
    .copy()
)

stage_validation_records = (
    stage_validation
    .to_dict(orient="records")
)


# ============================================================
# 5. Season-level ensemble performance
# ============================================================

season_validation_records = (
    season_metrics[
        [
            "season_id",
            "rows",
            "roc_auc",
            "brier_score",
            "ece",
            "mean_predicted",
            "actual_playoff_rate",
            "mean_bias",
        ]
    ]
    .copy()
    .to_dict(orient="records")
)


# ============================================================
# 6. Validation conclusions
# ============================================================

validation_conclusions = {

    "core_validation_status":
        "PASS",

    "temporal_integrity":
        "PASS",

    "target_integrity":
        "PASS",

    "feature_availability":
        "PASS",

    "overall_calibration":
        "PASS",

    "prediction_stability":
        "PASS",

    "latest_holdout_season":
        20252026,

    "latest_holdout_auc":
        float(
            season_metrics.loc[
                season_metrics["season_id"] == 20252026,
                "roc_auc",
            ].iloc[0]
        ),

    "latest_holdout_brier":
        float(
            season_metrics.loc[
                season_metrics["season_id"] == 20252026,
                "brier_score",
            ].iloc[0]
        ),

    "latest_holdout_ece":
        float(
            season_metrics.loc[
                season_metrics["season_id"] == 20252026,
                "ece",
            ].iloc[0]
        ),

    "notes": [
        "Early-season performance is materially weaker than mid/late season.",
        "Feature signal strengthens substantially as the season progresses.",
        "2025-26 degradation is concentrated in several unusual team trajectories.",
        "Random Forest has the best aggregate Brier score.",
        "Ensemble has the best aggregate calibration.",
        "Ensemble retained as production probability.",
    ],
}


# ============================================================
# 7. Build validation artifact
# ============================================================

validation_artifact = {

    "model_version":
        bundle.get("version"),

    "model_type":
        bundle.get("model_type"),

    "validated_at":
        datetime.now(timezone.utc).isoformat(),

    "source_table":
        bundle.get("source_table"),

    "target":
        bundle.get("target"),

    "features":
        bundle["features"],

    "n_features":
        len(bundle["features"]),

    "coverage":
        validation_coverage,

    "overall_metrics":
        overall_model_metrics,

    "component_comparison":
        component_comparison,

    "stage_performance":
        stage_validation_records,

    "season_performance":
        season_validation_records,

    "conclusions":
        validation_conclusions,
}


# ============================================================
# 8. Save validation artifact
# ============================================================

validation_file = (
    MODEL_PATH.parent
    / f"playoff_model_{bundle.get('version')}_validation.joblib"
)

joblib.dump(
    validation_artifact,
    validation_file,
)


# ============================================================
# 9. Final verification
# ============================================================

print("=" * 70)
print("💾 VALIDATION ARTIFACT SAVED")
print("=" * 70)

print(f"Model file:      {MODEL_PATH}")
print(f"Validation file: {validation_file}")
print(f"Exists:          {validation_file.exists()}")

print("\nSaved sections:")
for key in validation_artifact.keys():
    print(f"  ✓ {key}")

# ============================================================
# 9. Final verification
# ============================================================

saved_validation = joblib.load(
    validation_file
)

print(f"\nSaved: {validation_file}")

print(
    "Model version:",
    saved_validation["model_version"],
)

print(
    "Features:",
    saved_validation["n_features"],
)

print(
    "Holdout seasons:",
    saved_validation["coverage"]["n_holdout_seasons"],
)

print(
    "OOS rows:",
    f"{saved_validation['coverage']['oos_rows']:,}",
)

print("\nOverall metrics:")

for model_name, metrics in (
    saved_validation["overall_metrics"].items()
):

    print(
        f"  {model_name}: "
        f"AUC={metrics['roc_auc']:.4f}, "
        f"Brier={metrics['brier_score']:.4f}, "
        f"ECE={metrics['ece']:.4f}"
    )

print(
    "\nProduction probability:",
    saved_validation[
        "component_comparison"
    ]["production_probability"],
)

print(
    "Validation status:",
    saved_validation[
        "conclusions"
    ]["core_validation_status"],
)

print(
    "\n✅ V6 validation artifact finalized."
)

In [ ]:
# ============================================================
# 8. Save validation artifact
# ============================================================

validation_file = (
    MODEL_PATH.parent
    / f"playoff_model_{bundle.get('version')}_validation.joblib"
)

joblib.dump(
    validation_artifact,
    validation_file,
)


# ============================================================
# 9. Final verification
# ============================================================

print("=" * 70)
print("💾 VALIDATION ARTIFACT SAVED")
print("=" * 70)

print(f"Model file:      {MODEL_PATH}")
print(f"Validation file: {validation_file}")
print(f"Exists:          {validation_file.exists()}")

print("\nSaved sections:")
for key in validation_artifact.keys():
    print(f"  ✓ {key}")

In [ ]:
# ============================================================
# 34. Production Model Verification
# ============================================================

print("=" * 70)
print("🚀 BLOCK 34 — PRODUCTION MODEL VERIFICATION")
print("=" * 70)


# ============================================================
# 1. Inspect production bundle
# ============================================================

required_bundle_keys = [
    "xgb_pipeline",
    "rf_pipeline",
    "features",
    "trained_on",
    "model_type",
    "version",
    "source_table",
    "target",
]

missing_bundle_keys = [
    key
    for key in required_bundle_keys
    if key not in bundle
]


print("\n📦 PRODUCTION BUNDLE")
print("-" * 70)

print(f"Version:          {bundle.get('version')}")
print(f"Model type:       {bundle.get('model_type')}")
print(f"Trained on:       {bundle.get('trained_on')}")
print(f"Source table:     {bundle.get('source_table')}")
print(f"Target:           {bundle.get('target')}")
print(f"Feature count:    {len(bundle.get('features', []))}")
print(f"Missing keys:     {missing_bundle_keys}")


# ============================================================
# 2. Validate saved feature list
# ============================================================

production_features = bundle["features"]

duplicate_features = (
    pd.Series(production_features)
    .duplicated()
    .sum()
)

missing_from_scored = [
    feature
    for feature in production_features
    if feature not in df_scored.columns
]


print("\n🧩 FEATURE CONTRACT")
print("-" * 70)

print(f"Saved features:               {len(production_features)}")
print(f"Duplicate feature names:      {duplicate_features}")
print(f"Missing from df_scored:       {len(missing_from_scored)}")

if missing_from_scored:
    print("\nMissing features:")
    print(missing_from_scored)


# ============================================================
# 3. Build production scoring matrix
# ============================================================

X_production_test = (
    df_scored
    .reindex(columns=production_features)
    .copy()
)


print("\n📐 SCORING MATRIX")
print("-" * 70)

print(f"Rows:                         {len(X_production_test):,}")
print(f"Columns:                      {X_production_test.shape[1]}")
print(
    f"Feature order exact:          "
    f"{list(X_production_test.columns) == production_features}"
)


# ============================================================
# 4. Check nulls / infinities
# ============================================================

null_counts = (
    X_production_test
    .isna()
    .sum()
)

features_with_nulls = (
    null_counts[
        null_counts > 0
    ]
    .sort_values(ascending=False)
)


numeric_test = X_production_test.select_dtypes(
    include="number"
)

infinite_counts = pd.Series(
    {
        col: np.isinf(numeric_test[col]).sum()
        for col in numeric_test.columns
    }
)

features_with_infinity = (
    infinite_counts[
        infinite_counts > 0
    ]
    .sort_values(ascending=False)
)


print("\n🧹 INPUT QUALITY")
print("-" * 70)

print(
    f"Features containing nulls:    "
    f"{len(features_with_nulls)}"
)

print(
    f"Features containing infinity: "
    f"{len(features_with_infinity)}"
)

if len(features_with_nulls):
    print("\nTop null-containing features:")
    print(features_with_nulls.head(20))

if len(features_with_infinity):
    print("\nFeatures containing infinity:")
    print(features_with_infinity)


# ============================================================
# 5. Match production preprocessing behavior
#
# Training replaces missing values with 0 before fitting.
# Do the same here before scoring.
# ============================================================

X_production_test = (
    X_production_test
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)


# ============================================================
# 6. Score using ACTUAL saved production models
# ============================================================

production_xgb_prob = (
    bundle["xgb_pipeline"]
    .predict_proba(X_production_test)[:, 1]
)

production_rf_prob = (
    bundle["rf_pipeline"]
    .predict_proba(X_production_test)[:, 1]
)

production_ensemble_prob = (
    production_xgb_prob
    + production_rf_prob
) / 2


# ============================================================
# 7. Validate probability output
# ============================================================

probability_check = pd.DataFrame({

    "xgb": production_xgb_prob,
    "rf": production_rf_prob,
    "ensemble": production_ensemble_prob,

})


probability_summary = (
    probability_check
    .agg([
        "min",
        "max",
        "mean",
        "std",
    ])
    .T
)


print("\n🎯 PRODUCTION PROBABILITY OUTPUT")
print("-" * 70)

display(
    probability_summary.round(4)
)


invalid_probabilities = {

    model: int(
        (
            (~np.isfinite(probability_check[model]))
            |
            (probability_check[model] < 0)
            |
            (probability_check[model] > 1)
        ).sum()
    )

    for model in probability_check.columns

}


print("\nInvalid probabilities:")

for model, count in invalid_probabilities.items():
    print(f"  {model:10}: {count:,}")


# ============================================================
# 8. Compare component disagreement
# ============================================================

probability_check["xgb_rf_difference"] = (
    probability_check["xgb"]
    - probability_check["rf"]
)

probability_check["abs_xgb_rf_difference"] = (
    probability_check["xgb_rf_difference"]
    .abs()
)


print("\n🤝 MODEL AGREEMENT")
print("-" * 70)

print(
    "Mean absolute XGB/RF difference: "
    f"{probability_check['abs_xgb_rf_difference'].mean():.4f}"
)

print(
    "Median absolute difference:      "
    f"{probability_check['abs_xgb_rf_difference'].median():.4f}"
)

print(
    "95th percentile difference:      "
    f"{probability_check['abs_xgb_rf_difference'].quantile(.95):.4f}"
)

print(
    "Difference >= 10 points:         "
    f"{(probability_check['abs_xgb_rf_difference'] >= .10).mean():.2%}"
)

print(
    "Difference >= 20 points:         "
    f"{(probability_check['abs_xgb_rf_difference'] >= .20).mean():.2%}"
)


# ============================================================
# 9. Production verification flags
# ============================================================

checks = {

    "Required bundle keys present":
        len(missing_bundle_keys) == 0,

    "No duplicate saved features":
        duplicate_features == 0,

    "All production features available":
        len(missing_from_scored) == 0,

    "Feature order matches saved contract":
        list(X_production_test.columns) == production_features,

    "XGB probabilities valid":
        invalid_probabilities["xgb"] == 0,

    "RF probabilities valid":
        invalid_probabilities["rf"] == 0,

    "Ensemble probabilities valid":
        invalid_probabilities["ensemble"] == 0,

}


print("\n" + "=" * 70)
print("🚦 PRODUCTION VERIFICATION")
print("=" * 70)

for check, passed in checks.items():

    symbol = "✅ PASS" if passed else "❌ FAIL"

    print(
        f"{symbol:8} — {check}"
    )


# ============================================================
# 10. Final result
# ============================================================

passed_checks = sum(checks.values())
failed_checks = len(checks) - passed_checks


print("\n" + "=" * 70)
print("🏁 BLOCK 34 RESULT")
print("=" * 70)

print(f"PASS: {passed_checks}")
print(f"FAIL: {failed_checks}")


if failed_checks == 0:

    print(
        "\n✅ SAVED PRODUCTION MODEL PASSES "
        "ARTIFACT AND SCORING VERIFICATION."
    )

else:

    print(
        "\n❌ PRODUCTION VERIFICATION FOUND "
        "ISSUES THAT SHOULD BE INVESTIGATED."
    )

In [ ]:
# ============================================================
# 35. End-to-End Daily Inference Validation
# ============================================================

print("=" * 70)
print("🚀 BLOCK 35 — END-TO-END DAILY INFERENCE VALIDATION")
print("=" * 70)


# ============================================================
# 1. Configuration
# ============================================================

season_id = 20252026
SOURCE_TABLE = "team_daily_stats"

production_features = FEATURE_COLS


# ============================================================
# 2. Build the daily team-date spine
#
# This mirrors the production concept:
# every team receives a row for every league date.
# ============================================================

query = f"""
WITH league_dates AS (

    SELECT DISTINCT
        game_date

    FROM {SOURCE_TABLE}

    WHERE season_id = ?
),

teams AS (

    SELECT DISTINCT
        team_id

    FROM {SOURCE_TABLE}

    WHERE season_id = ?
),

team_dates AS (

    SELECT
        ? AS season_id,
        t.team_id,
        d.game_date

    FROM teams t
    CROSS JOIN league_dates d
),

source_rows AS (

    SELECT *

    FROM {SOURCE_TABLE}

    WHERE season_id = ?
),

joined AS (

    SELECT
        td.season_id,
        td.team_id,
        td.game_date AS asof_date,

        s.* EXCLUDE (
            season_id,
            team_id
        )

    FROM team_dates td

    ASOF LEFT JOIN source_rows s
        ON td.team_id = s.team_id
       AND td.game_date >= s.game_date
)

SELECT *

FROM joined

ORDER BY
    team_id,
    asof_date
"""

df_daily = con.execute(
    query,
    [
        season_id,   # league_dates
        season_id,   # teams
        season_id,   # team_dates season_id
        season_id,   # source_rows
    ],
).df()


print(f"\nDaily spine rows: {len(df_daily):,}")
print(
    f"Teams: "
    f"{df_daily['team_id'].nunique():,}"
)
print(
    f"League dates: "
    f"{df_daily['asof_date'].nunique():,}"
)


# ============================================================
# 3. Clean ASOF output
#
# Preserve the actual ASOF date as game_date so that
# date-relative / league-relative features are calculated
# for the current daily snapshot.
# ============================================================

if "game_date" in df_daily.columns:
    df_daily = df_daily.drop(
        columns=["game_date"]
    )

df_daily = df_daily.rename(
    columns={
        "asof_date": "game_date",
    }
)

df_daily["game_date"] = pd.to_datetime(
    df_daily["game_date"]
)


# ============================================================
# 4. Check for duplicate team-day rows
# ============================================================

duplicate_team_days = (
    df_daily
    .duplicated(
        subset=[
            "team_id",
            "game_date",
        ],
        keep=False,
    )
)

n_duplicate_team_days = int(
    duplicate_team_days.sum()
)

print(
    f"\nDuplicate team-day rows: "
    f"{n_duplicate_team_days:,}"
)

# ============================================================
# Remove pre-first-game team-days
#
# ASOF cannot carry a team snapshot forward until that team
# has actually played its first game.
# ============================================================

pre_first_game_rows = df_daily["games_played"].isna()

print(
    f"Pre-first-game team-days removed: "
    f"{pre_first_game_rows.sum():,}"
)

df_daily = (
    df_daily[
        ~pre_first_game_rows
    ]
    .copy()
    .reset_index(drop=True)
)

# ============================================================
# 5. Build production features
# ============================================================

df_daily_features = build_features(
    df_daily
)


print(
    f"Rows after feature engineering: "
    f"{len(df_daily_features):,}"
)


# ============================================================
# 6. Validate model feature contract
# ============================================================

missing_features = [
    col
    for col in production_features
    if col not in df_daily_features.columns
]

print(
    f"\nRequired model features: "
    f"{len(production_features):,}"
)

print(
    f"Missing model features: "
    f"{len(missing_features):,}"
)

if missing_features:
    print("\nMissing features:")
    for col in missing_features:
        print(f"  - {col}")


# ============================================================
# 7. Check model input for infinity
# ============================================================

if not missing_features:

    model_input = (
        df_daily_features[
            production_features
        ]
        .copy()
    )

    numeric_input = (
        model_input
        .select_dtypes(
            include="number"
        )
    )

    infinity_mask = np.isinf(
        numeric_input
    )

    n_infinity = int(
        infinity_mask.sum().sum()
    )

else:

    n_infinity = -1


print(
    f"\nInfinite model-input values: "
    f"{n_infinity:,}"
)


# ============================================================
# 8. Score using the ACTUAL production inference function
# ============================================================

df_daily_scored = score_dataframe(
    df_daily_features
)


print(
    f"\nScored rows: "
    f"{len(df_daily_scored):,}"
)


# ============================================================
# 9. Validate probability ranges
# ============================================================

probability_cols = [
    "playoff_prob_xgb",
    "playoff_prob_rf",
    "playoff_prob",
]

invalid_probability_counts = {}

for col in probability_cols:

    invalid = (
        df_daily_scored[col].isna()
        |
        ~df_daily_scored[col].between(
            0,
            1,
            inclusive="both",
        )
    )

    invalid_probability_counts[col] = int(
        invalid.sum()
    )


print("\nInvalid probabilities:")

for col, count in invalid_probability_counts.items():
    print(
        f"  {col:<20}: {count:,}"
    )


# ============================================================
# 10. Validate ensemble calculation
# ============================================================

expected_ensemble = (
    df_daily_scored["playoff_prob_xgb"]
    + df_daily_scored["playoff_prob_rf"]
) / 2


incorrect_ensemble = ~np.isclose(
    df_daily_scored["playoff_prob"],
    expected_ensemble,
    atol=1e-12,
    rtol=0,
)


n_incorrect_ensemble = int(
    incorrect_ensemble.sum()
)


print("\n🤝 ENSEMBLE")
print("-" * 70)

print(
    f"Rows with incorrect ensemble: "
    f"{n_incorrect_ensemble:,}"
)


# ============================================================
# 11. Validate carried-forward prediction consistency
#
# IMPORTANT:
#
# A team's probability MAY change on a day when that team
# does not play.
#
# League-relative and playoff-race features can change because
# OTHER teams played.
#
# Therefore the correct integrity test is:
#
# If probability changes, at least one MODEL INPUT must also
# have changed.
# ============================================================

ordered = (
    df_daily_scored
    .sort_values(
        [
            "team_id",
            "game_date",
        ]
    )
    .copy()
)


# ------------------------------------------------------------
# Previous GP and probability
# ------------------------------------------------------------

ordered["prev_games_played"] = (
    ordered
    .groupby("team_id")[
        "games_played"
    ]
    .shift()
)


ordered["prev_playoff_prob"] = (
    ordered
    .groupby("team_id")[
        "playoff_prob"
    ]
    .shift()
)


# ------------------------------------------------------------
# Identify carried-forward team-days
#
# Same games_played as previous league date means this team
# did not play another game.
# ------------------------------------------------------------

carried_mask = (
    ordered["games_played"]
    ==
    ordered["prev_games_played"]
)


# ============================================================
# 12. Determine whether model features changed
# ============================================================

feature_changed = pd.DataFrame(
    False,
    index=ordered.index,
    columns=production_features,
)


for col in production_features:

    previous = (
        ordered
        .groupby("team_id")[col]
        .shift()
    )

    current = ordered[col]

    feature_changed[col] = ~(
        current.eq(previous)
        |
        (
            current.isna()
            & previous.isna()
        )
    )


ordered["any_model_feature_changed"] = (
    feature_changed.any(axis=1)
)


# ============================================================
# 13. Determine whether probability changed
# ============================================================

ordered["probability_changed"] = ~np.isclose(
    ordered["playoff_prob"],
    ordered["prev_playoff_prob"],
    atol=1e-12,
    rtol=0,
)


# ============================================================
# 14. Find unexplained probability changes
#
# This is the actual invalid condition:
#
# team did not play
# +
# probability changed
# +
# none of the model inputs changed
# ============================================================

unexplained_changes = ordered[
    carried_mask
    & ordered["probability_changed"]
    & ~ordered["any_model_feature_changed"]
].copy()


carried_feature_changes = ordered[
    carried_mask
    & ordered["any_model_feature_changed"]
].copy()


carried_probability_changes = ordered[
    carried_mask
    & ordered["probability_changed"]
].copy()


print("\n📆 CARRIED-FORWARD SNAPSHOTS")
print("-" * 70)

print(
    f"Carried-forward team-days: "
    f"{int(carried_mask.sum()):,}"
)

print(
    f"Carried rows with model-input changes: "
    f"{len(carried_feature_changes):,}"
)

print(
    f"Carried rows with probability changes: "
    f"{len(carried_probability_changes):,}"
)

print(
    f"Unexplained probability changes: "
    f"{len(unexplained_changes):,}"
)


if not unexplained_changes.empty:

    display(
        unexplained_changes[
            [
                "season_id",
                "tri_code",
                "game_date",
                "games_played",
                "prev_playoff_prob",
                "playoff_prob",
            ]
        ]
        .head(20)
    )


carried_snapshot_consistency_pass = (
    len(unexplained_changes) == 0
)


# ============================================================
# 15. Production checks
# ============================================================

checks = {

    "No duplicate team-day rows":
        n_duplicate_team_days == 0,

    "All model features available":
        len(missing_features) == 0,

    "No infinity in model input":
        n_infinity == 0,

    "XGB probabilities valid":
        invalid_probability_counts[
            "playoff_prob_xgb"
        ] == 0,

    "RF probabilities valid":
        invalid_probability_counts[
            "playoff_prob_rf"
        ] == 0,

    "Ensemble probabilities valid":
        invalid_probability_counts[
            "playoff_prob"
        ] == 0,

    "Ensemble calculation exact":
        n_incorrect_ensemble == 0,

    "Carried probability changes explained by model inputs":
        carried_snapshot_consistency_pass,
}


# ============================================================
# 16. Print final results
# ============================================================

print("\n" + "=" * 70)
print("🚦 END-TO-END PRODUCTION CHECKS")
print("=" * 70)


pass_count = 0
fail_count = 0


for check_name, passed in checks.items():

    if passed:

        print(
            f"✅ PASS   — {check_name}"
        )

        pass_count += 1

    else:

        print(
            f"❌ FAIL   — {check_name}"
        )

        fail_count += 1


# ============================================================
# 17. Final Block 35 result
# ============================================================

print("\n" + "=" * 70)
print("🏁 BLOCK 35 RESULT")
print("=" * 70)

print(
    f"PASS: {pass_count}"
)

print(
    f"FAIL: {fail_count}"
)


if fail_count == 0:
    print(
        "\n✅ DAILY INFERENCE PIPELINE PASSES "
        "END-TO-END PRODUCTION VALIDATION."
    )
else:
    print(
        "\n⚠️ DAILY INFERENCE PIPELINE HAS "
        "ITEMS TO INVESTIGATE."
    )
    

In [ ]:

# ============================================================
# 36. Validate Daily Probability Database Write / Upsert
# ============================================================

print("=" * 70)
print("💾 BLOCK 36 — DAILY PROBABILITY DATABASE WRITE VALIDATION")
print("=" * 70)


# ============================================================
# 1. Configuration
# ============================================================

TEST_TABLE = "_validation_daily_playoff_probability"

key_cols = [
    "season_id",
    "team_id",
    "game_date",
]

prob_cols = [
    "playoff_prob_xgb",
    "playoff_prob_rf",
    "playoff_prob",
]


# ============================================================
# 2. Build validation output
#
# Use the exact scored output from Block 35.
# Pre-first-game rows have already been removed.
# ============================================================

write_test = df_daily_scored[
    key_cols + prob_cols
].copy()

write_test["game_date"] = pd.to_datetime(
    write_test["game_date"]
).dt.date

print(f"\nRows prepared for write: {len(write_test):,}")
print(f"Unique seasons: {write_test['season_id'].nunique():,}")
print(f"Unique teams: {write_test['team_id'].nunique():,}")
print(f"Unique dates: {write_test['game_date'].nunique():,}")


# ============================================================
# 3. Validate source rows
# ============================================================

n_source_duplicates = int(
    write_test.duplicated(
        subset=key_cols,
        keep=False
    ).sum()
)

n_source_null_keys = int(
    write_test[key_cols].isna().any(axis=1).sum()
)

print(f"\nDuplicate source keys: {n_source_duplicates:,}")
print(f"Rows with null keys: {n_source_null_keys:,}")


# ============================================================
# 4. Create isolated validation table
#
# Does NOT modify daily_playoff_probability.
# ============================================================

con.execute(f"DROP TABLE IF EXISTS {TEST_TABLE}")

con.execute(f"""
    CREATE TABLE {TEST_TABLE} (
        season_id BIGINT,
        team_id BIGINT,
        game_date DATE,
        playoff_prob_xgb DOUBLE,
        playoff_prob_rf DOUBLE,
        playoff_prob DOUBLE
    )
""")


# ============================================================
# 5. First write
# ============================================================

con.register("_validation_write_df", write_test)

con.execute(f"""
    INSERT INTO {TEST_TABLE}
    SELECT
        season_id,
        team_id,
        game_date,
        playoff_prob_xgb,
        playoff_prob_rf,
        playoff_prob
    FROM _validation_write_df
""")

first_write_count = con.execute(f"""
    SELECT COUNT(*)
    FROM {TEST_TABLE}
""").fetchone()[0]

print("\n" + "=" * 70)
print("1️⃣ FIRST WRITE")
print("=" * 70)

print(f"Expected rows: {len(write_test):,}")
print(f"Stored rows:   {first_write_count:,}")


# ============================================================
# 6. Simulate rerunning ONE production date
#
# Delete existing date rows and insert replacements.
# ============================================================

test_date = write_test["game_date"].max()

rerun_df = write_test[
    write_test["game_date"] == test_date
].copy()

print("\n" + "=" * 70)
print("🔄 SIMULATED DAILY RERUN")
print("=" * 70)

print(f"Test date: {test_date}")
print(f"Rows being rerun: {len(rerun_df):,}")


# Delete existing rows for the selected date

con.execute(
    f"""
    DELETE FROM {TEST_TABLE}
    WHERE season_id = ?
      AND game_date = ?
    """,
    [season_id, test_date]
)


# Insert replacement rows

con.unregister("_validation_write_df")

con.register("_validation_rerun_df", rerun_df)

con.execute(f"""
    INSERT INTO {TEST_TABLE}
    SELECT
        season_id,
        team_id,
        game_date,
        playoff_prob_xgb,
        playoff_prob_rf,
        playoff_prob
    FROM _validation_rerun_df
""")

second_write_count = con.execute(f"""
    SELECT COUNT(*)
    FROM {TEST_TABLE}
""").fetchone()[0]

print(f"Rows after rerun: {second_write_count:,}")


# ============================================================
# 7. Check database key integrity
# ============================================================

db_duplicate_keys = con.execute(f"""
    SELECT
        season_id,
        team_id,
        game_date,
        COUNT(*) AS rows
    FROM {TEST_TABLE}
    GROUP BY
        season_id,
        team_id,
        game_date
    HAVING COUNT(*) > 1
""").df()

db_null_keys = con.execute(f"""
    SELECT COUNT(*)
    FROM {TEST_TABLE}
    WHERE season_id IS NULL
       OR team_id IS NULL
       OR game_date IS NULL
""").fetchone()[0]


# ============================================================
# 8. Validate team coverage by date
#
# CORRECTED:
# Compare against valid source coverage, NOT 32 teams
# on every date.
#
# Teams without a first-game snapshot are not eligible
# for scoring.
# ============================================================

expected_coverage = (
    write_test
    .groupby(["season_id", "game_date"])["team_id"]
    .nunique()
    .rename("expected_teams")
    .reset_index()
)

date_coverage = con.execute(f"""
    SELECT
        season_id,
        game_date,
        COUNT(*) AS rows,
        COUNT(DISTINCT team_id) AS stored_teams
    FROM {TEST_TABLE}
    GROUP BY
        season_id,
        game_date
    ORDER BY
        season_id,
        game_date
""").df()

# Normalize date types for comparison

expected_coverage["game_date"] = pd.to_datetime(
    expected_coverage["game_date"]
).dt.date

date_coverage["game_date"] = pd.to_datetime(
    date_coverage["game_date"]
).dt.date

# Compare source coverage against stored coverage

coverage_comparison = expected_coverage.merge(
    date_coverage,
    on=["season_id", "game_date"],
    how="outer",
    indicator=True
)

coverage_comparison["coverage_matches"] = (
    coverage_comparison["_merge"].eq("both")
    & coverage_comparison["expected_teams"].eq(
        coverage_comparison["stored_teams"]
    )
)

bad_date_coverage = coverage_comparison[
    ~coverage_comparison["coverage_matches"]
].copy()

coverage_matches = bad_date_coverage.empty

print("\n" + "=" * 70)
print("📅 TEAM COVERAGE VALIDATION")
print("=" * 70)

print(f"Expected dates: {len(expected_coverage):,}")
print(f"Stored dates:   {len(date_coverage):,}")
print(f"Coverage mismatches: {len(bad_date_coverage):,}")
print(f"Coverage matches source: {coverage_matches}")


# ============================================================
# 9. Verify stored probabilities match source
# ============================================================

stored = con.execute(f"""
    SELECT *
    FROM {TEST_TABLE}
    ORDER BY
        season_id,
        team_id,
        game_date
""").df()

expected = (
    write_test
    .sort_values(key_cols)
    .reset_index(drop=True)
)

stored = (
    stored
    .sort_values(key_cols)
    .reset_index(drop=True)
)

same_keys = (
    stored[key_cols]
    .astype(str)
    .equals(
        expected[key_cols].astype(str)
    )
)

# Compare probabilities only when row counts match.
# This prevents shape errors if a database write fails.

same_xgb = (
    len(stored) == len(expected)
    and np.allclose(
        stored["playoff_prob_xgb"],
        expected["playoff_prob_xgb"],
        atol=1e-12,
        rtol=0,
        equal_nan=False
    )
)

same_rf = (
    len(stored) == len(expected)
    and np.allclose(
        stored["playoff_prob_rf"],
        expected["playoff_prob_rf"],
        atol=1e-12,
        rtol=0,
        equal_nan=False
    )
)

same_ensemble = (
    len(stored) == len(expected)
    and np.allclose(
        stored["playoff_prob"],
        expected["playoff_prob"],
        atol=1e-12,
        rtol=0,
        equal_nan=False
    )
)


# ============================================================
# 10. Validation checks
# ============================================================

checks_36 = {

    "No duplicate source keys":
        n_source_duplicates == 0,

    "No null source keys":
        n_source_null_keys == 0,

    "First write stored all rows":
        first_write_count == len(write_test),

    "Rerun did not change total row count":
        second_write_count == first_write_count,

    "No duplicate database keys":
        len(db_duplicate_keys) == 0,

    "No null database keys":
        db_null_keys == 0,

    "Database team coverage matches valid source":
        coverage_matches,

    "Stored keys match source":
        same_keys,

    "Stored XGB probabilities match source":
        same_xgb,

    "Stored RF probabilities match source":
        same_rf,

    "Stored ensemble probabilities match source":
        same_ensemble,
}


# ============================================================
# 11. Results
# ============================================================

print("\n" + "=" * 70)
print("🚦 DATABASE WRITE CHECKS")
print("=" * 70)

pass_count_36 = 0
fail_count_36 = 0

for check_name, passed in checks_36.items():

    if passed:
        print(f"✅ PASS   — {check_name}")
        pass_count_36 += 1

    else:
        print(f"❌ FAIL   — {check_name}")
        fail_count_36 += 1


# ============================================================
# 12. Show problems if any
# ============================================================

if not db_duplicate_keys.empty:
    print("\nDuplicate database keys:")
    display(db_duplicate_keys.head(20))

if not bad_date_coverage.empty:
    print("\nDates with mismatched team coverage:")
    display(bad_date_coverage.head(20))


# ============================================================
# 13. Clean up validation objects
# ============================================================

con.unregister("_validation_rerun_df")

con.execute(
    f"DROP TABLE IF EXISTS {TEST_TABLE}"
)


# ============================================================
# 14. Final result
# ============================================================

print("\n" + "=" * 70)
print("🏁 BLOCK 36 RESULT")
print("=" * 70)

print(f"PASS: {pass_count_36}")
print(f"FAIL: {fail_count_36}")

if fail_count_36 == 0:
    print(
        "\n✅ DAILY PROBABILITY DATABASE WRITE "
        "PASSES VALIDATION."
    )
else:
    print(
        "\n⚠️ DATABASE WRITE LOGIC HAS "
        "ITEMS TO INVESTIGATE."
    )


In [ ]:
# ============================================================
# 37E. Production runner interface validation
# ============================================================

import inspect
import importlib
import prediction_model.inference.daily_playoff_probability as runner

importlib.reload(runner)

signature = inspect.signature(runner.run_probability_scores)

print("Production runner signature:")
print(signature)

assert "refresh_existing" in signature.parameters
assert signature.parameters["refresh_existing"].default is False

print("✅ Production runner supports controlled refresh")

In [ ]:
# ============================================================
# 38. Isolated Production Runner Validation
# ============================================================

import duckdb
from config import SOURCE_TABLE, TARGET_TABLE
from prediction_model.inference.daily_playoff_probability import (
    run_probability_scores
)

TEST_SEASON = 20252026

# ------------------------------------------------------------
# 1. Create isolated in-memory database
# ------------------------------------------------------------

test_con = duckdb.connect(":memory:")

# Copy source data from the real database into the test database.
source_df = con.execute(
    f"""
    SELECT *
    FROM {SOURCE_TABLE}
    WHERE season_id = ?
    """,
    [TEST_SEASON]
).df()

print(f"Source rows copied: {len(source_df):,}")

test_con.register("_source_copy", source_df)

test_con.execute(f"""
    CREATE TABLE {SOURCE_TABLE} AS
    SELECT * FROM _source_copy
""")

test_con.unregister("_source_copy")

print("✅ Isolated source table created")

# ------------------------------------------------------------
# 2. First production-runner execution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRST RUN — INSERT")
print("=" * 70)

run_probability_scores(
    con=test_con,
    season_id=TEST_SEASON,
    refresh_existing=False,
)

first_count = test_con.execute(
    f"SELECT COUNT(*) FROM {TARGET_TABLE}"
).fetchone()[0]

print(f"Rows after first run: {first_count:,}")

# ------------------------------------------------------------
# 3. Second execution — incremental mode
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SECOND RUN — INCREMENTAL")
print("=" * 70)

run_probability_scores(
    con=test_con,
    season_id=TEST_SEASON,
    refresh_existing=False,
)

second_count = test_con.execute(
    f"SELECT COUNT(*) FROM {TARGET_TABLE}"
).fetchone()[0]

print(f"Rows after incremental rerun: {second_count:,}")

# ------------------------------------------------------------
# 4. Third execution — refresh mode
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("THIRD RUN — REFRESH")
print("=" * 70)

run_probability_scores(
    con=test_con,
    season_id=TEST_SEASON,
    refresh_existing=True,
)

third_count = test_con.execute(
    f"SELECT COUNT(*) FROM {TARGET_TABLE}"
).fetchone()[0]

print(f"Rows after refresh: {third_count:,}")

# ------------------------------------------------------------
# 5. Database integrity checks
# ------------------------------------------------------------

duplicate_count = test_con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT
            season_id,
            game_date,
            team_id,
            model_version
        FROM {TARGET_TABLE}
        GROUP BY
            season_id,
            game_date,
            team_id,
            model_version
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

null_probability_count = test_con.execute(f"""
    SELECT COUNT(*)
    FROM {TARGET_TABLE}
    WHERE playoff_prob IS NULL
       OR playoff_prob_xgb IS NULL
       OR playoff_prob_rf IS NULL
""").fetchone()[0]

checks_38 = {
    "First run wrote 5,299 predictions": first_count == 5299,
    "Incremental rerun preserved row count": second_count == first_count,
    "Refresh preserved row count": third_count == first_count,
    "No duplicate prediction keys": duplicate_count == 0,
    "No null model probabilities": null_probability_count == 0,
}

print("\n" + "=" * 70)
print("BLOCK 38 — RESULTS")
print("=" * 70)

for name, passed in checks_38.items():
    print(f"{'✅ PASS' if passed else '❌ FAIL'} — {name}")

print(
    f"\nPASS: {sum(checks_38.values())} | "
    f"FAIL: {len(checks_38) - sum(checks_38.values())}"
)

In [ ]:
# ============================================================
# 39. Compare production predictions against validated output
# ============================================================

from config import TARGET_TABLE

production_probs = test_con.execute(f"""
    SELECT
        season_id,
        team_id,
        game_date,
        playoff_prob_xgb,
        playoff_prob_rf,
        playoff_prob
    FROM {TARGET_TABLE}
    WHERE season_id = 20252026
""").df()

validated_probs = df_daily_scored[
    [
        "season_id",
        "team_id",
        "game_date",
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob",
    ]
].copy()

for df in (production_probs, validated_probs):
    df["game_date"] = pd.to_datetime(df["game_date"]).dt.date

comparison = production_probs.merge(
    validated_probs,
    on=["season_id", "team_id", "game_date"],
    how="outer",
    suffixes=("_production", "_validated"),
    indicator=True,
    validate="one_to_one",
)

checks_39 = {
    "All validated team-days exist in production":
        comparison["_merge"].eq("both").all(),

    "Production row count matches validation":
        len(production_probs) == len(validated_probs),
}

for col in [
    "playoff_prob_xgb",
    "playoff_prob_rf",
    "playoff_prob",
]:
    checks_39[f"{col} matches validated output"] = (
        comparison["_merge"].eq("both").all()
        and np.allclose(
            comparison[f"{col}_production"],
            comparison[f"{col}_validated"],
            atol=1e-12,
            rtol=0,
            equal_nan=False,
        )
    )

print("=" * 70)
print("BLOCK 39 — PRODUCTION VS VALIDATED PREDICTIONS")
print("=" * 70)

for name, passed in checks_39.items():
    print(f"{'✅ PASS' if passed else '❌ FAIL'} — {name}")

print(
    f"\nPASS: {sum(checks_39.values())} | "
    f"FAIL: {len(checks_39) - sum(checks_39.values())}"
)

In [ ]:

# ============================================================
# 40. Validate Production Metadata and Presentation Fields
# ============================================================

from config import TARGET_TABLE
from prediction_model.metadata import (
    MODEL_METADATA,
    PROBABILITY_TIERS,
)

print("=" * 70)
print("BLOCK 40 — PRODUCTION METADATA VALIDATION")
print("=" * 70)


# ============================================================
# 1. Load production predictions from isolated database
# ============================================================

df_prod = test_con.execute(
    f"""
    SELECT *
    FROM {TARGET_TABLE}
    WHERE season_id = 20252026
    """
).df()

print(f"\nProduction rows loaded: {len(df_prod):,}")


# ============================================================
# 2. Prepare validated reference data
#
# Normalize game_date in both DataFrames to avoid
# datetime64 vs object merge errors.
# ============================================================

gp_reference = df_daily_scored[
    [
        "season_id",
        "team_id",
        "game_date",
        "games_played",
        "tri_code",
        "conference",
        "division",
        "conference_rank",
        "division_rank",
        "wildcard_rank",
        "points",
        "projected_points",
    ]
].copy()

df_prod["game_date"] = pd.to_datetime(
    df_prod["game_date"]
).dt.normalize()

gp_reference["game_date"] = pd.to_datetime(
    gp_reference["game_date"]
).dt.normalize()

key_cols_40 = [
    "season_id",
    "team_id",
    "game_date",
]

df_check = df_prod.merge(
    gp_reference,
    on=key_cols_40,
    how="left",
    suffixes=("_production", "_reference"),
    validate="one_to_one",
    indicator=True,
)

matched_rows = int(
    df_check["_merge"].eq("both").sum()
)

unmatched_rows = int(
    df_check["_merge"].ne("both").sum()
)

print(f"Matched team-days:   {matched_rows:,}")
print(f"Unmatched team-days: {unmatched_rows:,}")


# ============================================================
# 3. Validate season bucket
#
# Production rules:
#   <= 20 games: early
#   <= 50 games: mid
#   > 50 games: late
# ============================================================

valid_buckets = (
    df_check["season_bucket"]
    .isin(["early", "mid", "late"])
    .all()
)

expected_bucket = np.select(
    [
        df_check["games_played"].le(20),
        df_check["games_played"].le(50),
    ],
    [
        "early",
        "mid",
    ],
    default="late",
)

bucket_matches = (
    df_check["season_bucket"].to_numpy()
    == expected_bucket
).all()


# ============================================================
# 4. Validate display probabilities
#
# Early season:
#   display = 0.45 * raw_probability + 0.275
#
# Mid/late:
#   display = raw_probability
# ============================================================

expected_display = np.where(
    df_check["season_bucket"].eq("early"),
    0.45 * df_check["playoff_prob"] + 0.275,
    df_check["playoff_prob"],
)

display_matches = np.allclose(
    df_check["playoff_prob_display"],
    expected_display,
    atol=1e-12,
    rtol=0,
    equal_nan=False,
)


# ============================================================
# 5. Validate probability tiers
# ============================================================

def expected_tier_40(p):
    for tier, (low, high) in PROBABILITY_TIERS.items():
        if low <= p < high:
            return tier
    return "Unknown"


expected_tiers = (
    df_check["playoff_prob_display"]
    .apply(expected_tier_40)
)

tier_matches = (
    df_check["playoff_tier"]
    .eq(expected_tiers)
    .all()
)


# ============================================================
# 6. Validate model metadata
# ============================================================

model_name_matches = (
    df_check["model_name"]
    .eq(MODEL_METADATA["model_name"])
    .all()
)

model_version_matches = (
    df_check["model_version"]
    .eq(MODEL_METADATA["version"])
    .all()
)

valid_timestamps = (
    pd.to_datetime(
        df_check["model_run_ts"],
        errors="coerce",
    )
    .notna()
    .all()
)


# ============================================================
# 7. Validate probability bounds
# ============================================================

probability_cols_40 = [
    "playoff_prob",
    "playoff_prob_xgb",
    "playoff_prob_rf",
    "playoff_prob_display",
]

valid_probability_bounds = all(
    df_check[col].between(0, 1).all()
    for col in probability_cols_40
)


# ============================================================
# 8. Validate standings and ranking fields
#
# Confirm values stored by the production runner match
# the corresponding validated source snapshots.
# ============================================================

standings_cols_40 = [
    "tri_code",
    "conference",
    "division",
    "conference_rank",
    "division_rank",
    "wildcard_rank",
    "points",
    "projected_points",
]

standings_results_40 = {}

for col in standings_cols_40:

    production_col = f"{col}_production"
    reference_col = f"{col}_reference"

    if (
        production_col not in df_check.columns
        or reference_col not in df_check.columns
    ):
        standings_results_40[col] = False
        continue

    left = df_check[production_col]
    right = df_check[reference_col]

    if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
        matches = np.allclose(
            pd.to_numeric(left, errors="coerce"),
            pd.to_numeric(right, errors="coerce"),
            atol=1e-12,
            rtol=0,
            equal_nan=True,
        )
    else:
        matches = left.fillna("<NULL>").astype(str).equals(
            right.fillna("<NULL>").astype(str)
        )

    standings_results_40[col] = bool(matches)


# ============================================================
# 9. Validation checks
# ============================================================

checks_40 = {

    "All production rows match validated team-days":
        unmatched_rows == 0
        and len(df_prod) == len(gp_reference),

    "All season buckets are valid":
        bool(valid_buckets),

    "Season buckets match games played":
        bool(bucket_matches),

    "Display probability adjustment is correct":
        bool(display_matches),

    "Probability tiers match display values":
        bool(tier_matches),

    "Model name matches metadata":
        bool(model_name_matches),

    "Model version matches metadata":
        bool(model_version_matches),

    "All run timestamps are populated":
        bool(valid_timestamps),

    "All probabilities are between 0 and 1":
        bool(valid_probability_bounds),

    "Standings fields match validated snapshots":
        all(standings_results_40.values()),
}


# ============================================================
# 10. Results
# ============================================================

print("\n" + "=" * 70)
print("🚦 BLOCK 40 — VALIDATION CHECKS")
print("=" * 70)

pass_count_40 = 0
fail_count_40 = 0

for check_name, passed in checks_40.items():

    if passed:
        print(f"✅ PASS — {check_name}")
        pass_count_40 += 1

    else:
        print(f"❌ FAIL — {check_name}")
        fail_count_40 += 1


# ============================================================
# 11. Diagnostics
# ============================================================

if unmatched_rows:
    print("\nUnmatched production team-days:")
    display(
        df_check.loc[
            df_check["_merge"].ne("both"),
            key_cols_40 + ["_merge"],
        ].head(20)
    )

failed_standings_cols = [
    col
    for col, passed in standings_results_40.items()
    if not passed
]

if failed_standings_cols:
    print("\nStandings columns with mismatches:")
    print(failed_standings_cols)

    for col in failed_standings_cols:
        production_col = f"{col}_production"
        reference_col = f"{col}_reference"

        if (
            production_col in df_check.columns
            and reference_col in df_check.columns
        ):
            display(
                df_check[
                    key_cols_40 + [
                        production_col,
                        reference_col,
                    ]
                ].head(10)
            )


# ============================================================
# 12. Final result
# ============================================================

print("\n" + "=" * 70)
print("🏁 BLOCK 40 RESULT")
print("=" * 70)

print(f"PASS: {pass_count_40}")
print(f"FAIL: {fail_count_40}")

if fail_count_40 == 0:
    print(
        "\n✅ PRODUCTION METADATA AND PRESENTATION "
        "FIELDS PASS VALIDATION."
    )
else:
    print(
        "\n⚠️ PRODUCTION METADATA VALIDATION HAS "
        "ITEMS TO INVESTIGATE."
    )


In [ ]:

# ============================================================
# 41. Validate Incomplete-Date Recovery and Transaction Rollback
# ============================================================

import importlib
import prediction_model.inference.daily_playoff_probability as runner

from config import TARGET_TABLE
from prediction_model.metadata import MODEL_METADATA

runner = importlib.reload(runner)

TEST_SEASON = 20252026
MODEL_VERSION = MODEL_METADATA["version"]

print("=" * 70)
print("BLOCK 41 — OPERATIONAL RECOVERY AND ROLLBACK")
print("=" * 70)


# ============================================================
# 1. Preserve original isolated database state
# ============================================================

original = test_con.execute(
    f"""
    SELECT *
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
    ORDER BY game_date, team_id
    """,
    [TEST_SEASON, MODEL_VERSION]
).df()

original_count = len(original)

assert original_count > 0, "No test predictions found"

print(f"\nOriginal prediction rows: {original_count:,}")


# ============================================================
# 2. Test incomplete-date recovery
#
# Delete one team from a date that has 32 predictions.
# Incremental mode should detect and restore it.
# ============================================================

complete_date = test_con.execute(
    f"""
    SELECT game_date
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
    GROUP BY game_date
    HAVING COUNT(DISTINCT team_id) = 32
    ORDER BY game_date DESC
    LIMIT 1
    """,
    [TEST_SEASON, MODEL_VERSION]
).fetchone()

assert complete_date is not None, "No complete date found"

test_date = complete_date[0]

missing_team = test_con.execute(
    f"""
    SELECT team_id
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
      AND game_date = ?
    ORDER BY team_id
    LIMIT 1
    """,
    [TEST_SEASON, MODEL_VERSION, test_date]
).fetchone()[0]

print("\n" + "=" * 70)
print("TEST 1 — INCOMPLETE-DATE RECOVERY")
print("=" * 70)

print(f"Test date: {test_date}")
print(f"Team removed: {missing_team}")

test_con.execute(
    f"""
    DELETE FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
      AND game_date = ?
      AND team_id = ?
    """,
    [TEST_SEASON, MODEL_VERSION, test_date, missing_team]
)

count_after_delete = test_con.execute(
    f"""
    SELECT COUNT(*)
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
    """,
    [TEST_SEASON, MODEL_VERSION]
).fetchone()[0]

print(f"Rows after simulated data loss: {count_after_delete:,}")

runner.run_probability_scores(
    con=test_con,
    season_id=TEST_SEASON,
    refresh_existing=False,
)

count_after_recovery = test_con.execute(
    f"""
    SELECT COUNT(*)
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
    """,
    [TEST_SEASON, MODEL_VERSION]
).fetchone()[0]

restored_team = test_con.execute(
    f"""
    SELECT COUNT(*)
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
      AND game_date = ?
      AND team_id = ?
    """,
    [TEST_SEASON, MODEL_VERSION, test_date, missing_team]
).fetchone()[0]

recovery_passed = (
    count_after_delete == original_count - 1
    and count_after_recovery == original_count
    and restored_team == 1
)

print(f"Rows after recovery: {count_after_recovery:,}")
print(f"Missing team restored: {restored_team == 1}")


# ============================================================
# 3. Test transactional rollback
#
# Temporarily add a CHECK constraint that makes every
# inserted probability invalid, forcing the INSERT to fail.
#
# The transaction must restore the rows deleted during
# the attempted refresh.
# ============================================================

print("\n" + "=" * 70)
print("TEST 2 — TRANSACTION ROLLBACK")
print("=" * 70)

before_failure = test_con.execute(
    f"""
    SELECT
        season_id,
        game_date,
        team_id,
        model_version,
        playoff_prob
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
    ORDER BY game_date, team_id
    """,
    [TEST_SEASON, MODEL_VERSION]
).df()

# Create an isolated replacement table with a deliberately
# impossible probability constraint.

ROLLBACK_TABLE = "_validation_rollback_target"

test_con.execute(
    f"DROP TABLE IF EXISTS {ROLLBACK_TABLE}"
)

test_con.execute(
    f"""
    CREATE TABLE {ROLLBACK_TABLE} AS
    SELECT *
    FROM {TARGET_TABLE}
    WHERE 1 = 0
    """
)

# Populate before adding the failing constraint.
test_con.execute(
    f"""
    INSERT INTO {ROLLBACK_TABLE}
    SELECT *
    FROM {TARGET_TABLE}
    """
)

# Save the original target table name for the runner.
original_target = runner.TARGET_TABLE

try:
    # Redirect the runner to our isolated rollback table.
    runner.TARGET_TABLE = ROLLBACK_TABLE

    # Force INSERT to fail by replacing the target table
    # with a view that rejects inserts.
    test_con.execute(
        f"""
        CREATE VIEW _validation_rollback_view AS
        SELECT *
        FROM {ROLLBACK_TABLE}
        """
    )

    runner.TARGET_TABLE = "_validation_rollback_view"

    failure_caught = False

    try:
        runner.run_probability_scores(
            con=test_con,
            season_id=TEST_SEASON,
            refresh_existing=True,
        )

    except Exception as exc:
        failure_caught = True
        print(f"Expected write failure: {type(exc).__name__}")

finally:
    runner.TARGET_TABLE = original_target

# Confirm the original target was never affected.
after_failure = test_con.execute(
    f"""
    SELECT
        season_id,
        game_date,
        team_id,
        model_version,
        playoff_prob
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
    ORDER BY game_date, team_id
    """,
    [TEST_SEASON, MODEL_VERSION]
).df()

rollback_passed = (
    failure_caught
    and before_failure.equals(after_failure)
)

# Clean up rollback test objects
test_con.execute(
    "DROP VIEW IF EXISTS _validation_rollback_view"
)
test_con.execute(
    f"DROP TABLE IF EXISTS {ROLLBACK_TABLE}"
)


# ============================================================
# 4. Final integrity checks
# ============================================================

final_count = test_con.execute(
    f"""
    SELECT COUNT(*)
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
    """,
    [TEST_SEASON, MODEL_VERSION]
).fetchone()[0]

duplicate_count = test_con.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT
            season_id,
            game_date,
            team_id,
            model_version
        FROM {TARGET_TABLE}
        WHERE season_id = ?
          AND model_version = ?
        GROUP BY
            season_id,
            game_date,
            team_id,
            model_version
        HAVING COUNT(*) > 1
    )
    """,
    [TEST_SEASON, MODEL_VERSION]
).fetchone()[0]


# ============================================================
# 5. Results
# ============================================================

checks_41 = {
    "Incomplete date detected and recovered": recovery_passed,
    "Insert failure triggered": failure_caught,
    "Transaction rollback preserved predictions": rollback_passed,
    "Final row count unchanged": final_count == original_count,
    "No duplicate prediction keys": duplicate_count == 0,
}

print("\n" + "=" * 70)
print("BLOCK 41 — RESULTS")
print("=" * 70)

for name, passed in checks_41.items():
    print(f"{'✅ PASS' if passed else '❌ FAIL'} — {name}")

print(
    f"\nPASS: {sum(checks_41.values())} | "
    f"FAIL: {len(checks_41) - sum(checks_41.values())}"
)


In [ ]:

# ============================================================
# 42. Validate Transaction Rollback After Successful DELETE
# ============================================================

from config import TARGET_TABLE
from prediction_model.metadata import MODEL_METADATA

TEST_SEASON = 20252026
MODEL_VERSION = MODEL_METADATA["version"]

print("=" * 70)
print("BLOCK 42 — TRANSACTION ROLLBACK VALIDATION")
print("=" * 70)


# ============================================================
# 1. Select a complete date for testing
# ============================================================

test_date = test_con.execute(
    f"""
    SELECT game_date
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
    GROUP BY game_date
    HAVING COUNT(DISTINCT team_id) = 32
    ORDER BY game_date DESC
    LIMIT 1
    """,
    [TEST_SEASON, MODEL_VERSION]
).fetchone()

assert test_date is not None, "No complete date available"

test_date = test_date[0]

print(f"\nTest date: {test_date}")


# ============================================================
# 2. Save original predictions
# ============================================================

original_rows = test_con.execute(
    f"""
    SELECT *
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
      AND game_date = ?
    ORDER BY team_id
    """,
    [TEST_SEASON, MODEL_VERSION, test_date]
).df()

original_count = len(original_rows)

total_before = test_con.execute(
    f"""
    SELECT COUNT(*)
    FROM {TARGET_TABLE}
    """
).fetchone()[0]

print(f"Original rows for date: {original_count}")
print(f"Total database rows: {total_before:,}")


# ============================================================
# 3. Begin transaction and delete existing predictions
# ============================================================

delete_succeeded = False
insert_failed = False
rollback_succeeded = False

try:
    test_con.execute("BEGIN TRANSACTION")

    test_con.execute(
        f"""
        DELETE FROM {TARGET_TABLE}
        WHERE season_id = ?
          AND model_version = ?
          AND game_date = ?
        """,
        [TEST_SEASON, MODEL_VERSION, test_date]
    )

    remaining_after_delete = test_con.execute(
        f"""
        SELECT COUNT(*)
        FROM {TARGET_TABLE}
        WHERE season_id = ?
          AND model_version = ?
          AND game_date = ?
        """,
        [TEST_SEASON, MODEL_VERSION, test_date]
    ).fetchone()[0]

    delete_succeeded = remaining_after_delete == 0

    print("\n" + "=" * 70)
    print("TRANSACTION TEST")
    print("=" * 70)

    print(f"DELETE succeeded: {delete_succeeded}")
    print(f"Rows remaining after DELETE: {remaining_after_delete}")

    # --------------------------------------------------------
    # 4. Force INSERT failure
    #
    # Insert a row with a NULL primary-key value.
    # This must violate the table's PRIMARY KEY constraint.
    # --------------------------------------------------------

    try:
        test_con.execute(
            f"""
            INSERT INTO {TARGET_TABLE} (
                season_id,
                game_date,
                team_id,
                model_version
            )
            VALUES (?, ?, NULL, ?)
            """,
            [TEST_SEASON, test_date, MODEL_VERSION]
        )

    except Exception as exc:
        insert_failed = True
        print(f"Expected INSERT failure: {type(exc).__name__}")
        print(str(exc)[:300])

    # --------------------------------------------------------
    # 5. Roll back the transaction
    # --------------------------------------------------------

    test_con.execute("ROLLBACK")
    rollback_succeeded = True

except Exception:
    # Ensure a failed test cannot leave a transaction open.
    try:
        test_con.execute("ROLLBACK")
        rollback_succeeded = True
    except Exception:
        pass

    raise


# ============================================================
# 6. Verify original predictions were restored
# ============================================================

restored_rows = test_con.execute(
    f"""
    SELECT *
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
      AND game_date = ?
    ORDER BY team_id
    """,
    [TEST_SEASON, MODEL_VERSION, test_date]
).df()

total_after = test_con.execute(
    f"""
    SELECT COUNT(*)
    FROM {TARGET_TABLE}
    """
).fetchone()[0]

# Compare original and restored rows, including probabilities
# and metadata. They should be identical.

rows_restored_exactly = original_rows.equals(restored_rows)

print("\n" + "=" * 70)
print("POST-ROLLBACK VERIFICATION")
print("=" * 70)

print(f"Original date rows: {original_count}")
print(f"Restored date rows: {len(restored_rows)}")
print(f"Original total rows: {total_before:,}")
print(f"Final total rows: {total_after:,}")
print(f"All original values restored: {rows_restored_exactly}")


# ============================================================
# 7. Final validation checks
# ============================================================

checks_42 = {

    "DELETE succeeded inside transaction":
        delete_succeeded,

    "INSERT failed as intended":
        insert_failed,

    "ROLLBACK executed successfully":
        rollback_succeeded,

    "All deleted rows were restored":
        len(restored_rows) == original_count,

    "All original prediction values were preserved":
        rows_restored_exactly,

    "Total database row count unchanged":
        total_after == total_before,
}


# ============================================================
# 8. Results
# ============================================================

print("\n" + "=" * 70)
print("BLOCK 42 — RESULTS")
print("=" * 70)

for name, passed in checks_42.items():
    print(f"{'✅ PASS' if passed else '❌ FAIL'} — {name}")

print(
    f"\nPASS: {sum(checks_42.values())} | "
    f"FAIL: {len(checks_42) - sum(checks_42.values())}"
)


In [ ]:

# ============================================================
# 43. Production Readiness Preflight — 2026-27
# ============================================================

from config import SOURCE_TABLE, TARGET_TABLE
from prediction_model.metadata import MODEL_METADATA
from feature_engineering import build_features
from prediction_model.inference import score_dataframe

SEASON_ID = 20262027

print("=" * 70)
print("BLOCK 43 — PRODUCTION READINESS PREFLIGHT")
print("=" * 70)

checks_43 = {}


# ============================================================
# 1. Verify database connection and source table
# ============================================================

try:
    con.execute("SELECT 1").fetchone()

    source_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM {SOURCE_TABLE}
        WHERE season_id = ?
        """,
        [SEASON_ID]
    ).fetchone()[0]

    checks_43["Production database accessible"] = True
    checks_43["Current season source data exists"] = source_count > 0

    print(f"\nCurrent season source rows: {source_count:,}")

except Exception as exc:
    checks_43["Production database accessible"] = False
    checks_43["Current season source data exists"] = False
    print(f"Database/source error: {exc}")


# ============================================================
# 2. Inspect current-season source data
# ============================================================

if checks_43["Current season source data exists"]:

    df_source_43 = con.execute(
        f"""
        SELECT *
        FROM {SOURCE_TABLE}
        WHERE season_id = ?
        ORDER BY game_date, team_id
        """,
        [SEASON_ID]
    ).df()

    required_keys = [
        "season_id",
        "team_id",
        "game_date",
    ]

    checks_43["Required source keys exist"] = all(
        col in df_source_43.columns
        for col in required_keys
    )

    checks_43["No null source keys"] = (
        df_source_43[required_keys].notna().all().all()
    )

    checks_43["Unique source team-day keys"] = (
        not df_source_43.duplicated(required_keys).any()
    )

    print(
        f"Source date range: "
        f"{df_source_43['game_date'].min()} → "
        f"{df_source_43['game_date'].max()}"
    )

    print(
        f"Unique teams: "
        f"{df_source_43['team_id'].nunique()}"
    )

else:
    print("\nNo current-season source data available.")
    print("Source-specific checks will be skipped.")


# ============================================================
# 3. Verify model artifact
# ============================================================

checks_43["Model pipelines available"] = all(
    key in bundle
    for key in [
        "xgb_pipeline",
        "rf_pipeline",
        "features",
    ]
)

checks_43["Model feature list populated"] = (
    len(FEATURE_COLS) > 0
)

checks_43["Model metadata available"] = all(
    key in MODEL_METADATA
    for key in ["model_name", "version"]
)

print(f"\nModel: {MODEL_METADATA['model_name']}")
print(f"Version: {MODEL_METADATA['version']}")
print(f"Required model features: {len(FEATURE_COLS)}")


# ============================================================
# 4. Check feature availability
#
# This inspects source columns only.
# The production runner performs ASOF expansion and
# feature engineering before actual scoring.
# ============================================================

if checks_43.get("Current season source data exists", False):

    source_columns = set(df_source_43.columns)

    directly_available = [
        col for col in FEATURE_COLS
        if col in source_columns
    ]

    engineered_features = [
        col for col in FEATURE_COLS
        if col not in source_columns
    ]

    print(
        f"\nFeatures directly in source: "
        f"{len(directly_available)}"
    )

    print(
        f"Features requiring engineering: "
        f"{len(engineered_features)}"
    )

    # Missing source columns are not automatically failures:
    # build_features creates derived model inputs.
    checks_43["Feature inspection completed"] = True


# ============================================================
# 5. Inspect target table (read-only)
# ============================================================

target_exists = con.execute(
    """
    SELECT COUNT(*)
    FROM information_schema.tables
    WHERE table_name = ?
    """,
    [TARGET_TABLE.split(".")[-1]]
).fetchone()[0] > 0

checks_43["Target table status verified"] = True

if target_exists:

    target_columns = con.execute(
        f"""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_name = ?
        """,
        [TARGET_TABLE.split(".")[-1]]
    ).df()["column_name"].tolist()

    expected_target_columns = [
        "season_id",
        "game_date",
        "team_id",
        "model_version",
        "playoff_prob",
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob_display",
        "playoff_tier",
        "model_run_ts",
    ]

    missing_target_columns = [
        col for col in expected_target_columns
        if col not in target_columns
    ]

    checks_43["Target schema compatible"] = (
        len(missing_target_columns) == 0
    )

    existing_predictions = con.execute(
        f"""
        SELECT COUNT(*)
        FROM {TARGET_TABLE}
        WHERE season_id = ?
        """,
        [SEASON_ID]
    ).fetchone()[0]

    print(f"\nTarget table: {TARGET_TABLE}")
    print(f"Existing 2026-27 predictions: {existing_predictions:,}")
    print(f"Missing target columns: {missing_target_columns}")

else:
    # Runner creates target if it does not exist.
    checks_43["Target schema compatible"] = True
    print("\nTarget table does not exist yet.")
    print("Production runner will create it.")


# ============================================================
# 6. Final readiness report
# ============================================================

print("\n" + "=" * 70)
print("BLOCK 43 — RESULTS")
print("=" * 70)

for name, passed in checks_43.items():
    print(f"{'✅ PASS' if passed else '❌ FAIL'} — {name}")

pass_count_43 = sum(checks_43.values())
fail_count_43 = len(checks_43) - pass_count_43

print(f"\nPASS: {pass_count_43}")
print(f"FAIL: {fail_count_43}")

if fail_count_43 == 0:
    print("\n✅ PRODUCTION PREFLIGHT PASSED")
else:
    print("\n⚠️ REVIEW FAILED CHECKS BEFORE PRODUCTION SCORING")


In [ ]:

# ============================================================
# 45. Validate Game Summary Corrections and Transaction Safety
# ============================================================

import importlib
import duckdb
import pandas as pd
import scripts.ingestion.game_summary as gs

gs = importlib.reload(gs)

print("=" * 70)
print("BLOCK 45 — GAME SUMMARY PRODUCTION VALIDATION")
print("=" * 70)

checks_45 = {}

# ------------------------------------------------------------
# 1. Create isolated test database
# ------------------------------------------------------------

gs_test_con = duckdb.connect(":memory:")

gs_test_con.execute("""
    CREATE TABLE all_game_summary (
        game_id BIGINT PRIMARY KEY,
        season_id BIGINT,
        away_team BIGINT,
        away_tri_code VARCHAR,
        home_team BIGINT,
        home_tri_code VARCHAR,
        "awayTeam.score" BIGINT,
        "homeTeam.score" BIGINT,
        gameState VARCHAR,
        link VARCHAR,
        game_date DATE,
        api_snapshot_date DATE
    )
""")

def make_game_45(
    game_id,
    state="OFF",
    away_score=2,
    home_score=3,
):
    return {
        "game_id": game_id,
        "season_id": 20262027,
        "away_team": 6,
        "away_tri_code": "BOS",
        "home_team": 8,
        "home_tri_code": "MTL",
        "awayTeam.score": away_score,
        "homeTeam.score": home_score,
        "gameState": state,
        "link": (
            f"https://api-web.nhle.com/v1/gamecenter/"
            f"{game_id}/play-by-play"
        ),
        "game_date": pd.Timestamp("2026-10-08").date(),
        "api_snapshot_date": pd.Timestamp("2026-10-08").date(),
    }

# Existing database state:
# 101: completed
# 102: scheduled
# 103: completed

existing = pd.DataFrame([
    make_game_45(2026020101, "OFF", 2, 3),
    make_game_45(2026020102, "FUT", None, None),
    make_game_45(2026020103, "OFF", 1, 4),
])

gs.upsert_games(
    con=gs_test_con,
    api_games=existing,
)

# ------------------------------------------------------------
# 2. Simulate updated API responses
# ------------------------------------------------------------

api_games = pd.DataFrame([
    make_game_45(2026020101, "OFF", 2, 4),   # corrected
    make_game_45(2026020102, "OFF", 3, 2),   # newly OFF
    make_game_45(2026020103, "OFF", 1, 4),   # unchanged
    make_game_45(2026020104, "FUT", None, None),
    make_game_45(2026020105, "OFF", 5, 1),   # new completed
])

changed = gs.determine_changed_games(
    api_games=api_games,
    existing_games=existing,
)

changed_ids = set(changed["game_id"].astype(int))

checks_45["Corrected OFF game detected"] = (
    2026020101 in changed_ids
)

checks_45["FUT to OFF transition detected"] = (
    2026020102 in changed_ids
)

checks_45["Unchanged OFF game excluded"] = (
    2026020103 not in changed_ids
)

checks_45["Scheduled game excluded"] = (
    2026020104 not in changed_ids
)

checks_45["New completed game detected"] = (
    2026020105 in changed_ids
)

print("\nChanged game IDs:", sorted(changed_ids))

# ------------------------------------------------------------
# 3. Test successful upsert
# ------------------------------------------------------------

gs.upsert_games(
    con=gs_test_con,
    api_games=api_games,
)

stored = gs_test_con.execute("""
    SELECT *
    FROM all_game_summary
    ORDER BY game_id
""").df()

corrected_score = gs_test_con.execute("""
    SELECT "homeTeam.score"
    FROM all_game_summary
    WHERE game_id = 2026020101
""").fetchone()[0]

checks_45["Corrected score stored"] = corrected_score == 4

checks_45["All five games stored"] = len(stored) == 5

checks_45["No duplicate game IDs"] = (
    not stored["game_id"].duplicated().any()
)

# ------------------------------------------------------------
# 4. Test rollback after DELETE
#
# Force an INSERT constraint failure by providing a NULL
# game_id, after first deleting an existing valid game.
# ------------------------------------------------------------

before_rollback = gs_test_con.execute("""
    SELECT *
    FROM all_game_summary
    ORDER BY game_id
""").df()

bad_updates = pd.DataFrame([
    make_game_45(2026020101, "OFF", 9, 9),
    make_game_45(None, "OFF", 1, 1),
])

rollback_error = False

try:
    gs.upsert_games(
        con=gs_test_con,
        api_games=bad_updates,
    )
except Exception as exc:
    rollback_error = True
    print(
        "\nExpected write failure:",
        type(exc).__name__,
        str(exc)[:200],
    )

after_rollback = gs_test_con.execute("""
    SELECT *
    FROM all_game_summary
    ORDER BY game_id
""").df()

checks_45["Invalid INSERT rejected"] = rollback_error

checks_45["Original data preserved after rollback"] = (
    before_rollback.equals(after_rollback)
)

# ------------------------------------------------------------
# 5. Final results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BLOCK 45 — RESULTS")
print("=" * 70)

for name, passed in checks_45.items():
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — {name}"
    )

print(
    f"\nPASS: {sum(checks_45.values())} | "
    f"FAIL: {len(checks_45) - sum(checks_45.values())}"
)

gs_test_con.close()


In [ ]:

# ============================================================
# BLOCK 46 — PLAY-BY-PLAY PRODUCTION VALIDATION
# ============================================================

import ast
import inspect
import importlib
from datetime import date, timedelta
from unittest.mock import patch

import duckdb
import pandas as pd

import scripts.ingestion.play_by_play as pbp

pbp = importlib.reload(pbp)

SEASON_ID = 20262027
TODAY = date.today()

GAME_NEW = 2026020101
GAME_CORRECTED = 2026020102
GAME_PARTIAL = 2026020103
GAME_OLD = 2026020104

checks = []


def check(label, condition):
    passed = bool(condition)
    checks.append((label, passed))
    print(f"{'✅ PASS' if passed else '❌ FAIL'} — {label}")


# ------------------------------------------------------------
# 1. Read the actual production schema
# ------------------------------------------------------------

source = inspect.getsource(pbp.run_play_by_play)
tree = ast.parse(source)
runner = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "run_play_by_play"
)

schema_node = next(
    node.value for node in ast.walk(runner)
    if isinstance(node, ast.Assign)
    and any(
        isinstance(target, ast.Name)
        and target.id == "PBP_SCHEMA"
        for target in node.targets
    )
)

PBP_SCHEMA = ast.literal_eval(schema_node)

type_map = {
    "Int64": "BIGINT",
    "string": "VARCHAR",
    "float64": "DOUBLE",
}

# ------------------------------------------------------------
# 2. Create isolated database
# ------------------------------------------------------------

con = duckdb.connect(":memory:")

con.execute("""
    CREATE TABLE all_game_summary (
        season_id BIGINT,
        game_id BIGINT,
        gameState VARCHAR,
        game_date DATE,
        link VARCHAR,
        home_team BIGINT,
        away_team BIGINT
    )
""")

columns_sql = []

for col, dtype in PBP_SCHEMA.items():
    definition = f'"{col}" {type_map[dtype]}'

    # Used to deliberately trigger an INSERT failure.
    if col == "eventId":
        definition += " CHECK (eventId <> 999)"

    columns_sql.append(definition)

con.execute(
    "CREATE TABLE all_play_by_play ("
    + ", ".join(columns_sql)
    + ")"
)

games = [
    (GAME_NEW, TODAY - timedelta(days=1)),
    (GAME_CORRECTED, TODAY - timedelta(days=2)),
    (GAME_PARTIAL, TODAY - timedelta(days=3)),
    (GAME_OLD, TODAY - timedelta(days=20)),
]

for game_id, game_date in games:
    con.execute(
        """
        INSERT INTO all_game_summary
        VALUES (?, ?, 'OFF', ?, ?, 6, 10)
        """,
        [
            SEASON_ID,
            game_id,
            game_date,
            f"https://example.test/pbp/{game_id}",
        ],
    )


# ------------------------------------------------------------
# 3. Seed existing PBP records
# ------------------------------------------------------------

def insert_existing(game_id, event_id, event_type):
    row = {col: None for col in PBP_SCHEMA}

    row.update({
        "season_id": SEASON_ID,
        "game_id": game_id,
        "eventId": event_id,
        "typeDescKey": event_type,
        "timeInPeriod": "01:00",
    })

    con.register("seed_row", pd.DataFrame([row]))

    try:
        con.execute("""
            INSERT INTO all_play_by_play
            SELECT * FROM seed_row
        """)
    finally:
        con.unregister("seed_row")


# Corrected game has outdated event data.
insert_existing(GAME_CORRECTED, 1, "old_event")

# Partial game has only one of three events.
insert_existing(GAME_PARTIAL, 1, "faceoff")

# Older completed game should not be refreshed.
insert_existing(GAME_OLD, 1, "old_stable_event")


# ------------------------------------------------------------
# 4. Mock NHL API responses
# ------------------------------------------------------------

def event(event_id, event_type, time_in_period="01:00"):
    return {
        "eventId": event_id,
        "typeCode": 502,
        "typeDescKey": event_type,
        "timeInPeriod": time_in_period,
        "timeRemaining": "19:00",
        "sortOrder": event_id,
        "periodDescriptor": {
            "number": 1,
            "periodType": "REG",
            "maxRegulationPeriods": 3,
        },
        "situationCode": "1551",
    }


api_plays = {
    GAME_NEW: [
        event(1, "faceoff"),
        event(2, "shot-on-goal"),
        event(2, "shot-on-goal"),  # Duplicate
    ],
    GAME_CORRECTED: [
        event(1, "faceoff"),       # Corrected
        event(2, "goal"),
    ],
    GAME_PARTIAL: [
        event(1, "faceoff"),
        event(2, "hit"),
        event(3, "shot-on-goal"),
    ],
    GAME_OLD: [
        event(1, "new_api_value"),
    ],
}

requested_games = []


class FakeResponse:
    status_code = 200

    def __init__(self, game_id):
        self.game_id = game_id

    def json(self):
        return {"plays": api_plays[self.game_id]}


def fake_get(url, **kwargs):
    game_id = int(url.rstrip("/").split("/")[-1])
    requested_games.append(game_id)
    return FakeResponse(game_id)


# ------------------------------------------------------------
# 5. Run actual production function
# ------------------------------------------------------------

print("\nBLOCK 46 — PLAY-BY-PLAY PRODUCTION VALIDATION\n")

with patch.object(pbp.requests, "get", side_effect=fake_get):
    pbp.run_play_by_play(
        con=con,
        season_id=SEASON_ID,
        changed_games=pd.DataFrame(),
    )


# ------------------------------------------------------------
# 6. Verify recovery and normalization
# ------------------------------------------------------------

result = con.execute("""
    SELECT game_id, eventId, typeDescKey,
           home_team, away_team, situation,
           goalie_situation, game_in_seconds
    FROM all_play_by_play
    ORDER BY game_id, eventId
""").df()


def rows_for(game_id):
    return result[result["game_id"] == game_id]


check(
    "New game inserted",
    len(rows_for(GAME_NEW)) == 2,
)

check(
    "Duplicate events removed",
    not result.duplicated(["game_id", "eventId"]).any(),
)

check(
    "Corrected game refreshed",
    rows_for(GAME_CORRECTED)["typeDescKey"].tolist()
    == ["faceoff", "goal"],
)

check(
    "Partially stored game recovered",
    rows_for(GAME_PARTIAL)["eventId"].tolist()
    == [1, 2, 3],
)

check(
    "Old unchanged game preserved",
    rows_for(GAME_OLD)["typeDescKey"].tolist()
    == ["old_stable_event"],
)

check(
    "Seven-day refresh selected correct games",
    set(requested_games)
    == {GAME_NEW, GAME_CORRECTED, GAME_PARTIAL},
)

check(
    "Home and away team IDs populated",
    (
        (result[result.game_id != GAME_OLD]["home_team"] == 6)
        & (result[result.game_id != GAME_OLD]["away_team"] == 10)
    ).all(),
)

check(
    "Situation normalized correctly",
    (
        result[result.game_id != GAME_OLD]["situation"]
        == "5 on 5"
    ).all(),
)

check(
    "Goalie situation normalized correctly",
    (
        result[result.game_id != GAME_OLD]["goalie_situation"]
        == "in net"
    ).all(),
)

check(
    "Game time calculated correctly",
    (
        result[result.game_id != GAME_OLD]["game_in_seconds"]
        == 60
    ).all(),
)


# ------------------------------------------------------------
# 7. Verify corrected-game processing outside lookback
# ------------------------------------------------------------

api_plays[GAME_OLD] = [
    event(1, "corrected_old_event")
]

requested_games.clear()

with patch.object(pbp.requests, "get", side_effect=fake_get):
    pbp.run_play_by_play(
        con=con,
        season_id=SEASON_ID,
        changed_games=pd.DataFrame({
            "game_id": [GAME_OLD],
        }),
    )

old_value = con.execute("""
    SELECT typeDescKey
    FROM all_play_by_play
    WHERE game_id = ?
""", [GAME_OLD]).fetchone()[0]

check(
    "Changed old game refreshed",
    old_value == "corrected_old_event",
)


# ------------------------------------------------------------
# 8. Verify transaction rollback
# ------------------------------------------------------------

before = con.execute("""
    SELECT game_id, eventId, typeDescKey
    FROM all_play_by_play
    ORDER BY game_id, eventId
""").df()

# Invalid event 999 violates the CHECK constraint.
# This should fail AFTER the DELETE but BEFORE COMMIT.
api_plays[GAME_CORRECTED] = [
    event(999, "invalid_event")
]

failed_as_expected = False

try:
    with patch.object(pbp.requests, "get", side_effect=fake_get):
        pbp.run_play_by_play(
            con=con,
            season_id=SEASON_ID,
            changed_games=pd.DataFrame({
                "game_id": [GAME_CORRECTED],
            }),
        )

except duckdb.Error as exc:
    failed_as_expected = True
    print(
        "\nExpected write failure:",
        type(exc).__name__,
        str(exc).split("\n")[0],
    )

after = con.execute("""
    SELECT game_id, eventId, typeDescKey
    FROM all_play_by_play
    ORDER BY game_id, eventId
""").df()

check(
    "Invalid INSERT rejected",
    failed_as_expected,
)

check(
    "Original data preserved after rollback",
    before.equals(after),
)

check(
    "No invalid event persisted",
    con.execute("""
        SELECT COUNT(*)
        FROM all_play_by_play
        WHERE eventId = 999
    """).fetchone()[0] == 0,
)


# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

passed = sum(ok for _, ok in checks)
failed = len(checks) - passed

print("\n" + "=" * 55)
print(f"BLOCK 46 RESULTS — PASS: {passed} | FAIL: {failed}")
print("=" * 55)

if failed == 0:
    print("✅ PLAY-BY-PLAY PRODUCTION VALIDATION PASSED")
else:
    print("❌ REVIEW FAILED CHECKS BEFORE PROCEEDING")

con.close()


In [ ]:

# ============================================================
# BLOCK 47 — GOALIE STATS PRODUCTION VALIDATION
# ============================================================

import importlib
from datetime import date, timedelta
from unittest.mock import patch

import duckdb
import pandas as pd

import scripts.ingestion.goalie_stats as gs

gs = importlib.reload(gs)

SEASON_ID = 20252026

GAME_NEW = 2025020101
GAME_CORRECTED = 2025020102
GAME_PARTIAL = 2025020103
GAME_OLD = 2025020104
GAME_MISSING_OLD = 2025020105
GAME_FUTURE = 2025020106
GAME_PLAYOFF = 2025030107

TODAY = date.today()

checks = []


def check(label, condition):
    passed = bool(condition)
    checks.append((label, passed))
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — {label}"
    )


# ============================================================
# 1. Create isolated DuckDB database
# ============================================================

con = duckdb.connect(":memory:")

con.execute("""
    CREATE TABLE all_game_summary (
        season_id BIGINT,
        game_id BIGINT,
        gameState VARCHAR,
        game_date DATE
    )
""")

con.execute("""
    CREATE TABLE all_teams (
        team_id BIGINT,
        tri_code VARCHAR
    )
""")

con.execute("""
    INSERT INTO all_teams VALUES
        (6, 'BOS'),
        (10, 'TOR'),
        (22, 'EDM')
""")

sql_types = {
    "Int64": "BIGINT",
    "string": "VARCHAR",
    "float64": "DOUBLE",
}

columns = []

for name, dtype in gs.GOALIE_SCHEMA.items():
    definition = f'"{name}" {sql_types[dtype]}'

    # Force a database write failure in the rollback test.
    if name == "player_id":
        definition += " CHECK (player_id <> 999)"

    columns.append(definition)

con.execute(
    "CREATE TABLE all_goalie_games ("
    + ", ".join(columns)
    + ")"
)


# ============================================================
# 2. Seed game summary
# ============================================================

summary_games = [
    (GAME_NEW, "OFF", TODAY - timedelta(days=1)),
    (GAME_CORRECTED, "OFF", TODAY - timedelta(days=2)),
    (GAME_PARTIAL, "OFF", TODAY - timedelta(days=3)),
    (GAME_OLD, "OFF", TODAY - timedelta(days=20)),
    (GAME_MISSING_OLD, "OFF", TODAY - timedelta(days=25)),
    (GAME_FUTURE, "FUT", TODAY + timedelta(days=1)),
    (GAME_PLAYOFF, "OFF", TODAY - timedelta(days=1)),
]

for game_id, state, game_date in summary_games:
    con.execute(
        """
        INSERT INTO all_game_summary
        VALUES (?, ?, ?, ?)
        """,
        [SEASON_ID, game_id, state, game_date],
    )


# ============================================================
# 3. Seed existing goalie records
# ============================================================

def insert_existing(
    game_id,
    player_id,
    goalie_name,
    tri_code="BOS",
):
    row = {
        col: None
        for col in gs.GOALIE_SCHEMA
    }

    row.update({
        "game_id": game_id,
        "player_id": player_id,
        "goalie_name": goalie_name,
        "tri_code": tri_code,
        "season_id": SEASON_ID,
        "games_played": 1,
        "games_started": 1,
    })

    con.register(
        "seed_goalie",
        gs.enforce_goalie_schema(
            pd.DataFrame([row])
        ),
    )

    try:
        con.execute("""
            INSERT INTO all_goalie_games
            SELECT * FROM seed_goalie
        """)
    finally:
        con.unregister("seed_goalie")


# Corrected game has outdated goalie stats.
insert_existing(
    GAME_CORRECTED, 101, "Outdated Goalie"
)

# Partial game is missing one goalie.
insert_existing(
    GAME_PARTIAL, 201, "Goalie A"
)

# Older unchanged game should be preserved.
insert_existing(
    GAME_OLD, 301, "Old Stable Goalie"
)


# ============================================================
# 4. Mock API data
# ============================================================

def goalie_row(
    game_id,
    player_id,
    name,
    team,
    saves=25,
    save_pct=0.925,
):
    return {
        "game_id": game_id,
        "games_played": 1,
        "games_started": 1,
        "goalie_name": name,
        "player_id": player_id,
        "tri_code": team,
        "toi": 3600,
        "wins": 1,
        "losses": 0,
        "ot_losses": 0,
        "gaa": 2.0,
        "saves": saves,
        "shutouts": 0,
        "save_pct": save_pct,
        "shots_against": 27,
        "goals_against": 2,
        "season_id": SEASON_ID,
        "team_id": None,
    }


api_rows = [
    goalie_row(
        GAME_NEW, 1, "New Goalie", "BOS"
    ),
    goalie_row(
        GAME_NEW, 1, "Duplicate Goalie", "BOS"
    ),
    goalie_row(
        GAME_CORRECTED, 101, "Corrected Goalie", "TOR"
    ),
    goalie_row(
        GAME_PARTIAL, 201, "Goalie A", "BOS"
    ),
    goalie_row(
        GAME_PARTIAL, 202, "Goalie B", "UTA"
    ),
    goalie_row(
        GAME_OLD, 301, "Updated Old Goalie", "BOS"
    ),
    goalie_row(
        GAME_MISSING_OLD, 401, "Recovered Goalie", "EDM"
    ),
    goalie_row(
        GAME_FUTURE, 501, "Future Goalie", "BOS"
    ),
    goalie_row(
        GAME_PLAYOFF, 601, "Playoff Goalie", "BOS"
    ),
]


def fake_get_goalie_games(start_date, end_date):
    return pd.DataFrame(api_rows).copy()


# ============================================================
# 5. First production run
# ============================================================

print(
    "\nBLOCK 47 — GOALIE STATS PRODUCTION VALIDATION\n"
)

with patch.object(
    gs,
    "get_goalie_games",
    side_effect=fake_get_goalie_games,
):
    gs.run_goalie_stats(
        con=con,
        season_id=SEASON_ID,
        changed_games=pd.DataFrame(),
    )


def read_goalies():
    return con.execute("""
        SELECT *
        FROM all_goalie_games
        ORDER BY game_id, player_id
    """).df()


result = read_goalies()


def game_rows(game_id):
    return result[
        result["game_id"] == game_id
    ]


# ============================================================
# 6. Validate ingestion and recovery
# ============================================================

check(
    "New game inserted",
    len(game_rows(GAME_NEW)) == 1,
)

check(
    "Duplicate goalie rows removed",
    not result.duplicated(
        ["game_id", "player_id"]
    ).any(),
)

check(
    "Historical keep-first duplicate behavior",
    game_rows(GAME_NEW)["goalie_name"].iloc[0]
    == "New Goalie",
)

check(
    "Corrected game refreshed",
    game_rows(GAME_CORRECTED)["goalie_name"].iloc[0]
    == "Corrected Goalie",
)

check(
    "Partially stored game recovered",
    set(
        game_rows(GAME_PARTIAL)["player_id"]
    ) == {201, 202},
)

check(
    "Older missing game recovered",
    len(game_rows(GAME_MISSING_OLD)) == 1,
)

check(
    "Older unchanged game preserved",
    game_rows(GAME_OLD)["goalie_name"].iloc[0]
    == "Old Stable Goalie",
)

check(
    "Future game excluded",
    game_rows(GAME_FUTURE).empty,
)

check(
    "Playoff game excluded",
    game_rows(GAME_PLAYOFF).empty,
)

check(
    "Standard BOS team mapping",
    game_rows(GAME_NEW)["team_id"].iloc[0] == 6,
)

check(
    "Standard TOR team mapping",
    game_rows(GAME_CORRECTED)["team_id"].iloc[0] == 10,
)

check(
    "Utah 2025-26 team mapping",
    game_rows(GAME_PARTIAL)
    .loc[
        game_rows(GAME_PARTIAL)["tri_code"] == "UTA",
        "team_id",
    ]
    .iloc[0] == 68,
)

check(
    "EDM team mapping",
    game_rows(GAME_MISSING_OLD)["team_id"].iloc[0] == 22,
)


# ============================================================
# 7. Verify older corrected game
# ============================================================

with patch.object(
    gs,
    "get_goalie_games",
    side_effect=fake_get_goalie_games,
):
    gs.run_goalie_stats(
        con=con,
        season_id=SEASON_ID,
        changed_games=pd.DataFrame({
            "game_id": [GAME_OLD],
        }),
    )

old_name = con.execute("""
    SELECT goalie_name
    FROM all_goalie_games
    WHERE game_id = ?
""", [GAME_OLD]).fetchone()[0]

check(
    "Changed older game refreshed",
    old_name == "Updated Old Goalie",
)


# ============================================================
# 8. API failure preserves existing data
# ============================================================

before_api_failure = read_goalies()

with patch.object(
    gs,
    "get_goalie_games",
    side_effect=RuntimeError(
        "Simulated incomplete API pagination"
    ),
):
    api_failed = False

    try:
        gs.run_goalie_stats(
            con=con,
            season_id=SEASON_ID,
            changed_games=pd.DataFrame(),
        )

    except RuntimeError:
        api_failed = True

after_api_failure = read_goalies()

check(
    "Incomplete API download rejected",
    api_failed,
)

check(
    "Existing data preserved after API failure",
    before_api_failure.equals(after_api_failure),
)


# ============================================================
# BLOCK 47 — SECTION 9 (CORRECTED)
# Transaction Rollback
# ============================================================

before_rollback = read_goalies()

# Include all recently selected games so API coverage
# validation passes and the database write is attempted.

invalid_rows = [
    goalie_row(
        GAME_NEW,
        1,
        "New Goalie",
        "BOS",
    ),
    goalie_row(
        GAME_PARTIAL,
        201,
        "Goalie A",
        "BOS",
    ),
    goalie_row(
        GAME_PARTIAL,
        202,
        "Goalie B",
        "UTA",
    ),
    goalie_row(
        GAME_CORRECTED,
        999,
        "Invalid Goalie",
        "BOS",
    ),
]

with patch.object(
    gs,
    "get_goalie_games",
    return_value=pd.DataFrame(invalid_rows),
):

    write_failed = False

    try:
        gs.run_goalie_stats(
            con=con,
            season_id=SEASON_ID,
            changed_games=pd.DataFrame({
                "game_id": [GAME_CORRECTED],
            }),
        )

    except duckdb.Error as exc:
        write_failed = True

        print(
            "\nExpected write failure:",
            type(exc).__name__,
            str(exc).split("\n")[0],
        )

after_rollback = read_goalies()

check(
    "Invalid database INSERT rejected",
    write_failed,
)

check(
    "Transaction rollback preserved original data",
    before_rollback.equals(after_rollback),
)

check(
    "Invalid goalie row not persisted",
    con.execute("""
        SELECT COUNT(*)
        FROM all_goalie_games
        WHERE player_id = 999
    """).fetchone()[0] == 0,
)

print("\nCorrected transaction test complete.")


# ============================================================
# 10. Summary
# ============================================================

passed = sum(
    ok for _, ok in checks
)

failed = len(checks) - passed

print("\n" + "=" * 55)
print(
    f"BLOCK 47 RESULTS — "
    f"PASS: {passed} | FAIL: {failed}"
)
print("=" * 55)

if failed == 0:
    print(
        "✅ GOALIE STATS PRODUCTION VALIDATION PASSED"
    )
else:
    print(
        "❌ REVIEW FAILED CHECKS BEFORE PROCEEDING"
    )

con.close()


In [ ]:

# ============================================================
# BLOCK 48 — SPECIAL TEAMS PRODUCTION VALIDATION
# ============================================================

import importlib
from datetime import date, timedelta
from unittest.mock import patch

import duckdb
import pandas as pd

import scripts.ingestion.special_teams as st

st = importlib.reload(st)

SEASON_ID = 20252026
TODAY = date.today()

GAME_NEW = 2025020101
GAME_CORRECTED = 2025020102
GAME_PARTIAL = 2025020103
GAME_OLD = 2025020104
GAME_MISSING_OLD = 2025020105
GAME_DUPLICATE = 2025020106
GAME_FUTURE = 2025020107
GAME_PLAYOFF = 2025030108

checks = []


def check(label, condition):
    passed = bool(condition)
    checks.append((label, passed))
    print(
        f"{'✅ PASS' if passed else '❌ FAIL'} — {label}"
    )


# ============================================================
# 1. Create isolated DuckDB database
# ============================================================

con = duckdb.connect(":memory:")

con.execute("""
    CREATE TABLE all_game_summary (
        season_id BIGINT,
        game_id BIGINT,
        gameState VARCHAR,
        game_date DATE
    )
""")

sql_types = {
    "Int64": "BIGINT",
    "string": "VARCHAR",
    "float64": "DOUBLE",
    "DATE": "DATE",
}


def create_special_table(name, schema, constraint=False):
    columns = []

    for col, dtype in schema.items():
        definition = f'"{col}" {sql_types[dtype]}'

        if constraint and col == "team_id":
            definition += " CHECK (team_id <> 999)"

        columns.append(definition)

    con.execute(
        f"CREATE TABLE {name} ("
        + ", ".join(columns)
        + ")"
    )


create_special_table(
    "all_pk",
    st.PK_SCHEMA,
)

create_special_table(
    "all_pp",
    st.PP_SCHEMA,
    constraint=True,
)


# ============================================================
# 2. Seed completed and incomplete games
# ============================================================

summary_games = [
    (GAME_NEW, "OFF", TODAY - timedelta(days=1)),
    (GAME_CORRECTED, "OFF", TODAY - timedelta(days=2)),
    (GAME_PARTIAL, "OFF", TODAY - timedelta(days=3)),
    (GAME_OLD, "OFF", TODAY - timedelta(days=20)),
    (GAME_MISSING_OLD, "OFF", TODAY - timedelta(days=25)),
    (GAME_DUPLICATE, "OFF", TODAY - timedelta(days=30)),
    (GAME_FUTURE, "FUT", TODAY + timedelta(days=1)),
    (GAME_PLAYOFF, "OFF", TODAY - timedelta(days=1)),
]

for game_id, state, game_date in summary_games:
    con.execute(
        """
        INSERT INTO all_game_summary
        VALUES (?, ?, ?, ?)
        """,
        [SEASON_ID, game_id, state, game_date],
    )


# ============================================================
# 3. Helpers for synthetic canonical data
# ============================================================

def special_row(
    game_id,
    team_id,
    schema,
    *,
    marker=1,
):
    row = {
        col: None
        for col in schema
    }

    row.update({
        "season_id": SEASON_ID,
        "game_id": game_id,
        "team_id": team_id,
        "game_date": TODAY - timedelta(days=1),
        "games_played": 1,
        "home_road": "H" if team_id == 6 else "R",
        "losses": 0,
        "opponent": "TOR" if team_id == 6 else "BOS",
        "ot_losses": 0,
        "points": marker,
        "team_name": f"Team {team_id}",
        "ties": "0",
        "wins": 1,
    })

    if schema is st.PK_SCHEMA:
        row.update({
            "pk_pct": 80.0,
            "pk_net_pct": 80.0,
            "times_sh": 4,
            "pp_ga": 1,
            "sh_gf": 0,
        })
    else:
        row.update({
            "pp_pct": 25.0,
            "pp_net_pct": 25.0,
            "pp_gf": 1,
            "pp_opportunities": 4,
            "sh_ga": 0,
        })

    return row


def insert_existing(table, schema, rows):
    df = st.enforce_schema(
        pd.DataFrame(rows),
        schema,
    )

    con.register("seed_rows", df)

    try:
        con.execute(
            f"INSERT INTO {table} SELECT * FROM seed_rows"
        )
    finally:
        con.unregister("seed_rows")


def rows_for(table, game_id):
    return con.execute(
        f"""
        SELECT *
        FROM {table}
        WHERE game_id = ?
        ORDER BY team_id
        """,
        [game_id],
    ).df()


def snapshot(table):
    return con.execute(
        f"""
        SELECT *
        FROM {table}
        ORDER BY game_id, team_id
        """
    ).df()


# ============================================================
# 4. Seed existing PK and PP data
# ============================================================

for table, schema in (
    ("all_pk", st.PK_SCHEMA),
    ("all_pp", st.PP_SCHEMA),
):

    # Corrected game: existing data is outdated.
    insert_existing(
        table,
        schema,
        [
            special_row(
                GAME_CORRECTED, 6, schema, marker=99
            ),
            special_row(
                GAME_CORRECTED, 10, schema, marker=99
            ),
        ],
    )

    # Old unchanged game should be preserved.
    insert_existing(
        table,
        schema,
        [
            special_row(
                GAME_OLD, 6, schema, marker=77
            ),
            special_row(
                GAME_OLD, 10, schema, marker=77
            ),
        ],
    )

    # Duplicate game: same team stored twice.
    insert_existing(
        table,
        schema,
        [
            special_row(
                GAME_DUPLICATE, 6, schema
            ),
            special_row(
                GAME_DUPLICATE, 6, schema
            ),
        ],
    )

# Partial game: PK has one team, PP has two.
insert_existing(
    "all_pk",
    st.PK_SCHEMA,
    [
        special_row(
            GAME_PARTIAL, 6, st.PK_SCHEMA
        ),
    ],
)

insert_existing(
    "all_pp",
    st.PP_SCHEMA,
    [
        special_row(
            GAME_PARTIAL, 6, st.PP_SCHEMA
        ),
        special_row(
            GAME_PARTIAL, 10, st.PP_SCHEMA
        ),
    ],
)


# ============================================================
# 5. Mock raw NHL API data
# ============================================================

def raw_api_row(
    game_id,
    team_id,
    *,
    pk=True,
    marker=1,
):
    row = {
        "season": SEASON_ID,
        "gameId": game_id,
        "teamId": team_id,
        "gameDate": str(TODAY - timedelta(days=1)),
        "teamFullName": f"Team {team_id}",
        "gamesPlayed": 1,
        "homeRoad": "H" if team_id == 6 else "R",
        "losses": 0,
        "opponentTeamAbbrev": (
            "TOR" if team_id == 6 else "BOS"
        ),
        "otLosses": 0,
        "points": marker,
        "ties": "0",
        "wins": 1,
    }

    if pk:
        row.update({
            "penaltyKillNetPct": 80.0,
            "penaltyKillPct": 80.0,
            "pkNetGoals": -1,
            "pkNetGoalsPerGame": -1.0,
            "pkTimeOnIcePerGame": 480.0,
            "pointsPct": 1.0,
            "ppGoalsAgainst": 1,
            "ppGoalsAgainstPerGame": 1.0,
            "shGoalsFor": 0,
            "shGoalsForPerGame": 0.0,
            "timesShorthanded": 4,
            "timesShorthandedPerGame": 4.0,
        })
    else:
        row.update({
            "pointPct": 1.0,
            "powerPlayGoalsFor": 1,
            "powerPlayNetPct": 25.0,
            "powerPlayPct": 25.0,
            "ppGoalsPerGame": 1.0,
            "ppNetGoals": 1,
            "ppNetGoalsPerGame": 1.0,
            "ppOpportunities": 4,
            "ppOpportunitiesPerGame": 4.0,
            "ppTimeOnIcePerGame": 480.0,
            "shGoalsAgainst": 0,
            "shGoalsAgainstPerGame": 0.0,
        })

    return row


eligible_games = [
    GAME_NEW,
    GAME_CORRECTED,
    GAME_PARTIAL,
    GAME_OLD,
    GAME_MISSING_OLD,
    GAME_DUPLICATE,
]

pk_api_rows = []
pp_api_rows = []

for game_id in eligible_games:
    for team_id in (6, 10):
        pk_api_rows.append(
            raw_api_row(game_id, team_id, pk=True)
        )
        pp_api_rows.append(
            raw_api_row(game_id, team_id, pk=False)
        )

# Include games that must be excluded.
for game_id in (GAME_FUTURE, GAME_PLAYOFF):
    for team_id in (6, 10):
        pk_api_rows.append(
            raw_api_row(game_id, team_id, pk=True)
        )
        pp_api_rows.append(
            raw_api_row(game_id, team_id, pk=False)
        )

# Exact API duplicates should be safely removed.
pk_api_rows.append(
    raw_api_row(GAME_NEW, 6, pk=True)
)
pp_api_rows.append(
    raw_api_row(GAME_NEW, 6, pk=False)
)


# ============================================================
# 6. First production run
# ============================================================

print(
    "\nBLOCK 48 — SPECIAL TEAMS PRODUCTION VALIDATION\n"
)

with patch.object(
    st,
    "get_pk_by_game",
    return_value=pd.DataFrame(pk_api_rows),
), patch.object(
    st,
    "get_pp_by_game",
    return_value=pd.DataFrame(pp_api_rows),
):

    st.run_special_teams(
        con=con,
        season_id=SEASON_ID,
        changed_games=pd.DataFrame(),
    )


# ============================================================
# 7. Validate ingestion and recovery
# ============================================================

check(
    "New game inserted in both tables",
    len(rows_for("all_pk", GAME_NEW)) == 2
    and len(rows_for("all_pp", GAME_NEW)) == 2,
)

check(
    "Corrected game refreshed",
    set(rows_for("all_pk", GAME_CORRECTED)["points"]) == {1}
    and set(rows_for("all_pp", GAME_CORRECTED)["points"]) == {1},
)

check(
    "Partially stored game recovered",
    len(rows_for("all_pk", GAME_PARTIAL)) == 2
    and len(rows_for("all_pp", GAME_PARTIAL)) == 2,
)

check(
    "Older missing game recovered",
    len(rows_for("all_pk", GAME_MISSING_OLD)) == 2
    and len(rows_for("all_pp", GAME_MISSING_OLD)) == 2,
)

check(
    "Older duplicate records repaired",
    len(rows_for("all_pk", GAME_DUPLICATE)) == 2
    and len(rows_for("all_pp", GAME_DUPLICATE)) == 2
    and rows_for("all_pk", GAME_DUPLICATE)[
        "team_id"
    ].nunique() == 2,
)

check(
    "Older unchanged game preserved",
    set(rows_for("all_pk", GAME_OLD)["points"]) == {77}
    and set(rows_for("all_pp", GAME_OLD)["points"]) == {77},
)

check(
    "Future game excluded",
    rows_for("all_pk", GAME_FUTURE).empty
    and rows_for("all_pp", GAME_FUTURE).empty,
)

check(
    "Playoff game excluded",
    rows_for("all_pk", GAME_PLAYOFF).empty
    and rows_for("all_pp", GAME_PLAYOFF).empty,
)

check(
    "Exactly two unique teams per game",
    all(
        rows_for(table, game_id)["team_id"].nunique() == 2
        for table in ("all_pk", "all_pp")
        for game_id in (
            GAME_NEW,
            GAME_CORRECTED,
            GAME_PARTIAL,
            GAME_MISSING_OLD,
            GAME_DUPLICATE,
        )
    ),
)

check(
    "Canonical PK values preserved",
    rows_for("all_pk", GAME_NEW)[
        "pk_pct"
    ].eq(80.0).all()
    and rows_for("all_pk", GAME_NEW)[
        "times_sh"
    ].eq(4).all(),
)

check(
    "Canonical PP values preserved",
    rows_for("all_pp", GAME_NEW)[
        "pp_pct"
    ].eq(25.0).all()
    and rows_for("all_pp", GAME_NEW)[
        "pp_opportunities"
    ].eq(4).all(),
)

check(
    "Canonical PK and PP schemas preserved",
    list(snapshot("all_pk").columns)
        == list(st.PK_SCHEMA)
    and list(snapshot("all_pp").columns)
        == list(st.PP_SCHEMA),
)


# ============================================================
# 8. Older corrected game
# ============================================================

with patch.object(
    st,
    "get_pk_by_game",
    return_value=pd.DataFrame(pk_api_rows),
), patch.object(
    st,
    "get_pp_by_game",
    return_value=pd.DataFrame(pp_api_rows),
):

    st.run_special_teams(
        con=con,
        season_id=SEASON_ID,
        changed_games=pd.DataFrame({
            "game_id": [GAME_OLD],
        }),
    )

check(
    "Changed older game refreshed",
    set(rows_for("all_pk", GAME_OLD)["points"]) == {1}
    and set(rows_for("all_pp", GAME_OLD)["points"]) == {1},
)


# ============================================================
# 9. Incomplete API coverage
# ============================================================

before_pk = snapshot("all_pk")
before_pp = snapshot("all_pp")

incomplete_pk = pd.DataFrame([
    raw_api_row(GAME_NEW, 6, pk=True),
])

coverage_rejected = False

with patch.object(
    st,
    "get_pk_by_game",
    return_value=incomplete_pk,
), patch.object(
    st,
    "get_pp_by_game",
    return_value=pd.DataFrame(pp_api_rows),
):

    try:
        st.run_special_teams(
            con=con,
            season_id=SEASON_ID,
            changed_games=pd.DataFrame(),
        )

    except ValueError:
        coverage_rejected = True

check(
    "Incomplete API coverage rejected",
    coverage_rejected,
)

check(
    "Both tables preserved after coverage failure",
    before_pk.equals(snapshot("all_pk"))
    and before_pp.equals(snapshot("all_pp")),
)


# ============================================================
# 10. Conflicting duplicate API rows
# ============================================================

conflicting_pk = [
    raw_api_row(GAME_NEW, 6, pk=True),
    raw_api_row(GAME_NEW, 10, pk=True),
    raw_api_row(
        GAME_NEW, 6, pk=True, marker=999
    ),
]

duplicate_rejected = False

with patch.object(
    st,
    "get_pk_by_game",
    return_value=pd.DataFrame(conflicting_pk),
), patch.object(
    st,
    "get_pp_by_game",
    return_value=pd.DataFrame(pp_api_rows),
):

    try:
        st.run_special_teams(
            con=con,
            season_id=SEASON_ID,
            changed_games=pd.DataFrame(),
        )

    except ValueError:
        duplicate_rejected = True

check(
    "Conflicting duplicate team rows rejected",
    duplicate_rejected,
)

check(
    "Existing data preserved after duplicate failure",
    before_pk.equals(snapshot("all_pk"))
    and before_pp.equals(snapshot("all_pp")),
)


# ============================================================
# 11. API pagination failure
# ============================================================

api_failed = False

with patch.object(
    st,
    "get_pk_by_game",
    side_effect=RuntimeError(
        "Simulated incomplete PK pagination"
    ),
):

    try:
        st.run_special_teams(
            con=con,
            season_id=SEASON_ID,
            changed_games=pd.DataFrame(),
        )

    except RuntimeError:
        api_failed = True

check(
    "Incomplete API pagination rejected",
    api_failed,
)

check(
    "Existing data preserved after API failure",
    before_pk.equals(snapshot("all_pk"))
    and before_pp.equals(snapshot("all_pp")),
)


# ============================================================
# 12. Atomic PK/PP transaction rollback
# ============================================================

# The PP table has a CHECK constraint rejecting team 999.
# Supply all selected recent games so coverage passes,
# then trigger a failure during the PP INSERT.

rollback_pk_rows = []
rollback_pp_rows = []

for game_id in (
    GAME_NEW,
    GAME_CORRECTED,
    GAME_PARTIAL,
):

    rollback_pk_rows.extend([
        raw_api_row(game_id, 6, pk=True),
        raw_api_row(game_id, 999, pk=True),
    ])

    rollback_pp_rows.extend([
        raw_api_row(game_id, 6, pk=False),
        raw_api_row(game_id, 999, pk=False),
    ])

before_rollback_pk = snapshot("all_pk")
before_rollback_pp = snapshot("all_pp")

write_failed = False

with patch.object(
    st,
    "get_pk_by_game",
    return_value=pd.DataFrame(rollback_pk_rows),
), patch.object(
    st,
    "get_pp_by_game",
    return_value=pd.DataFrame(rollback_pp_rows),
):

    try:
        st.run_special_teams(
            con=con,
            season_id=SEASON_ID,
            changed_games=pd.DataFrame(),
        )

    except duckdb.Error as exc:
        write_failed = True

        print(
            "\nExpected write failure:",
            type(exc).__name__,
            str(exc).split("\n")[0],
        )

check(
    "Invalid PP INSERT rejected",
    write_failed,
)

check(
    "PK rollback preserved existing records",
    before_rollback_pk.equals(snapshot("all_pk")),
)

check(
    "PP rollback preserved existing records",
    before_rollback_pp.equals(snapshot("all_pp")),
)

check(
    "Invalid team 999 not persisted",
    con.execute("""
        SELECT
            (SELECT COUNT(*) FROM all_pk
             WHERE team_id = 999)
            +
            (SELECT COUNT(*) FROM all_pp
             WHERE team_id = 999)
    """).fetchone()[0] == 0,
)


# ============================================================
# 13. Final Summary
# ============================================================

passed = sum(ok for _, ok in checks)
failed = len(checks) - passed

print("\n" + "=" * 55)
print(
    f"BLOCK 48 RESULTS — "
    f"PASS: {passed} | FAIL: {failed}"
)
print("=" * 55)

if failed == 0:
    print(
        "✅ SPECIAL TEAMS PRODUCTION VALIDATION PASSED"
    )
else:
    print(
        "❌ REVIEW FAILED CHECKS BEFORE PROCEEDING"
    )

con.close()


In [ ]:

# ============================================================
# BLOCK 49 — STANDINGS PRODUCTION VALIDATION
# ============================================================

import importlib
from datetime import date, timedelta
from unittest.mock import patch

import duckdb
import pandas as pd

import scripts.ingestion.standings_snaps as ss

ss = importlib.reload(ss)

SEASON_ID = 20252026
TODAY = pd.Timestamp.today().normalize()
START = TODAY - pd.Timedelta(days=5)
END = TODAY

checks = []


def check(label, condition):
    passed = bool(condition)
    checks.append((label, passed))
    print(f"{'✅ PASS' if passed else '❌ FAIL'} — {label}")


def count_rows(con, table, day=None):
    if day is None:
        return con.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]

    return con.execute(
        f"""
        SELECT COUNT(*)
        FROM {table}
        WHERE game_date = ?
        """,
        [day],
    ).fetchone()[0]


def snapshot(con, table):
    return con.execute(
        f"SELECT * FROM {table} ORDER BY ALL"
    ).df()


# ============================================================
# 1. Create isolated DuckDB database
# ============================================================

con = duckdb.connect(":memory:")

con.execute("""
    CREATE TABLE important_dates (
        season_id BIGINT,
        regular_start DATE,
        regular_end DATE
    )
""")

con.execute(
    """
    INSERT INTO important_dates
    VALUES (?, ?, ?)
    """,
    [SEASON_ID, START.date(), END.date()],
)

sql_types = {
    "Int64": "BIGINT",
    "float64": "DOUBLE",
    "string": "VARCHAR",
    "DATE": "DATE",
    "boolean": "BOOLEAN",
}


def create_table(name, schema, constraints=None):
    columns = []

    for col, dtype in schema.items():
        definition = f'"{col}" {sql_types[dtype]}'

        if constraints and col in constraints:
            definition += f" {constraints[col]}"

        columns.append(definition)

    con.execute(
        f"CREATE TABLE {name} ("
        + ", ".join(columns)
        + ")"
    )


create_table(
    "all_standings_daily",
    ss.SNAP_SCHEMA,
)

create_table(
    "all_clinch_history",
    ss.CLINCH_SCHEMA,
)

con.execute("""
    CREATE TABLE all_teams (
        tri_code VARCHAR,
        team_id BIGINT
    )
""")

con.execute("""
    INSERT INTO all_teams VALUES
        ('BOS', 6),
        ('TOR', 10),
        ('UTA', 59)
""")


# ============================================================
# 2. Synthetic NHL API responses
# ============================================================

def api_team(
    tri_code,
    team_id,
    *,
    gp=10,
    points=15,
    division_rank=2,
    wildcard_rank=5,
    indicator=None,
):

    return {
        "teamAbbrev": {"default": tri_code},
        "teamName": {"default": f"Team {tri_code}"},
        "teamId": team_id,

        "gamesPlayed": gp,
        "roadGamesPlayed": 5,

        "divisionName": "Atlantic",
        "divisionSequence": division_rank,
        "divisionHomeSequence": 2,
        "divisionRoadSequence": 3,
        "divisionL10Sequence": 2,

        "conferenceName": "Eastern",
        "conferenceSequence": 4,
        "conferenceHomeSequence": 3,
        "conferenceRoadSequence": 5,
        "conferenceL10Sequence": 4,

        "leagueSequence": 8,
        "leagueHomeSequence": 7,
        "leagueRoadSequence": 9,
        "leagueL10Sequence": 8,

        "wildcardSequence": wildcard_rank,

        "divisionRank": division_rank,
        "wildCardRank": wildcard_rank,

        "wins": 7,
        "winPctg": 0.7,
        "regulationWins": 5,
        "losses": 2,
        "otLosses": 1,
        "regulationPlusOtWins": 6,
        "regulationPlusOtWinPctg": 0.6,

        "roadLosses": 1,
        "roadOtLosses": 1,
        "roadRegulationPlusOtWins": 3,
        "roadRegulationWins": 2,

        "points": points,
        "pointPctg": 0.75,
        "homePoints": 8,
        "roadPoints": 7,

        "goalFor": 35,
        "goalsForPctg": 0.55,
        "homeGoalsFor": 20,
        "roadGoalsFor": 15,

        "goalAgainst": 28,
        "homeGoalsAgainst": 12,
        "roadGoalsAgainst": 16,

        "goalDifferential": 7,
        "goalDifferentialPctg": 0.2,
        "homeGoalDifferential": 8,
        "roadGoalDifferential": -1,

        "streakCode": "W",
        "streakCount": 3,

        "l10GamesPlayed": 10,
        "l10Wins": 7,
        "l10GoalDifferential": 7,
        "l10GoalsAgainst": 28,
        "l10GoalsFor": 35,
        "l10Losses": 2,
        "l10OtLosses": 1,
        "l10Points": 15,
        "l10RegulationPlusOtWins": 6,
        "l10RegulationWins": 5,

        "homeGamesPlayed": 5,
        "homeWins": 4,
        "homeLosses": 1,
        "homeOtLosses": 0,
        "homeRegulationPlusOtWins": 4,
        "homeRegulationWins": 3,

        "roadWins": 3,

        "shootoutLosses": 0,
        "shootoutWins": 1,

        "clinchIndicator": indicator,
    }


def api_response(day):
    return [
        api_team(
            "BOS", 6,
            gp=10,
            points=15,
            division_rank=2,
        ),
        api_team(
            "TOR", 10,
            gp=10,
            points=14,
            division_rank=4,
            wildcard_rank=1,
        ),
        api_team(
            "UTA", 59,
            gp=0,
            points=0,
            division_rank=5,
            wildcard_rank=6,
        ),
    ]


# ============================================================
# 3. Seed existing snapshots
# ============================================================

def seed_snapshot(day, teams, *, points_override=None):

    rows = api_response(day)

    rows = [
        row for row in rows
        if row["teamAbbrev"]["default"] in teams
    ]

    df, _ = ss.normalize_standings(
        rows,
        pd.Timestamp(day),
        SEASON_ID,
    )

    if points_override is not None:
        df["points"] = points_override

    con.register("seed_df", df)

    try:
        con.execute(
            """
            INSERT INTO all_standings_daily
            SELECT * FROM seed_df
            """
        )
    finally:
        con.unregister("seed_df")


# Complete old date.
seed_snapshot(
    START.date(),
    ["BOS", "TOR", "UTA"],
    points_override=99,
)

# Incomplete old date.
seed_snapshot(
    (START + pd.Timedelta(days=1)).date(),
    ["BOS"],
)

# Leave day 2 missing.

# Complete day 3.
seed_snapshot(
    (START + pd.Timedelta(days=3)).date(),
    ["BOS", "TOR", "UTA"],
)

# Days 4 and 5 will be refreshed.


# ============================================================
# 4. Canonical schema and calculations
# ============================================================

print("\nBLOCK 49 — STANDINGS PRODUCTION VALIDATION\n")

check(
    "Historical canonical schema retained",
    list(ss.SNAP_SCHEMA.keys())
    == list(
        con.execute(
            "SELECT * FROM all_standings_daily LIMIT 0"
        ).df().columns
    ),
)

check(
    "Season length: 2020-21 = 56",
    ss.get_regular_season_games(20202021) == 56,
)

check(
    "Season length: standard = 82",
    ss.get_regular_season_games(20252026) == 82,
)

check(
    "Season length: 2026-27 onward = 84",
    ss.get_regular_season_games(20262027) == 84,
)

test_df, _ = ss.normalize_standings(
    api_response(TODAY),
    TODAY,
    SEASON_ID,
)

bos = test_df.loc[
    test_df["tri_code"] == "BOS"
].iloc[0]

tor = test_df.loc[
    test_df["tri_code"] == "TOR"
].iloc[0]

uta = test_df.loc[
    test_df["tri_code"] == "UTA"
].iloc[0]

check(
    "Canonical games remaining calculation",
    bos["games_remaining"] == 72,
)

check(
    "Canonical PPG calculation",
    abs(bos["ppg"] - 1.5) < 0.00001,
)

check(
    "Projected points rounded to integer",
    bos["projected_points"] == 123,
)

check(
    "Zero-game PPG is 0",
    uta["ppg"] == 0.0,
)

check(
    "Division top-three playoff qualification",
    bos["in_playoff_spot"] == 1,
)

check(
    "Wildcard playoff qualification",
    tor["in_playoff_spot"] == 1,
)

check(
    "Utah 2025-26 team ID corrected to 68",
    uta["team_id"] == 68,
)

check(
    "Historical detailed fields retained",
    (
        bos["division_home_rank"] == 2
        and bos["last_10_goal_diff"] == 7
        and bos["home_gf"] == 20
        and bos["road_rw"] == 2
    ),
)


# ============================================================
# 5. Main runner: missing and incomplete recovery
# ============================================================

with patch.object(
    ss,
    "fetch_standings",
    side_effect=lambda day: api_response(day),
), patch.object(
    ss.time,
    "sleep",
    return_value=None,
):

    ss.run_standings_snaps(
        con=con,
        season_id=SEASON_ID,
    )


# ============================================================
# 6. Validate recovery
# ============================================================

check(
    "Missing snapshot date recovered",
    count_rows(
        con,
        "all_standings_daily",
        (START + pd.Timedelta(days=2)).date(),
    ) == 3,
)

check(
    "Incomplete snapshot recovered",
    count_rows(
        con,
        "all_standings_daily",
        (START + pd.Timedelta(days=1)).date(),
    ) == 3,
)

check(
    "Recent snapshots refreshed",
    all(
        count_rows(
            con,
            "all_standings_daily",
            day.date(),
        ) == 3
        for day in pd.date_range(
            START + pd.Timedelta(days=3),
            END,
        )
    ),
)

check(
    "Older unchanged snapshot preserved",
    set(
        con.execute(
            """
            SELECT points
            FROM all_standings_daily
            WHERE game_date = ?
            """,
            [START.date()],
        ).df()["points"]
    ) == {99},
)

check(
    "No duplicate team/date records",
    con.execute("""
        SELECT COUNT(*)
        FROM (
            SELECT
                season_id,
                game_date,
                tri_code,
                COUNT(*) AS n
            FROM all_standings_daily
            GROUP BY 1, 2, 3
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0] == 0,
)

check(
    "All expected dates exist",
    con.execute("""
        SELECT COUNT(DISTINCT game_date)
        FROM all_standings_daily
    """).fetchone()[0] == 6,
)


# ============================================================
# 7. Clinch history
# ============================================================

clinch_day = TODAY

clinch_rows = [
    api_team(
        "BOS", 6,
        indicator="x",
    ),
    api_team(
        "TOR", 10,
        indicator="e",
    ),
    api_team(
        "UTA", 59,
    ),
]

with patch.object(
    ss,
    "fetch_standings",
    return_value=clinch_rows,
), patch.object(
    ss.time,
    "sleep",
    return_value=None,
):

    ss.run_standings_snaps(
        con=con,
        season_id=SEASON_ID,
    )

check(
    "Clinch and elimination events recorded",
    con.execute("""
        SELECT COUNT(*)
        FROM all_clinch_history
    """).fetchone()[0] == 2,
)

check(
    "Canonical first event per team",
    con.execute("""
        SELECT COUNT(*)
        FROM (
            SELECT season_id, tri_code
            FROM all_clinch_history
            GROUP BY 1, 2
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0] == 0,
)

check(
    "Clinch types x and e preserved",
    set(
        con.execute(
            "SELECT clinch_type FROM all_clinch_history"
        ).df()["clinch_type"]
    ) == {"x", "e"},
)


# ============================================================
# 8. Earlier clinch event discovered
# ============================================================

earlier_day = TODAY - pd.Timedelta(days=2)

earlier_clinch = [{
    "clinch_indicator": True,
    "tri_code": "BOS",
    "clinch_type": "x",
    "clinch_date": earlier_day.date(),
    "season_id": SEASON_ID,
}]

prepared = ss.prepare_clinch_history(
    earlier_clinch
)

check(
    "Earlier clinch candidate normalized",
    prepared.iloc[0]["clinch_date"]
    == earlier_day.date(),
)


# ============================================================
# 9. API failure protection
# ============================================================

before_api_pk = snapshot(
    con,
    "all_standings_daily",
)

before_api_clinch = snapshot(
    con,
    "all_clinch_history",
)

api_failed = False

with patch.object(
    ss,
    "fetch_standings",
    side_effect=RuntimeError(
        "Simulated NHL API failure"
    ),
), patch.object(
    ss.time,
    "sleep",
    return_value=None,
):

    try:
        ss.run_standings_snaps(
            con=con,
            season_id=SEASON_ID,
        )

    except RuntimeError:
        api_failed = True

check(
    "API failure rejected",
    api_failed,
)

check(
    "Standings preserved after API failure",
    before_api_pk.equals(
        snapshot(con, "all_standings_daily")
    ),
)

check(
    "Clinch history preserved after API failure",
    before_api_clinch.equals(
        snapshot(con, "all_clinch_history")
    ),
)


# ============================================================
# 10. Incomplete API snapshot protection
# ============================================================

incomplete_rejected = False

with patch.object(
    ss,
    "fetch_standings",
    return_value=[
        api_team("BOS", 6),
    ],
), patch.object(
    ss.time,
    "sleep",
    return_value=None,
):

    try:
        ss.run_standings_snaps(
            con=con,
            season_id=SEASON_ID,
        )

    except ValueError:
        incomplete_rejected = True

check(
    "Incomplete API snapshot rejected",
    incomplete_rejected,
)

check(
    "Existing standings preserved after incomplete API",
    before_api_pk.equals(
        snapshot(con, "all_standings_daily")
    ),
)


# ============================================================
# 11. Atomic transaction rollback
# ============================================================

# Add a CHECK constraint to a separate in-memory table
# by rebuilding the standings table with a constraint.

con.execute("""
    CREATE TABLE standings_backup AS
    SELECT * FROM all_standings_daily
""")

con.execute("""
    CREATE TABLE clinch_backup AS
    SELECT * FROM all_clinch_history
""")

# Rebuild all_standings_daily with a constraint
# that rejects the special test team ID 999.

con.execute(
    "DROP TABLE all_standings_daily"
)

columns = []

for col, dtype in ss.SNAP_SCHEMA.items():

    definition = (
        f'"{col}" {sql_types[dtype]}'
    )

    if col == "team_id":
        definition += " CHECK (team_id <> 999)"

    columns.append(definition)

con.execute(
    "CREATE TABLE all_standings_daily ("
    + ", ".join(columns)
    + ")"
)

con.execute("""
    INSERT INTO all_standings_daily
    SELECT * FROM standings_backup
""")

before_rollback_standings = snapshot(
    con,
    "all_standings_daily",
)

before_rollback_clinch = snapshot(
    con,
    "all_clinch_history",
)

invalid_rows = [
    api_team("BOS", 6),
    api_team("TOR", 10),
    api_team("UTA", 59),
]

# The canonical Utah correction turns UTA into 68.
# Instead, use an additional invalid team 999,
# while maintaining a unique team abbreviation.

invalid_rows[2]["teamId"] = 999
invalid_rows[2]["teamAbbrev"] = {
    "default": "BAD"
}

rollback_failed = False

with patch.object(
    ss,
    "fetch_standings",
    return_value=invalid_rows,
), patch.object(
    ss.time,
    "sleep",
    return_value=None,
):

    try:
        ss.run_standings_snaps(
            con=con,
            season_id=SEASON_ID,
        )

    except duckdb.Error as exc:
        rollback_failed = True

        print(
            "\nExpected rollback error:",
            type(exc).__name__,
            str(exc).split("\n")[0],
        )

check(
    "Invalid standings INSERT rejected",
    rollback_failed,
)

check(
    "Standings rollback preserved existing records",
    before_rollback_standings.equals(
        snapshot(con, "all_standings_daily")
    ),
)

check(
    "Clinch rollback preserved existing records",
    before_rollback_clinch.equals(
        snapshot(con, "all_clinch_history")
    ),
)

check(
    "Invalid team ID not persisted",
    con.execute("""
        SELECT COUNT(*)
        FROM all_standings_daily
        WHERE team_id = 999
    """).fetchone()[0] == 0,
)


# ============================================================
# 12. Final Results
# ============================================================

passed = sum(ok for _, ok in checks)
failed = len(checks) - passed

print("\n" + "=" * 60)
print(
    f"BLOCK 49 RESULTS — "
    f"PASS: {passed} | FAIL: {failed}"
)
print("=" * 60)

if failed == 0:
    print(
        "✅ STANDINGS PRODUCTION VALIDATION PASSED"
    )
else:
    print(
        "❌ REVIEW FAILED CHECKS BEFORE PROCEEDING"
    )

con.close()


In [ ]:

# ============================================================
# BLOCK 50 — PREDICTION DATA VALIDATION
# ============================================================

import ast
import inspect
import importlib
import traceback

import duckdb
import numpy as np
import pandas as pd

from pathlib import Path

# ------------------------------------------------------------
# Load the actual pipeline module
# ------------------------------------------------------------
import scripts.features.prediction_data as prediction_data

prediction_data = importlib.reload(prediction_data)

from scripts.features.prediction_data import (
    DATA_SCHEMA,
    load_prediction_offsets,
    build_games_played_spine,
    cumulative_shot_attempts,
    build_danger_corsi_and_scf_stats,
    enforce_data_schema,
    run_prediction_data,
)

# ------------------------------------------------------------
# Test runner
# ------------------------------------------------------------
results = []

def check(number, description, test_fn):
    try:
        test_fn()
        results.append((number, description, "PASS", ""))
        print(f"✅ {number:02d} PASS | {description}")
    except Exception as exc:
        results.append((number, description, "FAIL", str(exc)))
        print(f"❌ {number:02d} FAIL | {description}")
        print(f"   {type(exc).__name__}: {exc}")

def require(condition, message):
    if not condition:
        raise AssertionError(message)

def equal(actual, expected, label):
    require(
        actual == expected,
        f"{label}: expected {expected!r}, got {actual!r}"
    )

def get_team(df, team_id):
    rows = df.loc[df["team_id"] == team_id]
    require(len(rows) == 1, f"Expected one row for team {team_id}")
    return rows.iloc[0]

# ------------------------------------------------------------
# Synthetic game fixtures
# ------------------------------------------------------------
SEASON = 20252026

games = pd.DataFrame([
    {
        "season_id": SEASON,
        "game_id": 2025020001,
        "game_date": "2025-10-07",
        "home_team": 1,
        "away_team": 2,
    },
    {
        "season_id": SEASON,
        "game_id": 2025020002,
        "game_date": "2025-10-09",
        "home_team": 2,
        "away_team": 1,
    },
])

offsets = pd.DataFrame([
    {
        "team_id": 1,
        "games_played_offset": 10,
    },
    {
        "team_id": 2,
        "games_played_offset": 8,
    },
])

# ------------------------------------------------------------
# 01 — Module imports
# ------------------------------------------------------------
def test_01():
    require(
        callable(run_prediction_data),
        "run_prediction_data is not callable"
    )

check(1, "Module imports and runner exists", test_01)

# ------------------------------------------------------------
# 02 — Required output schema
# ------------------------------------------------------------
def test_02():
    required = {
        "season_id",
        "game_id",
        "game_date",
        "team_id",
        "games_played",
        "games_before",
        "wins_before",
        "game_win",
        "points",
        "c_SCF",
        "c_SCA",
        "c_corsi_for",
        "c_corsi_against",
    }

    missing = required - set(DATA_SCHEMA)

    require(
        not missing,
        f"Missing schema columns: {sorted(missing)}"
    )

check(2, "Required prediction schema columns", test_02)

# ------------------------------------------------------------
# 03 — Two team rows per game
# ------------------------------------------------------------
def test_03():
    spine = build_games_played_spine(games)

    equal(len(spine), 4, "Total rows")

    counts = spine.groupby("game_id").size()

    require(
        counts.eq(2).all(),
        "Every game must produce exactly two team rows"
    )

    require(
        not spine.duplicated(
            ["season_id", "game_id", "team_id"]
        ).any(),
        "Duplicate team-game rows"
    )

check(3, "Team-game spine completeness", test_03)

# ------------------------------------------------------------
# 04 — Games played without offsets
# ------------------------------------------------------------
def test_04():
    spine = build_games_played_spine(games)

    for team_id in (1, 2):
        team = spine.loc[
            spine["team_id"] == team_id
        ].sort_values("game_date")

        equal(
            team["games_played"].tolist(),
            [1, 2],
            f"Team {team_id} games_played"
        )

        equal(
            team["games_before"].tolist(),
            [0, 1],
            f"Team {team_id} games_before"
        )

check(4, "Initial-season games-played sequence", test_04)

# ------------------------------------------------------------
# 05 — Games played with historical offsets
# ------------------------------------------------------------
def test_05():
    spine = build_games_played_spine(games, offsets)

    expected = {
        1: ([10, 11], [11, 12]),
        2: ([8, 9], [9, 10]),
    }

    for team_id, (before, played) in expected.items():
        team = spine.loc[
            spine["team_id"] == team_id
        ].sort_values("game_date")

        equal(
            team["games_before"].tolist(),
            before,
            f"Team {team_id} games_before"
        )

        equal(
            team["games_played"].tolist(),
            played,
            f"Team {team_id} games_played"
        )

check(5, "Historical games-played offsets", test_05)

# ------------------------------------------------------------
# 06 — Wins offset includes previous game
# ------------------------------------------------------------
def test_06():
    con = duckdb.connect(":memory:")

    try:
        previous = pd.DataFrame([
            {
                "season_id": SEASON,
                "game_id": 2025020000,
                "team_id": 1,
                "games_played": 10,
                "wins_before": 5,
                "game_win": 1,
            },
            {
                "season_id": SEASON,
                "game_id": 2025020000,
                "team_id": 2,
                "games_played": 8,
                "wins_before": 3,
                "game_win": 0,
            },
        ])

        con.register("previous", previous)
        con.execute("""
            CREATE TABLE team_daily_stats AS
            SELECT * FROM previous
        """)

        result = load_prediction_offsets(
            con,
            season_id=SEASON,
            start_game_id=2025020001,
        )

        equal(
            int(get_team(result, 1)["wins_completed_offset"]),
            6,
            "Team 1 completed wins"
        )

        equal(
            int(get_team(result, 2)["wins_completed_offset"]),
            3,
            "Team 2 completed wins"
        )

    finally:
        con.close()

check(6, "Previous-game win included in offset", test_06)

# ------------------------------------------------------------
# 07 — Cumulative shot attempts
# ------------------------------------------------------------
def test_07():
    shots = pd.DataFrame([
        {
            "season_id": SEASON,
            "game_id": 1,
            "team_id": 1,
            "game_date": pd.Timestamp("2025-10-07"),
            "games_played": 1,
            "shot_attempts_for": 12,
            "shot_attempts_against": 9,
            "shot_attempts_differential": 3,
            "corsi_for": 12,
            "corsi_against": 9,
        },
        {
            "season_id": SEASON,
            "game_id": 2,
            "team_id": 1,
            "game_date": pd.Timestamp("2025-10-09"),
            "games_played": 2,
            "shot_attempts_for": 15,
            "shot_attempts_against": 11,
            "shot_attempts_differential": 4,
            "corsi_for": 15,
            "corsi_against": 11,
        },
    ])

    result = cumulative_shot_attempts(shots)

    result = result.sort_values("game_id")

    equal(
        result["c_shots_for"].tolist(),
        [12, 27],
        "Cumulative attempts for"
    )

    equal(
        result["c_shots_against"].tolist(),
        [9, 20],
        "Cumulative attempts against"
    )

    equal(
        result["c_corsi_diff"].tolist(),
        [3, 7],
        "Cumulative Corsi differential"
    )

check(7, "Cumulative shot-attempt arithmetic", test_07)

# ------------------------------------------------------------
# 08 — Cumulative shot attempts with offsets
# ------------------------------------------------------------
def test_08():
    shots = pd.DataFrame([{
        "season_id": SEASON,
        "game_id": 100,
        "team_id": 1,
        "game_date": pd.Timestamp("2025-10-10"),
        "games_played": 11,
        "shot_attempts_for": 12,
        "shot_attempts_against": 9,
        "shot_attempts_differential": 3,
        "corsi_for": 12,
        "corsi_against": 9,
    }])

    historical = pd.DataFrame([{
        "team_id": 1,
        "c_shots_for_offset": 100,
        "c_shots_against_offset": 90,
        "c_shots_diff_offset": 10,
        "c_corsi_for_offset": 100,
        "c_corsi_against_offset": 90,
    }])

    result = cumulative_shot_attempts(
        shots,
        offsets=historical
    )

    row = result.iloc[0]

    equal(int(row["c_shots_for"]), 112, "Shots for")
    equal(int(row["c_shots_against"]), 99, "Shots against")
    equal(int(row["c_corsi_diff"]), 13, "Corsi differential")

check(8, "Cumulative shot-attempt restart offsets", test_08)

# ------------------------------------------------------------
# 09 — Scoring chances attributed to correct team
# ------------------------------------------------------------
def test_09():
    events = pd.DataFrame([
        {
            "season_id": SEASON,
            "game_id": 1,
            "team_id": 1,
            "opponent_team_id": 2,
            "type_code": 506,
            "scoring_chance": 1,
            "hd_shot": 1,
            "md_shot": 0,
            "ld_shot": 0,
        },
        {
            "season_id": SEASON,
            "game_id": 1,
            "team_id": 2,
            "opponent_team_id": 1,
            "type_code": 506,
            "scoring_chance": 1,
            "hd_shot": 0,
            "md_shot": 1,
            "ld_shot": 0,
        },
    ])

    result = build_danger_corsi_and_scf_stats(events)

    team1 = get_team(result, 1)
    team2 = get_team(result, 2)

    equal(int(team1["SCF"]), 1, "Team 1 SCF")
    equal(int(team1["SCA"]), 1, "Team 1 SCA")
    equal(int(team2["SCF"]), 1, "Team 2 SCF")
    equal(int(team2["SCA"]), 1, "Team 2 SCA")

    equal(int(team1["HDCF"]), 1, "Team 1 HDCF")
    equal(int(team1["MDCA"]), 1, "Team 1 MDCA")

check(9, "Normal shot scoring-chance attribution", test_09)

# ------------------------------------------------------------
# 10 — Blocked-shot attribution
# ------------------------------------------------------------
def test_10():
    # Fixture assumes blocked-shot team_id identifies
    # the blocking team, not the shooting team.

    events = pd.DataFrame([
        {
            "season_id": SEASON,
            "game_id": 1,
            "team_id": 2,
            "opponent_team_id": 1,
            "type_code": 508,
            "scoring_chance": 1,
            "hd_shot": 1,
            "md_shot": 0,
            "ld_shot": 0,
        },
        {
            "season_id": SEASON,
            "game_id": 1,
            "team_id": 2,
            "opponent_team_id": 1,
            "type_code": 506,
            "scoring_chance": 0,
            "hd_shot": 0,
            "md_shot": 0,
            "ld_shot": 1,
        },
    ])

    result = build_danger_corsi_and_scf_stats(events)

    team1 = get_team(result, 1)
    team2 = get_team(result, 2)

    equal(int(team1["SCF"]), 1, "Shooting team SCF")
    equal(int(team2["SCA"]), 1, "Blocking team SCA")

    equal(int(team1["HDCF"]), 1, "Shooting team HDCF")
    equal(int(team2["HDCA"]), 1, "Blocking team HDCA")

check(10, "Blocked-shot shooting-team attribution", test_10)

# ------------------------------------------------------------
# 11 — No duplicate schema column names
# ------------------------------------------------------------
def test_11():
    cols = list(DATA_SCHEMA.keys())

    equal(
        len(cols),
        len(set(cols)),
        "Schema column uniqueness"
    )

check(11, "Output schema column uniqueness", test_11)

# ------------------------------------------------------------
# 12 — Canonical game count is required
# ------------------------------------------------------------
def test_12():
    df = pd.DataFrame([{
        "season_id": SEASON,
        "game_id": 1,
        "team_id": 1,
    }])

    try:
        enforce_data_schema(df)
    except KeyError:
        return

    raise AssertionError(
        "Missing games_played should raise KeyError"
    )

check(12, "Missing canonical games_played rejected", test_12)

# ------------------------------------------------------------
# 13 — Canonical wins_before is required
# ------------------------------------------------------------
def test_13():
    df = pd.DataFrame([{
        "season_id": SEASON,
        "game_id": 1,
        "team_id": 1,
        "games_played": 1,
        "games_before": 0,
    }])

    try:
        enforce_data_schema(df)
    except KeyError:
        return

    raise AssertionError(
        "Missing wins_before should raise KeyError"
    )

check(13, "Missing canonical wins_before rejected", test_13)

# ------------------------------------------------------------
# 14 — Output schema enforcement
# ------------------------------------------------------------
def test_14():
    required_special = [
        "times_sh",
        "c_times_sh",
        "times_sh_pg",
        "pp_opp",
        "c_pp_opps",
        "pp_opp_pg",
        "pp_net_g",
        "c_pp_net_g",
        "pp_net_pct",
    ]

    row = {
        "season_id": SEASON,
        "game_id": 1,
        "game_date": "2025-10-07",
        "team_id": 1,
        "games_played": 1,
        "games_before": 0,
        "wins_before": 0,
        "game_win": 1,
    }

    row.update({c: 0 for c in required_special})

    result = enforce_data_schema(pd.DataFrame([row]))

    equal(
        result.columns.tolist(),
        list(DATA_SCHEMA.keys()),
        "Schema column order"
    )

    equal(
        int(result.iloc[0]["games_played"]),
        1,
        "Preserved games_played"
    )

check(14, "DuckDB-aligned output schema enforcement", test_14)

# ------------------------------------------------------------
# 15 — Transaction protection
# ------------------------------------------------------------
def test_15():
    source = inspect.getsource(run_prediction_data)

    required = [
        "BEGIN TRANSACTION",
        "COMMIT",
        "ROLLBACK",
        "prediction_updates",
    ]

    missing = [x for x in required if x not in source]

    require(
        not missing,
        f"Missing transaction safeguards: {missing}"
    )

check(15, "Transactional database write safeguards", test_15)

# ------------------------------------------------------------
# 16 — Rebuild detects missing/incomplete games
# ------------------------------------------------------------
def test_16():
    source = inspect.getsource(run_prediction_data)

    require(
        "team_rows" in source and "total_rows" in source,
        "Expected missing/incomplete game detection"
    )

    require(
        "missing.empty" in source,
        "Expected missing-game selection"
    )

    require(
        "Rebuilding" in source,
        "Expected suffix rebuild path"
    )

check(16, "Missing-game rebuild logic present", test_16)

# ------------------------------------------------------------
# 17 — Source game coverage validation
# ------------------------------------------------------------
def test_17():
    source = inspect.getsource(run_prediction_data)

    require(
        "expected_games - actual_games" in source,
        "Missing expected-game coverage validation"
    )

    require(
        "rows_per_game" in source,
        "Missing two-rows-per-game validation"
    )

check(17, "Final game coverage checks present", test_17)

# ------------------------------------------------------------
# 18 — Rebuild uses chronology-safe deletion
# ------------------------------------------------------------
def test_18():
    source = inspect.getsource(run_prediction_data)

    # This test deliberately flags a rebuild that selects
    # games by date but deletes rows only by numeric game_id.
    #
    # A chronological suffix should be selected/deleted
    # using compatible boundaries.

    date_based = (
        'game_summary["game_date"] > earliest["game_date"]'
        in source
    )

    id_only_delete = (
        "WHERE season_id = ? AND game_id >= ?" in source
    )

    require(
        not (date_based and id_only_delete),
        "Chronological rebuild uses game_date, but deletion "
        "uses game_id only. Verify or align the boundaries."
    )

check(18, "Chronological rebuild/deletion consistency", test_18)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------
summary = pd.DataFrame(
    results,
    columns=["Test", "Description", "Status", "Details"]
)

passed = int((summary["Status"] == "PASS").sum())
failed = int((summary["Status"] == "FAIL").sum())

print("\n" + "=" * 65)
print("BLOCK 50 — PREDICTION DATA VALIDATION SUMMARY")
print("=" * 65)

print(f"TOTAL:  {len(summary)}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")

print("\nFAILED TESTS:")
failures = summary.loc[summary["Status"] == "FAIL"]

if failures.empty:
    print("None")
else:
    print(failures[["Test", "Description", "Details"]].to_string(index=False))

print("\n" + "=" * 65)

if failed == 0:
    print("✅ BLOCK 50 SYNTHETIC TESTS PASSED")
    print("⚠️ Live-data reconciliation is still required.")
else:
    print("❌ BLOCK 50 NEEDS CORRECTIONS")

print("=" * 65)

summary


In [ ]:

# ============================================================
# BLOCK 50B — LIVE DATA RECONCILIATION (READ-ONLY)
# ============================================================

import duckdb
import pandas as pd
import numpy as np

from pathlib import Path
from collections import Counter

from scripts.features.prediction_data import DATA_SCHEMA

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

SEASON_ID = 20252026

# Leave as None to use your existing notebook connection "con".
# Otherwise enter your DuckDB database path.
DB_PATH = None

results = []

def check(number, description, fn):
    try:
        details = fn()
        results.append({
            "Test": number,
            "Description": description,
            "Status": "PASS",
            "Details": "" if details is None else str(details),
        })
        print(f"✅ {number:02d} PASS | {description}")
        if details is not None:
            print(f"   {details}")
    except Exception as exc:
        results.append({
            "Test": number,
            "Description": description,
            "Status": "FAIL",
            "Details": f"{type(exc).__name__}: {exc}",
        })
        print(f"❌ {number:02d} FAIL | {description}")
        print(f"   {type(exc).__name__}: {exc}")

def require(condition, message):
    if not condition:
        raise AssertionError(message)

# ------------------------------------------------------------
# CONNECTION — READ ONLY
# ------------------------------------------------------------

existing_con = globals().get("con")
own_connection = False

if DB_PATH is not None:
    db = duckdb.connect(str(DB_PATH), read_only=True)
    own_connection = True
elif existing_con is not None:
    db = existing_con
    print("Using existing notebook connection.")
else:
    raise RuntimeError(
        "No existing 'con' connection found. "
        "Set DB_PATH to your DuckDB database file."
    )

# Read-only SQL statements only.
# No CREATE, INSERT, UPDATE, DELETE, or pipeline calls.

try:
    # --------------------------------------------------------
    # 01 — Source and target tables exist
    # --------------------------------------------------------
    def test_01():
        tables = set(
            db.execute("""
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = 'main'
            """).df()["table_name"]
        )

        required = {"all_game_summary", "team_daily_stats"}
        missing = required - tables

        require(not missing, f"Missing tables: {sorted(missing)}")

    check(1, "Required source and target tables", test_01)

    # --------------------------------------------------------
    # 02 — Source/target key columns
    # --------------------------------------------------------
    def test_02():
        source_cols = set(
            db.execute("DESCRIBE all_game_summary")
              .df()["column_name"]
        )

        target_cols = set(
            db.execute("DESCRIBE team_daily_stats")
              .df()["column_name"]
        )

        required_source = {
            "season_id", "game_id", "game_date", "gameState"
        }
        required_target = {
            "season_id", "game_id", "game_date",
            "team_id", "games_played", "games_before",
            "wins_before", "game_win"
        }

        require(
            required_source <= source_cols,
            f"Missing source columns: {sorted(required_source-source_cols)}"
        )
        require(
            required_target <= target_cols,
            f"Missing target columns: {sorted(required_target-target_cols)}"
        )

    check(2, "Required key and cumulative columns", test_02)

    # --------------------------------------------------------
    # 03 — Season has completed source games
    # --------------------------------------------------------
    def test_03():
        n = db.execute("""
            SELECT COUNT(*)
            FROM all_game_summary
            WHERE season_id = ?
              AND gameState = 'OFF'
        """, [SEASON_ID]).fetchone()[0]

        require(n > 0, "No completed source games")
        return f"{n:,} completed source rows"

    check(3, "Completed source games available", test_03)

    # --------------------------------------------------------
    # 04 — Source game IDs are unique
    # --------------------------------------------------------
    def test_04():
        duplicates = db.execute("""
            SELECT game_id, COUNT(*) AS n
            FROM all_game_summary
            WHERE season_id = ?
              AND gameState = 'OFF'
            GROUP BY game_id
            HAVING COUNT(*) > 1
            LIMIT 5
        """, [SEASON_ID]).df()

        require(
            duplicates.empty,
            f"Duplicate source game IDs:\n{duplicates}"
        )

    check(4, "Source game IDs unique", test_04)

    # --------------------------------------------------------
    # 05 — Target team-game keys are unique
    # --------------------------------------------------------
    def test_05():
        duplicates = db.execute("""
            SELECT game_id, team_id, COUNT(*) AS n
            FROM team_daily_stats
            WHERE season_id = ?
            GROUP BY game_id, team_id
            HAVING COUNT(*) > 1
            LIMIT 5
        """, [SEASON_ID]).df()

        require(
            duplicates.empty,
            f"Duplicate target keys:\n{duplicates}"
        )

    check(5, "Target team-game keys unique", test_05)

    # --------------------------------------------------------
    # 06 — Every stored game has two team rows
    # --------------------------------------------------------
    def test_06():
        bad = db.execute("""
            SELECT game_id,
                   COUNT(*) AS rows_found,
                   COUNT(DISTINCT team_id) AS teams_found
            FROM team_daily_stats
            WHERE season_id = ?
            GROUP BY game_id
            HAVING COUNT(*) <> 2
                OR COUNT(DISTINCT team_id) <> 2
            LIMIT 10
        """, [SEASON_ID]).df()

        require(
            bad.empty,
            f"Games without exactly two teams:\n{bad}"
        )

    check(6, "Two distinct team rows per stored game", test_06)

    # --------------------------------------------------------
    # 07 — Every completed source game is represented
    # --------------------------------------------------------
    def test_07():
        missing = db.execute("""
            SELECT DISTINCT s.game_id
            FROM all_game_summary AS s
            WHERE s.season_id = ?
              AND s.gameState = 'OFF'
              AND NOT EXISTS (
                  SELECT 1
                  FROM team_daily_stats AS t
                  WHERE t.season_id = s.season_id
                    AND t.game_id = s.game_id
              )
            ORDER BY s.game_id
            LIMIT 20
        """, [SEASON_ID]).df()

        require(
            missing.empty,
            f"Missing completed games (first 20):\n{missing}"
        )

    check(7, "Completed source game coverage", test_07)

    # --------------------------------------------------------
    # 08 — No target games absent from completed source
    # --------------------------------------------------------
    def test_08():
        extra = db.execute("""
            SELECT DISTINCT t.game_id
            FROM team_daily_stats AS t
            WHERE t.season_id = ?
              AND NOT EXISTS (
                  SELECT 1
                  FROM all_game_summary AS s
                  WHERE s.season_id = t.season_id
                    AND s.game_id = t.game_id
                    AND s.gameState = 'OFF'
              )
            ORDER BY t.game_id
            LIMIT 20
        """, [SEASON_ID]).df()

        require(
            extra.empty,
            f"Target games absent from completed source:\n{extra}"
        )

    check(8, "No orphaned target games", test_08)

    # --------------------------------------------------------
    # 09 — Key fields are non-null
    # --------------------------------------------------------
    def test_09():
        n = db.execute("""
            SELECT COUNT(*)
            FROM team_daily_stats
            WHERE season_id = ?
              AND (
                  game_id IS NULL
                  OR game_date IS NULL
                  OR team_id IS NULL
                  OR games_played IS NULL
                  OR games_before IS NULL
                  OR wins_before IS NULL
              )
        """, [SEASON_ID]).fetchone()[0]

        require(n == 0, f"{n} rows have null key fields")

    check(9, "No null target key fields", test_09)

    # --------------------------------------------------------
    # 10 — games_before = games_played - 1
    # --------------------------------------------------------
    def test_10():
        bad = db.execute("""
            SELECT game_id, team_id,
                   games_before, games_played
            FROM team_daily_stats
            WHERE season_id = ?
              AND games_before <> games_played - 1
            LIMIT 10
        """, [SEASON_ID]).df()

        require(
            bad.empty,
            f"Inconsistent game counts:\n{bad}"
        )

    check(10, "Games-before arithmetic", test_10)

    # --------------------------------------------------------
    # 11 — Games played progress sequentially per team
    # --------------------------------------------------------
    def test_11():
        bad = db.execute("""
            WITH ordered AS (
                SELECT
                    game_id,
                    team_id,
                    game_date,
                    games_played,
                    ROW_NUMBER() OVER (
                        PARTITION BY team_id
                        ORDER BY game_date, game_id
                    ) AS expected_gp
                FROM team_daily_stats
                WHERE season_id = ?
            )
            SELECT *
            FROM ordered
            WHERE games_played <> expected_gp
            LIMIT 10
        """, [SEASON_ID]).df()

        require(
            bad.empty,
            f"Nonsequential games_played:\n{bad}"
        )

    check(11, "Sequential games-played by team", test_11)

    # --------------------------------------------------------
    # 12 — Wins before equals cumulative prior wins
    # --------------------------------------------------------
    def test_12():
        bad = db.execute("""
            WITH ordered AS (
                SELECT
                    game_id,
                    team_id,
                    game_date,
                    wins_before,
                    COALESCE(
                        SUM(game_win) OVER (
                            PARTITION BY team_id
                            ORDER BY game_date, game_id
                            ROWS BETWEEN UNBOUNDED PRECEDING
                                     AND 1 PRECEDING
                        ),
                        0
                    ) AS expected_wins_before
                FROM team_daily_stats
                WHERE season_id = ?
            )
            SELECT *
            FROM ordered
            WHERE wins_before <> expected_wins_before
            LIMIT 10
        """, [SEASON_ID]).df()

        require(
            bad.empty,
            f"Incorrect cumulative wins:\n{bad}"
        )

    check(12, "Wins-before cumulative reconciliation", test_12)

    # --------------------------------------------------------
    # 13 — game_win is binary
    # --------------------------------------------------------
    def test_13():
        n = db.execute("""
            SELECT COUNT(*)
            FROM team_daily_stats
            WHERE season_id = ?
              AND (
                  game_win IS NULL
                  OR game_win NOT IN (0, 1)
              )
        """, [SEASON_ID]).fetchone()[0]

        require(n == 0, f"{n} invalid game_win values")

    check(13, "Binary game-win values", test_13)

    # --------------------------------------------------------
    # 14 — Each game has exactly one winner
    # --------------------------------------------------------
    def test_14():
        bad = db.execute("""
            SELECT game_id, SUM(game_win) AS wins
            FROM team_daily_stats
            WHERE season_id = ?
            GROUP BY game_id
            HAVING SUM(game_win) <> 1
            LIMIT 10
        """, [SEASON_ID]).df()

        require(
            bad.empty,
            f"Games without exactly one winner:\n{bad}"
        )

    check(14, "One winner per completed game", test_14)

    # --------------------------------------------------------
    # 15 — Historical output schema matches DATA_SCHEMA
    # --------------------------------------------------------
    def test_15():
        target_cols = set(
            db.execute("DESCRIBE team_daily_stats")
              .df()["column_name"]
        )

        missing = set(DATA_SCHEMA) - target_cols

        require(
            not missing,
            f"Missing model feature columns: {sorted(missing)}"
        )

        return f"{len(DATA_SCHEMA)} required columns checked"

    check(15, "DATA_SCHEMA compatibility", test_15)

    # --------------------------------------------------------
    # 16 — No negative cumulative counts
    # --------------------------------------------------------
    def test_16():
        target_cols = set(
            db.execute("DESCRIBE team_daily_stats")
              .df()["column_name"]
        )

        candidates = [
            "c_SCF",
            "c_SCA",
            "c_corsi_for",
            "c_corsi_against",
            "c_shots_for",
            "c_shots_against",
        ]

        cols = [c for c in candidates if c in target_cols]

        require(cols, "No cumulative columns available")

        conditions = " OR ".join(
            f'"{c}" < 0' for c in cols
        )

        bad = db.execute(f"""
            SELECT game_id, team_id, {", ".join(cols)}
            FROM team_daily_stats
            WHERE season_id = ?
              AND ({conditions})
            LIMIT 10
        """, [SEASON_ID]).df()

        require(
            bad.empty,
            f"Negative cumulative statistics:\n{bad}"
        )

        return f"{len(cols)} cumulative columns checked"

    check(16, "Nonnegative cumulative statistics", test_16)

    # --------------------------------------------------------
    # 17 — Cumulative statistics never decrease
    # --------------------------------------------------------
    def test_17():
        target_cols = set(
            db.execute("DESCRIBE team_daily_stats")
              .df()["column_name"]
        )

        candidates = [
            "c_SCF",
            "c_SCA",
            "c_corsi_for",
            "c_corsi_against",
            "c_shots_for",
            "c_shots_against",
        ]

        cols = [c for c in candidates if c in target_cols]

        require(cols, "No cumulative columns available")

        lagged = ",\n".join(
            f"""
            LAG("{c}") OVER (
                PARTITION BY team_id
                ORDER BY game_date, game_id
            ) AS prev_{c}
            """
            for c in cols
        )

        comparisons = " OR ".join(
            f'("{c}" < prev_{c})'
            for c in cols
        )

        bad = db.execute(f"""
            WITH ordered AS (
                SELECT
                    game_id,
                    team_id,
                    {", ".join(f'"{c}"' for c in cols)},
                    {lagged}
                FROM team_daily_stats
                WHERE season_id = ?
            )
            SELECT game_id, team_id
            FROM ordered
            WHERE {comparisons}
            LIMIT 10
        """, [SEASON_ID]).df()

        require(
            bad.empty,
            f"Cumulative statistics decreased:\n{bad}"
        )

        return f"{len(cols)} cumulative columns checked"

    check(17, "Cumulative statistics are monotonic", test_17)

    # --------------------------------------------------------
    # 18 — Report coverage and latest stored game
    # --------------------------------------------------------
    def test_18():
        row = db.execute("""
            SELECT
                COUNT(*) AS team_rows,
                COUNT(DISTINCT game_id) AS games,
                COUNT(DISTINCT team_id) AS teams,
                MAX(game_date) AS latest_date
            FROM team_daily_stats
            WHERE season_id = ?
        """, [SEASON_ID]).fetchone()

        require(row[0] > 0, "No stored team-game records")

        return (
            f"{row[0]:,} rows | "
            f"{row[1]:,} games | "
            f"{row[2]} teams | "
            f"latest {row[3]}"
        )

    check(18, "Season coverage summary", test_18)

finally:
    if own_connection:
        db.close()

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

summary_50b = pd.DataFrame(results)

passed = int((summary_50b["Status"] == "PASS").sum())
failed = int((summary_50b["Status"] == "FAIL").sum())

print("\n" + "=" * 65)
print("BLOCK 50B — LIVE DATA RECONCILIATION")
print("=" * 65)
print(f"TOTAL:  {len(summary_50b)}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")

if failed:
    print("\nFAILED TESTS:")
    print(
        summary_50b.loc[
            summary_50b["Status"] == "FAIL",
            ["Test", "Description", "Details"]
        ].to_string(index=False)
    )
else:
    print("\nAll read-only reconciliation checks passed.")

print("=" * 65)

summary_50b


In [ ]:

# ============================================================
# BLOCK 51A — FEATURE ENGINEERING VALIDATION
# Read-only | Existing DuckDB data | No model changes
# ============================================================

import importlib
import warnings
import traceback

import duckdb
import numpy as np
import pandas as pd

import feature_engineering as fe

fe = importlib.reload(fe)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

SEASON_ID = 20252026
DB_PATH = None  # Uses existing notebook connection "con"

results = []
artifacts = {}

def check(number, description, fn):
    try:
        detail = fn()
        status = "PASS"
        message = "" if detail is None else str(detail)
        print(f"✅ {number:02d} PASS | {description}")
        if message:
            print(f"   {message}")
    except AssertionError as exc:
        status = "FAIL"
        message = str(exc)
        print(f"❌ {number:02d} FAIL | {description}")
        print(f"   {message}")
    except Exception as exc:
        status = "ERROR"
        message = f"{type(exc).__name__}: {exc}"
        print(f"❌ {number:02d} ERROR | {description}")
        print(f"   {message}")

    results.append({
        "Test": number,
        "Description": description,
        "Status": status,
        "Details": message,
    })

def require(condition, message):
    if not bool(condition):
        raise AssertionError(message)

def numeric(series):
    return pd.to_numeric(series, errors="coerce")

def same_values(a, b, atol=1e-9):
    aa = numeric(a).to_numpy(dtype=float, na_value=np.nan)
    bb = numeric(b).to_numpy(dtype=float, na_value=np.nan)
    return np.allclose(aa, bb, rtol=1e-9, atol=atol, equal_nan=True)

def compare_feature(left, right, column, keys):
    merged = left[keys + [column]].merge(
        right[keys + [column]],
        on=keys,
        how="inner",
        suffixes=("_full", "_partial"),
        validate="one_to_one",
    )
    require(len(merged) > 0, f"No comparable rows for {column}")

    a = numeric(merged[f"{column}_full"])
    b = numeric(merged[f"{column}_partial"])

    different = ~np.isclose(
        a.to_numpy(dtype=float, na_value=np.nan),
        b.to_numpy(dtype=float, na_value=np.nan),
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )

    return int(different.sum()), len(merged)

# ------------------------------------------------------------
# CONNECTION
# ------------------------------------------------------------

existing_con = globals().get("con")
own_connection = False

if DB_PATH is not None:
    db = duckdb.connect(str(DB_PATH), read_only=True)
    own_connection = True
elif existing_con is not None:
    db = existing_con
    print("Using existing notebook connection.")
else:
    raise RuntimeError(
        "No existing DuckDB connection 'con'. "
        "Set DB_PATH to your DuckDB file."
    )

try:
    # --------------------------------------------------------
    # LOAD REAL HISTORICAL DATA
    # --------------------------------------------------------

    raw = db.execute("""
        SELECT *
        FROM team_daily_stats
        WHERE season_id = ?
        ORDER BY game_date, game_id, team_id
    """, [SEASON_ID]).df()

    require(not raw.empty, "No team_daily_stats records for season")

    raw["game_date"] = pd.to_datetime(raw["game_date"])

    print(
        f"Loaded {len(raw):,} team-game rows | "
        f"{raw['team_id'].nunique()} teams | "
        f"{raw['game_date'].nunique()} dates"
    )

    # --------------------------------------------------------
    # 01 — Required functions
    # --------------------------------------------------------

    def test_01():
        names = [
            "normalize_made_playoffs",
            "add_conference_baseline",
            "add_streak_features",
            "add_rank_bucket",
            "add_wc_field_pressure_features",
            "add_wc_density_features",
            "add_special_teams_features",
            "add_season_context",
            "add_interaction_features",
            "add_rush_features",
            "add_rush_interactions",
            "add_corsi_features",
            "add_corsi_relative_features",
            "add_corsi_interactions",
            "add_shot_quality_features",
            "add_shot_relative_features",
            "add_shot_interactions",
            "add_standings_features",
            "add_standings_interactions",
            "build_features",
        ]

        missing = [
            name for name in names
            if not callable(getattr(fe, name, None))
        ]

        require(not missing, f"Missing functions: {missing}")
        return f"{len(names)} functions available"

    check(1, "Feature-engineering functions import", test_01)

    # --------------------------------------------------------
    # 02 — Required source columns
    # --------------------------------------------------------

    def test_02():
        required = [
            "season_id", "game_id", "game_date", "team_id",
            "tri_code", "conference", "division_rank",
            "conference_rank", "wildcard_rank",
            "points", "points_pct",
            "games_played", "games_remaining",
        ]

        missing = [c for c in required if c not in raw.columns]

        require(not missing, f"Missing source columns: {missing}")
        return f"{len(required)} required source columns"

    check(2, "Feature input contract", test_02)

    # --------------------------------------------------------
    # 03 — Build all historical features
    # --------------------------------------------------------

    def test_03():
        built = fe.build_features(raw.copy())
        artifacts["full"] = built

        require(
            len(built) == len(raw),
            f"Row count changed: {len(raw)} -> {len(built)}"
        )

        require(
            not built.duplicated(
                ["season_id", "game_id", "team_id"]
            ).any(),
            "Duplicate team-game keys"
        )

        return (
            f"{len(built):,} rows | "
            f"{len(built.columns)} columns"
        )

    check(3, "Full historical feature build", test_03)

    # --------------------------------------------------------
    # 04 — Original keys preserved
    # --------------------------------------------------------

    def test_04():
        built = artifacts["full"]

        keys = ["season_id", "game_id", "team_id"]

        a = raw[keys].sort_values(keys).reset_index(drop=True)
        b = built[keys].sort_values(keys).reset_index(drop=True)

        require(a.equals(b), "Original team-game keys changed")

    check(4, "Team-game identity preserved", test_04)

    # --------------------------------------------------------
    # 05 — Conference baseline sanity
    # --------------------------------------------------------

    def test_05():
        built = artifacts["full"]

        baseline = numeric(
            built["conf_playoff_chance_baseline"]
        )

        require(
            baseline.notna().all(),
            "Null conference baseline values"
        )

        require(
            baseline.between(0, 1).all(),
            "Conference baseline outside [0, 1]"
        )

        return (
            f"min={baseline.min():.4f}, "
            f"max={baseline.max():.4f}"
        )

    check(5, "Conference baseline validity", test_05)

    # --------------------------------------------------------
    # 06 — Wild-card cutline arithmetic
    # --------------------------------------------------------

    def test_06():
        built = artifacts["full"]

        eligible = built[
            built["division_rank"] > 3
        ].copy()

        require(not eligible.empty, "No WC-eligible rows")

        valid = eligible[
            eligible["wc2_points"].notna()
        ]

        require(not valid.empty, "No valid WC2 cutline rows")

        expected = (
            numeric(valid["points"])
            - numeric(valid["wc2_points"])
        )

        require(
            same_values(valid["wc_points_margin"], expected),
            "WC point margin calculation mismatch"
        )

        return f"{len(valid):,} eligible rows checked"

    check(6, "Wild-card margin arithmetic", test_06)

    # --------------------------------------------------------
    # 07 — Wild-card density
    # --------------------------------------------------------

    def test_07():
        built = artifacts["full"]

        for w in (2, 4, 6):
            col = f"wc_teams_within_{w}pts"

            require(col in built.columns, f"Missing {col}")

            values = numeric(built[col])

            require(
                values.notna().all() and (values >= 0).all(),
                f"Invalid density values in {col}"
            )

        return "2-, 4-, and 6-point windows checked"

    check(7, "Wild-card density feature validity", test_07)

    # --------------------------------------------------------
    # 08 — Season percentage bounds
    # --------------------------------------------------------

    def test_08():
        built = artifacts["full"]

        season_pct = numeric(built["season_pct"])

        require(
            season_pct.notna().all(),
            "Null season_pct values"
        )

        require(
            season_pct.between(0, 1).all(),
            (
                "season_pct outside [0,1]: "
                f"min={season_pct.min()}, max={season_pct.max()}"
            )
        )

        return (
            f"min={season_pct.min():.4f}, "
            f"max={season_pct.max():.4f}"
        )

    check(8, "Season-progress feature bounds", test_08)

    # --------------------------------------------------------
    # 09 — Season percentage denominator
    # --------------------------------------------------------

    def test_09():
        built = artifacts["full"]

        gp = numeric(raw["games_played"])
        remaining = numeric(raw["games_remaining"])

        standard = gp / (gp + remaining).replace(0, np.nan)
        actual = numeric(built["season_pct"])

        mismatch = ~np.isclose(
            actual.to_numpy(dtype=float, na_value=np.nan),
            standard.to_numpy(dtype=float, na_value=np.nan),
            rtol=1e-9,
            atol=1e-9,
            equal_nan=True,
        )

        artifacts["season_pct_mismatches"] = int(mismatch.sum())

        # Report rather than assume the source definition.
        return (
            f"{int(mismatch.sum()):,}/{len(raw):,} rows differ "
            "from gp/(gp+remaining); verify games_remaining semantics"
        )

    check(9, "Season-progress denominator diagnostic", test_09)

    # --------------------------------------------------------
    # 10 — Finite engineered numeric features
    # --------------------------------------------------------

    def test_10():
        built = artifacts["full"]

        new_cols = [
            c for c in built.columns
            if c not in raw.columns
        ]

        numeric_cols = [
            c for c in new_cols
            if pd.api.types.is_numeric_dtype(built[c])
        ]

        require(numeric_cols, "No new numeric features")

        values = built[numeric_cols].apply(
            pd.to_numeric, errors="coerce"
        ).to_numpy(dtype=float, na_value=np.nan)

        inf_count = int(np.isinf(values).sum())

        require(
            inf_count == 0,
            f"{inf_count} infinite feature values"
        )

        artifacts["new_features"] = new_cols

        return f"{len(numeric_cols)} numeric engineered columns checked"

    check(10, "No infinite engineered feature values", test_10)

    # --------------------------------------------------------
    # 11 — Deterministic feature generation
    # --------------------------------------------------------

    def test_11():
        a = artifacts["full"]
        b = fe.build_features(raw.copy())

        require(
            a.columns.tolist() == b.columns.tolist(),
            "Column order changed between runs"
        )

        pd.testing.assert_frame_equal(
            a.reset_index(drop=True),
            b.reset_index(drop=True),
            check_dtype=False,
        )

    check(11, "Deterministic repeated feature build", test_11)

    # --------------------------------------------------------
    # 12 — Input DataFrame not mutated
    # --------------------------------------------------------

    def test_12():
        sample = raw.head(100).copy(deep=True)
        original = sample.copy(deep=True)

        fe.build_features(sample)

        pd.testing.assert_frame_equal(
            sample,
            original,
            check_dtype=True,
        )

    check(12, "Input DataFrame immutability", test_12)

    # --------------------------------------------------------
    # 13 — Conference baseline batch consistency
    # --------------------------------------------------------

    def test_13():
        full = artifacts["full"]

        latest = raw["game_date"].max()
        partial = raw[
            raw["game_date"] == latest
        ].copy()

        require(
            not partial.empty,
            "No latest-date records"
        )

        subset = fe.build_features(partial)

        keys = ["season_id", "game_id", "team_id"]

        n, total = compare_feature(
            full, subset,
            "conf_playoff_chance_baseline",
            keys,
        )

        require(
            n == 0,
            f"{n}/{total} same rows change baseline "
            "when scored as a latest-date batch"
        )

        return f"{total} rows compared"

    check(13, "Conference baseline batch invariance", test_13)

    # --------------------------------------------------------
    # 14 — WC cutline batch consistency
    # --------------------------------------------------------

    def test_14():
        full = artifacts["full"]

        latest = raw["game_date"].max()
        partial = raw[
            raw["game_date"] == latest
        ].copy()

        subset = fe.build_features(partial)

        keys = ["season_id", "game_id", "team_id"]

        n, total = compare_feature(
            full, subset,
            "wc_points_margin",
            keys,
        )

        require(
            n == 0,
            f"{n}/{total} WC margins change with batch composition"
        )

        return f"{total} rows compared"

    check(14, "Wild-card cutline batch invariance", test_14)

    # --------------------------------------------------------
    # 15 — Relative Corsi batch consistency
    # --------------------------------------------------------

    def test_15():
        full = artifacts["full"]

        latest = raw["game_date"].max()
        partial = raw[
            raw["game_date"] == latest
        ].copy()

        subset = fe.build_features(partial)

        keys = ["season_id", "game_id", "team_id"]

        n, total = compare_feature(
            full, subset,
            "corsi_net_rel",
            keys,
        )

        require(
            n == 0,
            f"{n}/{total} relative Corsi values change "
            "with batch composition"
        )

        return f"{total} rows compared"

    check(15, "Relative Corsi batch invariance", test_15)

    # --------------------------------------------------------
    # 16 — Relative shot-quality batch consistency
    # --------------------------------------------------------

    def test_16():
        full = artifacts["full"]

        latest = raw["game_date"].max()
        partial = raw[
            raw["game_date"] == latest
        ].copy()

        subset = fe.build_features(partial)

        keys = ["season_id", "game_id", "team_id"]

        n, total = compare_feature(
            full, subset,
            "danger_net_rel",
            keys,
        )

        require(
            n == 0,
            f"{n}/{total} relative danger values change "
            "with batch composition"
        )

        return f"{total} rows compared"

    check(16, "Relative shot-quality batch invariance", test_16)

    # --------------------------------------------------------
    # 17 — Streak feature bounds
    # --------------------------------------------------------

    def test_17():
        built = artifacts["full"]

        streak = numeric(built["streak_signed"])

        require(
            streak.notna().all(),
            "Null streak_signed values"
        )

        require(
            streak.between(-7, 7).all(),
            "Streak feature exceeds clipping bounds"
        )

    check(17, "Streak clipping bounds", test_17)

    # --------------------------------------------------------
    # 18 — Rank bucket domain
    # --------------------------------------------------------

    def test_18():
        built = artifacts["full"]

        allowed = {
            "tier1_elite",
            "tier2_strong",
            "tier3_bubble",
            "tier4_bottom",
        }

        unexpected = (
            set(built["rank_bucket"].dropna()) - allowed
        )

        require(
            not unexpected,
            f"Unexpected rank buckets: {unexpected}"
        )

    check(18, "Rank bucket values", test_18)

    # --------------------------------------------------------
    # 19 — Engineered feature columns unique
    # --------------------------------------------------------

    def test_19():
        built = artifacts["full"]

        require(
            built.columns.is_unique,
            "Duplicate engineered column names"
        )

    check(19, "Feature column uniqueness", test_19)

    # --------------------------------------------------------
    # 20 — Report output feature dimensions
    # --------------------------------------------------------

    def test_20():
        built = artifacts["full"]

        new_cols = [
            c for c in built.columns
            if c not in raw.columns
        ]

        require(
            len(new_cols) > 0,
            "No new feature columns"
        )

        return (
            f"{len(raw.columns)} input columns -> "
            f"{len(built.columns)} output columns | "
            f"{len(new_cols)} new columns"
        )

    check(20, "Feature dimension summary", test_20)

finally:
    if own_connection:
        db.close()

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

summary_51a = pd.DataFrame(results)

passed = int((summary_51a["Status"] == "PASS").sum())
failed = int((summary_51a["Status"] == "FAIL").sum())
errors = int((summary_51a["Status"] == "ERROR").sum())

print("\n" + "=" * 68)
print("BLOCK 51A — FEATURE ENGINEERING VALIDATION SUMMARY")
print("=" * 68)

print(f"TOTAL:  {len(summary_51a)}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")
print(f"ERRORS: {errors}")

if failed or errors:
    print("\nISSUES:")
    print(
        summary_51a.loc[
            summary_51a["Status"] != "PASS",
            ["Test", "Description", "Details"]
        ].to_string(index=False)
    )
else:
    print("\nAll validation checks passed.")

print("=" * 68)

summary_51a


In [ ]:

# ============================================================
# BLOCK 51B — FEATURE BATCH CONSISTENCY
# Read-only | No database or model modifications
# ============================================================

import importlib
import numpy as np
import pandas as pd
import duckdb

import feature_engineering as fe
fe = importlib.reload(fe)

SEASON_ID = 20252026
DB_PATH = None

results = []
diagnostics = {}

def check(n, label, fn):
    try:
        detail = fn()
        status = "PASS"
        message = str(detail) if detail is not None else ""
        print(f"✅ {n:02d} PASS | {label}")
        if message:
            print("  ", message)
    except AssertionError as exc:
        status = "FAIL"
        message = str(exc)
        print(f"❌ {n:02d} FAIL | {label}")
        print("  ", message)
    except Exception as exc:
        status = "ERROR"
        message = f"{type(exc).__name__}: {exc}"
        print(f"❌ {n:02d} ERROR | {label}")
        print("  ", message)

    results.append({
        "Test": n,
        "Description": label,
        "Status": status,
        "Details": message,
    })

def require(ok, message):
    if not bool(ok):
        raise AssertionError(message)

KEYS = ["season_id", "game_id", "team_id"]

def differences(full, partial, column):
    merged = full[KEYS + [column]].merge(
        partial[KEYS + [column]],
        on=KEYS,
        how="inner",
        suffixes=("_full", "_partial"),
        validate="one_to_one",
    )

    a = pd.to_numeric(
        merged[column + "_full"], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)

    b = pd.to_numeric(
        merged[column + "_partial"], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)

    changed = ~np.isclose(
        a, b, atol=1e-9, rtol=1e-9, equal_nan=True
    )

    return int(changed.sum()), len(merged)

existing_con = globals().get("con")
own_connection = False

if DB_PATH is not None:
    db = duckdb.connect(str(DB_PATH), read_only=True)
    own_connection = True
elif existing_con is not None:
    db = existing_con
else:
    raise RuntimeError("Provide existing 'con' or set DB_PATH.")

try:
    raw = db.execute("""
        SELECT *
        FROM team_daily_stats
        WHERE season_id = ?
        ORDER BY game_date, game_id, team_id
    """, [SEASON_ID]).df()

    require(not raw.empty, "No season data")

    raw["game_date"] = pd.to_datetime(raw["game_date"])

    full = fe.build_features(raw.copy())

    # Construct a deliberately incomplete scoring batch:
    # one team per conference on each date, where available.
    partial_raw = (
        raw.sort_values(
            ["game_date", "conference", "team_id", "game_id"]
        )
        .groupby(
            ["season_id", "game_date", "conference"],
            dropna=False
        )
        .head(1)
        .copy()
    )

    partial = fe.build_features(partial_raw)

    print(
        f"Full: {len(full):,} rows | "
        f"Partial: {len(partial):,} rows | "
        f"Dates: {raw['game_date'].nunique()}"
    )

    # --------------------------------------------------------
    # 01 — Same team-game rows retained
    # --------------------------------------------------------

    def test_01():
        require(
            len(partial) == len(partial_raw),
            "Partial build changed row count"
        )

        require(
            not partial.duplicated(KEYS).any(),
            "Duplicate partial keys"
        )

        return f"{len(partial)} retained team-game rows"

    check(1, "Partial-batch identity", test_01)

    # --------------------------------------------------------
    # 02 — Conference baseline independent of batch
    # --------------------------------------------------------

    def test_02():
        changed, total = differences(
            full, partial, "conf_playoff_chance_baseline"
        )

        require(
            changed == 0,
            f"{changed}/{total} baseline values changed"
        )

        return f"{total} records compared"

    check(2, "Conference baseline invariance", test_02)

    # --------------------------------------------------------
    # 03 — Wild-card points margin
    # --------------------------------------------------------

    def test_03():
        changed, total = differences(
            full, partial, "wc_points_margin"
        )

        diagnostics["wc_points_margin"] = (changed, total)

        require(
            changed == 0,
            f"{changed}/{total} margins changed"
        )

        return f"{total} records compared"

    check(3, "Wild-card points-margin invariance", test_03)

    # --------------------------------------------------------
    # 04 — Wild-card points percentage margin
    # --------------------------------------------------------

    def test_04():
        changed, total = differences(
            full, partial, "wc_points_pct_margin"
        )

        diagnostics["wc_points_pct_margin"] = (changed, total)

        require(
            changed == 0,
            f"{changed}/{total} margins changed"
        )

        return f"{total} records compared"

    check(4, "Wild-card percentage-margin invariance", test_04)

    # --------------------------------------------------------
    # 05 — Wild-card density
    # --------------------------------------------------------

    def test_05():
        failures = []

        for w in (2, 4, 6):
            col = f"wc_teams_within_{w}pts"
            changed, total = differences(full, partial, col)

            diagnostics[col] = (changed, total)

            if changed:
                failures.append(f"{col}: {changed}/{total}")

        require(
            not failures,
            "; ".join(failures)
        )

    check(5, "Wild-card density invariance", test_05)

    # --------------------------------------------------------
    # 06 — Relative Corsi net
    # --------------------------------------------------------

    def test_06():
        changed, total = differences(
            full, partial, "corsi_net_rel"
        )

        diagnostics["corsi_net_rel"] = (changed, total)

        require(
            changed == 0,
            f"{changed}/{total} values changed"
        )

        return f"{total} records compared"

    check(6, "Relative Corsi-net invariance", test_06)

    # --------------------------------------------------------
    # 07 — Relative Corsi ratio
    # --------------------------------------------------------

    def test_07():
        changed, total = differences(
            full, partial, "corsi_ratio_rel"
        )

        diagnostics["corsi_ratio_rel"] = (changed, total)

        require(
            changed == 0,
            f"{changed}/{total} values changed"
        )

        return f"{total} records compared"

    check(7, "Relative Corsi-ratio invariance", test_07)

    # --------------------------------------------------------
    # 08 — Relative danger
    # --------------------------------------------------------

    def test_08():
        changed, total = differences(
            full, partial, "danger_net_rel"
        )

        diagnostics["danger_net_rel"] = (changed, total)

        require(
            changed == 0,
            f"{changed}/{total} values changed"
        )

        return f"{total} records compared"

    check(8, "Relative danger invariance", test_08)

    # --------------------------------------------------------
    # 09 — Relative PDO
    # --------------------------------------------------------

    def test_09():
        changed, total = differences(
            full, partial, "pdo_dev_rel"
        )

        diagnostics["pdo_dev_rel"] = (changed, total)

        require(
            changed == 0,
            f"{changed}/{total} values changed"
        )

        return f"{total} records compared"

    check(9, "Relative PDO invariance", test_09)

    # --------------------------------------------------------
    # 10 — Row-local season progress
    # --------------------------------------------------------

    def test_10():
        changed, total = differences(
            full, partial, "season_pct"
        )

        require(
            changed == 0,
            f"{changed}/{total} season values changed"
        )

        return f"{total} records compared"

    check(10, "Season progress invariance", test_10)

    # --------------------------------------------------------
    # 11 — Row-local special teams
    # --------------------------------------------------------

    def test_11():
        changed, total = differences(
            full, partial, "special_teams_net"
        )

        require(
            changed == 0,
            f"{changed}/{total} special-teams values changed"
        )

        return f"{total} records compared"

    check(11, "Special-teams invariance", test_11)

    # --------------------------------------------------------
    # 12 — Row-local rush
    # --------------------------------------------------------

    def test_12():
        changed, total = differences(
            full, partial, "rush_net"
        )

        require(
            changed == 0,
            f"{changed}/{total} rush values changed"
        )

        return f"{total} records compared"

    check(12, "Rush-feature invariance", test_12)

    # --------------------------------------------------------
    # 13 — Output keys unchanged
    # --------------------------------------------------------

    def test_13():
        expected = partial_raw[KEYS].sort_values(
            KEYS
        ).reset_index(drop=True)

        actual = partial[KEYS].sort_values(
            KEYS
        ).reset_index(drop=True)

        pd.testing.assert_frame_equal(
            expected, actual, check_dtype=False
        )

    check(13, "Partial-batch key preservation", test_13)

    # --------------------------------------------------------
    # 14 — Engineered feature finiteness
    # --------------------------------------------------------

    def test_14():
        new_cols = [
            c for c in partial.columns
            if c not in raw.columns
        ]

        numeric_cols = [
            c for c in new_cols
            if pd.api.types.is_numeric_dtype(partial[c])
        ]

        vals = partial[numeric_cols].apply(
            pd.to_numeric, errors="coerce"
        ).to_numpy(dtype=float, na_value=np.nan)

        require(
            not np.isinf(vals).any(),
            "Infinite engineered values"
        )

        return f"{len(numeric_cols)} numeric columns checked"

    check(14, "Partial-batch feature finiteness", test_14)

finally:
    if own_connection:
        db.close()

summary_51b = pd.DataFrame(results)

passed = (summary_51b["Status"] == "PASS").sum()
failed = (summary_51b["Status"] == "FAIL").sum()
errors = (summary_51b["Status"] == "ERROR").sum()

print("\n" + "=" * 66)
print("BLOCK 51B — BATCH CONSISTENCY SUMMARY")
print("=" * 66)
print(f"TOTAL:  {len(summary_51b)}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")
print(f"ERRORS: {errors}")

if failed or errors:
    print("\nFEATURES REQUIRING INVESTIGATION:")
    print(
        summary_51b.loc[
            summary_51b["Status"] != "PASS",
            ["Test", "Description", "Details"]
        ].to_string(index=False)
    )

print("=" * 66)

summary_51b


In [ ]:

# ============================================================
# BLOCK 51C — MODEL FEATURE CONTRACT & INFERENCE CONSISTENCY
# Read-only | No model/database modifications
# ============================================================

import importlib
import numpy as np
import pandas as pd

from joblib import load
from config import MODEL_PATH

import feature_engineering as fe
fe = importlib.reload(fe)

# Import the module containing score_dataframe.
# Adjust this import only if your project uses a different path.
from prediction_model.inference.inference import (
    score_dataframe,
    FEATURE_COLS,
    XGB_PIPELINE,
    RF_PIPELINE,
)

SEASON_ID = 20252026

results = []
diagnostics = {}

def check(n, description, fn):
    try:
        detail = fn()
        status = "PASS"
        message = "" if detail is None else str(detail)
        print(f"✅ {n:02d} PASS | {description}")
        if message:
            print("   ", message)
    except AssertionError as exc:
        status = "FAIL"
        message = str(exc)
        print(f"❌ {n:02d} FAIL | {description}")
        print("   ", message)
    except Exception as exc:
        status = "ERROR"
        message = f"{type(exc).__name__}: {exc}"
        print(f"❌ {n:02d} ERROR | {description}")
        print("   ", message)

    results.append({
        "Test": n,
        "Description": description,
        "Status": status,
        "Details": message,
    })

def require(condition, message):
    if not bool(condition):
        raise AssertionError(message)

def compare_numeric(full, partial, column):
    keys = ["season_id", "game_id", "team_id"]

    merged = full[keys + [column]].merge(
        partial[keys + [column]],
        on=keys,
        how="inner",
        suffixes=("_full", "_partial"),
        validate="one_to_one",
    )

    a = pd.to_numeric(
        merged[column + "_full"], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)

    b = pd.to_numeric(
        merged[column + "_partial"], errors="coerce"
    ).to_numpy(dtype=float, na_value=np.nan)

    changed = ~np.isclose(
        a, b,
        atol=1e-9,
        rtol=1e-9,
        equal_nan=True,
    )

    return int(changed.sum()), len(merged)

# ------------------------------------------------------------
# LOAD MODEL CONTRACT
# ------------------------------------------------------------

bundle = load(MODEL_PATH)
model_features = list(bundle["features"])

BATCH_DEPENDENT = [
    "wc_points_margin",
    "wc_points_pct_margin",
    "wc_teams_within_2pts",
    "wc_teams_within_4pts",
    "wc_teams_within_6pts",
    "corsi_net_rel",
    "corsi_ratio_rel",
    "danger_net_rel",
    "pdo_dev_rel",
]

# ------------------------------------------------------------
# LOAD REAL DATA — READ ONLY
# ------------------------------------------------------------

if "con" not in globals():
    raise RuntimeError(
        "This cell requires your existing DuckDB connection 'con'."
    )

raw = con.execute("""
    SELECT *
    FROM team_daily_stats
    WHERE season_id = ?
    ORDER BY game_date, game_id, team_id
""", [SEASON_ID]).df()

require(not raw.empty, "No team_daily_stats data found")

raw["game_date"] = pd.to_datetime(raw["game_date"])

full_features = fe.build_features(raw.copy())

partial_raw = (
    raw.sort_values(
        ["game_date", "conference", "team_id", "game_id"]
    )
    .groupby(
        ["season_id", "game_date", "conference"],
        dropna=False
    )
    .head(1)
    .copy()
)

partial_features = fe.build_features(partial_raw)

print(
    f"Loaded {len(raw):,} team-game rows | "
    f"{len(model_features)} model features | "
    f"{len(partial_raw)} partial-batch rows"
)

# ------------------------------------------------------------
# 01 — Model artifact structure
# ------------------------------------------------------------

def test_01():
    required = {"xgb_pipeline", "rf_pipeline", "features"}

    missing = required - set(bundle)

    require(not missing, f"Missing bundle keys: {missing}")

    return f"{len(model_features)} model features"

check(1, "Model bundle structure", test_01)

# ------------------------------------------------------------
# 02 — Model feature names unique
# ------------------------------------------------------------

def test_02():
    require(
        len(model_features) == len(set(model_features)),
        "Duplicate model feature names"
    )

check(2, "Unique model feature names", test_02)

# ------------------------------------------------------------
# 03 — Inference module uses artifact feature list
# ------------------------------------------------------------

def test_03():
    require(
        list(FEATURE_COLS) == model_features,
        "Inference feature list differs from saved artifact"
    )

check(3, "Inference feature-list alignment", test_03)

# ------------------------------------------------------------
# 04 — XGB pipeline feature count
# ------------------------------------------------------------

def test_04():
    n = getattr(XGB_PIPELINE, "n_features_in_", None)

    require(
        n is not None,
        "XGB pipeline has no n_features_in_"
    )

    require(
        n == len(model_features),
        f"XGB expects {n}, artifact has {len(model_features)}"
    )

    return f"{n} features"

check(4, "XGBoost feature-count alignment", test_04)

# ------------------------------------------------------------
# 05 — RF pipeline feature count
# ------------------------------------------------------------

def test_05():
    n = getattr(RF_PIPELINE, "n_features_in_", None)

    require(
        n is not None,
        "RF pipeline has no n_features_in_"
    )

    require(
        n == len(model_features),
        f"RF expects {n}, artifact has {len(model_features)}"
    )

    return f"{n} features"

check(5, "Random Forest feature-count alignment", test_05)

# ------------------------------------------------------------
# 06 — Model-required features available
# ------------------------------------------------------------

def test_06():
    missing = [
        c for c in model_features
        if c not in full_features.columns
    ]

    require(
        not missing,
        f"Missing model features: {missing}"
    )

check(6, "Engineered feature coverage", test_06)

# ------------------------------------------------------------
# 07 — Model-required features numeric
# ------------------------------------------------------------

def test_07():
    nonnumeric = [
        c for c in model_features
        if not pd.api.types.is_numeric_dtype(
            full_features[c]
        )
    ]

    require(
        not nonnumeric,
        f"Nonnumeric model features: {nonnumeric}"
    )

check(7, "Numeric model input types", test_07)

# ------------------------------------------------------------
# 08 — Batch-dependent features used by model
# ------------------------------------------------------------

def test_08():
    used = [
        c for c in BATCH_DEPENDENT
        if c in model_features
    ]

    diagnostics["batch_dependent_used"] = used

    # Diagnostic: presence is important but not an error.
    return (
        f"{len(used)} of {len(BATCH_DEPENDENT)} affected features "
        f"used by model: {used}"
    )

check(8, "Batch-dependent model features diagnostic", test_08)

# ------------------------------------------------------------
# 09 — Complete-batch scoring
# ------------------------------------------------------------

def test_09():
    scored = score_dataframe(full_features.copy())

    require(
        len(scored) == len(full_features),
        "Scoring changed row count"
    )

    diagnostics["full_scored"] = scored

    return f"{len(scored):,} predictions generated"

check(9, "Complete-batch model scoring", test_09)

# ------------------------------------------------------------
# 10 — Partial-batch scoring
# ------------------------------------------------------------

def test_10():
    scored = score_dataframe(partial_features.copy())

    require(
        len(scored) == len(partial_features),
        "Partial scoring changed row count"
    )

    diagnostics["partial_scored"] = scored

    return f"{len(scored):,} predictions generated"

check(10, "Partial-batch model scoring", test_10)

# ------------------------------------------------------------
# 11 — Probability validity
# ------------------------------------------------------------

def test_11():
    for name in ("full_scored", "partial_scored"):
        scored = diagnostics[name]

        for col in [
            "playoff_prob_xgb",
            "playoff_prob_rf",
            "playoff_prob",
        ]:
            values = pd.to_numeric(
                scored[col], errors="coerce"
            )

            require(
                values.notna().all(),
                f"{name}: null values in {col}"
            )

            require(
                values.between(0, 1).all(),
                f"{name}: probabilities outside [0,1] in {col}"
            )

check(11, "Predicted probability bounds", test_11)

# ------------------------------------------------------------
# 12 — Identical records produce identical probabilities
# ------------------------------------------------------------

def test_12():
    full = diagnostics["full_scored"]
    partial = diagnostics["partial_scored"]

    failures = []

    for col in [
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob",
    ]:
        changed, total = compare_numeric(full, partial, col)

        diagnostics[f"{col}_changed"] = (changed, total)

        if changed:
            failures.append(f"{col}: {changed}/{total}")

    require(
        not failures,
        " | ".join(failures)
    )

    return "All matched predictions invariant"

check(12, "Prediction batch invariance", test_12)

# ------------------------------------------------------------
# 13 — Feature alignment is deterministic
# ------------------------------------------------------------

def test_13():
    scored_a = score_dataframe(
        full_features.head(50).copy()
    )
    scored_b = score_dataframe(
        full_features.head(50).copy()
    )

    for col in [
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob",
    ]:
        require(
            np.allclose(
                scored_a[col],
                scored_b[col],
                atol=1e-12,
                rtol=1e-12,
            ),
            f"Nondeterministic {col}"
        )

check(13, "Deterministic model inference", test_13)

# ------------------------------------------------------------
# 14 — Explicit feature ordering
# ------------------------------------------------------------

def test_14():
    X = (
        full_features
        .drop(
            columns=[
                "game_id", "game_date", "date",
                "season_id", "team_name", "opponent",
                "clinch_type", "made_playoffs",
                "team_id", "opponent_id", "tri_code",
            ],
            errors="ignore",
        )
        .select_dtypes(include="number")
        .fillna(0)
    )

    missing = [
        c for c in model_features
        if c not in X.columns
    ]

    require(not missing, f"Missing aligned features: {missing}")

    X = X.reindex(columns=model_features)

    require(
        list(X.columns) == model_features,
        "Feature ordering mismatch"
    )

check(14, "Model input feature ordering", test_14)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary_51c = pd.DataFrame(results)

passed = int((summary_51c["Status"] == "PASS").sum())
failed = int((summary_51c["Status"] == "FAIL").sum())
errors = int((summary_51c["Status"] == "ERROR").sum())

print("\n" + "=" * 68)
print("BLOCK 51C — MODEL CONTRACT SUMMARY")
print("=" * 68)

print(f"TOTAL:  {len(summary_51c)}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")
print(f"ERRORS: {errors}")

if diagnostics.get("batch_dependent_used") is not None:
    print(
        "\nBatch-dependent features used:",
        diagnostics["batch_dependent_used"]
    )

for col in [
    "playoff_prob_xgb",
    "playoff_prob_rf",
    "playoff_prob",
]:
    key = f"{col}_changed"
    if key in diagnostics:
        changed, total = diagnostics[key]
        print(f"{col} changed: {changed}/{total}")

if failed or errors:
    print("\nFAILED CHECKS:")
    print(
        summary_51c.loc[
            summary_51c["Status"] != "PASS",
            ["Test", "Description", "Details"]
        ].to_string(index=False)
    )

print("=" * 68)

summary_51c


In [ ]:

# ============================================================
# BLOCK 52A — INFERENCE.PY VALIDATION
# Read-only | 20 tests | No database or model writes
# ============================================================

import importlib
import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal
from joblib import load

from config import MODEL_PATH

import feature_engineering as fe
import prediction_model.inference.inference as inf

fe = importlib.reload(fe)
inf = importlib.reload(inf)

SEASON_ID = 20252026

results_52a = []
diagnostics_52a = {}

def check(n, description, fn):
    try:
        detail = fn()
        status = "PASS"
        message = "" if detail is None else str(detail)
        print(f"✅ {n:02d} PASS | {description}")
        if message:
            print(f"   {message}")
    except AssertionError as exc:
        status = "FAIL"
        message = str(exc)
        print(f"❌ {n:02d} FAIL | {description}")
        print(f"   {message}")
    except Exception as exc:
        status = "ERROR"
        message = f"{type(exc).__name__}: {exc}"
        print(f"❌ {n:02d} ERROR | {description}")
        print(f"   {message}")

    results_52a.append({
        "Test": n,
        "Description": description,
        "Status": status,
        "Details": message,
    })

def require(condition, message):
    if not bool(condition):
        raise AssertionError(message)

def model_input(df):
    """Replicate inference.py's feature-selection contract."""
    X = (
        df.drop(
            columns=inf.LEAK_COLS + inf.ID_COLS,
            errors="ignore"
        )
        .select_dtypes(include="number")
        .copy()
    )
    return X.fillna(0).reindex(columns=inf.FEATURE_COLS)

def expect_error(fn, exception_type, contains=None):
    try:
        fn()
    except exception_type as exc:
        if contains is not None:
            require(
                contains.lower() in str(exc).lower(),
                f"Unexpected error message: {exc}"
            )
        return f"Correctly raised {exception_type.__name__}"
    except Exception as exc:
        raise AssertionError(
            f"Expected {exception_type.__name__}, "
            f"got {type(exc).__name__}: {exc}"
        )
    raise AssertionError(
        f"Expected {exception_type.__name__}, but no error occurred"
    )

# ------------------------------------------------------------
# LOAD DATA AND ENGINEER FEATURES
# ------------------------------------------------------------

if "con" not in globals():
    raise RuntimeError(
        "An existing DuckDB connection named 'con' is required."
    )

raw_52a = con.execute("""
    SELECT *
    FROM team_daily_stats
    WHERE season_id = ?
    ORDER BY game_date, game_id, team_id
""", [SEASON_ID]).df()

require(
    not raw_52a.empty,
    f"No team_daily_stats rows for {SEASON_ID}"
)

features_52a = fe.build_features(raw_52a.copy())

# Use a representative subset for faster repeated scoring.
sample_52a = features_52a.head(100).copy(deep=True)

bundle_52a = load(MODEL_PATH)
feature_cols_52a = list(inf.FEATURE_COLS)

print("=" * 68)
print("BLOCK 52A — INFERENCE VALIDATION")
print("=" * 68)
print(
    f"Source rows: {len(raw_52a):,} | "
    f"Sample rows: {len(sample_52a):,} | "
    f"Model features: {len(feature_cols_52a)}"
)

# ------------------------------------------------------------
# 01 — Artifact keys
# ------------------------------------------------------------

def test_01():
    required = {"xgb_pipeline", "rf_pipeline", "features"}
    missing = required - set(bundle_52a)
    require(not missing, f"Missing artifact keys: {missing}")
    return "Required model objects present"

check(1, "Model artifact structure", test_01)

# ------------------------------------------------------------
# 02 — Feature list consistency
# ------------------------------------------------------------

def test_02():
    require(
        feature_cols_52a == list(bundle_52a["features"]),
        "Inference feature list differs from artifact"
    )
    require(
        len(feature_cols_52a) == len(set(feature_cols_52a)),
        "Duplicate feature names"
    )
    return f"{len(feature_cols_52a)} unique features"

check(2, "Saved feature contract", test_02)

# ------------------------------------------------------------
# 03 — Both models support probability inference
# ------------------------------------------------------------

def test_03():
    for name, pipeline in [
        ("XGBoost", inf.XGB_PIPELINE),
        ("Random Forest", inf.RF_PIPELINE),
    ]:
        require(
            callable(getattr(pipeline, "predict_proba", None)),
            f"{name} does not expose predict_proba"
        )
    return "Both pipelines support predict_proba"

check(3, "Model probability interfaces", test_03)

# ------------------------------------------------------------
# 04 — All required input features exist
# ------------------------------------------------------------

def test_04():
    X = model_input(sample_52a)
    missing = [
        c for c in feature_cols_52a
        if c not in sample_52a.columns
    ]
    require(not missing, f"Missing features: {missing}")
    require(
        X.shape == (len(sample_52a), len(feature_cols_52a)),
        f"Unexpected feature matrix shape: {X.shape}"
    )
    return f"Feature matrix: {X.shape}"

check(4, "Required feature availability", test_04)

# ------------------------------------------------------------
# 05 — Model feature order
# ------------------------------------------------------------

def test_05():
    X = model_input(sample_52a)
    require(
        list(X.columns) == feature_cols_52a,
        "Incorrect model feature ordering"
    )
    return "Exact saved feature order"

check(5, "Feature alignment", test_05)

# ------------------------------------------------------------
# 06 — Missing required feature rejected
# ------------------------------------------------------------

def test_06():
    missing_col = feature_cols_52a[0]
    bad = sample_52a.drop(columns=[missing_col])
    return expect_error(
        lambda: inf.score_dataframe(bad),
        ValueError,
        "Missing required model features"
    )

check(6, "Missing-feature rejection", test_06)

# ------------------------------------------------------------
# 07 — Extra numeric feature ignored
# ------------------------------------------------------------

def test_07():
    base = inf.score_dataframe(sample_52a)
    extra = sample_52a.copy()
    extra["unexpected_numeric_feature_52a"] = 999999.0

    scored = inf.score_dataframe(extra)

    for col in [
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob",
    ]:
        require(
            np.allclose(
                base[col].to_numpy(),
                scored[col].to_numpy(),
                atol=1e-12,
                rtol=1e-12
            ),
            f"Extra feature changed {col}"
        )

    return "Extra numeric column does not affect predictions"

check(7, "Extra-feature isolation", test_07)

# ------------------------------------------------------------
# 08 — Identifier and leakage columns excluded
# ------------------------------------------------------------

def test_08():
    X = model_input(sample_52a)

    excluded = set(inf.LEAK_COLS + inf.ID_COLS)
    present = excluded.intersection(X.columns)

    require(
        not present,
        f"Excluded columns entered model matrix: {present}"
    )

    return "Explicitly excluded columns absent"

check(8, "Identifier and leakage exclusion", test_08)

# ------------------------------------------------------------
# 09 — Input DataFrame unchanged
# ------------------------------------------------------------

def test_09():
    original = sample_52a.copy(deep=True)
    before = sample_52a.copy(deep=True)

    inf.score_dataframe(original)

    assert_frame_equal(
        original,
        before,
        check_dtype=True,
        check_exact=True
    )

    return "Input remains unchanged"

check(9, "Input immutability", test_09)

# ------------------------------------------------------------
# 10 — Output row count preserved
# ------------------------------------------------------------

def test_10():
    scored = inf.score_dataframe(sample_52a)
    diagnostics_52a["scored"] = scored

    require(
        len(scored) == len(sample_52a),
        "Scoring changed row count"
    )

    return f"{len(scored)} rows returned"

check(10, "Output row preservation", test_10)

# ------------------------------------------------------------
# 11 — Original columns preserved
# ------------------------------------------------------------

def test_11():
    scored = diagnostics_52a["scored"]

    missing = [
        c for c in sample_52a.columns
        if c not in scored.columns
    ]

    require(
        not missing,
        f"Original columns removed: {missing}"
    )

    return f"{len(sample_52a.columns)} source columns preserved"

check(11, "Output column preservation", test_11)

# ------------------------------------------------------------
# 12 — All probability columns produced
# ------------------------------------------------------------

def test_12():
    scored = diagnostics_52a["scored"]

    expected = {
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob",
    }

    require(
        expected.issubset(scored.columns),
        f"Missing prediction columns: {expected - set(scored.columns)}"
    )

    return "XGB, RF, and ensemble probabilities present"

check(12, "Probability output schema", test_12)

# ------------------------------------------------------------
# 13 — Finite, bounded probabilities
# ------------------------------------------------------------

def test_13():
    scored = diagnostics_52a["scored"]

    for col in [
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob",
    ]:
        values = scored[col].to_numpy(dtype=float)

        require(
            np.isfinite(values).all(),
            f"{col} contains nonfinite values"
        )

        require(
            ((values >= 0) & (values <= 1)).all(),
            f"{col} contains values outside [0, 1]"
        )

    return "All probabilities finite and within [0,1]"

check(13, "Probability validity", test_13)

# ------------------------------------------------------------
# 14 — XGBoost predictions match direct model call
# ------------------------------------------------------------

def test_14():
    X = model_input(sample_52a)

    expected = inf.XGB_PIPELINE.predict_proba(X)[:, 1]
    actual = diagnostics_52a["scored"]["playoff_prob_xgb"]

    require(
        np.allclose(actual, expected, atol=1e-12, rtol=1e-12),
        "XGBoost probability mismatch"
    )

    return "Matches direct XGBoost inference"

check(14, "XGBoost probability calculation", test_14)

# ------------------------------------------------------------
# 15 — RF predictions match direct model call
# ------------------------------------------------------------

def test_15():
    X = model_input(sample_52a)

    expected = inf.RF_PIPELINE.predict_proba(X)[:, 1]
    actual = diagnostics_52a["scored"]["playoff_prob_rf"]

    require(
        np.allclose(actual, expected, atol=1e-12, rtol=1e-12),
        "Random Forest probability mismatch"
    )

    return "Matches direct Random Forest inference"

check(15, "Random Forest probability calculation", test_15)

# ------------------------------------------------------------
# 16 — Ensemble equals arithmetic mean
# ------------------------------------------------------------

def test_16():
    scored = diagnostics_52a["scored"]

    expected = (
        scored["playoff_prob_xgb"]
        + scored["playoff_prob_rf"]
    ) / 2

    require(
        np.allclose(
            scored["playoff_prob"],
            expected,
            atol=1e-12,
            rtol=1e-12
        ),
        "Ensemble does not equal XGB/RF mean"
    )

    return "Ensemble = (XGB + RF) / 2"

check(16, "Ensemble calculation", test_16)

# ------------------------------------------------------------
# 17 — Deterministic repeated scoring
# ------------------------------------------------------------

def test_17():
    a = inf.score_dataframe(sample_52a)
    b = inf.score_dataframe(sample_52a)

    for col in [
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob",
    ]:
       require(
        np.allclose(
            a[col].to_numpy(),
            b[col].to_numpy(),
            atol=1e-12,
            rtol=1e-12,
        ),
        f"Repeated scoring changed {col} beyond tolerance"
    )
    return "Repeated calls produce identical predictions"

check(17, "Inference determinism", test_17)

# ------------------------------------------------------------
# 18 — Missing numeric values handled consistently
# ------------------------------------------------------------

def test_18():
    col = feature_cols_52a[0]

    with_nan = sample_52a.head(5).copy()
    with_zero = sample_52a.head(5).copy()

    with_nan[col] = np.nan
    with_zero[col] = 0.0

    a = inf.score_dataframe(with_nan)
    b = inf.score_dataframe(with_zero)

    for pcol in [
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob",
    ]:
        require(
            np.allclose(
                a[pcol],
                b[pcol],
                atol=1e-12,
                rtol=1e-12
            ),
            f"NaN handling differs from zero for {pcol}"
        )

    return f"NaN in {col} treated as zero"

check(18, "Missing-value behavior", test_18)

# ------------------------------------------------------------
# 19 — Infinite numeric values rejected
# ------------------------------------------------------------

def test_19():
    bad = sample_52a.head(5).copy()
    bad[feature_cols_52a[0]] = np.inf

    # This is a desired production safety behavior.
    # Current implementation may fail this check.
    try:
        scored = inf.score_dataframe(bad)
    except (ValueError, OverflowError) as exc:
        return f"Nonfinite input rejected: {type(exc).__name__}"

    values = scored[
        ["playoff_prob_xgb", "playoff_prob_rf", "playoff_prob"]
    ].to_numpy(dtype=float)

    require(
        False,
        "Infinite model input was accepted without explicit validation"
    )

check(19, "Nonfinite-input rejection", test_19)

# ------------------------------------------------------------
# 20 — Empty input handling
# ------------------------------------------------------------

def test_20():
    empty = sample_52a.iloc[0:0].copy()

    try:
        scored = inf.score_dataframe(empty)
    except ValueError as exc:
        return f"Empty input rejected: {type(exc).__name__}"

    require(
        scored.empty,
        "Empty input produced nonempty predictions"
    )

    return "Empty input returned an empty DataFrame"

check(20, "Empty-input behavior", test_20)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary_52a = pd.DataFrame(results_52a)

passed = int((summary_52a["Status"] == "PASS").sum())
failed = int((summary_52a["Status"] == "FAIL").sum())
errors = int((summary_52a["Status"] == "ERROR").sum())

print("\n" + "=" * 68)
print("BLOCK 52A — INFERENCE VALIDATION SUMMARY")
print("=" * 68)

print(f"TOTAL:  {len(summary_52a)}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")
print(f"ERRORS: {errors}")

if failed or errors:
    print("\nCHECKS REQUIRING ATTENTION:")
    print(
        summary_52a.loc[
            summary_52a["Status"] != "PASS",
            ["Test", "Description", "Details"]
        ].to_string(index=False)
    )

print("=" * 68)

summary_52a


In [ ]:

# ============================================================
# BLOCK 52B — RANDOM FOREST DETERMINISM DIAGNOSTIC
# Read-only | Does not modify saved model or database
# ============================================================

import copy
import numpy as np
import pandas as pd

# Reuse variables from Block 52A:
# inf, sample_52a, model_input

X = model_input(sample_52a)

results_52b = []

def report(name, a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    exact = np.array_equal(a, b)
    abs_diff = np.abs(a - b)
    max_diff = float(np.max(abs_diff))
    mean_diff = float(np.mean(abs_diff))
    changed = int(np.count_nonzero(a != b))

    print(f"\n{name}")
    print(f"  Exact equality: {exact}")
    print(f"  Changed rows:   {changed}/{len(a)}")
    print(f"  Max difference: {max_diff:.15g}")
    print(f"  Mean difference:{mean_diff:.15g}")
    print(
        "  Close at 1e-12:",
        np.allclose(a, b, atol=1e-12, rtol=1e-12)
    )
    print(
        "  Close at 1e-9: ",
        np.allclose(a, b, atol=1e-9, rtol=1e-9)
    )

    results_52b.append({
        "Test": name,
        "Exact": exact,
        "Changed": changed,
        "MaxDifference": max_diff,
        "MeanDifference": mean_diff,
        "Close_1e12": bool(
            np.allclose(a, b, atol=1e-12, rtol=1e-12)
        ),
        "Close_1e9": bool(
            np.allclose(a, b, atol=1e-9, rtol=1e-9)
        ),
    })

# ------------------------------------------------------------
# 01 — Direct Random Forest repeatability
# ------------------------------------------------------------

rf_a = inf.RF_PIPELINE.predict_proba(X)[:, 1]
rf_b = inf.RF_PIPELINE.predict_proba(X)[:, 1]

report("01 — Direct RF repeated predictions", rf_a, rf_b)

# ------------------------------------------------------------
# 02 — Full inference repeatability
# ------------------------------------------------------------

score_a = inf.score_dataframe(sample_52a)
score_b = inf.score_dataframe(sample_52a)

report(
    "02 — score_dataframe RF predictions",
    score_a["playoff_prob_rf"],
    score_b["playoff_prob_rf"]
)

# ------------------------------------------------------------
# 03 — XGBoost control
# ------------------------------------------------------------

report(
    "03 — XGBoost repeated predictions",
    score_a["playoff_prob_xgb"],
    score_b["playoff_prob_xgb"]
)

# ------------------------------------------------------------
# 04 — Ensemble impact
# ------------------------------------------------------------

report(
    "04 — Ensemble repeated predictions",
    score_a["playoff_prob"],
    score_b["playoff_prob"]
)

# ------------------------------------------------------------
# 05 — Inspect Random Forest parameters
# ------------------------------------------------------------

rf_model = inf.RF_PIPELINE.named_steps.get("model")

if rf_model is None:
    print("\n05 — Could not locate pipeline step named 'model'")
else:
    print("\n05 — Random Forest configuration")
    print("  Model class:", type(rf_model).__name__)
    print("  n_estimators:", getattr(rf_model, "n_estimators", None))
    print("  n_jobs:", getattr(rf_model, "n_jobs", None))
    print("  random_state:", getattr(rf_model, "random_state", None))

# ------------------------------------------------------------
# 06 — Single-threaded Random Forest comparison
# ------------------------------------------------------------

try:
    rf_single = copy.deepcopy(inf.RF_PIPELINE)

    single_model = rf_single.named_steps["model"]

    if "n_jobs" not in single_model.get_params():
        print(
            "\n06 — Single-thread comparison unavailable: "
            "model has no n_jobs parameter"
        )
    else:
        single_model.set_params(n_jobs=1)

        single_a = rf_single.predict_proba(X)[:, 1]
        single_b = rf_single.predict_proba(X)[:, 1]

        report(
            "06 — Single-thread RF repeated predictions",
            single_a,
            single_b
        )

        report(
            "07 — Original RF vs single-thread RF",
            rf_a,
            single_a
        )

except Exception as exc:
    print(
        "\n06 — Single-thread comparison error:",
        type(exc).__name__,
        str(exc)
    )

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary_52b = pd.DataFrame(results_52b)

print("\n" + "=" * 68)
print("BLOCK 52B — DETERMINISM DIAGNOSTIC SUMMARY")
print("=" * 68)

print(summary_52b.to_string(index=False))

print("=" * 68)

summary_52b


In [ ]:

# ============================================================
# BLOCK 53A — DAILY SCORING SNAPSHOT VALIDATION
# Read-only | No writes to DuckDB
# ============================================================

import importlib
import numpy as np
import pandas as pd

from config import SOURCE_TABLE
import feature_engineering as fe
import prediction_model.inference.inference as inference

fe = importlib.reload(fe)
inference = importlib.reload(inference)

SEASON_ID = 20252026
KEYS = ["season_id", "game_date", "team_id"]

results = []
diagnostics_53a = {}

def check(n, name, fn):
    try:
        detail = fn()
        status, msg = "PASS", str(detail or "")
        print(f"✅ {n:02d} PASS | {name}")
        if msg:
            print("   ", msg)
    except AssertionError as exc:
        status, msg = "FAIL", str(exc)
        print(f"❌ {n:02d} FAIL | {name}")
        print("   ", msg)
    except Exception as exc:
        status = "ERROR"
        msg = f"{type(exc).__name__}: {exc}"
        print(f"❌ {n:02d} ERROR | {name}")
        print("   ", msg)
    results.append((n, name, status, msg))

def require(condition, message):
    if not bool(condition):
        raise AssertionError(message)

# Original source data
source = con.execute(f"""
    SELECT *
    FROM {SOURCE_TABLE}
    WHERE season_id = ?
    ORDER BY game_date, team_id, game_id
""", [SEASON_ID]).df()

source["game_date"] = pd.to_datetime(source["game_date"])

# Replicate the production ASOF query, adding the source
# snapshot date for validation only.
daily = con.execute(f"""
    WITH teams AS (
        SELECT DISTINCT team_id
        FROM {SOURCE_TABLE}
        WHERE season_id = ?
    ),
    dates AS (
        SELECT DISTINCT game_date
        FROM {SOURCE_TABLE}
        WHERE season_id = ?
    ),
    spine AS (
        SELECT
            ?::BIGINT AS season_id,
            d.game_date,
            t.team_id
        FROM dates d
        CROSS JOIN teams t
    )
    SELECT
        snap.* EXCLUDE (
            season_id, game_date, team_id
        ),
        snap.game_date AS source_snapshot_date,
        spine.season_id,
        spine.game_date,
        spine.team_id
    FROM spine
    ASOF LEFT JOIN {SOURCE_TABLE} AS snap
        ON spine.team_id = snap.team_id
        AND spine.season_id = snap.season_id
        AND spine.game_date >= snap.game_date
    ORDER BY spine.game_date, spine.team_id
""", [SEASON_ID, SEASON_ID, SEASON_ID]).df()

daily["game_date"] = pd.to_datetime(daily["game_date"])
daily["source_snapshot_date"] = pd.to_datetime(
    daily["source_snapshot_date"]
)

valid = daily.loc[
    daily["games_played"].notna()
    & daily["tri_code"].notna()
].copy()

print(
    f"Source: {len(source):,} rows | "
    f"Daily spine: {len(daily):,} rows | "
    f"Usable: {len(valid):,} rows"
)

# 01 — Source key uniqueness
check(1, "Source team-game uniqueness", lambda: (
    require(
        not source.duplicated(["season_id", "game_id", "team_id"]).any(),
        "Duplicate team-game source records"
    ),
    f"{len(source):,} records"
)[1])

# 02 — Expected spine size
check(2, "Complete team-date spine", lambda: (
    require(
        len(daily) ==
        source["team_id"].nunique() * source["game_date"].nunique(),
        "Unexpected spine row count"
    ),
    f"{source['team_id'].nunique()} teams × "
    f"{source['game_date'].nunique()} dates"
)[1])

# 03 — Team-date uniqueness
check(3, "Unique daily prediction keys", lambda: (
    require(
        not valid.duplicated(KEYS).any(),
        "Duplicate team-date records"
    ),
    f"{len(valid):,} unique rows"
)[1])

# 04 — No future source snapshots
check(4, "No future-dated source snapshots", lambda: (
    require(
        (
            valid["source_snapshot_date"]
            <= valid["game_date"]
        ).all(),
        "A source snapshot is dated after its prediction date"
    ),
    "All source snapshot dates <= prediction dates"
)[1])

# 05 — Complete team coverage by date
def test_05():
    expected = source["team_id"].nunique()
    counts = valid.groupby("game_date")["team_id"].nunique()
    incomplete = counts[counts != expected]

    diagnostics_53a["incomplete_dates"] = incomplete

    require(
        incomplete.empty,
        f"{len(incomplete)} of {len(counts)} dates "
        f"have fewer than {expected} teams; "
        f"first examples: {incomplete.head(5).to_dict()}"
    )
    return f"All {len(counts)} dates have {expected} teams"

check(5, "Full team coverage on every date", test_05)

# 06 — No missing league-context fields
def test_06():
    cols = [
        "conference", "division", "division_rank",
        "points", "points_pct"
    ]
    missing = valid[cols].isna().sum()
    missing = missing[missing > 0]

    require(
        missing.empty,
        f"Missing context fields: {missing.to_dict()}"
    )
    return "League-context inputs populated"

check(6, "League-context field completeness", test_06)

# 07 — Each snapshot matches the latest source date
def test_07():
    latest = (
        source[["season_id", "team_id", "game_date"]]
        .rename(columns={"game_date": "snapshot_date"})
        .sort_values("snapshot_date")
    )

    expected = pd.merge_asof(
        valid[KEYS].sort_values("game_date"),
        latest,
        left_on="game_date",
        right_on="snapshot_date",
        by=["season_id", "team_id"],
        direction="backward"
    )

    actual = valid[KEYS + ["source_snapshot_date"]]

    merged = actual.merge(
        expected,
        on=KEYS,
        how="left",
        validate="one_to_one"
    )

    mismatches = (
        merged["source_snapshot_date"]
        != merged["snapshot_date"]
    ).sum()

    require(
        mismatches == 0,
        f"{mismatches} incorrect carried-forward dates"
    )
    return f"{len(merged):,} snapshots checked"

check(7, "ASOF snapshot recency", test_07)

# 08 — Source team-date ties
def test_08():
    ties = source.duplicated(
        ["season_id", "team_id", "game_date"],
        keep=False
    )

    diagnostics_53a["source_ties"] = int(ties.sum())

    require(
        not ties.any(),
        f"{ties.sum()} rows share a team-date; "
        "ASOF selection may be ambiguous"
    )
    return "No ambiguous source team-date matches"

check(8, "ASOF source-date uniqueness", test_08)

# 09 — Engineered daily feature coverage
def test_09():
    features = fe.build_features(valid.copy())
    diagnostics_53a["features"] = features

    missing = [
        c for c in inference.FEATURE_COLS
        if c not in features.columns
    ]
    require(not missing, f"Missing model features: {missing}")
    return f"{len(inference.FEATURE_COLS)} features available"

check(9, "Daily model feature contract", test_09)

# 10 — Daily scoring succeeds
def test_10():
    features = diagnostics_53a["features"]
    scored = inference.score_dataframe(features)
    diagnostics_53a["scored"] = scored

    require(
        len(scored) == len(valid),
        "Daily scoring changed row count"
    )
    return f"{len(scored):,} daily predictions"

check(10, "Daily model scoring", test_10)

# 11 — Daily probability validity
def test_11():
    scored = diagnostics_53a["scored"]

    for col in [
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob"
    ]:
        values = scored[col].to_numpy(dtype=float)

        require(
            np.isfinite(values).all(),
            f"Nonfinite values in {col}"
        )
        require(
            ((values >= 0) & (values <= 1)).all(),
            f"Out-of-range probabilities in {col}"
        )

    return "All probabilities finite and within [0,1]"

check(11, "Daily prediction probability validity", test_11)

# 12 — Source versus scoring context comparison
def test_12():
    source_features = fe.build_features(source.copy())
    daily_features = diagnostics_53a["features"]

    used = [
        c for c in [
            "wc_points_margin",
            "wc_points_pct_margin",
            "wc_teams_within_2pts",
            "wc_teams_within_4pts",
            "wc_teams_within_6pts",
            "corsi_net_rel",
        ]
        if c in inference.FEATURE_COLS
    ]

    # Only compare original team-game rows with their
    # corresponding team-day rows.
    original = source[KEYS].drop_duplicates()

    a = source_features.merge(
        original,
        on=KEYS,
        how="inner"
    )
    b = daily_features.merge(
        original,
        on=KEYS,
        how="inner"
    )

    merged = a[KEYS + used].merge(
        b[KEYS + used],
        on=KEYS,
        suffixes=("_training", "_daily"),
        validate="one_to_one"
    )

    changed_counts = {}

    for col in used:
        x = pd.to_numeric(
            merged[col + "_training"],
            errors="coerce"
        ).to_numpy(dtype=float, na_value=np.nan)

        y = pd.to_numeric(
            merged[col + "_daily"],
            errors="coerce"
        ).to_numpy(dtype=float, na_value=np.nan)

        changed_counts[col] = int(
            (~np.isclose(
                x, y,
                atol=1e-9,
                rtol=1e-9,
                equal_nan=True
            )).sum()
        )

    diagnostics_53a["context_differences"] = changed_counts

    changed = {
        k: v for k, v in changed_counts.items()
        if v > 0
    }

    require(
        not changed,
        f"Training-grain versus daily-grain differences: {changed}"
    )

    return f"{len(merged):,} matching team-game records"

check(12, "Training-grain vs daily-grain feature consistency", test_12)

# Summary
summary_53a = pd.DataFrame(
    results,
    columns=["Test", "Description", "Status", "Details"]
)

print("\n" + "=" * 68)
print("BLOCK 53A — DAILY SCORING SNAPSHOT SUMMARY")
print("=" * 68)
print(f"TOTAL:  {len(summary_53a)}")
print(f"PASSED: {(summary_53a.Status == 'PASS').sum()}")
print(f"FAILED: {(summary_53a.Status == 'FAIL').sum()}")
print(f"ERRORS: {(summary_53a.Status == 'ERROR').sum()}")

print("\nContext feature differences:")
print(diagnostics_53a.get("context_differences", "Not computed"))

print("\nFailed checks:")
print(
    summary_53a.loc[
        summary_53a.Status != "PASS",
        ["Test", "Description", "Details"]
    ].to_string(index=False)
)
print("=" * 68)

summary_53a


In [ ]:

# ============================================================
# BLOCK 54 — MODEL TRAINING VALIDATION
# Read-only | 24 tests | No model fitting or database writes
# ============================================================

import numpy as np
import pandas as pd
import importlib

from joblib import load
from pandas.testing import assert_frame_equal

from config import (
    MODEL_PATH,
    EXCLUDE_SEASON_ID,
)

import feature_engineering as fe
import prediction_model.inference.inference as inf

fe = importlib.reload(fe)
inf = importlib.reload(inf)

SOURCE_TABLE_54 = "team_daily_stats"
TARGET_COL_54 = "made_playoffs"
TEST_SEASON_54 = 20252026

LEAK_COLS_54 = [
    "game_id", "game_date", "date", "season_id",
    "team_name", "opponent", "clinch_type",
    TARGET_COL_54,
]

ID_COLS_54 = [
    "team_id", "opponent_id", "tri_code",
]

results_54 = []
diagnostics_54 = {}

def check(n, description, fn):
    try:
        detail = fn()
        status = "PASS"
        message = "" if detail is None else str(detail)
        print(f"✅ {n:02d} PASS | {description}")
        if message:
            print(f"   {message}")
    except AssertionError as exc:
        status = "FAIL"
        message = str(exc)
        print(f"❌ {n:02d} FAIL | {description}")
        print(f"   {message}")
    except Exception as exc:
        status = "ERROR"
        message = f"{type(exc).__name__}: {exc}"
        print(f"❌ {n:02d} ERROR | {description}")
        print(f"   {message}")

    results_54.append({
        "Test": n,
        "Description": description,
        "Status": status,
        "Details": message,
    })

def require(condition, message):
    if not bool(condition):
        raise AssertionError(message)

# ------------------------------------------------------------
# LOAD DATA AND SAVED MODEL
# ------------------------------------------------------------

if "con" not in globals():
    raise RuntimeError(
        "Block 54 requires the existing DuckDB connection 'con'."
    )

bundle_54 = load(MODEL_PATH)
saved_features_54 = list(bundle_54["features"])

raw_all_54 = con.execute(f"""
    SELECT *
    FROM {SOURCE_TABLE_54}
    ORDER BY season_id, game_date, game_id, team_id
""").df()

excluded_54 = (
    [EXCLUDE_SEASON_ID]
    if isinstance(EXCLUDE_SEASON_ID, (int, np.integer))
    else list(EXCLUDE_SEASON_ID)
    if EXCLUDE_SEASON_ID is not None
    else []
)

raw_54 = raw_all_54.loc[
    ~raw_all_54["season_id"].isin(excluded_54)
].copy()

# Match training script's final ordering.
if "game_id" in raw_54.columns:
    raw_54 = (
        raw_54.sort_values("game_id")
        .reset_index(drop=True)
    )

features_54 = fe.build_features(raw_54.copy())

train_mask_54 = (
    features_54["season_id"] < TEST_SEASON_54
)

test_mask_54 = (
    features_54["season_id"] == TEST_SEASON_54
)

numeric_54 = (
    features_54
    .drop(
        columns=LEAK_COLS_54 + ID_COLS_54,
        errors="ignore"
    )
    .select_dtypes(include="number")
    .copy()
)

# Reproduce the training script's feature-matrix behavior.
missing_rate_all_54 = numeric_54.isna().mean()
eligible_all_54 = missing_rate_all_54[
    missing_rate_all_54 < 0.20
].index.tolist()

X_54 = numeric_54[eligible_all_54].fillna(0)

print("=" * 72)
print("BLOCK 54 — MODEL TRAINING VALIDATION")
print("=" * 72)
print(
    f"Source rows: {len(raw_all_54):,} | "
    f"Included rows: {len(raw_54):,} | "
    f"Seasons: {raw_54['season_id'].nunique()}"
)
print(
    f"Training: {int(train_mask_54.sum()):,} | "
    f"Holdout: {int(test_mask_54.sum()):,} | "
    f"Saved model features: {len(saved_features_54)}"
)

# ------------------------------------------------------------
# 01 — Required training schema
# ------------------------------------------------------------

def test_01():
    required = {
        "season_id", "team_id", "game_id",
        "game_date", "games_played", TARGET_COL_54,
    }
    missing = required - set(raw_54.columns)
    require(not missing, f"Missing columns: {missing}")
    return "Required training columns present"

check(1, "Training source schema", test_01)

# ------------------------------------------------------------
# 02 — Training data not empty
# ------------------------------------------------------------

def test_02():
    require(not raw_54.empty, "Training data is empty")
    return f"{len(raw_54):,} included rows"

check(2, "Nonempty training data", test_02)

# ------------------------------------------------------------
# 03 — Excluded season removed
# ------------------------------------------------------------

def test_03():
    present = set(raw_54["season_id"].unique())
    overlap = present.intersection(excluded_54)

    require(
        not overlap,
        f"Excluded seasons remain: {overlap}"
    )
    return f"Excluded season IDs: {excluded_54}"

check(3, "Excluded-season filtering", test_03)

# ------------------------------------------------------------
# 04 — Unique team-game records
# ------------------------------------------------------------

def test_04():
    duplicates = raw_54.duplicated(
        ["season_id", "game_id", "team_id"]
    )
    require(
        not duplicates.any(),
        f"{duplicates.sum()} duplicate team-game rows"
    )
    return "Unique team-game keys"

check(4, "Training record uniqueness", test_04)

# ------------------------------------------------------------
# 05 — Playoff targets populated
# ------------------------------------------------------------

def test_05():
    nulls = int(raw_54[TARGET_COL_54].isna().sum())
    require(nulls == 0, f"{nulls} null playoff labels")
    return "No null target labels"

check(5, "Target completeness", test_05)

# ------------------------------------------------------------
# 06 — Target constant within team-season
# ------------------------------------------------------------

def test_06():
    counts = (
        raw_54.groupby(["season_id", "team_id"])[TARGET_COL_54]
        .nunique(dropna=False)
    )
    bad = counts[counts != 1]

    require(
        bad.empty,
        f"{len(bad)} team-seasons have inconsistent labels"
    )
    return f"{len(counts)} team-seasons checked"

check(6, "Team-season target consistency", test_06)

# ------------------------------------------------------------
# 07 — Binary target values
# ------------------------------------------------------------

def test_07():
    values = set(
        raw_54[TARGET_COL_54].dropna().unique().tolist()
    )
    require(
        values.issubset({0, 1, False, True}),
        f"Unexpected target values: {values}"
    )
    return f"Target values: {values}"

check(7, "Binary playoff target", test_07)

# ------------------------------------------------------------
# 08 — Season playoff team counts
# ------------------------------------------------------------

def test_08():
    per_team = (
        raw_54.groupby(["season_id", "team_id"])[TARGET_COL_54]
        .max()
        .reset_index()
    )
    summary = per_team.groupby("season_id").agg(
        teams=("team_id", "nunique"),
        playoff_teams=(TARGET_COL_54, "sum"),
    )

    diagnostics_54["season_targets"] = summary

    # Diagnostic: unusual season structures are reported.
    unusual = summary[
        (summary["teams"] != 32)
        | (summary["playoff_teams"] != 16)
    ]

    return (
        f"{len(summary)} seasons | "
        f"Unusual season counts: {len(unusual)}\n"
        f"{unusual.to_string() if len(unusual) else 'All seasons 32 teams / 16 playoff teams'}"
    )

check(8, "Season playoff-count diagnostic", test_08)

# ------------------------------------------------------------
# 09 — Nonempty training split
# ------------------------------------------------------------

def test_09():
    require(
        train_mask_54.any(),
        "No pre-holdout training records"
    )
    return f"{train_mask_54.sum():,} training rows"

check(9, "Training split availability", test_09)

# ------------------------------------------------------------
# 10 — Nonempty holdout split
# ------------------------------------------------------------

def test_10():
    require(
        test_mask_54.any(),
        "No holdout-season records"
    )
    return f"{test_mask_54.sum():,} holdout rows"

check(10, "Holdout split availability", test_10)

# ------------------------------------------------------------
# 11 — Train and holdout season separation
# ------------------------------------------------------------

def test_11():
    train_seasons = set(
        features_54.loc[train_mask_54, "season_id"]
    )
    test_seasons = set(
        features_54.loc[test_mask_54, "season_id"]
    )

    require(
        train_seasons.isdisjoint(test_seasons),
        "Training and holdout share season IDs"
    )

    require(
        max(train_seasons) < min(test_seasons),
        "Training contains future seasons"
    )

    return (
        f"Train seasons: {sorted(train_seasons)} | "
        f"Holdout: {sorted(test_seasons)}"
    )

check(11, "Temporal holdout separation", test_11)

# ------------------------------------------------------------
# 12 — Both classes present in training
# ------------------------------------------------------------

def test_12():
    labels = features_54.loc[
        train_mask_54, TARGET_COL_54
    ]
    require(
        labels.nunique() == 2,
        "Training data lacks both target classes"
    )
    return f"Positive rate: {labels.mean():.3f}"

check(12, "Training class coverage", test_12)

# ------------------------------------------------------------
# 13 — Both classes present in holdout
# ------------------------------------------------------------

def test_13():
    labels = features_54.loc[
        test_mask_54, TARGET_COL_54
    ]
    require(
        labels.nunique() == 2,
        "Holdout data lacks both target classes"
    )
    return f"Positive rate: {labels.mean():.3f}"

check(13, "Holdout class coverage", test_13)

# ------------------------------------------------------------
# 14 — Explicit leakage-column exclusion
# ------------------------------------------------------------

def test_14():
    forbidden = set(
        LEAK_COLS_54 + ID_COLS_54
    )
    leaked = forbidden.intersection(X_54.columns)

    require(
        not leaked,
        f"Excluded columns in model matrix: {leaked}"
    )
    return "Explicit leakage and identifier columns absent"

check(14, "Leakage-column exclusion", test_14)

# ------------------------------------------------------------
# 15 — Saved model features available
# ------------------------------------------------------------

def test_15():
    missing = [
        c for c in saved_features_54
        if c not in X_54.columns
    ]
    require(
        not missing,
        f"Saved model features missing: {missing}"
    )
    return f"{len(saved_features_54)} features available"

check(15, "Saved feature coverage", test_15)

# ------------------------------------------------------------
# 16 — Model feature types numeric
# ------------------------------------------------------------

def test_16():
    bad = [
        c for c in saved_features_54
        if not pd.api.types.is_numeric_dtype(
            X_54[c]
        )
    ]
    require(not bad, f"Nonnumeric features: {bad}")
    return "All saved features numeric"

check(16, "Saved feature numeric types", test_16)

# ------------------------------------------------------------
# 17 — Train-only missingness eligibility
# ------------------------------------------------------------

def test_17():
    train_missing = (
        numeric_54.loc[train_mask_54]
        .isna()
        .mean()
    )

    eligible_train = set(
        train_missing[train_missing < 0.20].index
    )
    eligible_all = set(eligible_all_54)

    different = eligible_train.symmetric_difference(
        eligible_all
    )

    diagnostics_54["eligibility_difference"] = sorted(different)

    require(
        not different,
        f"{len(different)} features have different eligibility "
        f"under train-only vs full-data filtering: {sorted(different)}"
    )
    return "Holdout data does not change eligibility decisions"

check(17, "Holdout influence on feature eligibility", test_17)

# ------------------------------------------------------------
# 18 — Missingness threshold applied
# ------------------------------------------------------------

def test_18():
    violations = [
        c for c in X_54.columns
        if missing_rate_all_54[c] >= 0.20
    ]
    require(
        not violations,
        f"Features exceed missingness threshold: {violations}"
    )
    return f"{len(X_54.columns)} eligible numeric features"

check(18, "Training missingness threshold", test_18)

# ------------------------------------------------------------
# 19 — No nonfinite training values
# ------------------------------------------------------------

def test_19():
    values = X_54.to_numpy(dtype=float)
    require(
        np.isfinite(values).all(),
        "Training matrix contains NaN or infinity"
    )
    return "All training matrix values finite"

check(19, "Finite model inputs", test_19)

# ------------------------------------------------------------
# 20 — Model feature count alignment
# ------------------------------------------------------------

def test_20():
    for name, pipeline in [
        ("XGB", bundle_54["xgb_pipeline"]),
        ("RF", bundle_54["rf_pipeline"]),
    ]:
        n = getattr(pipeline, "n_features_in_", None)
        require(
            n == len(saved_features_54),
            f"{name} expects {n} features; "
            f"bundle has {len(saved_features_54)}"
        )
    return f"Both models expect {len(saved_features_54)} features"

check(20, "Saved model input dimensions", test_20)

# ------------------------------------------------------------
# 21 — Model training metadata
# ------------------------------------------------------------

def test_21():
    required = {
        "trained_on", "version", "source_table",
        "target", "metrics",
    }
    missing = required - set(bundle_54)
    require(not missing, f"Missing metadata: {missing}")
    return (
        f"Version: {bundle_54['version']} | "
        f"Source: {bundle_54['source_table']}"
    )

check(21, "Saved model metadata", test_21)

# ------------------------------------------------------------
# 22 — Batch-dependent feature exposure
# ------------------------------------------------------------

def test_22():
    affected = [
        "wc_points_margin",
        "wc_points_pct_margin",
        "wc_teams_within_2pts",
        "wc_teams_within_4pts",
        "wc_teams_within_6pts",
        "corsi_net_rel",
    ]
    used = [
        c for c in affected
        if c in saved_features_54
    ]

    diagnostics_54["affected_features"] = used

    # Diagnostic, not a failure.
    return (
        f"{len(used)} batch-dependent features used: {used}"
    )

check(22, "Batch-dependent feature diagnostic", test_22)

# ------------------------------------------------------------
# 23 — Model artifact matches inference module
# ------------------------------------------------------------

def test_23():
    require(
        list(inf.FEATURE_COLS) == saved_features_54,
        "Inference uses a different feature contract"
    )
    return "Inference and training artifact agree"

check(23, "Training/inference model contract", test_23)

# ------------------------------------------------------------
# 24 — Training grain diagnostic
# ------------------------------------------------------------

def test_24():
    rows = len(raw_54)
    team_dates = (
        raw_54[["season_id", "game_date", "team_id"]]
        .drop_duplicates()
        .shape[0]
    )

    dates = (
        raw_54[["season_id", "game_date"]]
        .drop_duplicates()
        .shape[0]
    )

    team_seasons = (
        raw_54[["season_id", "team_id"]]
        .drop_duplicates()
        .shape[0]
    )

    diagnostics_54["training_grain"] = "team-game"

    return (
        f"Rows: {rows:,} | "
        f"Unique team-dates: {team_dates:,} | "
        f"Season-dates: {dates:,} | "
        f"Team-seasons: {team_seasons:,}\n"
        "Training uses team-game observations, "
        "not a complete team-date spine."
    )

check(24, "Training observation grain", test_24)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary_54 = pd.DataFrame(results_54)

passed = int((summary_54["Status"] == "PASS").sum())
failed = int((summary_54["Status"] == "FAIL").sum())
errors = int((summary_54["Status"] == "ERROR").sum())

print("\n" + "=" * 72)
print("BLOCK 54 — MODEL TRAINING VALIDATION SUMMARY")
print("=" * 72)

print(f"TOTAL:  {len(summary_54)}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")
print(f"ERRORS: {errors}")

print("\nAffected model features:")
print(diagnostics_54.get("affected_features", []))

print("\nFeature eligibility differences:")
print(diagnostics_54.get("eligibility_difference", "Not evaluated"))

print("\nSeason target summary:")
if "season_targets" in diagnostics_54:
    print(diagnostics_54["season_targets"].to_string())

if failed or errors:
    print("\nCHECKS REQUIRING ATTENTION:")
    print(
        summary_54.loc[
            summary_54["Status"] != "PASS",
            ["Test", "Description", "Details"]
        ].to_string(index=False)
    )

print("=" * 72)

summary_54


In [ ]:

# ============================================================
# BLOCK 55A — DAILY SHAP FEATURE VALIDATION
# Read-only | 18 tests | No database writes
# ============================================================

import numpy as np
import pandas as pd
import shap
from joblib import load

from config import MODEL_PATH
from feature_engineering import build_features

SEASON_55 = 20252026
SOURCE_55 = "team_daily_stats"
TARGET_55 = "playoff_shap_values"

bundle_55 = load(MODEL_PATH)
pipeline_55 = bundle_55["xgb_pipeline"]
model_55 = pipeline_55.named_steps["model"]
scaler_55 = pipeline_55.named_steps.get("scaler")
feature_cols_55 = list(bundle_55["features"])

results_55 = []
diagnostics_55 = {}

def require_55(condition, message):
    if not bool(condition):
        raise AssertionError(message)

def check_55(number, name, fn):
    try:
        detail = fn()
        status = "PASS"
        message = "" if detail is None else str(detail)
        print(f"✅ {number:02d} PASS | {name}")
        if message:
            print(f"   {message}")
    except AssertionError as exc:
        status = "FAIL"
        message = str(exc)
        print(f"❌ {number:02d} FAIL | {name}")
        print(f"   {message}")
    except Exception as exc:
        status = "ERROR"
        message = f"{type(exc).__name__}: {exc}"
        print(f"❌ {number:02d} ERROR | {name}")
        print(f"   {message}")

    results_55.append({
        "Test": number,
        "Description": name,
        "Status": status,
        "Details": message,
    })

# ------------------------------------------------------------
# Reproduce the script's latest-per-team selection
# ------------------------------------------------------------

latest_55 = con.execute(f"""
    SELECT *
    FROM {SOURCE_55}
    WHERE season_id = ?
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY team_id
        ORDER BY game_date DESC, game_id DESC
    ) = 1
    ORDER BY team_id
""", [SEASON_55]).df()

if latest_55.empty:
    raise RuntimeError("No source snapshots found")

features_55 = build_features(latest_55.copy())

X_55 = (
    features_55[feature_cols_55]
    .copy()
    .fillna(0)
)

if scaler_55 is not None:
    transformed_55 = pd.DataFrame(
        scaler_55.transform(X_55),
        columns=feature_cols_55,
        index=X_55.index,
    )
else:
    transformed_55 = X_55.copy()

print("=" * 72)
print("BLOCK 55A — DAILY SHAP FEATURE VALIDATION")
print("=" * 72)
print(
    f"Season: {SEASON_55} | "
    f"Latest team snapshots: {len(latest_55)} | "
    f"Model features: {len(feature_cols_55)}"
)

# ------------------------------------------------------------
# 01 — One latest snapshot per team
# ------------------------------------------------------------

def test_01():
    duplicates = latest_55["team_id"].duplicated().sum()
    require_55(duplicates == 0, f"{duplicates} duplicate teams")
    return f"{len(latest_55)} distinct teams"

check_55(1, "Latest team snapshot uniqueness", test_01)

# ------------------------------------------------------------
# 02 — Full league team coverage
# ------------------------------------------------------------

def test_02():
    expected = con.execute(f"""
        SELECT COUNT(DISTINCT team_id)
        FROM {SOURCE_55}
        WHERE season_id = ?
    """, [SEASON_55]).fetchone()[0]

    require_55(
        len(latest_55) == expected,
        f"Expected {expected} teams, got {len(latest_55)}"
    )
    return f"{expected} teams represented"

check_55(2, "Latest snapshot team coverage", test_02)

# ------------------------------------------------------------
# 03 — Snapshot dates aligned
# ------------------------------------------------------------

def test_03():
    dates = pd.to_datetime(latest_55["game_date"])
    count = dates.nunique()

    diagnostics_55["snapshot_dates"] = (
        dates.value_counts().sort_index()
    )

    require_55(
        count == 1,
        f"Latest team snapshots span {count} dates "
        f"({dates.min().date()} to {dates.max().date()})"
    )
    return "All teams share one snapshot date"

check_55(3, "Common SHAP snapshot date", test_03)

# ------------------------------------------------------------
# 04 — Latest snapshot selection accuracy
# ------------------------------------------------------------

def test_04():
    expected = con.execute(f"""
        SELECT team_id, MAX(game_date) AS latest_date
        FROM {SOURCE_55}
        WHERE season_id = ?
        GROUP BY team_id
    """, [SEASON_55]).df()

    actual = latest_55[["team_id", "game_date"]].merge(
        expected,
        on="team_id",
        validate="one_to_one",
    )

    matches = (
        pd.to_datetime(actual["game_date"])
        == pd.to_datetime(actual["latest_date"])
    )

    require_55(
        matches.all(),
        f"{(~matches).sum()} teams not at latest date"
    )
    return "All teams use their latest available game"

check_55(4, "Latest game selection accuracy", test_04)

# ------------------------------------------------------------
# 05 — Required SHAP identifiers
# ------------------------------------------------------------

def test_05():
    required = {
        "team_id", "tri_code", "season_id", "game_date"
    }
    missing = required - set(features_55.columns)

    require_55(not missing, f"Missing: {missing}")
    return "All output identifiers available"

check_55(5, "SHAP identifier availability", test_05)

# ------------------------------------------------------------
# 06 — Saved model features available
# ------------------------------------------------------------

def test_06():
    missing = [
        c for c in feature_cols_55
        if c not in features_55.columns
    ]

    require_55(not missing, f"Missing features: {missing}")
    return f"{len(feature_cols_55)} features available"

check_55(6, "Saved feature contract", test_06)

# ------------------------------------------------------------
# 07 — Feature order
# ------------------------------------------------------------

def test_07():
    require_55(
        list(X_55.columns) == feature_cols_55,
        "Feature order differs from saved model"
    )
    return "Exact saved feature order"

check_55(7, "SHAP feature alignment", test_07)

# ------------------------------------------------------------
# 08 — Finite model inputs
# ------------------------------------------------------------

def test_08():
    require_55(
        np.isfinite(X_55.to_numpy(dtype=float)).all(),
        "SHAP inputs contain NaN or infinity"
    )
    return "All model inputs finite"

check_55(8, "Finite SHAP feature matrix", test_08)

# ------------------------------------------------------------
# 09 — Preprocessing matches pipeline
# ------------------------------------------------------------

def test_09():
    direct = (
        scaler_55.transform(X_55)
        if scaler_55 is not None
        else X_55.to_numpy()
    )

    require_55(
        np.allclose(
            transformed_55.to_numpy(),
            np.asarray(direct),
            atol=1e-12,
            rtol=1e-12,
        ),
        "Transformed inputs differ from pipeline"
    )
    return "SHAP receives pipeline-transformed features"

check_55(9, "XGBoost preprocessing consistency", test_09)

# ------------------------------------------------------------
# 10 — SHAP output dimensions
# ------------------------------------------------------------

def test_10():
    values = np.asarray(shap_values_55)

    require_55(
        values.shape == X_55.shape,
        f"SHAP shape {values.shape}; expected {X_55.shape}"
    )
    return f"SHAP matrix: {values.shape}"

# Compute SHAP once for tests 10–13.
shap_error_55 = None
shap_values_55 = None
base_values_55 = None

try:
    explainer_55 = shap.TreeExplainer(model_55)
    explanation_55 = explainer_55(transformed_55)
    shap_values_55 = np.asarray(explanation_55.values)
    base_values_55 = np.asarray(explanation_55.base_values)
except Exception as exc:
    shap_error_55 = f"{type(exc).__name__}: {exc}"

def shap_ready_55():
    require_55(
        shap_error_55 is None,
        f"SHAP calculation failed: {shap_error_55}"
    )

def test_10_wrapper():
    shap_ready_55()
    return test_10()

check_55(10, "SHAP output dimensions", test_10_wrapper)

# ------------------------------------------------------------
# 11 — SHAP values finite
# ------------------------------------------------------------

def test_11():
    shap_ready_55()
    require_55(
        np.isfinite(shap_values_55).all(),
        "Nonfinite SHAP values"
    )
    return "All SHAP contributions finite"

check_55(11, "Finite SHAP values", test_11)

# ------------------------------------------------------------
# 12 — SHAP reconstructs XGBoost raw margin
# ------------------------------------------------------------

def test_12():
    shap_ready_55()

    from xgboost import DMatrix

    margin = model_55.get_booster().predict(
        DMatrix(
            transformed_55.to_numpy(),
            feature_names=feature_cols_55,
        ),
        output_margin=True,
    )

    reconstructed = (
        shap_values_55.sum(axis=1)
        + base_values_55.reshape(-1)
    )

    difference = np.abs(
        reconstructed - margin
    )

    max_difference = float(difference.max())

    diagnostics_55["shap_margin_max_diff"] = max_difference

    require_55(
        np.allclose(
            reconstructed,
            margin,
            atol=1e-4,
            rtol=1e-4,
        ),
        f"SHAP does not reconstruct model margin; "
        f"max difference={max_difference:.8g}"
    )

    return (
        f"Raw-margin reconstruction max error: "
        f"{max_difference:.8g}"
    )

check_55(12, "SHAP additivity", test_12)

# ------------------------------------------------------------
# 13 — SHAP explanations reproduce model probabilities
# ------------------------------------------------------------

def test_13():
    shap_ready_55()

    reconstructed_margin = (
        shap_values_55.sum(axis=1)
        + base_values_55.reshape(-1)
    )

    reconstructed_prob = (
        1 / (1 + np.exp(-reconstructed_margin))
    )

    actual_prob = pipeline_55.predict_proba(X_55)[:, 1]

    difference = np.abs(
        reconstructed_prob - actual_prob
    )

    max_difference = float(difference.max())

    require_55(
        np.allclose(
            reconstructed_prob,
            actual_prob,
            atol=1e-4,
            rtol=1e-4,
        ),
        f"Probability mismatch: {max_difference:.8g}"
    )

    return f"Maximum probability error: {max_difference:.8g}"

check_55(13, "SHAP-to-probability reconstruction", test_13)

# ------------------------------------------------------------
# 14 — Compare SHAP features with daily scoring features
# ------------------------------------------------------------

def test_14():
    # Recreate a complete as-of league snapshot at the
    # most recent game date, rather than independently
    # engineering features on latest team-game rows.

    asof_date = pd.to_datetime(
        latest_55["game_date"]
    ).max()

    daily_raw = con.execute(f"""
        SELECT *
        FROM {SOURCE_55}
        WHERE season_id = ?
          AND game_date <= ?
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY team_id
            ORDER BY game_date DESC, game_id DESC
        ) = 1
        ORDER BY team_id
    """, [SEASON_55, asof_date]).df()

    daily_features = build_features(daily_raw.copy())

    compare_cols = [
        "wc_points_margin",
        "wc_points_pct_margin",
        "wc_teams_within_2pts",
        "wc_teams_within_4pts",
        "wc_teams_within_6pts",
        "corsi_net_rel",
    ]

    merged = features_55[
        ["team_id"] + compare_cols
    ].merge(
        daily_features[
            ["team_id"] + compare_cols
        ],
        on="team_id",
        suffixes=("_shap", "_daily"),
        validate="one_to_one",
    )

    differences = {}

    for col in compare_cols:
        a = merged[f"{col}_shap"].to_numpy(float)
        b = merged[f"{col}_daily"].to_numpy(float)

        differences[col] = int(
            (~np.isclose(
                a, b,
                atol=1e-10,
                rtol=1e-10,
                equal_nan=True,
            )).sum()
        )

    diagnostics_55["shap_daily_differences"] = differences

    require_55(
        all(v == 0 for v in differences.values()),
        f"SHAP vs daily feature differences: {differences}"
    )

    return "SHAP and daily feature values match"

check_55(14, "SHAP versus daily feature consistency", test_14)

# ------------------------------------------------------------
# 15 — Output row uniqueness
# ------------------------------------------------------------

def test_15():
    keys = features_55[
        ["season_id", "game_date", "team_id"]
    ]

    duplicates = keys.duplicated().sum()

    require_55(
        duplicates == 0,
        f"{duplicates} duplicate output keys"
    )

    return f"{len(keys)} unique SHAP rows"

check_55(15, "SHAP output key uniqueness", test_15)

# ------------------------------------------------------------
# 16 — Existing SHAP table schema
# ------------------------------------------------------------

def test_16():
    exists = con.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_name = ?
    """, [TARGET_55]).fetchone()[0]

    if not exists:
        return (
            "Target table does not yet exist; "
            "schema comparison deferred"
        )

    schema = con.execute(f"""
        DESCRIBE {TARGET_55}
    """).df()

    existing_cols = schema["column_name"].tolist()

    expected_cols = (
        feature_cols_55
        + [
            "team_id", "tri_code",
            "season_id", "game_date", "created_at"
        ]
    )

    require_55(
        existing_cols == expected_cols,
        "Existing table column order/schema differs "
        "from positional INSERT SELECT *"
    )

    return "Existing table column order matches SHAP output"

check_55(16, "SHAP target schema alignment", test_16)

# ------------------------------------------------------------
# 17 — Existing duplicate SHAP keys
# ------------------------------------------------------------

def test_17():
    exists = con.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_name = ?
    """, [TARGET_55]).fetchone()[0]

    if not exists:
        return "Target table not created yet"

    duplicates = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT
                season_id, game_date, team_id,
                COUNT(*) AS n
            FROM {TARGET_55}
            GROUP BY 1, 2, 3
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    require_55(
        duplicates == 0,
        f"{duplicates} duplicate SHAP keys"
    )

    return "No duplicate stored SHAP keys"

check_55(17, "Stored SHAP key uniqueness", test_17)

# ------------------------------------------------------------
# 18 — Stored SHAP date freshness
# ------------------------------------------------------------

def test_18():
    exists = con.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_name = ?
    """, [TARGET_55]).fetchone()[0]

    if not exists:
        return "Target table not created yet"

    stored = con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            MAX(game_date) AS latest_date
        FROM {TARGET_55}
        WHERE season_id = ?
    """, [SEASON_55]).df().iloc[0]

    if stored["rows"] == 0:
        return "No stored SHAP rows for this season"

    source_latest = pd.to_datetime(
        latest_55["game_date"]
    ).max()

    stored_latest = pd.to_datetime(
        stored["latest_date"]
    )

    require_55(
        stored_latest >= source_latest,
        f"Stored SHAP latest date {stored_latest.date()} "
        f"lags source {source_latest.date()}"
    )

    return (
        f"{stored['rows']} stored rows | "
        f"Latest: {stored_latest.date()}"
    )

check_55(18, "Stored SHAP freshness", test_18)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary_55a = pd.DataFrame(results_55)

passed = int((summary_55a["Status"] == "PASS").sum())
failed = int((summary_55a["Status"] == "FAIL").sum())
errors = int((summary_55a["Status"] == "ERROR").sum())

print("\n" + "=" * 72)
print("BLOCK 55A — DAILY SHAP VALIDATION SUMMARY")
print("=" * 72)

print(f"TOTAL:  {len(summary_55a)}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")
print(f"ERRORS: {errors}")

print("\nDiagnostics:")
for key, value in diagnostics_55.items():
    print(f"{key}: {value}")

if failed or errors:
    print("\nCHECKS REQUIRING ATTENTION:")
    print(
        summary_55a.loc[
            summary_55a["Status"] != "PASS",
            ["Test", "Description", "Details"]
        ].to_string(index=False)
    )

print("=" * 72)

summary_55a


In [ ]:

# ============================================================
# BLOCK 55B — SHAP STORAGE DIAGNOSTICS
# Read-only | No INSERT, DELETE, ALTER, or model fitting
# ============================================================

import pandas as pd
import numpy as np

TARGET_55B = "playoff_shap_values"
SEASON_55B = 20252026

results_55b = []

def check_55b(n, name, fn):
    try:
        detail = fn()
        status = "PASS"
        message = str(detail)
        print(f"✅ {n:02d} PASS | {name}")
        print(f"   {message}")
    except AssertionError as exc:
        status = "FAIL"
        message = str(exc)
        print(f"❌ {n:02d} FAIL | {name}")
        print(f"   {message}")
    except Exception as exc:
        status = "ERROR"
        message = f"{type(exc).__name__}: {exc}"
        print(f"❌ {n:02d} ERROR | {name}")
        print(f"   {message}")

    results_55b.append({
        "Test": n,
        "Description": name,
        "Status": status,
        "Details": message,
    })

def require_55b(condition, message):
    if not bool(condition):
        raise AssertionError(message)

print("=" * 72)
print("BLOCK 55B — SHAP STORAGE DIAGNOSTICS")
print("=" * 72)

# ------------------------------------------------------------
# Existing table structure
# ------------------------------------------------------------

schema_55b = con.execute(f"""
    DESCRIBE {TARGET_55B}
""").df()

stored_cols_55b = schema_55b["column_name"].tolist()

expected_cols_55b = (
    list(feature_cols_55)
    + [
        "team_id",
        "tri_code",
        "season_id",
        "game_date",
        "created_at",
    ]
)

# ------------------------------------------------------------
# 01 — Column count
# ------------------------------------------------------------

def test_01():
    require_55b(
        len(stored_cols_55b) == len(expected_cols_55b),
        f"Stored: {len(stored_cols_55b)} | "
        f"Expected: {len(expected_cols_55b)}"
    )
    return f"{len(stored_cols_55b)} columns"

check_55b(1, "SHAP table column count", test_01)

# ------------------------------------------------------------
# 02 — Required columns exist
# ------------------------------------------------------------

def test_02():
    missing = sorted(
        set(expected_cols_55b) - set(stored_cols_55b)
    )
    require_55b(
        not missing,
        f"Missing stored columns: {missing}"
    )
    return "All expected columns exist"

check_55b(2, "SHAP table column coverage", test_02)

# ------------------------------------------------------------
# 03 — Unexpected stored columns
# ------------------------------------------------------------

def test_03():
    extra = sorted(
        set(stored_cols_55b) - set(expected_cols_55b)
    )
    require_55b(
        not extra,
        f"Unexpected stored columns: {extra}"
    )
    return "No unexpected columns"

check_55b(3, "SHAP table extra columns", test_03)

# ------------------------------------------------------------
# 04 — Positional column alignment
# ------------------------------------------------------------

def test_04():
    mismatches = [
        (i, stored, expected)
        for i, (stored, expected) in enumerate(
            zip(stored_cols_55b, expected_cols_55b),
            start=1
        )
        if stored != expected
    ]

    require_55b(
        not mismatches,
        f"{len(mismatches)} position mismatches; "
        f"first 10: {mismatches[:10]}"
    )

    return "Column order matches"

check_55b(4, "Positional INSERT safety", test_04)

# ------------------------------------------------------------
# 05 — Required identifier types
# ------------------------------------------------------------

def test_05():
    types = dict(
        zip(
            schema_55b["column_name"],
            schema_55b["column_type"]
        )
    )

    required = [
        "team_id",
        "tri_code",
        "season_id",
        "game_date",
        "created_at",
    ]

    missing = [c for c in required if c not in types]

    require_55b(
        not missing,
        f"Missing identifiers: {missing}"
    )

    return {c: types[c] for c in required}

check_55b(5, "SHAP identifier column types", test_05)

# ------------------------------------------------------------
# 06 — Existing row count
# ------------------------------------------------------------

def test_06():
    count = con.execute(f"""
        SELECT COUNT(*)
        FROM {TARGET_55B}
        WHERE season_id = ?
    """, [SEASON_55B]).fetchone()[0]

    return f"{count:,} stored rows for season {SEASON_55B}"

check_55b(6, "Stored SHAP row count", test_06)

# ------------------------------------------------------------
# 07 — Stored date distribution
# ------------------------------------------------------------

def test_07():
    dates = con.execute(f"""
        SELECT
            game_date,
            COUNT(*) AS rows,
            COUNT(DISTINCT team_id) AS teams
        FROM {TARGET_55B}
        WHERE season_id = ?
        GROUP BY game_date
        ORDER BY game_date DESC
        LIMIT 15
    """, [SEASON_55B]).df()

    print("\nRecent stored SHAP dates:")
    print(dates.to_string(index=False))

    return f"{len(dates)} recent dates displayed"

check_55b(7, "Stored SHAP date distribution", test_07)

# ------------------------------------------------------------
# 08 — Stored SHAP rows per team
# ------------------------------------------------------------

def test_08():
    counts = con.execute(f"""
        SELECT
            team_id,
            COUNT(*) AS rows,
            MAX(game_date) AS latest_date
        FROM {TARGET_55B}
        WHERE season_id = ?
        GROUP BY team_id
        ORDER BY team_id
    """, [SEASON_55B]).df()

    print("\nStored SHAP team coverage:")
    print(counts.to_string(index=False))

    require_55b(
        len(counts) == len(latest_55),
        f"Stored {len(counts)} teams; "
        f"source has {len(latest_55)}"
    )

    return f"{len(counts)} teams represented"

check_55b(8, "Stored SHAP team coverage", test_08)

# ------------------------------------------------------------
# 09 — Source versus stored latest team dates
# ------------------------------------------------------------

def test_09():
    source_dates = latest_55[
        ["team_id", "game_date"]
    ].rename(
        columns={"game_date": "source_date"}
    )

    stored_dates = con.execute(f"""
        SELECT
            team_id,
            MAX(game_date) AS stored_date
        FROM {TARGET_55B}
        WHERE season_id = ?
        GROUP BY team_id
    """, [SEASON_55B]).df()

    compare = source_dates.merge(
        stored_dates,
        on="team_id",
        how="left",
        validate="one_to_one",
    )

    compare["source_date"] = pd.to_datetime(
        compare["source_date"]
    )

    compare["stored_date"] = pd.to_datetime(
        compare["stored_date"]
    )

    behind = compare[
        compare["stored_date"].isna()
        | (compare["stored_date"] < compare["source_date"])
    ]

    print("\nTeams with stale or missing SHAP results:")
    print(
        behind.to_string(index=False)
        if not behind.empty
        else "None"
    )

    require_55b(
        behind.empty,
        f"{len(behind)} teams missing current SHAP results"
    )

    return "All team SHAP dates current"

check_55b(9, "Team-level SHAP freshness", test_09)

# ------------------------------------------------------------
# 10 — Current SHAP output key conflicts
# ------------------------------------------------------------

def test_10():
    current_keys = latest_55[
        ["season_id", "game_date", "team_id"]
    ].copy()

    stored_keys = con.execute(f"""
        SELECT
            season_id,
            game_date,
            team_id
        FROM {TARGET_55B}
        WHERE season_id = ?
    """, [SEASON_55B]).df()

    overlap = current_keys.merge(
        stored_keys,
        on=["season_id", "game_date", "team_id"],
        how="inner"
    )

    return (
        f"{len(overlap)} existing matching keys; "
        f"{len(current_keys) - len(overlap)} new keys"
    )

check_55b(10, "Current SHAP key overlap", test_10)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary_55b = pd.DataFrame(results_55b)

passed = int((summary_55b["Status"] == "PASS").sum())
failed = int((summary_55b["Status"] == "FAIL").sum())
errors = int((summary_55b["Status"] == "ERROR").sum())

print("\n" + "=" * 72)
print("BLOCK 55B — SUMMARY")
print("=" * 72)

print(f"TOTAL:  {len(summary_55b)}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")
print(f"ERRORS: {errors}")

if failed or errors:
    print("\nCHECKS REQUIRING ATTENTION:")
    print(
        summary_55b.loc[
            summary_55b["Status"] != "PASS",
            ["Test", "Description", "Details"]
        ].to_string(index=False)
    )

print("=" * 72)

summary_55b


In [ ]:
# ============================================================
# BLOCK 55C — Historical Daily SHAP Validation (READ ONLY)
# ============================================================

import importlib
import numpy as np
import pandas as pd

import prediction_model.explainability.daily_feature_predictions as dfp

importlib.reload(dfp)

results = []

def check(name, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | {name}"
        + (f" | {detail}" if detail else "")
    )

season_id = 20252026  # Test input only, not hard-coded in the script

print("\n=== 55C: HISTORICAL DAILY SHAP ===\n")

# 01 — Saved model metadata
check("01 Model features loaded", len(dfp.FEATURE_COLS) > 0)
check("02 Model version loaded", bool(dfp.MODEL_VERSION_USED))
check("03 New target table", dfp.FEATURE_TARGET_TABLE == "playoff_shap_values_v2")

# 04 — Historical game dates
dates = dfp._source_dates(con, season_id)
check("04 Historical dates available", len(dates) > 1, f"{len(dates)} dates")

# 05 — Team coverage
counts = dfp._snapshot_team_counts(con, season_id)
check("05 Coverage matches dates", set(counts) == set(dates))
check("06 Latest date has 32 teams", counts[dates[-1]] == 32)

# 07 — ASOF snapshots
latest = dfp._load_asof_snapshot(con, season_id, dates[-1])
check("07 Latest snapshot has 32 rows", len(latest) == 32)
check("08 One row per team", latest["team_id"].is_unique)
check(
    "09 No future game dates",
    (pd.to_datetime(latest["game_date"]).dt.date <= dates[-1]).all()
)

# 10 — Historical snapshot
early = dfp._load_asof_snapshot(con, season_id, dates[0])
check("10 Early snapshot available", len(early) > 0)
check("11 Early snapshot has unique teams", early["team_id"].is_unique)

# 12 — SHAP calculation
shap_df = dfp._build_shap_dataframe(
    latest,
    season_id=season_id,
    snapshot_date=dates[-1],
)

check("12 SHAP output has 32 rows", len(shap_df) == 32)
check("13 Output columns match contract", list(shap_df.columns) == dfp.OUTPUT_COLS)
check("14 All model SHAP columns present", len(dfp.SHAP_COLS) == len(dfp.FEATURE_COLS))
check("15 Unique daily snapshot keys", not shap_df.duplicated(
    ["season_id", "snapshot_date", "team_id", "model_version"]
).any())

# 16 — Probability reconstruction
reconstructed = 1 / (
    1 + np.exp(
        -(
            shap_df["shap_base_value"].to_numpy()
            + shap_df[dfp.SHAP_COLS].sum(axis=1).to_numpy()
        )
    )
)

check(
    "16 SHAP reconstructs XGB probability",
    np.allclose(
        reconstructed,
        shap_df["xgb_probability"].to_numpy(),
        atol=1e-6,
    )
)

check(
    "17 Probabilities in [0, 1]",
    shap_df["xgb_probability"].between(0, 1).all()
)

# 18 — Historical dates to process
pending = dfp._dates_to_process(con, season_id)
check("18 Historical processing dates resolved", len(pending) <= len(dates))

print("\n" + "=" * 55)
print(f"BLOCK 55C: {sum(results)}/{len(results)} PASS")
print("=" * 55)

In [ ]:
# ============================================================
# BLOCK 55D — SHAP DATABASE WRITE VALIDATION
# ============================================================

import importlib
import pandas as pd
import prediction_model.explainability.daily_feature_predictions as dfp

importlib.reload(dfp)

results = []

def check(name, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | {name}"
        + (f" | {detail}" if detail else "")
    )

season_id = 20252026
snapshot_date = dfp._source_dates(con, season_id)[-1]

print("\n=== 55D: SHAP DATABASE WRITE ===\n")

# 01 — Execute a single-date write
write_error = None

try:
    dfp.run_prediction_features(
        con=con,
        season_id=season_id,
        snapshot_date=snapshot_date,
    )
except Exception as exc:
    write_error = exc

check(
    "01 Single-date write completed",
    write_error is None,
    str(write_error) if write_error else "",
)

# 02–10 — Validate stored results
if write_error is None:

    stored = con.execute(
        """
        SELECT *
        FROM playoff_shap_values_v2
        WHERE season_id = ?
          AND snapshot_date = ?
          AND model_version = ?
        ORDER BY team_id
        """,
        [season_id, snapshot_date, dfp.MODEL_VERSION_USED],
    ).df()

    check("02 Stored 32 teams", len(stored) == 32)
    check("03 Unique teams", stored["team_id"].is_unique)
    check("04 Correct schema", list(stored.columns) == dfp.OUTPUT_COLS)
    check("05 All SHAP columns present", all(c in stored for c in dfp.SHAP_COLS))
    check("06 No null SHAP values", stored[dfp.SHAP_COLS].notna().all().all())
    check("07 Base values populated", stored["shap_base_value"].notna().all())
    check("08 Raw margins populated", stored["xgb_raw_margin"].notna().all())
    check("09 Probabilities populated", stored["xgb_probability"].between(0, 1).all())

    # Verify the incremental runner recognizes this date as complete.
    pending = dfp._dates_to_process(con, season_id)

    check(
        "10 Completed date excluded from backfill",
        snapshot_date not in pending,
    )

print("\n" + "=" * 55)
print(f"BLOCK 55D: {sum(results)}/{len(results)} PASS")
print("=" * 55)

In [ ]:
# ============================================================
# BLOCK 55E — HISTORICAL DAILY SHAP BACKFILL
# ============================================================

import prediction_model.explainability.daily_feature_predictions as dfp

dfp.run_prediction_features(
    con=con,
    season_id=20252026,
    refresh_existing=False,
)

In [ ]:

# ============================================================
# BLOCK 55F — HISTORICAL DAILY SHAP VALIDATION (READ ONLY)
# ============================================================

import numpy as np
import pandas as pd
from scipy.special import expit

import prediction_model.explainability.daily_feature_predictions as dfp

season_id = 20252026
version = dfp.MODEL_VERSION_USED

results = []

def check(name, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | {name}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== 55F: HISTORICAL SHAP VALIDATION ===\n")

# ------------------------------------------------------------
# Load stored SHAP data
# ------------------------------------------------------------

stored = con.execute(
    """
    SELECT *
    FROM playoff_shap_values_v2
    WHERE season_id = ?
      AND model_version = ?
    ORDER BY snapshot_date, team_id
    """,
    [season_id, version],
).df()

dates = dfp._source_dates(con, season_id)
expected_counts = dfp._snapshot_team_counts(con, season_id)

actual_counts = (
    stored.assign(
        snapshot_date=pd.to_datetime(
            stored["snapshot_date"]
        ).dt.date
    )
    .groupby("snapshot_date")
    .size()
    .to_dict()
)

expected_rows = sum(expected_counts.values())

# ------------------------------------------------------------
# Historical coverage
# ------------------------------------------------------------

check(
    "01 Historical rows present",
    len(stored) > 32,
    f"{len(stored):,} rows"
)

check(
    "02 Total row count",
    len(stored) == expected_rows,
    f"actual={len(stored):,}, expected={expected_rows:,}"
)

check(
    "03 All game dates covered",
    set(actual_counts) == set(dates),
    f"{len(actual_counts)}/{len(dates)} dates"
)

check(
    "04 Correct teams per date",
    actual_counts == expected_counts
)

# ------------------------------------------------------------
# Keys and schema
# ------------------------------------------------------------

check(
    "05 Unique snapshot keys",
    not stored.duplicated([
        "season_id",
        "snapshot_date",
        "team_id",
        "model_version"
    ]).any()
)

check(
    "06 Schema matches saved model",
    list(stored.columns) == dfp.OUTPUT_COLS
)

check(
    "07 All SHAP values populated",
    stored[dfp.SHAP_COLS].notna().all().all()
)

check(
    "08 Base values populated",
    stored["shap_base_value"].notna().all()
)

check(
    "09 Raw margins populated",
    stored["xgb_raw_margin"].notna().all()
)

check(
    "10 Probabilities in [0, 1]",
    stored["xgb_probability"].between(0, 1).all()
)

# ------------------------------------------------------------
# SHAP probability reconstruction
# ------------------------------------------------------------

if not stored.empty:

    shap_sum = stored[dfp.SHAP_COLS].sum(axis=1).to_numpy()

    reconstructed_margin = (
        stored["shap_base_value"].to_numpy()
        + shap_sum
    )

    reconstructed_probability = expit(
        reconstructed_margin
    )

    margin_error = np.max(
        np.abs(
            reconstructed_margin
            - stored["xgb_raw_margin"].to_numpy()
        )
    )

    probability_error = np.max(
        np.abs(
            reconstructed_probability
            - stored["xgb_probability"].to_numpy()
        )
    )

    check(
        "11 SHAP reconstructs raw margin",
        np.allclose(
            reconstructed_margin,
            stored["xgb_raw_margin"].to_numpy(),
            atol=1e-5
        ),
        f"max error={margin_error:.2e}"
    )

    check(
        "12 SHAP reconstructs probability",
        np.allclose(
            reconstructed_probability,
            stored["xgb_probability"].to_numpy(),
            atol=1e-6
        ),
        f"max error={probability_error:.2e}"
    )

    # --------------------------------------------------------
    # Point-in-time integrity
    # --------------------------------------------------------

    check(
        "13 No future game dates",
        (
            pd.to_datetime(stored["game_date"]).dt.date
            <= pd.to_datetime(stored["snapshot_date"]).dt.date
        ).all()
    )

    check(
        "14 Latest snapshot has 32 teams",
        actual_counts.get(dates[-1]) == 32
    )

else:
    for i in range(11, 15):
        check(f"{i:02d} Historical data validation", False)

# ------------------------------------------------------------
# Incremental processing
# ------------------------------------------------------------

pending = dfp._dates_to_process(
    con,
    season_id
)

check(
    "15 No incomplete snapshots remaining",
    len(pending) == 0,
    f"{len(pending)} pending dates"
)

# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print(
    f"BLOCK 55F: {sum(results)}/{len(results)} PASS"
)
print("=" * 55)


In [ ]:

# ============================================================
# BLOCK 56A — COMPUTE_SHAP VALIDATION (READ ONLY)
# ============================================================

import importlib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

import prediction_model.explainability.compute_shap as cs
import prediction_model.explainability.daily_feature_predictions as dfp

importlib.reload(cs)

season_id = 20252026
results = []

def check(name, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | {name}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== BLOCK 56A: COMPUTE_SHAP VALIDATION ===\n")

# ------------------------------------------------------------
# 1. Model contract
# ------------------------------------------------------------

check(
    "01 Saved features match daily SHAP",
    list(cs.FEATURE_COLS) == dfp.FEATURE_COLS
)

check(
    "02 Same XGBoost model type",
    type(cs.xgb_model) is type(dfp.xgb_model)
)

check(
    "03 Scaler configuration matches",
    (cs.scaler is None) == (dfp.scaler is None)
)

# ------------------------------------------------------------
# 2. Latest team snapshots
# ------------------------------------------------------------

raw = con.execute(
    """
    SELECT *
    FROM team_daily_stats
    WHERE season_id = ?
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY team_id
        ORDER BY game_date DESC, game_id DESC
    ) = 1
    ORDER BY team_id
    """,
    [season_id],
).df()

check("04 Latest snapshot has 32 teams", len(raw) == 32)
check("05 Unique teams", raw["team_id"].is_unique)

snapshot_date = pd.to_datetime(raw["game_date"]).max().date()

check(
    "06 Latest date matches stored SHAP",
    snapshot_date == dfp._source_dates(con, season_id)[-1]
)

# ------------------------------------------------------------
# 3. Feature engineering
# ------------------------------------------------------------

features = cs.build_features(raw.copy())

missing = [
    col for col in cs.FEATURE_COLS
    if col not in features.columns
]

check("07 No missing model features", not missing)

X = features[cs.FEATURE_COLS].copy().fillna(0)

check(
    "08 Feature order matches model",
    list(X.columns) == list(cs.FEATURE_COLS)
)

check(
    "09 Feature values finite",
    np.isfinite(X.to_numpy(dtype=float)).all()
)

if cs.scaler is not None:
    X_in = pd.DataFrame(
        cs.scaler.transform(X),
        columns=cs.FEATURE_COLS,
        index=X.index,
    )
else:
    X_in = X.copy()

check(
    "10 Transformed values finite",
    np.isfinite(X_in.to_numpy(dtype=float)).all()
)

# ------------------------------------------------------------
# 4. SHAP calculation
# ------------------------------------------------------------

explainer = shap.TreeExplainer(cs.xgb_model)
values = np.asarray(explainer(X_in).values)

check(
    "11 SHAP matrix shape",
    values.shape == (len(raw), len(cs.FEATURE_COLS)),
    str(values.shape)
)

check(
    "12 SHAP values finite",
    np.isfinite(values).all()
)

# ------------------------------------------------------------
# 5. Compare with stored SHAP
# ------------------------------------------------------------

stored = con.execute(
    """
    SELECT *
    FROM playoff_shap_values_v2
    WHERE season_id = ?
      AND snapshot_date = ?
      AND model_version = ?
    ORDER BY team_id
    """,
    [season_id, snapshot_date, dfp.MODEL_VERSION_USED],
).df()

check(
    "13 Stored snapshot has 32 teams",
    len(stored) == 32
)

check(
    "14 Team ordering matches",
    raw["team_id"].tolist() == stored["team_id"].tolist()
)

stored_values = stored[dfp.SHAP_COLS].to_numpy(dtype=float)

max_diff = (
    float(np.max(np.abs(values - stored_values)))
    if values.shape == stored_values.shape
    else float("inf")
)

check(
    "15 SHAP values match stored results",
    np.allclose(values, stored_values, atol=1e-5),
    f"max difference={max_diff:.2e}"
)

# ------------------------------------------------------------
# 6. Plot generation
# ------------------------------------------------------------

plot_error = None

try:
    shap.summary_plot(
        values,
        X_in,
        feature_names=cs.FEATURE_COLS,
        show=False,
    )
    plt.close("all")
except Exception as exc:
    plot_error = exc

check(
    "16 SHAP summary plot renders",
    plot_error is None,
    str(plot_error) if plot_error else ""
)

# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print(f"BLOCK 56A: {sum(results)}/{len(results)} PASS")
print("=" * 55)


In [ ]:

# ============================================================
# BLOCK 56B — REFACTORED COMPUTE_SHAP VALIDATION (READ ONLY)
# ============================================================

import importlib
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit

import prediction_model.explainability.compute_shap as cs
import prediction_model.explainability.daily_feature_predictions as dfp

importlib.reload(cs)

season_id = 20252026
results = []

def check(name, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | {name}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== BLOCK 56B: REFACTORED COMPUTE_SHAP ===\n")

# ------------------------------------------------------------
# 1. Configuration and function contracts
# ------------------------------------------------------------

check(
    "01 Uses historical SHAP table",
    cs.FEATURE_TARGET_TABLE == "playoff_shap_values_v2"
)

check(
    "02 Model features match SHAP storage",
    cs.FEATURE_COLS == dfp.FEATURE_COLS
)

check(
    "03 Model version matches",
    cs.MODEL_VERSION_USED == dfp.MODEL_VERSION_USED
)

check(
    "04 Summary accepts snapshot_date",
    "snapshot_date" in inspect.signature(
        cs.run_shap_summary
    ).parameters
)

check(
    "05 Loader accepts snapshot_date",
    "snapshot_date" in inspect.signature(
        cs.load_shap_summary_data
    ).parameters
)

# ------------------------------------------------------------
# 2. Latest snapshot resolution
# ------------------------------------------------------------

dates = dfp._source_dates(con, season_id)
latest_date = dates[-1]

resolved = cs._resolve_snapshot_date(
    con,
    season_id
)

check(
    "06 Defaults to latest stored date",
    resolved == latest_date,
    str(resolved)
)

# ------------------------------------------------------------
# 3. Load latest stored explanations
# ------------------------------------------------------------

stored, X, X_in = cs.load_shap_summary_data(
    con=con,
    season_id=season_id
)

check(
    "07 Latest snapshot has 32 teams",
    len(stored) == 32
)

check(
    "08 Team IDs unique",
    stored["team_id"].is_unique
)

check(
    "09 Feature matrix shape",
    X.shape == (32, len(cs.FEATURE_COLS)),
    str(X.shape)
)

check(
    "10 Transformed matrix shape",
    X_in.shape == X.shape
)

check(
    "11 SHAP columns match saved model",
    all(c in stored.columns for c in cs.SHAP_COLS)
)

# ------------------------------------------------------------
# 4. Stored SHAP probability reconstruction
# ------------------------------------------------------------

shap_values = stored[
    cs.SHAP_COLS
].to_numpy(dtype=float)

base_values = stored[
    "shap_base_value"
].to_numpy(dtype=float)

reconstructed = expit(
    base_values + shap_values.sum(axis=1)
)

check(
    "12 SHAP reconstructs probability",
    np.allclose(
        reconstructed,
        stored["xgb_probability"].to_numpy(),
        atol=1e-6
    )
)

predicted = cs.xgb_pipeline.predict_proba(X)[:, 1]

check(
    "13 Reconstructed features reproduce XGB",
    np.allclose(
        predicted,
        stored["xgb_probability"].to_numpy(),
        atol=1e-4
    )
)

check(
    "14 All stored SHAP values finite",
    np.isfinite(shap_values).all()
)

check(
    "15 Feature values finite",
    np.isfinite(X_in.to_numpy(dtype=float)).all()
)

# ------------------------------------------------------------
# 5. Historical snapshot
# ------------------------------------------------------------

historical_date = pd.Timestamp(
    "2026-01-15"
).date()

historical, X_hist, X_hist_in = (
    cs.load_shap_summary_data(
        con=con,
        season_id=season_id,
        snapshot_date=historical_date
    )
)

check(
    "16 Historical snapshot has 32 teams",
    len(historical) == 32
)

check(
    "17 Historical snapshot date correct",
    (
        pd.to_datetime(
            historical["snapshot_date"]
        ).dt.date == historical_date
    ).all()
)

check(
    "18 Historical SHAP matrix shape",
    historical[cs.SHAP_COLS].shape ==
    (32, len(cs.FEATURE_COLS))
)

# ------------------------------------------------------------
# 6. Early-season partial coverage
# ------------------------------------------------------------

early_date = dates[0]

early, X_early, X_early_in = (
    cs.load_shap_summary_data(
        con=con,
        season_id=season_id,
        snapshot_date=early_date
    )
)

check(
    "19 Early snapshot has 6 teams",
    len(early) == 6,
    f"{len(early)} teams"
)

check(
    "20 Early snapshot unique teams",
    early["team_id"].is_unique
)

check(
    "21 Early feature matrix matches teams",
    len(X_early) == len(early)
)

# ------------------------------------------------------------
# 7. Plot generation
# ------------------------------------------------------------

plot_error = None

try:
    cs.run_shap_summary(
        con=con,
        season_id=season_id,
        snapshot_date=historical_date,
        max_display=15,
        show=False
    )
except Exception as exc:
    plot_error = exc
finally:
    plt.close("all")

check(
    "22 Historical plot renders",
    plot_error is None,
    str(plot_error) if plot_error else ""
)

# ------------------------------------------------------------
# 8. Invalid date handling
# ------------------------------------------------------------

invalid_date_rejected = False

try:
    cs.load_shap_summary_data(
        con=con,
        season_id=season_id,
        snapshot_date="2020-01-01"
    )
except ValueError:
    invalid_date_rejected = True

check(
    "23 Missing snapshot rejected",
    invalid_date_rejected
)

# ------------------------------------------------------------
# 9. No SHAP recalculation in visualization script
# ------------------------------------------------------------

source = inspect.getsource(
    cs.load_shap_summary_data
)

check(
    "24 Loader does not calculate SHAP",
    "TreeExplainer" not in source
    and "shap.Explainer" not in source
    and "shap_values =" not in source
)

# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print(
    f"BLOCK 56B: {sum(results)}/{len(results)} PASS"
)
print("=" * 55)


In [ ]:

# ============================================================
# BLOCK 57A — MODEL FEATURE CONSISTENCY DIAGNOSTIC
# READ ONLY — NO DATABASE WRITES OR MODEL CHANGES
# ============================================================

import numpy as np
import pandas as pd
from joblib import load

from config import MODEL_PATH
from feature_engineering import build_features

SEASON_ID = 20252026
ATOL = 1e-8
RTOL = 1e-6

bundle = load(MODEL_PATH)
FEATURE_COLS = list(bundle["features"])

KEY_COLS = ["season_id", "game_date", "team_id"]

WATCH_FEATURES = [
    "wc_points_margin",
    "wc_points_pct_margin",
    "wc_teams_within_2pts",
    "wc_teams_within_4pts",
    "wc_teams_within_6pts",
    "corsi_net_rel",
    "corsi_ratio_rel",
    "danger_net_rel",
    "pdo_dev_rel",
]

print("\n=== BLOCK 57A: FEATURE CONSISTENCY DIAGNOSTIC ===")
print(f"Season: {SEASON_ID}")
print(f"Saved model: {bundle.get('version', 'unknown')}")
print(f"Model features: {len(FEATURE_COLS)}")

# ------------------------------------------------------------
# 1. Load original team-game observations
# ------------------------------------------------------------

df_game_raw = con.execute(
    """
    SELECT *
    FROM team_daily_stats
    WHERE season_id = ?
    ORDER BY game_date, team_id, game_id
    """,
    [SEASON_ID],
).df()

if df_game_raw.empty:
    raise RuntimeError("No team-game data found")

df_game_raw["game_date"] = pd.to_datetime(
    df_game_raw["game_date"]
).dt.date

print(f"\nTeam-game rows: {len(df_game_raw):,}")

# ------------------------------------------------------------
# 2. Build daily as-of snapshots
#
# Mirrors the daily prediction runner's ASOF join.
# Teams without a previous game are excluded.
# ------------------------------------------------------------

df_daily_raw = con.execute(
    """
    WITH teams AS (
        SELECT DISTINCT team_id
        FROM team_daily_stats
        WHERE season_id = ?
    ),
    dates AS (
        SELECT DISTINCT game_date
        FROM team_daily_stats
        WHERE season_id = ?
    ),
    spine AS (
        SELECT
            ?::BIGINT AS season_id,
            d.game_date,
            t.team_id
        FROM dates d
        CROSS JOIN teams t
    )
    SELECT
        snap.* EXCLUDE (
            season_id,
            game_date,
            team_id
        ),
        spine.season_id,
        spine.game_date,
        spine.team_id
    FROM spine
    ASOF LEFT JOIN team_daily_stats AS snap
        ON spine.team_id = snap.team_id
        AND spine.season_id = snap.season_id
        AND spine.game_date >= snap.game_date
    ORDER BY spine.game_date, spine.team_id
    """,
    [SEASON_ID, SEASON_ID, SEASON_ID],
).df()

df_daily_raw = df_daily_raw.loc[
    df_daily_raw["games_played"].notna()
    & df_daily_raw["tri_code"].notna()
].copy()

df_daily_raw["game_date"] = pd.to_datetime(
    df_daily_raw["game_date"]
).dt.date

print(f"Daily as-of rows: {len(df_daily_raw):,}")

# ------------------------------------------------------------
# 3. Check raw data uniqueness
# ------------------------------------------------------------

for name, frame in [
    ("team-game", df_game_raw),
    ("daily-asof", df_daily_raw),
]:
    duplicates = frame.duplicated(KEY_COLS).sum()

    print(
        f"{name} duplicate team-date keys: {duplicates}"
    )

    if duplicates:
        raise RuntimeError(
            f"{name} contains duplicate comparison keys"
        )

# ------------------------------------------------------------
# 4. Build features independently
# ------------------------------------------------------------

print("\nBuilding team-game features...")
df_game = build_features(df_game_raw.copy())

print("Building daily-snapshot features...")
df_daily = build_features(df_daily_raw.copy())

# ------------------------------------------------------------
# 5. Validate saved model feature availability
# ------------------------------------------------------------

missing_game = [
    c for c in FEATURE_COLS if c not in df_game.columns
]

missing_daily = [
    c for c in FEATURE_COLS if c not in df_daily.columns
]

if missing_game or missing_daily:
    raise RuntimeError(
        f"Missing team-game features: {missing_game}\n"
        f"Missing daily features: {missing_daily}"
    )

print("PASS | All saved model features available")

# ------------------------------------------------------------
# 6. Align matching team-date observations
# ------------------------------------------------------------

compare = df_game[
    KEY_COLS + FEATURE_COLS
].merge(
    df_daily[KEY_COLS + FEATURE_COLS],
    on=KEY_COLS,
    how="inner",
    suffixes=("_game", "_daily"),
    validate="one_to_one",
)

if compare.empty:
    raise RuntimeError("No matching team-date observations")

print(
    f"PASS | Matching team-date observations: "
    f"{len(compare):,}"
)

# ------------------------------------------------------------
# 7. Compare every saved model feature
# ------------------------------------------------------------

results = []

for feature in FEATURE_COLS:

    game = pd.to_numeric(
        compare[f"{feature}_game"],
        errors="coerce",
    ).to_numpy(dtype=float)

    daily = pd.to_numeric(
        compare[f"{feature}_daily"],
        errors="coerce",
    ).to_numpy(dtype=float)

    # Treat paired NaNs as equivalent.
    same = np.isclose(
        game,
        daily,
        atol=ATOL,
        rtol=RTOL,
        equal_nan=True,
    )

    mismatch = ~same
    count = int(mismatch.sum())

    abs_diff = np.abs(game - daily)
    finite_diff = abs_diff[np.isfinite(abs_diff)]

    max_diff = (
        float(finite_diff.max())
        if len(finite_diff)
        else np.nan
    )

    results.append({
        "feature": feature,
        "mismatch_rows": count,
        "mismatch_pct": round(
            100 * count / len(compare), 2
        ),
        "max_abs_diff": max_diff,
        "status": "FAIL" if count else "PASS",
    })

feature_results = pd.DataFrame(results)

feature_results = feature_results.sort_values(
    ["mismatch_rows", "max_abs_diff"],
    ascending=[False, False],
).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

different = feature_results[
    feature_results["mismatch_rows"] > 0
].copy()

matching = feature_results[
    feature_results["mismatch_rows"] == 0
].copy()

print("\n" + "=" * 60)
print("ALL MODEL FEATURES")
print("=" * 60)

print(f"Features tested: {len(FEATURE_COLS)}")
print(f"Matching features: {len(matching)}")
print(f"Differing features: {len(different)}")
print(f"Compared observations: {len(compare):,}")

print("\nFEATURES WITH DIFFERENCES")

if different.empty:
    print("None")
else:
    print(
        different.to_string(index=False)
    )

# ------------------------------------------------------------
# 9. Watchlist diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("WATCHLIST: ORIGINAL SIX + THREE ADDITIONAL FEATURES")
print("=" * 60)

watch_results = feature_results[
    feature_results["feature"].isin(WATCH_FEATURES)
].copy()

print(
    watch_results.to_string(index=False)
    if not watch_results.empty
    else "No watchlist features are in the saved model"
)

not_in_model = [
    f for f in WATCH_FEATURES
    if f not in FEATURE_COLS
]

if not_in_model:
    print("\nWatchlist features NOT in saved model:")
    print(not_in_model)

# ------------------------------------------------------------
# 10. Detailed mismatch examples
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MISMATCH EXAMPLES")
print("=" * 60)

for feature in different["feature"].head(10):

    game_col = f"{feature}_game"
    daily_col = f"{feature}_daily"

    a = pd.to_numeric(
        compare[game_col], errors="coerce"
    )

    b = pd.to_numeric(
        compare[daily_col], errors="coerce"
    )

    mismatch = ~np.isclose(
        a.to_numpy(dtype=float),
        b.to_numpy(dtype=float),
        atol=ATOL,
        rtol=RTOL,
        equal_nan=True,
    )

    examples = compare.loc[
        mismatch,
        KEY_COLS + [game_col, daily_col],
    ].head(5)

    print(f"\n{feature}")
    print(examples.to_string(index=False))

# ------------------------------------------------------------
# 11. Final diagnostic result
# ------------------------------------------------------------

print("\n" + "=" * 60)

if different.empty:
    print(
        "BLOCK 57A: PASS — All model features are consistent"
    )
else:
    print(
        f"BLOCK 57A: DIAGNOSTIC COMPLETE — "
        f"{len(different)} of {len(FEATURE_COLS)} "
        "features differ"
    )

print("=" * 60)

# Results remain available for inspection:
# feature_results
# different
# watch_results
# compare


In [ ]:

# ============================================================
# BLOCK 57B — CANONICAL SNAPSHOT FEATURE VALIDATION
# READ ONLY — NO DATABASE WRITES
# ============================================================

import numpy as np
import pandas as pd
from joblib import load

from config import MODEL_PATH
from feature_engineering import build_features

SEASON_ID = 20252026
ATOL = 1e-8
RTOL = 1e-6

bundle = load(MODEL_PATH)
FEATURE_COLS = list(bundle["features"])
KEY_COLS = ["season_id", "game_date", "team_id"]

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== BLOCK 57B: CANONICAL SNAPSHOT VALIDATION ===\n")

# ------------------------------------------------------------
# 1. Load original team-game observations
# ------------------------------------------------------------

df_games = con.execute(
    """
    SELECT *
    FROM team_daily_stats
    WHERE season_id = ?
    ORDER BY game_date, team_id, game_id
    """,
    [SEASON_ID],
).df()

df_games["game_date"] = pd.to_datetime(
    df_games["game_date"]
).dt.date

check(
    1, "Original team-game observations loaded",
    len(df_games) > 0,
    f"{len(df_games):,} rows"
)

check(
    2, "Original team-game keys unique",
    not df_games.duplicated(KEY_COLS).any()
)

# ------------------------------------------------------------
# 2. Construct canonical daily as-of snapshots
# ------------------------------------------------------------

df_canonical = con.execute(
    """
    WITH teams AS (
        SELECT DISTINCT team_id
        FROM team_daily_stats
        WHERE season_id = ?
    ),
    dates AS (
        SELECT DISTINCT game_date
        FROM team_daily_stats
        WHERE season_id = ?
    ),
    spine AS (
        SELECT
            ?::BIGINT AS season_id,
            d.game_date,
            t.team_id
        FROM dates d
        CROSS JOIN teams t
    )
    SELECT
        snap.* EXCLUDE (
            season_id,
            game_date,
            team_id
        ),
        spine.season_id,
        spine.game_date,
        spine.team_id
    FROM spine
    ASOF LEFT JOIN team_daily_stats AS snap
        ON spine.team_id = snap.team_id
        AND spine.season_id = snap.season_id
        AND spine.game_date >= snap.game_date
    ORDER BY spine.game_date, spine.team_id
    """,
    [SEASON_ID, SEASON_ID, SEASON_ID],
).df()

df_canonical = df_canonical.loc[
    df_canonical["games_played"].notna()
    & df_canonical["tri_code"].notna()
].copy()

df_canonical["game_date"] = pd.to_datetime(
    df_canonical["game_date"]
).dt.date

check(
    3, "Canonical snapshots loaded",
    len(df_canonical) > 0,
    f"{len(df_canonical):,} rows"
)

check(
    4, "Canonical team-date keys unique",
    not df_canonical.duplicated(KEY_COLS).any()
)

check(
    5, "Canonical snapshots cover all game dates",
    set(df_canonical["game_date"]) ==
    set(df_games["game_date"])
)

# ------------------------------------------------------------
# 3. Calculate features using complete daily population
# ------------------------------------------------------------

df_canonical_features = build_features(
    df_canonical.copy()
)

missing_features = [
    c for c in FEATURE_COLS
    if c not in df_canonical_features.columns
]

check(
    6, "All saved model features available",
    not missing_features,
    str(missing_features) if missing_features else ""
)

# ------------------------------------------------------------
# 4. Extract original team-game observations
# ------------------------------------------------------------

game_keys = df_games[KEY_COLS].drop_duplicates()

df_training_candidate = (
    df_canonical_features.merge(
        game_keys,
        on=KEY_COLS,
        how="inner",
        validate="one_to_one"
    )
)

check(
    7, "Training candidate preserves game-row count",
    len(df_training_candidate) == len(df_games),
    f"{len(df_training_candidate):,} vs {len(df_games):,}"
)

check(
    8, "Training candidate keys unique",
    not df_training_candidate.duplicated(KEY_COLS).any()
)

check(
    9, "Training candidate retains every game key",
    set(map(tuple, df_training_candidate[KEY_COLS].to_numpy())) ==
    set(map(tuple, df_games[KEY_COLS].to_numpy()))
)

# ------------------------------------------------------------
# 5. Compare against independently calculated daily features
# ------------------------------------------------------------

df_daily_reference = build_features(
    df_canonical.copy()
)

compare = df_training_candidate[
    KEY_COLS + FEATURE_COLS
].merge(
    df_daily_reference[KEY_COLS + FEATURE_COLS],
    on=KEY_COLS,
    suffixes=("_training", "_daily"),
    how="inner",
    validate="one_to_one"
)

check(
    10, "Comparison includes all game observations",
    len(compare) == len(df_games),
    f"{len(compare):,} rows"
)

# ------------------------------------------------------------
# 6. Compare all 121 features
# ------------------------------------------------------------

feature_differences = []

for feature in FEATURE_COLS:

    a = pd.to_numeric(
        compare[f"{feature}_training"],
        errors="coerce"
    ).to_numpy(dtype=float)

    b = pd.to_numeric(
        compare[f"{feature}_daily"],
        errors="coerce"
    ).to_numpy(dtype=float)

    mismatches = ~np.isclose(
        a, b,
        atol=ATOL,
        rtol=RTOL,
        equal_nan=True
    )

    count = int(mismatches.sum())

    if count:
        feature_differences.append({
            "feature": feature,
            "mismatch_rows": count,
            "mismatch_pct": round(
                100 * count / len(compare), 2
            )
        })

check(
    11, "All 121 features match daily inference",
    len(feature_differences) == 0,
    f"{len(feature_differences)} differing features"
)

# ------------------------------------------------------------
# 7. Validate nine previously inconsistent features
# ------------------------------------------------------------

AFFECTED_FEATURES = [
    "corsi_net_rel",
    "wc2_points",
    "wc2_points_pct",
    "wc_points_margin",
    "wc_points_pct_margin",
    "wc_teams_within_2pts",
    "wc_teams_within_4pts",
    "wc_teams_within_6pts",
    "wc_pressure_x_performance",
]

different_names = {
    item["feature"] for item in feature_differences
}

check(
    12, "All nine previously inconsistent features resolved",
    all(
        f in FEATURE_COLS and f not in different_names
        for f in AFFECTED_FEATURES
    )
)

# ------------------------------------------------------------
# 8. Snapshot coverage diagnostics
# ------------------------------------------------------------

coverage = (
    df_canonical.groupby("game_date")["team_id"]
    .nunique()
    .sort_index()
)

check(
    13, "No date exceeds 32 teams",
    coverage.max() <= 32
)

check(
    14, "Latest snapshot includes 32 teams",
    coverage.iloc[-1] == 32,
    f"{coverage.iloc[-1]} teams"
)

check(
    15, "Expected early-season partial coverage",
    coverage.iloc[0] == 6
    and coverage.iloc[1] == 13,
    f"{coverage.iloc[0]}, {coverage.iloc[1]} teams"
)

# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 60)

if feature_differences:
    print("FEATURE MISMATCHES:")
    print(pd.DataFrame(feature_differences).to_string(index=False))
else:
    print("All 121 saved model features match.")

print("\nCANONICAL SNAPSHOT COVERAGE")
print(
    f"Dates: {len(coverage)}"
    f" | Total rows: {len(df_canonical):,}"
    f" | Training observations: {len(df_training_candidate):,}"
)

print("\n" + "=" * 60)
print(f"BLOCK 57B: {sum(results)}/{len(results)} PASS")
print("=" * 60)

# DataFrames retained for inspection:
# df_canonical
# df_canonical_features
# df_training_candidate
# compare
# coverage


In [ ]:

# ============================================================
# BLOCK 57C — HISTORICAL SNAPSHOT INTEGRITY
# READ ONLY — NO DATABASE WRITES
# Requires Block 57B DataFrames
# ============================================================

import numpy as np
import pandas as pd

SEASON_ID = 20252026
KEY_COLS = ["season_id", "game_date", "team_id"]

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== BLOCK 57C: HISTORICAL SNAPSHOT INTEGRITY ===\n")

# ------------------------------------------------------------
# 1. Prepare source and canonical snapshots
# ------------------------------------------------------------

games = df_games.copy()
canonical = df_canonical.copy()
training = df_training_candidate.copy()

for frame in [games, canonical, training]:
    frame["game_date"] = pd.to_datetime(
        frame["game_date"]
    ).dt.date

check(
    1, "Season IDs consistent",
    all(
        (frame["season_id"] == SEASON_ID).all()
        for frame in [games, canonical, training]
    )
)

check(
    2, "Canonical team-date keys unique",
    not canonical.duplicated(KEY_COLS).any()
)

check(
    3, "Training team-date keys unique",
    not training.duplicated(KEY_COLS).any()
)

# ------------------------------------------------------------
# 2. Independently identify source game dates
# ------------------------------------------------------------

source_dates = (
    games.groupby("team_id")["game_date"]
    .apply(lambda x: set(x))
    .to_dict()
)

canonical["is_game_day"] = canonical.apply(
    lambda r: r["game_date"] in source_dates[r["team_id"]],
    axis=1
)

game_day_count = int(canonical["is_game_day"].sum())

check(
    4, "Game-day row count matches source",
    game_day_count == len(games),
    f"{game_day_count:,} vs {len(games):,}"
)

# ------------------------------------------------------------
# 3. Validate point-in-time snapshot selection
# ------------------------------------------------------------

# Compute the expected source game date for each canonical row.
# This is independent of the ASOF JOIN used in Block 57B.

expected = (
    canonical[KEY_COLS]
    .merge(
        games[
            ["team_id", "game_date", "game_id", "games_played"]
        ].rename(columns={
            "game_date": "source_game_date",
            "game_id": "source_game_id",
            "games_played": "source_games_played",
        }),
        on="team_id",
        how="left"
    )
)

expected = expected.loc[
    expected["source_game_date"] <= expected["game_date"]
].copy()

expected = (
    expected.sort_values(
        ["season_id", "game_date", "team_id",
         "source_game_date", "source_game_id"]
    )
    .drop_duplicates(
        KEY_COLS,
        keep="last"
    )
)

expected = expected[
    KEY_COLS + [
        "source_game_date",
        "source_game_id",
        "source_games_played",
    ]
]

actual = canonical.merge(
    expected,
    on=KEY_COLS,
    how="left",
    validate="one_to_one"
)

check(
    5, "Every snapshot has a historical source",
    actual["source_game_date"].notna().all()
)

check(
    6, "No future source game selected",
    (
        actual["source_game_date"] <=
        actual["game_date"]
    ).all()
)

check(
    7, "Games played matches latest historical game",
    np.allclose(
        pd.to_numeric(
            actual["games_played"], errors="coerce"
        ),
        pd.to_numeric(
            actual["source_games_played"], errors="coerce"
        ),
        equal_nan=True
    )
)

# ------------------------------------------------------------
# 4. Validate carried-forward statistics
# ------------------------------------------------------------

STAT_COLS = [
    "games_played",
    "points",
    "points_pct",
    "wins",
    "losses",
    "goal_diff",
    "corsi_for",
    "corsi_against",
    "corsi_diff_per_game",
]

STAT_COLS = [
    c for c in STAT_COLS
    if c in games.columns and c in canonical.columns
]

source_values = games[
    ["team_id", "game_date"] +
    [c for c in STAT_COLS if c != "games_played"]
].rename(
    columns={
        "game_date": "source_game_date",
        **{
            c: f"source_{c}"
            for c in STAT_COLS
            if c != "games_played"
        }
    }
)

actual = actual.merge(
    source_values,
    on=["team_id", "source_game_date"],
    how="left",
    validate="many_to_one"
)

stat_mismatches = {}

for col in STAT_COLS:

    left = pd.to_numeric(
        actual[col], errors="coerce"
    ).to_numpy(dtype=float)

    right = pd.to_numeric(
        actual[f"source_{col}"], errors="coerce"
    ).to_numpy(dtype=float)

    mismatch = ~np.isclose(
        left,
        right,
        atol=1e-8,
        rtol=1e-6,
        equal_nan=True
    )

    stat_mismatches[col] = int(mismatch.sum())

check(
    8, "Carried-forward statistics match source",
    sum(stat_mismatches.values()) == 0,
    str(stat_mismatches)
)

# ------------------------------------------------------------
# 5. Check non-game days
# ------------------------------------------------------------

non_game = actual.loc[
    ~actual["is_game_day"]
].copy()

check(
    9, "Non-game-day snapshots exist",
    len(non_game) > 0,
    f"{len(non_game):,} rows"
)

check(
    10, "Non-game days use an earlier source date",
    (
        non_game["source_game_date"] <
        non_game["game_date"]
    ).all()
)

check(
    11, "Non-game-day games_played carried forward",
    np.allclose(
        pd.to_numeric(
            non_game["games_played"], errors="coerce"
        ),
        pd.to_numeric(
            non_game["source_games_played"], errors="coerce"
        ),
        equal_nan=True
    )
)

# ------------------------------------------------------------
# 6. Check actual game days
# ------------------------------------------------------------

game_day = actual.loc[
    actual["is_game_day"]
].copy()

check(
    12, "Game-day snapshots use current game",
    (
        game_day["source_game_date"] ==
        game_day["game_date"]
    ).all()
)

# ------------------------------------------------------------
# 7. Training observations match canonical snapshots
# ------------------------------------------------------------

train_compare = training[
    KEY_COLS + STAT_COLS
].merge(
    canonical[KEY_COLS + STAT_COLS],
    on=KEY_COLS,
    suffixes=("_training", "_canonical"),
    validate="one_to_one"
)

training_mismatches = {}

for col in STAT_COLS:

    a = pd.to_numeric(
        train_compare[f"{col}_training"],
        errors="coerce"
    ).to_numpy(dtype=float)

    b = pd.to_numeric(
        train_compare[f"{col}_canonical"],
        errors="coerce"
    ).to_numpy(dtype=float)

    training_mismatches[col] = int((
        ~np.isclose(
            a, b,
            atol=1e-8,
            rtol=1e-6,
            equal_nan=True
        )
    ).sum())

check(
    13, "Training observations retain canonical statistics",
    sum(training_mismatches.values()) == 0,
    str(training_mismatches)
)

# ------------------------------------------------------------
# 8. Verify expected team coverage
# ------------------------------------------------------------

coverage = (
    canonical.groupby("game_date")["team_id"]
    .nunique()
    .sort_index()
)

check(
    14, "Team coverage never decreases",
    (coverage.diff().dropna() >= 0).all()
)

check(
    15, "Full league coverage eventually reached",
    (coverage == 32).any()
)

check(
    16, "No snapshot has more than 32 teams",
    (coverage <= 32).all()
)

# ------------------------------------------------------------
# 9. Final results
# ------------------------------------------------------------

print("\n" + "=" * 60)

print(f"Source game rows: {len(games):,}")
print(f"Canonical snapshots: {len(canonical):,}")
print(f"Training observations: {len(training):,}")
print(f"Non-game-day snapshots: {len(non_game):,}")
print(f"Statistics checked: {len(STAT_COLS)}")

print("\n" + "=" * 60)
print(
    f"BLOCK 57C: {sum(results)}/{len(results)} PASS"
)
print("=" * 60)

# Diagnostic DataFrames remain available:
# actual
# non_game
# game_day
# train_compare
# coverage


In [ ]:

# ============================================================
# BLOCK 58A — HISTORICAL CANONICAL TRAINING VALIDATION
# READ ONLY — NO DATABASE WRITES OR MODEL SAVES
# ============================================================

import numpy as np
import pandas as pd

from config import EXCLUDE_SEASON_ID
from feature_engineering import build_features

SOURCE_TABLE = "team_daily_stats"
TARGET_COL = "made_playoffs"
KEY_COLS = ["season_id", "game_date", "team_id"]

results = []


def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )


print("\n=== BLOCK 58A: HISTORICAL CANONICAL TRAINING ===\n")

# ============================================================
# 1. LOAD ALL ELIGIBLE SOURCE OBSERVATIONS
# ============================================================

df_source = con.execute(
    f"SELECT * FROM {SOURCE_TABLE}"
).df()

if isinstance(EXCLUDE_SEASON_ID, (int, np.integer)):
    excluded = [EXCLUDE_SEASON_ID]
else:
    excluded = list(EXCLUDE_SEASON_ID)

df_source = df_source.loc[
    ~df_source["season_id"].isin(excluded)
].copy()

# IMPORTANT:
# Keep datetime64[ns] for pd.merge_asof().
# Do NOT use .dt.date here.

df_source["game_date"] = pd.to_datetime(
    df_source["game_date"]
).dt.normalize()

seasons = sorted(
    df_source["season_id"].unique()
)

check(
    1,
    "Historical source observations loaded",
    not df_source.empty,
    f"{len(df_source):,} rows"
)

check(
    2,
    "Excluded seasons absent",
    not df_source["season_id"].isin(excluded).any()
)

check(
    3,
    "Source team-date keys unique",
    not df_source.duplicated(KEY_COLS).any()
)

check(
    4,
    "Playoff target available",
    TARGET_COL in df_source.columns
    and df_source[TARGET_COL].notna().all()
)

check(
    5,
    "Game dates are datetime64",
    pd.api.types.is_datetime64_any_dtype(
        df_source["game_date"]
    )
)

check(
    6,
    "No null game dates",
    df_source["game_date"].notna().all()
)

# ============================================================
# 2. CONSTRUCT DAILY AS-OF SNAPSHOTS FOR EACH SEASON
# ============================================================

canonical_parts = []
season_summary_rows = []

print("\nBuilding historical daily snapshots...\n")

for season_id in seasons:

    season_games = df_source.loc[
        df_source["season_id"] == season_id
    ].copy()

    # All game dates in this season
    dates = pd.DataFrame({
        "game_date": sorted(
            season_games["game_date"].unique()
        )
    })

    dates["game_date"] = pd.to_datetime(
        dates["game_date"]
    )

    # All teams appearing in this season
    teams = pd.DataFrame({
        "team_id": sorted(
            season_games["team_id"].unique()
        )
    })

    # Date x team grid
    spine = dates.merge(
        teams,
        how="cross"
    )

    spine["season_id"] = season_id

    # IMPORTANT:
    # merge_asof requires the ON column to be
    # globally sorted, even when using BY.

    spine = spine.sort_values(
        ["game_date", "team_id"]
    ).reset_index(drop=True)

    history = season_games.sort_values(
        ["game_date", "team_id"]
    ).reset_index(drop=True)

    # Validate inputs before merging

    if not spine["game_date"].is_monotonic_increasing:
        raise RuntimeError(
            f"Spine dates not sorted: {season_id}"
        )

    if not history["game_date"].is_monotonic_increasing:
        raise RuntimeError(
            f"History dates not sorted: {season_id}"
        )

    if not pd.api.types.is_datetime64_any_dtype(
        spine["game_date"]
    ):
        raise TypeError(
            f"Spine date dtype invalid: {season_id}"
        )

    if not pd.api.types.is_datetime64_any_dtype(
        history["game_date"]
    ):
        raise TypeError(
            f"History date dtype invalid: {season_id}"
        )

    # --------------------------------------------------------
    # Point-in-time ASOF join
    #
    # Each team/date receives the latest record for
    # that team on or before that date.
    # --------------------------------------------------------

    snapshots = pd.merge_asof(
        spine,
        history.drop(columns=["season_id"]),
        on="game_date",
        by="team_id",
        direction="backward",
        allow_exact_matches=True
    )

    snapshots["season_id"] = season_id

    # Match daily inference eligibility rules

    snapshots = snapshots.loc[
        snapshots["games_played"].notna()
        & snapshots["tri_code"].notna()
    ].copy()

    canonical_parts.append(snapshots)

    season_summary_rows.append({
        "season_id": season_id,
        "source_rows": len(season_games),
        "snapshot_rows": len(snapshots),
        "dates": snapshots["game_date"].nunique(),
        "teams": snapshots["team_id"].nunique(),
    })

    print(
        f"Season {season_id}: "
        f"{len(season_games):,} game rows -> "
        f"{len(snapshots):,} snapshots"
    )

df_canonical_all = pd.concat(
    canonical_parts,
    ignore_index=True
)

season_summary = pd.DataFrame(
    season_summary_rows
)

check(
    7,
    "Canonical snapshots constructed",
    not df_canonical_all.empty,
    f"{len(df_canonical_all):,} rows"
)

check(
    8,
    "Canonical team-date keys unique",
    not df_canonical_all.duplicated(KEY_COLS).any()
)

check(
    9,
    "All eligible seasons represented",
    set(df_canonical_all["season_id"].unique())
    == set(seasons)
)

# ============================================================
# 3. VERIFY SOURCE GAME COVERAGE
# ============================================================

matched_source = df_source[
    KEY_COLS
].merge(
    df_canonical_all[KEY_COLS],
    on=KEY_COLS,
    how="inner",
    validate="one_to_one"
)

check(
    10,
    "Every source game has a canonical snapshot",
    len(matched_source) == len(df_source),
    f"{len(matched_source):,} / {len(df_source):,}"
)

# ============================================================
# 4. CALCULATE FEATURES ON COMPLETE SNAPSHOTS
# ============================================================

print("\nBuilding canonical historical features...")

df_canonical_features_all = build_features(
    df_canonical_all.copy()
)

check(
    11,
    "Feature engineering preserves snapshot count",
    len(df_canonical_features_all)
    == len(df_canonical_all)
)

check(
    12,
    "Feature engineering preserves unique keys",
    not df_canonical_features_all.duplicated(
        KEY_COLS
    ).any()
)

# ============================================================
# 5. RETAIN ORIGINAL TEAM-GAME OBSERVATIONS
# ============================================================

game_keys = df_source[
    KEY_COLS
].drop_duplicates()

df_training_canonical = (
    df_canonical_features_all.merge(
        game_keys,
        on=KEY_COLS,
        how="inner",
        validate="one_to_one"
    )
)

check(
    13,
    "Original training row count preserved",
    len(df_training_canonical) == len(df_source),
    f"{len(df_training_canonical):,} / {len(df_source):,}"
)

check(
    14,
    "Training observations unique",
    not df_training_canonical.duplicated(
        KEY_COLS
    ).any()
)

# ============================================================
# 6. VALIDATE SOURCE IDENTIFIERS AND TARGETS
# ============================================================

source_check = df_source[
    KEY_COLS + [
        "game_id",
        TARGET_COL,
        "games_played"
    ]
].merge(
    df_training_canonical[
        KEY_COLS + [
            "game_id",
            TARGET_COL,
            "games_played"
        ]
    ],
    on=KEY_COLS,
    suffixes=("_source", "_canonical"),
    validate="one_to_one"
)

check(
    15,
    "Game IDs preserved",
    (
        source_check["game_id_source"]
        == source_check["game_id_canonical"]
    ).all()
)

check(
    16,
    "Playoff targets preserved",
    (
        source_check[f"{TARGET_COL}_source"]
        == source_check[f"{TARGET_COL}_canonical"]
    ).all()
)

check(
    17,
    "Games played preserved",
    np.allclose(
        source_check["games_played_source"],
        source_check["games_played_canonical"],
        equal_nan=True
    )
)

# ============================================================
# 7. VALIDATE HISTORICAL TARGETS
# ============================================================

target_check = (
    df_training_canonical
    .groupby(
        ["season_id", "team_id"]
    )[TARGET_COL]
    .nunique(dropna=False)
)

check(
    18,
    "Playoff targets constant within team-season",
    target_check.eq(1).all()
)

check(
    19,
    "Training and holdout seasons represented",
    (
        df_training_canonical["season_id"]
        < 20252026
    ).any()
    and (
        df_training_canonical["season_id"]
        == 20252026
    ).any()
)

# ============================================================
# 8. VALIDATE HISTORICAL SEASON COVERAGE
# ============================================================

check(
    20,
    "Snapshots at least as numerous as game rows",
    (
        season_summary["snapshot_rows"]
        >= season_summary["source_rows"]
    ).all()
)

check(
    21,
    "No season exceeds 32 distinct teams",
    season_summary["teams"].le(32).all()
)

check(
    22,
    "No snapshot has a future source game",
    (
        df_canonical_all["game_date"]
        >= df_canonical_all.groupby(
            ["season_id", "team_id"]
        )["game_date"].transform("min")
    ).all()
)

# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("HISTORICAL SEASON COVERAGE")
print("=" * 65)

print(
    season_summary.to_string(
        index=False
    )
)

print("\n" + "=" * 65)
print("TRAINING DATASET SUMMARY")
print("=" * 65)

print(f"Seasons: {len(seasons)}")
print(
    f"Original team-game rows: "
    f"{len(df_source):,}"
)
print(
    f"Canonical daily snapshots: "
    f"{len(df_canonical_all):,}"
)
print(
    f"Final training rows: "
    f"{len(df_training_canonical):,}"
)
print(
    f"Feature dataframe columns: "
    f"{len(df_canonical_features_all.columns)}"
)

print("\n" + "=" * 65)
print(
    f"BLOCK 58A: {sum(results)}/{len(results)} PASS"
)
print("=" * 65)

# DataFrames retained for inspection:
#
# df_source
# df_canonical_all
# df_canonical_features_all
# df_training_canonical
# season_summary


In [ ]:

# ============================================================
# BLOCK 58B — HISTORICAL POINT-IN-TIME INTEGRITY
# READ ONLY — NO DATABASE WRITES OR MODEL SAVES
#
# Requires Block 58A:
#   df_source
#   df_canonical_all
#   df_training_canonical
# ============================================================

import numpy as np
import pandas as pd

KEY_COLS = ["season_id", "game_date", "team_id"]
TEAM_COLS = ["season_id", "team_id"]

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== BLOCK 58B: HISTORICAL POINT-IN-TIME INTEGRITY ===\n")

# ============================================================
# 1. PREPARE INDEPENDENT SOURCE AND SNAPSHOT DATA
# ============================================================

source = df_source.copy()
snapshots = df_canonical_all.copy()
training = df_training_canonical.copy()

for frame in [source, snapshots, training]:
    frame["game_date"] = pd.to_datetime(
        frame["game_date"]
    ).dt.normalize()

check(
    1,
    "Source team-date keys unique",
    not source.duplicated(KEY_COLS).any()
)

check(
    2,
    "Snapshot team-date keys unique",
    not snapshots.duplicated(KEY_COLS).any()
)

check(
    3,
    "Training team-date keys unique",
    not training.duplicated(KEY_COLS).any()
)

check(
    4,
    "All snapshot seasons exist in source",
    set(snapshots["season_id"].unique()).issubset(
        set(source["season_id"].unique())
    )
)

# ============================================================
# 2. INDEPENDENTLY IDENTIFY THE LATEST SOURCE GAME
#
# Uses a different method from Block 58A:
# groupby + forward fill of the latest source date.
# ============================================================

source_dates = source[
    KEY_COLS
].copy()

source_dates["is_source_game"] = True

timeline = snapshots[
    KEY_COLS
].merge(
    source_dates,
    on=KEY_COLS,
    how="left",
    validate="one_to_one"
)

timeline["is_source_game"] = (
    timeline["is_source_game"]
    .fillna(False)
    .astype(bool)
)

timeline = timeline.sort_values(
    ["season_id", "team_id", "game_date"]
).reset_index(drop=True)

timeline["expected_source_date"] = (
    timeline["game_date"]
    .where(timeline["is_source_game"])
)

timeline["expected_source_date"] = (
    timeline.groupby(TEAM_COLS)[
        "expected_source_date"
    ].ffill()
)

check(
    5,
    "Every snapshot has a historical source game",
    timeline["expected_source_date"].notna().all()
)

check(
    6,
    "No future source date selected",
    (
        timeline["expected_source_date"]
        <= timeline["game_date"]
    ).all()
)

# ============================================================
# 3. ATTACH INDEPENDENTLY EXPECTED SOURCE VALUES
# ============================================================

STAT_COLS = [
    "games_played",
    "points",
    "points_pct",
    "wins",
    "losses",
    "so_losses",
    "goal_diff",
    "corsi_for",
    "corsi_against",
    "corsi_diff_per_game",
    "conference_rank",
    "division_rank",
    "wildcard_rank",
    "sos_future",
]

STAT_COLS = [
    c for c in STAT_COLS
    if c in source.columns
    and c in snapshots.columns
]

source_values = source[
    KEY_COLS + ["game_id"] + STAT_COLS
].rename(
    columns={
        "game_date": "expected_source_date",
        "game_id": "expected_game_id",
        **{
            c: f"expected_{c}"
            for c in STAT_COLS
        }
    }
)

expected = timeline.merge(
    source_values,
    on=[
        "season_id",
        "team_id",
        "expected_source_date"
    ],
    how="left",
    validate="many_to_one"
)

actual = expected.merge(
    snapshots[
        KEY_COLS + ["game_id"] + STAT_COLS
    ],
    on=KEY_COLS,
    how="left",
    validate="one_to_one"
)

check(
    7,
    "Expected game IDs available",
    actual["expected_game_id"].notna().all()
)

check(
    8,
    "Snapshot game IDs match latest historical games",
    (
        actual["game_id"]
        == actual["expected_game_id"]
    ).all()
)

# ============================================================
# 4. VALIDATE ALL SELECTED STATISTICS
# ============================================================

stat_results = []

for col in STAT_COLS:

    observed = pd.to_numeric(
        actual[col],
        errors="coerce"
    ).to_numpy(dtype=float)

    expected_values = pd.to_numeric(
        actual[f"expected_{col}"],
        errors="coerce"
    ).to_numpy(dtype=float)

    mismatch = ~np.isclose(
        observed,
        expected_values,
        atol=1e-8,
        rtol=1e-6,
        equal_nan=True
    )

    stat_results.append({
        "feature": col,
        "mismatch_rows": int(mismatch.sum())
    })

stat_results = pd.DataFrame(stat_results)

check(
    9,
    "All selected statistics match historical source",
    stat_results["mismatch_rows"].sum() == 0,
    f"{len(STAT_COLS)} statistics checked"
)

# ============================================================
# 5. VALIDATE NON-GAME-DAY CARRY-FORWARD
# ============================================================

non_game = actual.loc[
    ~actual["is_source_game"]
].copy()

game_day = actual.loc[
    actual["is_source_game"]
].copy()

check(
    10,
    "Non-game-day snapshots exist",
    len(non_game) > 0,
    f"{len(non_game):,} rows"
)

check(
    11,
    "Non-game-day source dates precede snapshot dates",
    (
        non_game["expected_source_date"]
        < non_game["game_date"]
    ).all()
)

check(
    12,
    "Game-day source dates equal snapshot dates",
    (
        game_day["expected_source_date"]
        == game_day["game_date"]
    ).all()
)

check(
    13,
    "Game-day snapshot count matches source",
    len(game_day) == len(source),
    f"{len(game_day):,} / {len(source):,}"
)

# ============================================================
# 6. VALIDATE SEASON BOUNDARIES
# ============================================================

source_seasons = (
    source.groupby("team_id")["season_id"]
    .nunique()
)

check(
    14,
    "Multiple seasons represented",
    source["season_id"].nunique() > 1,
    f"{source['season_id'].nunique()} seasons"
)

check(
    15,
    "Snapshot game IDs belong to correct season",
    actual[
        ["season_id", "game_id"]
    ].merge(
        source[
            ["season_id", "game_id"]
        ].drop_duplicates(),
        on=["season_id", "game_id"],
        how="left",
        indicator=True
    )["_merge"].eq("both").all()
)

# ============================================================
# 7. VALIDATE TRAINING ROWS
# ============================================================

train_check = source[
    KEY_COLS + [
        "game_id",
        "made_playoffs",
        "games_played"
    ]
].merge(
    training[
        KEY_COLS + [
            "game_id",
            "made_playoffs",
            "games_played"
        ]
    ],
    on=KEY_COLS,
    suffixes=("_source", "_training"),
    validate="one_to_one"
)

check(
    16,
    "All source game rows retained for training",
    len(train_check) == len(source)
)

check(
    17,
    "Training game IDs preserved",
    (
        train_check["game_id_source"]
        == train_check["game_id_training"]
    ).all()
)

check(
    18,
    "Training playoff targets preserved",
    (
        train_check["made_playoffs_source"]
        == train_check["made_playoffs_training"]
    ).all()
)

check(
    19,
    "Training games played preserved",
    np.allclose(
        train_check["games_played_source"],
        train_check["games_played_training"],
        equal_nan=True
    )
)

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("HISTORICAL SOURCE VALIDATION")
print("=" * 65)

print(f"Seasons: {source['season_id'].nunique()}")
print(f"Source game rows: {len(source):,}")
print(f"Canonical snapshots: {len(snapshots):,}")
print(f"Game-day snapshots: {len(game_day):,}")
print(f"Non-game-day snapshots: {len(non_game):,}")
print(f"Statistics validated: {len(STAT_COLS)}")

print("\nSTATISTIC MISMATCHES")
print(
    stat_results.to_string(index=False)
)

print("\n" + "=" * 65)
print(
    f"BLOCK 58B: {sum(results)}/{len(results)} PASS"
)
print("=" * 65)

# DataFrames retained:
# timeline
# actual
# stat_results
# non_game
# game_day
# train_check


In [ ]:

# ============================================================
# BLOCK 59A — CANONICAL TRAINING FUNCTION VALIDATION
# READ ONLY — NO TRAINING OR MODEL SAVES
# ============================================================

import importlib
import pandas as pd
import numpy as np

from config import EXCLUDE_SEASON_ID

MODULE_NAME = "prediction_model.training.train_new_model"

training_module = importlib.import_module(MODULE_NAME)
importlib.reload(training_module)

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== BLOCK 59A: TRAINING FUNCTION VALIDATION ===\n")

# Reproduce the source ordering used in main()
source = df_source.copy()

if "game_id" in source.columns:
    source = source.sort_values(
        "game_id"
    ).reset_index(drop=True)

# Execute the updated training helper
actual = training_module.build_canonical_training_features(
    source
)

expected = df_training_canonical.copy()

keys = ["season_id", "game_date", "team_id"]

for frame in [actual, expected]:
    frame["game_date"] = pd.to_datetime(
        frame["game_date"]
    ).dt.normalize()

check(
    1,
    "Original training row count preserved",
    len(actual) == len(source),
    f"{len(actual):,} rows"
)

check(
    2,
    "Canonical training keys unique",
    not actual.duplicated(keys).any()
)

check(
    3,
    "All historical seasons retained",
    set(actual["season_id"])
    == set(expected["season_id"])
)

check(
    4,
    "Same columns as validated canonical dataset",
    set(actual.columns) == set(expected.columns),
    f"{len(actual.columns)} columns"
)

# Compare using team-date keys, independent of row order
common_cols = sorted(
    set(actual.columns).intersection(expected.columns)
    - set(keys)
)

comparison = actual.merge(
    expected,
    on=keys,
    how="outer",
    suffixes=("_actual", "_expected"),
    indicator=True,
    validate="one_to_one"
)

check(
    5,
    "Every canonical training key matches Block 58A",
    comparison["_merge"].eq("both").all()
)

mismatches = []

for col in common_cols:
    left = comparison[f"{col}_actual"]
    right = comparison[f"{col}_expected"]

    if pd.api.types.is_numeric_dtype(left) and \
       pd.api.types.is_numeric_dtype(right):

        equal = np.isclose(
            pd.to_numeric(left, errors="coerce"),
            pd.to_numeric(right, errors="coerce"),
            rtol=1e-9,
            atol=1e-9,
            equal_nan=True
        )
    else:
        equal = (
            left.eq(right)
            | (left.isna() & right.isna())
        ).fillna(False)

    count = int((~equal).sum())

    if count:
        mismatches.append({
            "column": col,
            "mismatch_rows": count
        })

check(
    6,
    "All canonical feature values match Block 58A",
    len(mismatches) == 0,
    f"{len(common_cols)} columns compared"
)

check(
    7,
    "Original game IDs preserved",
    actual["game_id"].reset_index(drop=True).equals(
        source["game_id"].reset_index(drop=True)
    )
)

check(
    8,
    "Original playoff targets preserved",
    actual["made_playoffs"].reset_index(drop=True).equals(
        source["made_playoffs"].reset_index(drop=True)
    )
)

check(
    9,
    "Original games-played values preserved",
    actual["games_played"].reset_index(drop=True).equals(
        source["games_played"].reset_index(drop=True)
    )
)

# Build the actual model input matrix
X, feature_cols = training_module._build_feature_matrix(
    actual
)

check(
    10,
    "Model feature matrix retains every training row",
    len(X) == len(source)
)

check(
    11,
    "Model feature matrix has no null values",
    not X.isna().any().any()
)

check(
    12,
    "Model feature names are unique",
    len(feature_cols) == len(set(feature_cols))
)

check(
    13,
    "Playoff target excluded from model inputs",
    "made_playoffs" not in feature_cols
)

check(
    14,
    "Identifiers excluded from model inputs",
    not any(
        col in feature_cols
        for col in [
            "season_id",
            "team_id",
            "game_id",
            "game_date"
        ]
    )
)

check(
    15,
    "Both training and holdout seasons represented",
    (actual["season_id"] < 20252026).any()
    and (actual["season_id"] == 20252026).any()
)

print("\n" + "=" * 60)

if mismatches:
    print("COLUMN MISMATCHES")
    print(pd.DataFrame(mismatches).to_string(index=False))
else:
    print("All canonical columns match.")

print(f"\nModel input features: {len(feature_cols)}")
print(f"Training observations: {len(X):,}")

print(
    f"\nBLOCK 59A: {sum(results)}/{len(results)} PASS"
)
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 59B — ORIGINAL VALUE PRESERVATION
# READ ONLY
# ============================================================

import numpy as np
import pandas as pd

print("\n=== BLOCK 59B: ORIGINAL VALUE PRESERVATION ===\n")

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

columns = [
    "game_id",
    "made_playoffs",
    "games_played"
]

for number, col in enumerate(columns, start=1):

    left = pd.to_numeric(
        source[col].reset_index(drop=True),
        errors="coerce"
    )

    right = pd.to_numeric(
        actual[col].reset_index(drop=True),
        errors="coerce"
    )

    equal = np.isclose(
        left.to_numpy(dtype=float),
        right.to_numpy(dtype=float),
        rtol=0,
        atol=0,
        equal_nan=True
    )

    mismatches = int((~equal).sum())

    print(f"\nColumn: {col}")
    print(f"  Source dtype: {source[col].dtype}")
    print(f"  Actual dtype: {actual[col].dtype}")
    print(f"  Numeric mismatches: {mismatches:,}")

    check(
        number,
        f"{col} values preserved",
        mismatches == 0
    )

    if mismatches:
        print("\nFirst mismatches:")
        print(
            pd.DataFrame({
                "source": left[~equal].head(10).to_numpy(),
                "actual": right[~equal].head(10).to_numpy()
            }).to_string(index=False)
        )

print("\n" + "=" * 60)
print(
    f"BLOCK 59B: {sum(results)}/{len(results)} PASS"
)
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 60A — V7 MODEL OUTPUT SAFETY
# READ ONLY — DOES NOT TRAIN OR SAVE A MODEL
# ============================================================

import importlib
from pathlib import Path
from config import MODEL_PATH, MODEL_VERSION

module = importlib.import_module(
    "prediction_model.training.train_new_model"
)
module = importlib.reload(module)

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

production_path = Path(MODEL_PATH).resolve()
candidate_path = Path(module.MODEL_OUT_PATH).resolve()

print("\n=== BLOCK 60A: V7 MODEL OUTPUT SAFETY ===\n")

print(f"Production version: {MODEL_VERSION}")
print(f"Production path: {production_path}")
print(f"Candidate version: {module.TRAINING_MODEL_VERSION}")
print(f"Candidate path: {candidate_path}")

check(
    1,
    "Candidate version is v7",
    module.TRAINING_MODEL_VERSION == "v7"
)

check(
    2,
    "Production version remains v6",
    MODEL_VERSION == "v6"
)

check(
    3,
    "Candidate and production paths differ",
    candidate_path != production_path
)

check(
    4,
    "Candidate model has a .joblib or .pkl extension",
    candidate_path.suffix.lower() in [".joblib", ".pkl"]
)

check(
    5,
    "Production model exists",
    production_path.is_file()
)

check(
    6,
    "Candidate output directory exists",
    candidate_path.parent.is_dir()
)

check(
    7,
    "Candidate file does not already exist",
    not candidate_path.exists()
)

print("\n" + "=" * 60)
print(f"BLOCK 60A: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:
import importlib
from prediction_model.training import train_new_model

importlib.reload(train_new_model)

train_new_model.main()

In [ ]:

# ============================================================
# BLOCK 61A — SAVED V7 MODEL BUNDLE VALIDATION
# READ ONLY
# ============================================================

import importlib
import numpy as np
from pathlib import Path
from joblib import load
from config import MODEL_PATH

training = importlib.import_module(
    "prediction_model.training.train_new_model"
)
training = importlib.reload(training)

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== BLOCK 61A: SAVED V7 MODEL VALIDATION ===\n")

v6_path = Path(MODEL_PATH)
v7_path = Path(training.MODEL_OUT_PATH)

check(1, "Production v6 exists", v6_path.is_file())
check(2, "Candidate v7 exists", v7_path.is_file())
check(3, "Model paths differ", v6_path.resolve() != v7_path.resolve())

bundle = load(v7_path)

required = [
    "xgb_pipeline",
    "rf_pipeline",
    "features",
    "version",
    "trained_on",
    "metrics",
    "importance_by_bucket",
]

check(4, "All required bundle keys present",
      all(k in bundle for k in required))

check(5, "Saved version is v7", bundle.get("version") == "v7")

features = bundle["features"]

check(6, "Exactly 121 selected features", len(features) == 121)
check(7, "Feature names unique", len(features) == len(set(features)))
check(8, "Playoff target excluded", "made_playoffs" not in features)
check(9, "Identifiers excluded",
      not any(c in features for c in ["game_id", "team_id", "season_id"]))

check(10, "XGBoost pipeline fitted",
      hasattr(bundle["xgb_pipeline"].named_steps["model"], "classes_"))

check(11, "Random Forest pipeline fitted",
      hasattr(bundle["rf_pipeline"].named_steps["model"], "classes_"))

metrics = bundle["metrics"]

check(12, "Final training rows correct",
      metrics.get("final_training_rows") == 31944)

check(13, "Holdout rows correct",
      metrics.get("evaluation_test_rows") == 2624)

check(14, "Holdout Brier scores valid",
      all(
          np.isfinite(metrics.get(k, np.nan))
          and 0 <= metrics[k] <= 1
          for k in ["holdout_xgb_brier", "holdout_rf_brier"]
      ))

check(15, "Season-stage SHAP importance saved",
      all(
          stage in bundle["importance_by_bucket"]
          for stage in ["early", "mid", "late"]
      ))

print("\n" + "=" * 60)
print(f"BLOCK 61A: {sum(results)}/{len(results)} PASS")
print("=" * 60)

print("\nSaved model:", v7_path)
print("Feature count:", len(features))
print("Metrics:", metrics)


In [ ]:

# ============================================================
# BLOCK 61B — V7 INFERENCE SMOKE TEST
# READ ONLY — NO DATABASE WRITES OR MODEL SAVES
# ============================================================

import numpy as np
import pandas as pd
from joblib import load
from config import MODEL_PATH
from pathlib import Path

v7_path = Path(MODEL_PATH).with_name("playoff_models_v7.joblib")
bundle = load(v7_path)

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== BLOCK 61B: V7 INFERENCE SMOKE TEST ===\n")

features = bundle["features"]

# Use the latest complete historical date for a 32-team snapshot
canonical = df_canonical_features_all.copy()
canonical["game_date"] = pd.to_datetime(canonical["game_date"])

season = 20252026
season_df = canonical[canonical["season_id"] == season].copy()
latest_date = season_df["game_date"].max()

snapshot = season_df[
    season_df["game_date"] == latest_date
].copy()

print(f"Season: {season}")
print(f"Snapshot date: {latest_date.date()}")
print(f"Teams: {snapshot['team_id'].nunique()}")

check(1, "Snapshot has 32 teams", snapshot["team_id"].nunique() == 32)
check(2, "One row per team", not snapshot["team_id"].duplicated().any())
check(3, "All v7 features available",
      all(col in snapshot.columns for col in features))

X = snapshot.reindex(columns=features).fillna(0)

check(4, "Feature order matches saved bundle",
      X.columns.tolist() == features)

check(5, "Feature matrix contains no null values",
      not X.isna().any().any())

xgb_prob = bundle["xgb_pipeline"].predict_proba(X)[:, 1]
rf_prob = bundle["rf_pipeline"].predict_proba(X)[:, 1]
ensemble_prob = (xgb_prob + rf_prob) / 2

check(6, "XGBoost returns one prediction per team",
      len(xgb_prob) == len(snapshot))

check(7, "Random Forest returns one prediction per team",
      len(rf_prob) == len(snapshot))

check(8, "XGBoost probabilities finite and within [0, 1]",
      np.isfinite(xgb_prob).all()
      and ((xgb_prob >= 0) & (xgb_prob <= 1)).all())

check(9, "Random Forest probabilities finite and within [0, 1]",
      np.isfinite(rf_prob).all()
      and ((rf_prob >= 0) & (rf_prob <= 1)).all())

check(10, "Ensemble probabilities finite and within [0, 1]",
      np.isfinite(ensemble_prob).all()
      and ((ensemble_prob >= 0) & (ensemble_prob <= 1)).all())

check(11, "Predictions vary across teams",
      np.unique(np.round(ensemble_prob, 6)).size > 1)

predictions = snapshot[
    ["team_id", "tri_code", "games_played"]
].copy()

predictions["xgb_prob"] = xgb_prob
predictions["rf_prob"] = rf_prob
predictions["ensemble_prob"] = ensemble_prob

print("\n=== V7 PREDICTIONS ===\n")
print(
    predictions.sort_values(
        "ensemble_prob", ascending=False
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "=" * 60)
print(f"BLOCK 61B: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 62A — V6 VS V7 MODEL COMPARISON
# READ ONLY — NO DATABASE WRITES OR MODEL SAVES
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from joblib import load
from config import MODEL_PATH

print("\n=== BLOCK 62A: V6 VS V7 COMPARISON ===\n")

v6_path = Path(MODEL_PATH)
v7_path = v6_path.with_name("playoff_models_v7.joblib")

v6 = load(v6_path)
v7 = load(v7_path)

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

features_v6 = list(v6["features"])
features_v7 = list(v7["features"])

common = sorted(set(features_v6) & set(features_v7))
only_v6 = sorted(set(features_v6) - set(features_v7))
only_v7 = sorted(set(features_v7) - set(features_v6))

print("FEATURE COMPARISON")
print(f"V6 features: {len(features_v6)}")
print(f"V7 features: {len(features_v7)}")
print(f"Shared features: {len(common)}")
print(f"Only in v6: {len(only_v6)}")
print(f"Only in v7: {len(only_v7)}")

print("\nFeatures only in v6:", only_v6)
print("Features only in v7:", only_v7)

check(1, "Both model bundles have feature lists",
      bool(features_v6) and bool(features_v7))

check(2, "Both feature lists contain unique names",
      len(features_v6) == len(set(features_v6))
      and len(features_v7) == len(set(features_v7)))

check(3, "All model features exist in canonical snapshots",
      all(f in snapshot.columns for f in set(features_v6 + features_v7)))

check(4, "Model versions correctly identified",
      v6.get("version") == "v6"
      and v7.get("version") == "v7")

# Use the same 32-team snapshot from Block 61B
X6 = snapshot.reindex(columns=features_v6).fillna(0)
X7 = snapshot.reindex(columns=features_v7).fillna(0)

p6_xgb = v6["xgb_pipeline"].predict_proba(X6)[:, 1]
p6_rf = v6["rf_pipeline"].predict_proba(X6)[:, 1]
p7_xgb = v7["xgb_pipeline"].predict_proba(X7)[:, 1]
p7_rf = v7["rf_pipeline"].predict_proba(X7)[:, 1]

p6 = (p6_xgb + p6_rf) / 2
p7 = (p7_xgb + p7_rf) / 2

check(5, "Both models predict all 32 teams",
      len(p6) == len(p7) == 32)

check(6, "All predictions finite",
      np.isfinite(p6).all() and np.isfinite(p7).all())

check(7, "All probabilities within [0, 1]",
      ((p6 >= 0) & (p6 <= 1)).all()
      and ((p7 >= 0) & (p7 <= 1)).all())

comparison = snapshot[
    ["team_id", "tri_code", "games_played"]
].copy()

comparison["v6_prob"] = p6
comparison["v7_prob"] = p7
comparison["change"] = comparison["v7_prob"] - comparison["v6_prob"]
comparison["abs_change"] = comparison["change"].abs()

check(8, "Comparison has unique teams",
      comparison["team_id"].is_unique)

check(9, "Probability differences are finite",
      np.isfinite(comparison["change"]).all())

print("\nPROBABILITY DIFFERENCE SUMMARY")
print(f"Mean absolute change: {comparison['abs_change'].mean():.4f}")
print(f"Median absolute change: {comparison['abs_change'].median():.4f}")
print(f"Maximum absolute change: {comparison['abs_change'].max():.4f}")

print("\nLARGEST V6 → V7 CHANGES\n")
print(
    comparison.sort_values(
        "abs_change", ascending=False
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "=" * 60)
print(f"BLOCK 62A: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 62B — V6 VS V7 ACROSS SEASON STAGES
# READ ONLY
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from joblib import load
from config import MODEL_PATH

print("\n=== BLOCK 62B: V6 VS V7 SEASON-STAGE COMPARISON ===\n")

v6 = load(Path(MODEL_PATH))
v7 = load(Path(MODEL_PATH).with_name("playoff_models_v7.joblib"))

canonical = df_canonical_features_all.copy()
canonical["game_date"] = pd.to_datetime(canonical["game_date"])

season_df = canonical[
    canonical["season_id"] == 20252026
].copy()

# Compare identical complete daily snapshots
checkpoints = {
    "early": "2025-11-15",
    "mid": "2026-01-15",
    "late": "2026-03-15",
}

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

all_comparisons = []

for stage, requested_date in checkpoints.items():

    available_dates = season_df.loc[
        season_df["game_date"] <= pd.Timestamp(requested_date),
        "game_date"
    ]

    if available_dates.empty:
        raise RuntimeError(f"No snapshot for {stage}")

    date = available_dates.max()

    snapshot_stage = season_df[
        season_df["game_date"] == date
    ].copy()

    X6 = snapshot_stage.reindex(
        columns=v6["features"]
    ).fillna(0)

    X7 = snapshot_stage.reindex(
        columns=v7["features"]
    ).fillna(0)

    p6 = (
        v6["xgb_pipeline"].predict_proba(X6)[:, 1]
        + v6["rf_pipeline"].predict_proba(X6)[:, 1]
    ) / 2

    p7 = (
        v7["xgb_pipeline"].predict_proba(X7)[:, 1]
        + v7["rf_pipeline"].predict_proba(X7)[:, 1]
    ) / 2

    comparison = snapshot_stage[
        ["team_id", "tri_code", "games_played"]
    ].copy()

    comparison["stage"] = stage
    comparison["date"] = date
    comparison["v6_prob"] = p6
    comparison["v7_prob"] = p7
    comparison["change"] = p7 - p6
    comparison["abs_change"] = np.abs(p7 - p6)

    all_comparisons.append(comparison)

    check(
        len(results) + 1,
        f"{stage} snapshot contains 32 teams",
        len(comparison) == 32
        and comparison["team_id"].is_unique,
        str(date.date())
    )

comparison_all = pd.concat(
    all_comparisons, ignore_index=True
)

summary = (
    comparison_all.groupby("stage")
    .agg(
        teams=("team_id", "nunique"),
        mean_abs_change=("abs_change", "mean"),
        median_abs_change=("abs_change", "median"),
        max_abs_change=("abs_change", "max"),
        mean_v6_prob=("v6_prob", "mean"),
        mean_v7_prob=("v7_prob", "mean"),
    )
    .reindex(["early", "mid", "late"])
)

print("\n=== SEASON-STAGE SUMMARY ===\n")
print(summary.to_string(float_format=lambda x: f"{x:.4f}"))

print("\n=== LARGEST DIFFERENCES BY STAGE ===\n")

for stage in checkpoints:

    print(f"\n{stage.upper()}")

    subset = comparison_all[
        comparison_all["stage"] == stage
    ]

    print(
        subset.nlargest(10, "abs_change")[
            ["tri_code", "games_played",
             "v6_prob", "v7_prob", "change"]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

check(
    4,
    "All 96 comparisons have finite probabilities",
    len(comparison_all) == 96
    and np.isfinite(
        comparison_all[["v6_prob", "v7_prob"]].to_numpy()
    ).all()
)

check(
    5,
    "All probabilities within [0, 1]",
    comparison_all[
        ["v6_prob", "v7_prob"]
    ].ge(0).all().all()
    and comparison_all[
        ["v6_prob", "v7_prob"]
    ].le(1).all().all()
)

print("\n" + "=" * 60)
print(f"BLOCK 62B: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 63A — DAILY INFERENCE FEATURE CONSISTENCY
# READ ONLY
# ============================================================

import numpy as np
import pandas as pd
from feature_engineering import build_features

print("\n=== BLOCK 63A: DAILY FEATURE CONSISTENCY ===\n")

batch_features = [
    "corsi_net_rel",
    "wc_teams_within_6pts",
    "wc2_points_pct",
    "wc_points_pct_margin",
    "wc_pressure_x_performance",
    "wc_teams_within_4pts",
    "wc_teams_within_2pts",
    "wc2_points",
    "wc_points_margin",
]

keys = ["season_id", "game_date", "team_id"]

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# Use a midseason date where wildcard calculations matter
season = 20252026
requested_date = pd.Timestamp("2026-01-15")

canonical = df_canonical_all.copy()
canonical["game_date"] = pd.to_datetime(canonical["game_date"])

reference = df_canonical_features_all.copy()
reference["game_date"] = pd.to_datetime(reference["game_date"])

available = canonical.loc[
    (canonical["season_id"] == season)
    & (canonical["game_date"] <= requested_date),
    "game_date"
]

if available.empty:
    raise RuntimeError("No eligible snapshot date found")

snapshot_date = available.max()

# IMPORTANT: use ALL teams, not just teams playing
raw_snapshot = canonical.loc[
    (canonical["season_id"] == season)
    & (canonical["game_date"] == snapshot_date)
].copy()

expected = reference.loc[
    (reference["season_id"] == season)
    & (reference["game_date"] == snapshot_date)
].copy()

actual = build_features(raw_snapshot.copy())

print(f"Snapshot date: {snapshot_date.date()}")
print(f"Raw teams: {raw_snapshot['team_id'].nunique()}")
print(f"Expected teams: {expected['team_id'].nunique()}")
print(f"Actual teams: {actual['team_id'].nunique()}")

check(1, "Raw snapshot has 32 teams",
      raw_snapshot["team_id"].nunique() == 32)

check(2, "Engineered snapshot has 32 teams",
      actual["team_id"].nunique() == 32)

check(3, "No duplicate team keys",
      not actual.duplicated(keys).any())

check(4, "All 9 batch-dependent features exist",
      all(c in actual.columns for c in batch_features))

check(5, "All 9 reference features exist",
      all(c in expected.columns for c in batch_features))

comparison = expected[
    keys + batch_features
].merge(
    actual[keys + batch_features],
    on=keys,
    suffixes=("_expected", "_actual"),
    validate="one_to_one"
)

check(6, "All 32 teams matched",
      len(comparison) == 32)

print("\n=== FEATURE COMPARISON ===\n")

for i, feature in enumerate(batch_features, start=7):
    left = pd.to_numeric(
        comparison[f"{feature}_expected"],
        errors="coerce"
    ).to_numpy(dtype=float)

    right = pd.to_numeric(
        comparison[f"{feature}_actual"],
        errors="coerce"
    ).to_numpy(dtype=float)

    equal = np.isclose(
        left, right,
        rtol=1e-8,
        atol=1e-8,
        equal_nan=True
    )

    check(
        i,
        f"{feature} matches canonical reference",
        equal.all(),
        f"{equal.sum()}/32 teams"
    )

print("\n" + "=" * 60)
print(f"BLOCK 63A: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 63B — PRODUCTION INFERENCE PATH INSPECTION
# READ ONLY — DOES NOT EXECUTE THE RUNNER
# ============================================================

import inspect
import importlib
import ast

print("\n=== BLOCK 63B: PRODUCTION INFERENCE PATH ===\n")

modules = {
    "daily_runner": "prediction_model.inference.run_daily_playoff_predictions",
    "inference": "prediction_model.inference.inference",
}

for label, module_path in modules.items():
    print("\n" + "=" * 70)
    print(f"MODULE: {label}")
    print("=" * 70)

    try:
        module = importlib.import_module(module_path)
        source = inspect.getsource(module)
        tree = ast.parse(source)

        print(f"File: {inspect.getfile(module)}")
        print(f"Source lines: {len(source.splitlines())}")

        print("\nFUNCTIONS:")
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                print(
                    f"  {node.name} "
                    f"(lines {node.lineno}-{node.end_lineno})"
                )

        print("\nRELEVANT SOURCE LINES:")
        keywords = [
            "build_features",
            "team_daily_stats",
            "team_id",
            "game_date",
            "predict_proba",
            "MODEL_PATH",
            "MODEL_VERSION",
            "playoff_models",
            "merge_asof",
            "groupby",
            "snapshot",
        ]

        lines = source.splitlines()
        relevant = set()

        for i, line in enumerate(lines):
            if any(word in line for word in keywords):
                relevant.update(
                    range(max(0, i - 4), min(len(lines), i + 5))
                )

        for i in sorted(relevant):
            print(f"{i + 1:4d} | {lines[i]}")

    except Exception as exc:
        print(f"INSPECTION ERROR: {type(exc).__name__}: {exc}")

print("\n=== BLOCK 63B COMPLETE ===")


In [ ]:

# ============================================================
# BLOCK 63D — MODEL VERSION ALIGNMENT
# READ ONLY
# ============================================================

from pathlib import Path
from joblib import load
import config
from prediction_model.metadata import MODEL_METADATA

print("\n=== BLOCK 63D: MODEL VERSION ALIGNMENT ===\n")

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

active_path = Path(config.MODEL_PATH)
active_bundle = load(active_path)

active_version = active_bundle.get("version")
config_version = str(config.MODEL_VERSION)
metadata_version = str(MODEL_METADATA["version"])

print(f"Config version:   {config_version}")
print(f"Metadata version: {metadata_version}")
print(f"Bundle version:   {active_version}")
print(f"Active model:     {active_path}")

check(1, "Active model file exists", active_path.is_file())

check(2, "Config matches saved model",
      config_version == active_version)

check(3, "Metadata matches saved model",
      metadata_version == active_version)

check(4, "Config and metadata agree",
      config_version == metadata_version)

check(5, "Production still points to v6",
      active_version == "v6")

v7_path = active_path.with_name("playoff_models_v7.joblib")

check(6, "Candidate v7 file exists",
      v7_path.is_file())

if v7_path.is_file():
    v7_bundle = load(v7_path)
    check(7, "Candidate bundle identifies as v7",
          v7_bundle.get("version") == "v7")

print("\n" + "=" * 60)
print(f"BLOCK 63D: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 64A — V7 INFERENCE INTEGRATION
# IN-MEMORY TEST ONLY — NO DATABASE WRITES
# ============================================================

import importlib
import numpy as np
import pandas as pd
from pathlib import Path
from joblib import load
from config import MODEL_PATH

inference_module = importlib.import_module(
    "prediction_model.inference.inference"
)

v7_path = Path(MODEL_PATH).with_name(
    "playoff_models_v7.joblib"
)
v7 = load(v7_path)

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== BLOCK 64A: V7 INFERENCE INTEGRATION ===\n")

# Save the currently active in-memory model references
original_xgb = inference_module.XGB_PIPELINE
original_rf = inference_module.RF_PIPELINE
original_features = inference_module.FEATURE_COLS

# Reuse validated canonical feature snapshot
canonical = df_canonical_features_all.copy()
canonical["game_date"] = pd.to_datetime(
    canonical["game_date"]
)

season_df = canonical[
    canonical["season_id"] == 20252026
]

test_date = pd.Timestamp("2026-01-15")

test_df = season_df[
    season_df["game_date"] == test_date
].copy()

check(1, "Integration snapshot has 32 teams",
      test_df["team_id"].nunique() == 32)

check(2, "All v7 features available",
      all(c in test_df.columns for c in v7["features"]))

try:
    # Substitute v7 ONLY within this Python process
    inference_module.XGB_PIPELINE = v7["xgb_pipeline"]
    inference_module.RF_PIPELINE = v7["rf_pipeline"]
    inference_module.FEATURE_COLS = v7["features"]

    scored = inference_module.score_dataframe(
        test_df.copy()
    )

finally:
    # Always restore original model references
    inference_module.XGB_PIPELINE = original_xgb
    inference_module.RF_PIPELINE = original_rf
    inference_module.FEATURE_COLS = original_features

check(3, "All 32 teams scored",
      len(scored) == 32)

check(4, "No duplicate teams",
      scored["team_id"].is_unique)

prob_cols = [
    "playoff_prob_xgb",
    "playoff_prob_rf",
    "playoff_prob",
]

check(5, "All probability columns returned",
      all(c in scored.columns for c in prob_cols))

check(6, "All probabilities finite",
      np.isfinite(scored[prob_cols].to_numpy()).all())

check(7, "All probabilities between 0 and 1",
      scored[prob_cols].ge(0).all().all()
      and scored[prob_cols].le(1).all().all())

expected_ensemble = (
    scored["playoff_prob_xgb"]
    + scored["playoff_prob_rf"]
) / 2

check(8, "Ensemble calculation correct",
      np.allclose(
          scored["playoff_prob"],
          expected_ensemble,
          atol=1e-12
      ))

check(9, "Original XGBoost reference restored",
      inference_module.XGB_PIPELINE is original_xgb)

check(10, "Original Random Forest reference restored",
      inference_module.RF_PIPELINE is original_rf)

check(11, "Original feature list restored",
      inference_module.FEATURE_COLS is original_features)

print("\n=== V7 INTEGRATION PREDICTIONS ===\n")
print(
    scored[
        ["tri_code", "games_played", "playoff_prob"]
    ]
    .sort_values("playoff_prob", ascending=False)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "=" * 60)
print(f"BLOCK 64A: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 64B — RUNNER SQL VS CANONICAL TRAINING SNAPSHOTS
# READ ONLY — NO PREDICTION OR DATABASE WRITES
# ============================================================

import duckdb
import numpy as np
import pandas as pd

from config import DB_PATH, SOURCE_TABLE

print("\n=== BLOCK 64B: RUNNER SNAPSHOT VALIDATION ===\n")

season_id = 20252026
keys = ["season_id", "game_date", "team_id"]

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# Reproduce the runner's snapshot query without executing
# the runner itself.
query = f"""
WITH teams AS (
    SELECT DISTINCT team_id
    FROM {SOURCE_TABLE}
    WHERE season_id = ?
),
dates AS (
    SELECT DISTINCT game_date
    FROM {SOURCE_TABLE}
    WHERE season_id = ?
),
spine AS (
    SELECT
        ?::BIGINT AS season_id,
        d.game_date,
        t.team_id
    FROM dates d
    CROSS JOIN teams t
)
SELECT
    snap.* EXCLUDE (season_id, game_date, team_id),
    spine.season_id,
    spine.game_date,
    spine.team_id
FROM spine
ASOF LEFT JOIN {SOURCE_TABLE} AS snap
    ON spine.team_id = snap.team_id
    AND spine.season_id = snap.season_id
    AND spine.game_date >= snap.game_date
ORDER BY spine.game_date, spine.team_id
"""

# ============================================================
# Use existing notebook connection
# ============================================================

runner_raw = con.execute(
    query,
    [season_id, season_id, season_id]
).df()

try:
    runner_raw = con.execute(
        query, [season_id, season_id, season_id]
    ).df()
finally:
    con.close()

runner_raw = runner_raw.loc[
    runner_raw["games_played"].notna()
    & runner_raw["tri_code"].notna()
].copy()

canonical = df_canonical_all.loc[
    df_canonical_all["season_id"] == season_id
].copy()

for df in [runner_raw, canonical]:
    df["game_date"] = pd.to_datetime(
        df["game_date"]
    ).dt.normalize()

check(1, "Runner returned nonempty data",
      not runner_raw.empty)

check(2, "Runner team-day keys unique",
      not runner_raw.duplicated(keys).any())

check(3, "Canonical team-day keys unique",
      not canonical.duplicated(keys).any())

check(4, "Runner and canonical row counts match",
      len(runner_raw) == len(canonical),
      f"runner={len(runner_raw)}, canonical={len(canonical)}")

runner_keys = pd.MultiIndex.from_frame(runner_raw[keys])
canonical_keys = pd.MultiIndex.from_frame(canonical[keys])

check(5, "All runner keys match canonical keys",
      runner_keys.symmetric_difference(canonical_keys).empty)

# Compare every shared numeric source column.
shared = sorted(
    set(runner_raw.columns)
    & set(canonical.columns)
    - set(keys)
)

numeric_cols = [
    col for col in shared
    if pd.api.types.is_numeric_dtype(runner_raw[col])
    and pd.api.types.is_numeric_dtype(canonical[col])
]

left = runner_raw.set_index(keys).sort_index()
right = canonical.set_index(keys).sort_index()

comparison_rows = []

for col in numeric_cols:
    a = pd.to_numeric(
        left[col], errors="coerce"
    ).to_numpy(dtype=float)

    b = pd.to_numeric(
        right[col], errors="coerce"
    ).to_numpy(dtype=float)

    equal = np.isclose(
        a, b, rtol=1e-8, atol=1e-8, equal_nan=True
    )

    comparison_rows.append({
        "feature": col,
        "matches": int(equal.sum()),
        "total": len(equal),
        "match_rate": float(equal.mean()),
    })

comparison_df = pd.DataFrame(comparison_rows)

check(6, "All shared numeric source columns match",
      comparison_df["match_rate"].eq(1).all(),
      f"{comparison_df['match_rate'].eq(1).sum()}/{len(comparison_df)} columns")

print("\n=== NUMERIC SOURCE COMPARISON ===\n")
print(
    comparison_df.sort_values(
        ["match_rate", "feature"]
    ).to_string(index=False)
)

print("\n" + "=" * 60)
print(f"BLOCK 64B: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 65A — PRODUCTION VERSION DEPENDENCY AUDIT
# READ ONLY — NO DATABASE WRITES OR MODEL CHANGES
# ============================================================

from pathlib import Path
import ast
import importlib.util
import config

print("\n=== BLOCK 65A: MODEL VERSION DEPENDENCY AUDIT ===\n")

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

project_root = Path(config.__file__).resolve().parent

files_to_inspect = {
    "config": project_root / "config.py",
    "metadata": (
        project_root / "prediction_model" / "metadata.py"
    ),
    "inference": (
        project_root / "prediction_model"
        / "inference" / "inference.py"
    ),
    "daily_probability": (
        project_root / "prediction_model"
        / "inference" / "run_daily_playoff_predictions.py"
    ),
    "daily_features": (
        project_root / "prediction_model"
        / "explainability" / "daily_feature_predictions.py"
    ),
    "compute_shap": (
        project_root / "prediction_model"
        / "explainability" / "compute_shap.py"
    ),
    "pipeline": project_root / "pipeline.py",
}

# Locate the daily probability script without assuming
# which directory contains it.
runner_matches = sorted(
    project_root.rglob("run_daily_playoff_predictions.py")
)

if runner_matches:
    files_to_inspect["daily_probability"] = runner_matches[0]

check(
    1,
    "Production config file exists",
    files_to_inspect["config"].is_file()
)

check(
    2,
    "Model metadata file exists",
    files_to_inspect["metadata"].is_file()
)

check(
    3,
    "Inference module exists",
    files_to_inspect["inference"].is_file()
)

check(
    4,
    "Daily probability runner located",
    files_to_inspect["daily_probability"].is_file()
)

check(
    5,
    "Daily feature prediction script exists",
    files_to_inspect["daily_features"].is_file()
)

check(
    6,
    "SHAP computation script exists",
    files_to_inspect["compute_shap"].is_file()
)

print("\n=== MODEL REFERENCES ===")

keywords = [
    "MODEL_PATH",
    "MODEL_VERSION",
    "MODEL_METADATA",
    "model_version",
    "playoff_models_",
    "score_dataframe",
    "run_probability_scores",
    "daily_feature_predictions",
    "compute_shap",
]

for name, path in files_to_inspect.items():

    print(f"\n{'=' * 70}")
    print(f"{name}: {path.relative_to(project_root)}")
    print("=" * 70)

    if not path.is_file():
        print("NOT FOUND")
        continue

    source = path.read_text(encoding="utf-8-sig")
    lines = source.splitlines()

    try:
        ast.parse(source)
        print("Syntax: PASS")
    except SyntaxError as exc:
        print(f"Syntax: FAIL | {exc}")

    matches = [
        (i + 1, line.strip())
        for i, line in enumerate(lines)
        if any(keyword in line for keyword in keywords)
    ]

    if not matches:
        print("No matching references")
    else:
        for line_number, line in matches:
            print(f"{line_number:4d} | {line}")

print("\n=== CURRENT CONFIGURATION ===")

print("MODEL_VERSION:", config.MODEL_VERSION)
print("MODEL_PATH:", config.MODEL_PATH)

check(
    7,
    "V6 remains active",
    str(config.MODEL_VERSION) == "v6"
)

check(
    8,
    "V7 model file exists",
    Path(config.MODEL_PATH)
    .with_name("playoff_models_v7.joblib")
    .is_file()
)

print("\n" + "=" * 60)
print(f"BLOCK 65A: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 65B — FINAL VERSION REFERENCE AUDIT
# READ ONLY
# ============================================================

from pathlib import Path
import re
import ast
import config

print("\n=== BLOCK 65B: FINAL VERSION REFERENCE AUDIT ===\n")

root = Path(config.__file__).resolve().parent
runner = (
    root / "prediction_model"
    / "inference"
    / "daily_playoff_probability.py"
)

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

check(1, "Actual production runner exists", runner.is_file())

if runner.is_file():
    source = runner.read_text(encoding="utf-8-sig")
    ast.parse(source)

    check(
        2,
        "Runner imports shared model metadata",
        "MODEL_METADATA" in source
    )

    check(
        3,
        "Runner uses shared inference function",
        "score_dataframe" in source
    )

    check(
        4,
        "Runner builds features before scoring",
        source.find("build_features(")
        < source.find("score_dataframe(")
    )

print("\n=== HARDCODED VERSION REFERENCES ===\n")

pattern = re.compile(
    r"""(?i)(playoff_models_v\d+|["']v[0-9]+["'])"""
)

matches = []

excluded_dirs = {
    ".git",
    ".venv",
    "venv",
    "new_hockey_env",
    "__pycache__",
    "site-packages",
}

for path in root.rglob("*.py"):

    if any(part in excluded_dirs for part in path.parts):
        continue

    lines = path.read_text(
        encoding="utf-8-sig"
    ).splitlines()

    for line_no, line in enumerate(lines, start=1):
        if pattern.search(line):
            matches.append(
                (path.relative_to(root), line_no, line.strip())
            )

for path, line_no, line in matches:
    print(f"{path}:{line_no} | {line}")

check(
    5,
    "Version references found",
    len(matches) > 0,
    f"{len(matches)} references"
)

print("\n=== ACTIVE MODEL ===\n")

print("Version:", config.MODEL_VERSION)
print("Path:", config.MODEL_PATH)

check(
    6,
    "V6 still active",
    config.MODEL_VERSION == "v6"
)

check(
    7,
    "V7 available for activation",
    Path(config.MODEL_PATH)
    .with_name("playoff_models_v7.joblib")
    .exists()
)

print("\n" + "=" * 60)
print(f"BLOCK 65B: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:
# ============================================================
# RESTORE NOTEBOOK DATABASE CONNECTION
# ============================================================

import duckdb
from config import DB_PATH

con = duckdb.connect(str(DB_PATH))

print("Connected:", DB_PATH)
print("Database check:", con.execute("SELECT 1").fetchone()[0])

In [ ]:

# ============================================================
# BLOCK 66A — MODEL VERSION DATABASE SAFETY
# READ ONLY — NO DATABASE OR CONFIG CHANGES
# ============================================================

import pandas as pd
from config import TARGET_TABLE
import config

print("\n=== BLOCK 66A: MODEL VERSION DATABASE SAFETY ===\n")

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# Use the existing notebook DuckDB connection: con.

table_info = con.execute(
    f"PRAGMA table_info('{TARGET_TABLE}')"
).df()

print("=== TARGET TABLE SCHEMA ===")
print(
    table_info[
        ["name", "type", "notnull", "pk"]
    ].to_string(index=False)
)

columns = set(table_info["name"])

check(1, "Prediction table has model_version",
      "model_version" in columns)

check(2, "Prediction table has team/date keys",
      {"season_id", "game_date", "team_id"}.issubset(columns))

pk_columns = set(
    table_info.loc[table_info["pk"] > 0, "name"]
)

expected_pk = {
    "season_id",
    "game_date",
    "team_id",
    "model_version",
}

check(3, "Primary key separates model versions",
      pk_columns == expected_pk,
      f"PK={sorted(pk_columns)}")

version_counts = con.execute(f"""
    SELECT
        model_version,
        COUNT(*) AS rows,
        COUNT(DISTINCT game_date) AS dates,
        COUNT(DISTINCT team_id) AS teams
    FROM {TARGET_TABLE}
    GROUP BY model_version
    ORDER BY model_version
""").df()

print("\n=== STORED MODEL VERSIONS ===")
print(version_counts.to_string(index=False))

check(4, "Existing predictions retain version labels",
      version_counts["model_version"].notna().all())

duplicates = con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT
            season_id,
            game_date,
            team_id,
            model_version,
            COUNT(*) AS n
        FROM {TARGET_TABLE}
        GROUP BY 1, 2, 3, 4
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

check(5, "No duplicate versioned prediction keys",
      duplicates == 0,
      f"duplicates={duplicates}")

check(6, "V6 remains the configured model",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 66A: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 66B — PREDICTION AND SHAP VERSION INVENTORY
# READ ONLY — NO DATABASE WRITES
# ============================================================

import pandas as pd
import config

print("\n=== BLOCK 66B: MODEL VERSION INVENTORY ===\n")

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# Identify all tables that have a model_version column.
tables = con.execute("""
    SELECT
        table_schema,
        table_name
    FROM information_schema.columns
    WHERE column_name = 'model_version'
      AND table_schema NOT IN (
          'information_schema',
          'pg_catalog'
      )
    GROUP BY table_schema, table_name
    ORDER BY table_schema, table_name
""").df()

check(
    1,
    "Versioned tables discovered",
    not tables.empty,
    f"{len(tables)} tables"
)

inventory = []

for row in tables.itertuples(index=False):

    schema = row.table_schema
    table = row.table_name

    # Quote identifiers safely.
    qualified = (
        '"' + schema.replace('"', '""') + '"."'
        + table.replace('"', '""') + '"'
    )

    counts = con.execute(f"""
        SELECT
            model_version,
            COUNT(*) AS rows
        FROM {qualified}
        GROUP BY model_version
        ORDER BY model_version
    """).df()

    for item in counts.itertuples(index=False):
        inventory.append({
            "table": f"{schema}.{table}",
            "model_version": item.model_version,
            "rows": item.rows,
        })

inventory_df = pd.DataFrame(
    inventory,
    columns=["table", "model_version", "rows"]
)

print("\n=== STORED VERSION COUNTS ===\n")

if inventory_df.empty:
    print("No versioned rows found")
else:
    print(inventory_df.to_string(index=False))

check(
    2,
    "All version labels populated",
    inventory_df.empty
    or inventory_df["model_version"].notna().all()
)

print("\n=== ACTIVE CONFIGURATION ===")
print("Configured version:", config.MODEL_VERSION)
print("Configured path:", config.MODEL_PATH)

check(
    3,
    "V6 still active",
    config.MODEL_VERSION == "v6"
)

print("\n" + "=" * 60)
print(f"BLOCK 66B: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 67A — PRODUCTION EXECUTION ORDER
# READ ONLY — DOES NOT RUN PIPELINE
# ============================================================

from pathlib import Path
import ast
import config

print("\n=== BLOCK 67A: PRODUCTION EXECUTION ORDER ===\n")

path = Path(config.__file__).resolve().parent / "pipeline.py"
source = path.read_text(encoding="utf-8-sig")
lines = source.splitlines()
tree = ast.parse(source)

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

check(1, "Pipeline exists", path.is_file())

check(2, "Pipeline syntax valid", tree is not None)

functions = [
    node for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
]

print("\n=== TOP-LEVEL FUNCTIONS ===")
for node in functions:
    print(f"{node.name}: lines {node.lineno}-{node.end_lineno}")

keywords = [
    "run_probability_scores",
    "run_prediction_features",
    "run_shap_summary",
    "refresh_existing",
    "season_id",
    "run_step",
]

print("\n=== RELEVANT PIPELINE LINES ===")

matches = set()

for i, line in enumerate(lines):
    if any(keyword in line for keyword in keywords):
        matches.update(
            range(max(0, i - 5), min(len(lines), i + 6))
        )

for i in sorted(matches):
    print(f"{i + 1:4d} | {lines[i]}")

# Find actual function calls rather than imports.
calls = []

for node in ast.walk(tree):
    if isinstance(node, ast.Call):
        func = node.func
        if isinstance(func, ast.Name):
            calls.append((node.lineno, func.id))
        elif isinstance(func, ast.Attribute):
            calls.append((node.lineno, func.attr))

probability_calls = [
    line for line, name in calls
    if name == "run_probability_scores"
]

shap_calls = [
    line for line, name in calls
    if name in {
        "run_prediction_features",
        "run_shap_summary"
    }
]

# Some pipelines pass functions as callbacks instead of
# calling them directly, so also inspect their references.
check(
    3,
    "Probability scoring referenced",
    "run_probability_scores" in source
)

check(
    4,
    "SHAP generation referenced",
    "run_prediction_features" in source
)

check(
    5,
    "SHAP summary referenced",
    "run_shap_summary" in source
)

print("\n=== DIRECT CALL LOCATIONS ===")
print("Probability:", probability_calls)
print("SHAP:", shap_calls)

print("\n" + "=" * 60)
print(f"BLOCK 67A: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 67B — V7 FIRST-RUN BACKFILL READINESS
# READ ONLY — NO DATABASE WRITES
# ============================================================

import pandas as pd
from config import SOURCE_TABLE, TARGET_TABLE

print("\n=== BLOCK 67B: V7 BACKFILL READINESS ===\n")

season_id = 20252026
candidate_version = "v7"
results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# Reuse the runner's validated team-day snapshots
# from Block 64B.
expected = runner_raw.copy()
expected["game_date"] = pd.to_datetime(
    expected["game_date"]
).dt.date

expected_counts = (
    expected.groupby("game_date")["team_id"]
    .nunique()
)

check(
    1,
    "Expected snapshot data available",
    not expected.empty
)

check(
    2,
    "Expected team-day keys unique",
    not expected.duplicated(
        ["season_id", "game_date", "team_id"]
    ).any()
)

stored = con.execute(f"""
    SELECT
        game_date,
        COUNT(DISTINCT team_id) AS stored_teams
    FROM {TARGET_TABLE}
    WHERE season_id = ?
      AND model_version = ?
    GROUP BY game_date
""", [season_id, candidate_version]).df()

stored["game_date"] = pd.to_datetime(
    stored["game_date"]
).dt.date

stored_counts = stored.set_index(
    "game_date"
)["stored_teams"]

incomplete_dates = {
    date
    for date, count in expected_counts.items()
    if stored_counts.get(date, 0) != count
}

check(
    3,
    "V7 has no existing probability coverage",
    stored.empty,
    f"stored_dates={len(stored)}"
)

check(
    4,
    "All available dates require v7 scoring",
    len(incomplete_dates) == len(expected_counts),
    f"{len(incomplete_dates)}/{len(expected_counts)} dates"
)

expected_rows = expected[
    expected["game_date"].isin(incomplete_dates)
]

check(
    5,
    "All canonical snapshots selected",
    len(expected_rows) == len(expected),
    f"{len(expected_rows):,}/{len(expected):,} rows"
)

# Inspect existing SHAP coverage by version.
shap_table = "playoff_shap_values_v2"

shap_versions = con.execute(f"""
    SELECT
        model_version,
        COUNT(*) AS rows
    FROM {shap_table}
    WHERE season_id = ?
    GROUP BY model_version
    ORDER BY model_version
""", [season_id]).df()

print("\n=== SHAP VERSION COVERAGE ===")
print(shap_versions.to_string(index=False))

check(
    6,
    "V7 has no existing SHAP rows",
    "v7" not in shap_versions["model_version"].values
)

check(
    7,
    "V6 SHAP rows remain available",
    "v6" in shap_versions["model_version"].values
)

print("\n=== EXPECTED V7 BACKFILL ===")
print(f"Dates: {len(incomplete_dates):,}")
print(f"Team-day rows: {len(expected_rows):,}")

print("\n" + "=" * 60)
print(f"BLOCK 67B: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:
# ============================================================
# BLOCK 67C — SHAP SUMMARY PIPELINE INTEGRATION AUDIT
# READ ONLY — NO DATABASE WRITES
# ============================================================

import inspect
from pathlib import Path
import config

from prediction_model.explainability.compute_shap import (
    run_shap_summary
)

print("\n=== BLOCK 67C: SHAP SUMMARY INTEGRATION AUDIT ===\n")

print("=== FUNCTION SIGNATURE ===")
print(inspect.signature(run_shap_summary))

print("\n=== FUNCTION SOURCE ===")
print(inspect.getsource(run_shap_summary))

pipeline_path = (
    Path(config.__file__).resolve().parent / "pipeline.py"
)

source = pipeline_path.read_text(encoding="utf-8-sig")
lines = source.splitlines()

print("\n=== PIPELINE FLAGS ===")
for i, line in enumerate(lines, start=1):
    if "add_argument(" in line or "skip_" in line:
        print(f"{i:4d} | {line}")

print("\n=== PIPELINE FINAL STEPS ===")
for i in range(199, min(len(lines), 240)):
    print(f"{i + 1:4d} | {lines[i]}")

print("\n=== BLOCK 67C COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 68A — V7 SHAP BACKFILL READINESS
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

import ast
import inspect
from pathlib import Path

import config
import prediction_model.explainability.daily_feature_predictions as shap_module

print("\n=== BLOCK 68A: V7 SHAP BACKFILL READINESS ===\n")

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# ------------------------------------------------------------
# 1. Inspect actual function and module
# ------------------------------------------------------------

source = inspect.getsource(shap_module)
tree = ast.parse(source)

function = shap_module.run_prediction_features
signature = inspect.signature(function)

print("Module:", inspect.getfile(shap_module))
print("Function signature:", signature)

check(
    1,
    "SHAP runner exists",
    callable(function)
)

check(
    2,
    "SHAP runner accepts database connection",
    "con" in signature.parameters
)

check(
    3,
    "SHAP runner accepts season_id",
    "season_id" in signature.parameters
)

# ------------------------------------------------------------
# 2. Check model version handling
# ------------------------------------------------------------

check(
    4,
    "SHAP module loads shared model path",
    "MODEL_PATH" in source
)

check(
    5,
    "SHAP module derives version from model bundle",
    'bundle["version"]' in source
)

print("\n=== CURRENT SHAP MODEL ===")
print("Configured version:", config.MODEL_VERSION)
print(
    "Loaded SHAP version:",
    shap_module.MODEL_VERSION_USED
)

check(
    6,
    "Current SHAP module remains on v6",
    shap_module.MODEL_VERSION_USED == "v6"
)

# ------------------------------------------------------------
# 3. Inspect version-specific database behavior
# ------------------------------------------------------------

print("\n=== RELEVANT SHAP SOURCE LINES ===")

lines = source.splitlines()

keywords = [
    "MODEL_VERSION_USED",
    "model_version",
    "run_prediction_features",
    "INSERT",
    "DELETE",
    "ON CONFLICT",
    "BEGIN TRANSACTION",
    "COMMIT",
    "ROLLBACK",
    "playoff_daily_predictions",
    "playoff_shap_values_v2",
    "refresh_existing",
]

relevant = set()

for i, line in enumerate(lines):
    if any(keyword in line for keyword in keywords):
        relevant.update(
            range(max(0, i - 3), min(len(lines), i + 4))
        )

for i in sorted(relevant):
    print(f"{i + 1:4d} | {lines[i]}")

# ------------------------------------------------------------
# 4. Verify existing database coverage
# ------------------------------------------------------------

season_id = 20252026

prediction_versions = con.execute("""
    SELECT model_version, COUNT(*) AS rows
    FROM playoff_daily_predictions
    WHERE season_id = ?
    GROUP BY model_version
    ORDER BY model_version
""", [season_id]).df()

shap_versions = con.execute("""
    SELECT model_version, COUNT(*) AS rows
    FROM playoff_shap_values_v2
    WHERE season_id = ?
    GROUP BY model_version
    ORDER BY model_version
""", [season_id]).df()

print("\n=== PREDICTION COVERAGE ===")
print(prediction_versions.to_string(index=False))

print("\n=== SHAP COVERAGE ===")
print(shap_versions.to_string(index=False))

check(
    7,
    "No existing v7 predictions",
    "v7" not in prediction_versions["model_version"].values
)

check(
    8,
    "No existing v7 SHAP records",
    "v7" not in shap_versions["model_version"].values
)

check(
    9,
    "Existing v6 SHAP records preserved",
    shap_versions.loc[
        shap_versions["model_version"] == "v6",
        "rows"
    ].sum() == 5299
)

print("\n" + "=" * 60)
print(f"BLOCK 68A: {sum(results)}/{len(results)} PASS")
print("=" * 60)

In [ ]:

# ============================================================
# BLOCK 68B — SHAP SOURCE AND SCORING DEPENDENCY AUDIT
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

import inspect
import ast
import textwrap

import prediction_model.explainability.daily_feature_predictions as shap_module

print("\n=== BLOCK 68B: SHAP SOURCE AND SCORING AUDIT ===\n")

source = inspect.getsource(shap_module)
tree = ast.parse(source)

functions = {
    node.name: node
    for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
}

print("=== MODULE FUNCTIONS ===")
for name in functions:
    print(" -", name)

print("\n=== SHAP RUNNER ===")
print(inspect.getsource(shap_module.run_prediction_features))

print("\n=== SOURCE / SCORING FUNCTIONS ===")

keywords = [
    "source",
    "load",
    "build",
    "score",
    "shap",
    "date",
    "snapshot",
]

selected = [
    name for name in functions
    if any(k in name.lower() for k in keywords)
    and name != "run_prediction_features"
]

for name in selected:
    print(f"\n--- {name} ---")
    fn = getattr(shap_module, name, None)
    if callable(fn):
        try:
            print(inspect.getsource(fn))
        except (OSError, TypeError):
            print("Source unavailable")

print("\n=== STATIC DEPENDENCY CHECKS ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
    )

check(
    1,
    "SHAP runner uses version-specific model",
    "MODEL_VERSION_USED" in source
)

check(
    2,
    "SHAP runner references canonical feature builder",
    "build_features" in source
)

check(
    3,
    "SHAP output includes model version",
    'out["model_version"] = MODEL_VERSION_USED' in source
)

check(
    4,
    "SHAP date selection checks existing coverage",
    "_dates_to_process" in source
)

check(
    5,
    "SHAP writes are transactional",
    "BEGIN TRANSACTION" in source
    and 'con.execute("COMMIT")' in source
    and 'con.execute("ROLLBACK")' in source
)

check(
    6,
    "SHAP target writes include version-specific keys",
    "season_id, snapshot_date, model_version" in source
)

print("\n" + "=" * 60)
print(f"BLOCK 68B: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 68C — V7 SHAP SCHEMA COMPATIBILITY
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

from joblib import load
from pathlib import Path
import config

print("\n=== BLOCK 68C: V7 SHAP SCHEMA COMPATIBILITY ===\n")

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# Load saved bundles without changing the active model.
model_dir = Path(config.MODEL_PATH).parent
v6 = load(model_dir / "playoff_models_v6.joblib")
v7 = load(model_dir / "playoff_models_v7.joblib")

v6_features = list(v6["features"])
v7_features = list(v7["features"])

check(1, "V6 bundle version correct", v6["version"] == "v6")
check(2, "V7 bundle version correct", v7["version"] == "v7")
check(3, "V7 feature names unique",
      len(v7_features) == len(set(v7_features)))

# Existing SHAP table schema
existing_cols = con.execute(
    "DESCRIBE playoff_shap_values_v2"
).df()["column_name"].tolist()

existing_shap = [
    c for c in existing_cols if c.startswith("shap_")
    and c not in ("shap_base_value",)
]

expected_v7_shap = [f"shap_{f}" for f in v7_features]

missing = sorted(set(expected_v7_shap) - set(existing_shap))
extra = sorted(set(existing_shap) - set(expected_v7_shap))

print("\n=== FEATURE COUNTS ===")
print("V6 features:", len(v6_features))
print("V7 features:", len(v7_features))
print("Existing SHAP columns:", len(existing_shap))

print("\n=== SCHEMA DIFFERENCES ===")
print("V7 columns missing from table:", len(missing))
print("Existing columns absent from V7:", len(extra))

if missing:
    print("\nMissing columns:")
    print("\n".join(missing[:30]))

if extra:
    print("\nExtra columns:")
    print("\n".join(extra[:30]))

check(
    4,
    "All V7 SHAP columns exist",
    len(missing) == 0
)

check(
    5,
    "No incompatible extra SHAP columns",
    len(extra) == 0
)

check(
    6,
    "V7 SHAP column order matches existing table",
    existing_shap == expected_v7_shap
)

print("\n" + "=" * 60)
print(f"BLOCK 68C: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 69A — SHAP TABLE DEPENDENCY AUDIT
# READ ONLY — NO FILE OR DATABASE CHANGES
# ============================================================

from pathlib import Path
import config

print("\n=== BLOCK 69A: SHAP TABLE DEPENDENCY AUDIT ===\n")

root = Path(config.__file__).resolve().parent

search_terms = [
    "playoff_shap_values_v2",
    "FEATURE_TARGET_TABLE",
    "run_prediction_features",
    "load_shap_summary_data",
]

results = []
matches = {}

excluded_dirs = {
    ".git", ".venv", "venv", "env",
    "__pycache__", "site-packages",
    "node_modules", ".ipynb_checkpoints",
}

for path in root.rglob("*.py"):
    if any(part in excluded_dirs for part in path.parts):
        continue

    try:
        lines = path.read_text(
            encoding="utf-8-sig"
        ).splitlines()
    except (UnicodeError, OSError):
        continue

    found = []

    for number, line in enumerate(lines, start=1):
        if any(term in line for term in search_terms):
            found.append((number, line.strip()))

    if found:
        matches[str(path.relative_to(root))] = found

print("=== FILE REFERENCES ===")

for filename, found in sorted(matches.items()):
    print(f"\n--- {filename} ---")
    for number, line in found:
        print(f"{number:4d} | {line}")

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

print("\n=== DEPENDENCY CHECKS ===")

check(
    1,
    "SHAP table references discovered",
    any(
        "playoff_shap_values_v2" in line
        for found in matches.values()
        for _, line in found
    )
)

check(
    2,
    "SHAP generation function discovered",
    any(
        "run_prediction_features" in line
        for found in matches.values()
        for _, line in found
    )
)

check(
    3,
    "SHAP summary loader discovered",
    any(
        "load_shap_summary_data" in line
        for found in matches.values()
        for _, line in found
    )
)

print("\n=== SUMMARY ===")
print("Python files containing references:", len(matches))

print("\n" + "=" * 60)
print(f"BLOCK 69A: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 69B — VERSIONED SHAP TABLE DESIGN
# READ ONLY — NO DATABASE OR FILE CHANGES
# ============================================================

from joblib import load
from pathlib import Path
import config
import prediction_model.explainability.daily_feature_predictions as shap_module
import prediction_model.explainability.compute_shap as summary_module

print("\n=== BLOCK 69B: VERSIONED SHAP TABLE DESIGN ===\n")

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

model_dir = Path(config.MODEL_PATH).parent
v6 = load(model_dir / "playoff_models_v6.joblib")
v7 = load(model_dir / "playoff_models_v7.joblib")

def target_for_version(version):
    return (
        "playoff_shap_values_v2"
        if version == "v6"
        else f"playoff_shap_values_{version}"
    )

v6_target = target_for_version(v6["version"])
v7_target = target_for_version(v7["version"])

print("V6 target:", v6_target)
print("V7 target:", v7_target)

check(
    1,
    "V6 retains existing table",
    v6_target == "playoff_shap_values_v2"
)

check(
    2,
    "V7 uses separate table",
    v7_target == "playoff_shap_values_v7"
)

check(
    3,
    "Targets are distinct",
    v6_target != v7_target
)

existing_tables = {
    row[0]
    for row in con.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'main'
    """).fetchall()
}

check(
    4,
    "Existing V6 SHAP table present",
    v6_target in existing_tables
)

check(
    5,
    "V7 SHAP table not yet created",
    v7_target not in existing_tables
)

check(
    6,
    "SHAP writer supports table creation",
    "CREATE TABLE" in __import__("inspect").getsource(
        shap_module._write_shap_dataframe
    )
)

check(
    7,
    "SHAP summary currently reads V6 table",
    summary_module.FEATURE_TARGET_TABLE == v6_target
)

check(
    8,
    "SHAP generator currently writes V6 table",
    shap_module.FEATURE_TARGET_TABLE == v6_target
)

print("\n=== EXPECTED V7 TABLE ===")
print("Name:", v7_target)
print("Feature columns:", len(v7["features"]))
print("SHAP columns:", len(v7["features"]))
print("Primary uniqueness:")
print("  season_id, snapshot_date, team_id, model_version")

print("\n" + "=" * 60)
print(f"BLOCK 69B: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:
# ============================================================
# BLOCK 69C — SHAP VERSIONING IMPLEMENTATION READINESS
# READ ONLY — NO FILE OR DATABASE CHANGES
# ============================================================

import inspect
import re

import prediction_model.explainability.daily_feature_predictions as generator
import prediction_model.explainability.compute_shap as summary

print("\n=== BLOCK 69C: IMPLEMENTATION READINESS ===\n")

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

generator_source = inspect.getsource(generator)
summary_source = inspect.getsource(summary)

# Verify both modules can use the same version-selection rule.
def target_for_version(version):
    if version == "v6":
        return "playoff_shap_values_v2"

    if not re.fullmatch(r"v[0-9]+", version):
        raise ValueError(f"Unexpected model version: {version}")

    return f"playoff_shap_values_{version}"

check(
    1,
    "V6 table mapping correct",
    target_for_version("v6") == "playoff_shap_values_v2"
)

check(
    2,
    "V7 table mapping correct",
    target_for_version("v7") == "playoff_shap_values_v7"
)

check(
    3,
    "Invalid versions rejected",
    all(
        _invalid
        for _invalid in [
            not re.fullmatch(r"v[0-9]+", x)
            for x in ["../v7", "v7;DROP", ""]
        ]
    )
)

# Verify the generator's table-creation and indexing behavior.
writer_source = inspect.getsource(generator._write_shap_dataframe)

check(
    4,
    "Writer creates missing target table",
    "CREATE TABLE" in writer_source
)

check(
    5,
    "Writer currently uses a fixed index name",
    "shap_v2_unique_snapshot" in writer_source
)

check(
    6,
    "Writer validates schema before inserting",
    "_validate_target_schema(con)" in writer_source
)

# Verify summary module uses a single target-table constant.
check(
    7,
    "Summary module has a target-table constant",
    'FEATURE_TARGET_TABLE = "playoff_shap_values_v2"'
    in summary_source
)

# Verify both modules obtain the version from their bundles.
check(
    8,
    "Generator uses bundle version",
    'bundle["version"]' in generator_source
)

check(
    9,
    "Summary uses bundle version",
    'bundle["version"]' in summary_source
)

print("\n=== PLANNED CHANGES ===")
print("1. Add identical version-to-table selection in both modules")
print("2. Preserve v6 -> playoff_shap_values_v2")
print("3. Route v7 -> playoff_shap_values_v7")
print("4. Make the unique index name table-specific")
print("5. Leave pipeline.py and config.py unchanged")

print("\n" + "=" * 60)
print(f"BLOCK 69C: {sum(results)}/{len(results)} PASS")
print("=" * 60)

In [ ]:

# ============================================================
# BLOCK 70A — IMPLEMENT VERSION-SPECIFIC SHAP TABLES
# FILE CHANGES ONLY — NO DATABASE WRITES
# V6 REMAINS ACTIVE
# ============================================================

from pathlib import Path
from datetime import datetime
import ast
import re
import shutil
import config

print("\n=== BLOCK 70A: IMPLEMENT SHAP TABLE VERSIONING ===\n")

root = Path(config.__file__).resolve().parent

paths = {
    "generator": root / "prediction_model" / "explainability" / "daily_feature_predictions.py",
    "summary": root / "prediction_model" / "explainability" / "compute_shap.py",
}

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

originals = {
    name: path.read_text(encoding="utf-8-sig")
    for name, path in paths.items()
}

# ------------------------------------------------------------
# 1. Prepare table-selection helper
# ------------------------------------------------------------

helper = '''
def _shap_table_for_version(version: str) -> str:
    """Preserve the v6 legacy table; use dedicated tables for newer models."""
    if version == "v6":
        return "playoff_shap_values_v2"
    if not re.fullmatch(r"v[0-9]+", version):
        raise ValueError(f"Unsupported model version: {version}")
    return f"playoff_shap_values_{version}"


'''

modified = {}

for name, source in originals.items():
    updated = source

    # Ensure re is imported.
    if not re.search(r"^import re\s*$", updated, re.MULTILINE):
        updated = "import re\n" + updated

    # Insert helper immediately before the existing table constant.
    old = 'FEATURE_TARGET_TABLE = "playoff_shap_values_v2"'

    if updated.count(old) != 1:
        raise RuntimeError(
            f"{name}: expected exactly one SHAP target assignment"
        )

    # Function is defined before the bundle is loaded, but is only
    # called after MODEL_VERSION_USED has been assigned.
    updated = updated.replace(
        old,
        helper + 'FEATURE_TARGET_TABLE = _shap_table_for_version(MODEL_VERSION_USED)',
        1,
    )

    # Move target assignment after the bundle version assignment.
    target_line = (
        'FEATURE_TARGET_TABLE = _shap_table_for_version(MODEL_VERSION_USED)'
    )

    updated = updated.replace(target_line + "\n", "", 1)

    version_line = 'MODEL_VERSION_USED = str(bundle["version"])'

    if updated.count(version_line) != 1:
        raise RuntimeError(
            f"{name}: expected exactly one bundle version assignment"
        )

    updated = updated.replace(
        version_line,
        version_line + "\n" + target_line,
        1,
    )

    if name == "generator":
        old_index = "shap_v2_unique_snapshot"
        new_index = "shap_{MODEL_VERSION_USED}_unique_snapshot"

        if updated.count(old_index) != 1:
            raise RuntimeError(
                "Generator: expected exactly one fixed index name"
            )

        updated = updated.replace(old_index, new_index, 1)

    # Validate syntax before writing anything.
    ast.parse(updated, filename=str(paths[name]))
    modified[name] = updated

check(1, "Both updated modules pass Python syntax validation",
      len(modified) == 2)

check(2, "Generator routes tables by model version",
      'FEATURE_TARGET_TABLE = _shap_table_for_version(MODEL_VERSION_USED)'
      in modified["generator"])

check(3, "Summary routes tables by model version",
      'FEATURE_TARGET_TABLE = _shap_table_for_version(MODEL_VERSION_USED)'
      in modified["summary"])

check(4, "Generator index name is version-specific",
      "shap_{MODEL_VERSION_USED}_unique_snapshot"
      in modified["generator"])

check(5, "V6 remains configured",
      config.MODEL_VERSION == "v6")

if not all(results):
    raise RuntimeError("Pre-write checks failed; no files modified")

# ------------------------------------------------------------
# 2. Back up and write files
# ------------------------------------------------------------

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for name, path in paths.items():
    backup = path.with_name(path.name + f".backup_{stamp}")
    shutil.copy2(path, backup)
    path.write_text(modified[name], encoding="utf-8")
    print(f"BACKUP | {backup.name}")
    print(f"UPDATED | {path.name}")

# ------------------------------------------------------------
# 3. Verify saved files
# ------------------------------------------------------------

for name, path in paths.items():
    saved = path.read_text(encoding="utf-8")
    ast.parse(saved, filename=str(path))
    check(
        6 if name == "generator" else 7,
        f"{name.capitalize()} file saved and syntax valid",
        saved == modified[name],
    )

print("\n" + "=" * 60)
print(f"BLOCK 70A: {sum(results)}/{len(results)} PASS")
print("=" * 60)

print("\nNOTE: Existing imported modules are not automatically reloaded.")
print("No database changes have been made.")


In [ ]:

# ============================================================
# BLOCK 70B — FRESH-PROCESS SHAP VERSION ROUTING
# READ ONLY — NO FILE OR DATABASE CHANGES
# ============================================================

import subprocess
import sys
import json
from pathlib import Path
import config

print("\n=== BLOCK 70B: FRESH-PROCESS SHAP ROUTING ===\n")

root = Path(config.__file__).resolve().parent

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

probe = r'''
import json
import sys
from pathlib import Path

import config

version = sys.argv[1]

if version == "v7":
    config.MODEL_PATH = (
        Path(config.MODEL_PATH).parent
        / "playoff_models_v7.joblib"
    )

import prediction_model.explainability.daily_feature_predictions as generator
import prediction_model.explainability.compute_shap as summary

print("ROUTING_RESULT=" + json.dumps({
    "requested": version,
    "generator_version": generator.MODEL_VERSION_USED,
    "summary_version": summary.MODEL_VERSION_USED,
    "generator_table": generator.FEATURE_TARGET_TABLE,
    "summary_table": summary.FEATURE_TARGET_TABLE,
    "generator_features": len(generator.FEATURE_COLS),
    "summary_features": len(summary.FEATURE_COLS),
}))
'''

outputs = {}

for version in ("v6", "v7"):
    completed = subprocess.run(
        [sys.executable, "-c", probe, version],
        cwd=root,
        capture_output=True,
        text=True,
        timeout=120,
    )

    if completed.returncode != 0:
        print(f"\nERROR | {version} subprocess")
        print(completed.stderr[-4000:])
        outputs[version] = None
        continue

    matching = [
        line.split("ROUTING_RESULT=", 1)[1]
        for line in completed.stdout.splitlines()
        if line.startswith("ROUTING_RESULT=")
    ]

    outputs[version] = (
        json.loads(matching[-1]) if matching else None
    )

    print(f"\n=== {version.upper()} ROUTING ===")
    print(json.dumps(outputs[version], indent=2))

v6 = outputs.get("v6")
v7 = outputs.get("v7")

check(1, "V6 process loads successfully", v6 is not None)
check(2, "V7 process loads successfully", v7 is not None)

if v6:
    check(
        3,
        "V6 generator and summary use legacy table",
        v6["generator_table"] == "playoff_shap_values_v2"
        and v6["summary_table"] == "playoff_shap_values_v2"
    )
    check(
        4,
        "V6 modules load v6 model",
        v6["generator_version"] == "v6"
        and v6["summary_version"] == "v6"
    )

if v7:
    check(
        5,
        "V7 generator and summary use v7 table",
        v7["generator_table"] == "playoff_shap_values_v7"
        and v7["summary_table"] == "playoff_shap_values_v7"
    )
    check(
        6,
        "V7 modules load v7 model",
        v7["generator_version"] == "v7"
        and v7["summary_version"] == "v7"
    )

if v6 and v7:
    check(
        7,
        "Both versions retain 121 features",
        all(
            output["generator_features"] == 121
            and output["summary_features"] == 121
            for output in (v6, v7)
        )
    )

print("\n" + "=" * 60)
print(f"BLOCK 70B: {sum(results)}/7 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 70C — REPAIR FUTURE IMPORT ORDER
# FILE FIX ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

from pathlib import Path
from datetime import datetime
import ast
import shutil
import config

print("\n=== BLOCK 70C: REPAIR IMPORT ORDER ===\n")

root = Path(config.__file__).resolve().parent
folder = root / "prediction_model" / "explainability"

paths = [
    folder / "compute_shap.py",
    folder / "daily_feature_predictions.py",
]

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
    )

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for path in paths:
    source = path.read_text(encoding="utf-8-sig")
    lines = source.splitlines(keepends=True)

    future_indices = [
        i for i, line in enumerate(lines)
        if line.strip().startswith("from __future__ import ")
    ]

    if not future_indices:
        print(f"UNCHANGED | {path.name}: no future imports")
        continue

    # Remove an incorrectly prepended `import re`.
    if lines[0].strip() == "import re":
        lines.pop(0)

        # Insert import re immediately after the final future import.
        future_indices = [
            i for i, line in enumerate(lines)
            if line.strip().startswith("from __future__ import ")
        ]

        insertion = max(future_indices) + 1
        lines.insert(insertion, "import re\n")

        updated = "".join(lines)

        # Verify syntax and future-import placement before writing.
        ast.parse(updated, filename=str(path))
        compile(updated, str(path), "exec")

        backup = path.with_name(
            path.name + f".backup_import_fix_{stamp}"
        )
        shutil.copy2(path, backup)
        path.write_text(updated, encoding="utf-8")

        print(f"BACKUP  | {backup.name}")
        print(f"REPAIRED | {path.name}")

    else:
        print(f"UNCHANGED | {path.name}: import re not prepended")

# ------------------------------------------------------------
# Verify both modules
# ------------------------------------------------------------

for number, path in enumerate(paths, start=1):
    source = path.read_text(encoding="utf-8")
    try:
        compile(source, str(path), "exec")
        valid = True
    except SyntaxError as exc:
        valid = False
        print(f"ERROR | {path.name}: {exc}")

    check(
        number,
        f"{path.name} compiles successfully",
        valid
    )

check(
    3,
    "Active model remains v6",
    config.MODEL_VERSION == "v6"
)

print("\n" + "=" * 60)
print(f"BLOCK 70C: {sum(results)}/3 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 71A — V7 SHAP TABLE CREATION READINESS
# READ ONLY — NO FILE OR DATABASE CHANGES
# ============================================================

import subprocess
import sys
import json
from pathlib import Path
import config

print("\n=== BLOCK 71A: V7 SHAP TABLE READINESS ===\n")

root = Path(config.__file__).resolve().parent

probe = r'''
import json
import inspect
from pathlib import Path
import config

config.MODEL_PATH = (
    Path(config.MODEL_PATH).parent
    / "playoff_models_v7.joblib"
)

import prediction_model.explainability.daily_feature_predictions as m

writer = inspect.getsource(m._write_shap_dataframe)

print("AUDIT_RESULT=" + json.dumps({
    "version": m.MODEL_VERSION_USED,
    "table": m.FEATURE_TARGET_TABLE,
    "feature_count": len(m.FEATURE_COLS),
    "output_columns": m.OUTPUT_COLS,
    "has_create": "CREATE TABLE" in writer,
    "has_index": "CREATE UNIQUE INDEX" in writer,
    "index_template": "shap_{MODEL_VERSION_USED}_unique_snapshot" in writer,
    "has_transaction": "BEGIN TRANSACTION" in writer,
    "has_rollback": "ROLLBACK" in writer,
    "has_schema_check": "_validate_target_schema(con)" in writer,
}))
'''

completed = subprocess.run(
    [sys.executable, "-c", probe],
    cwd=root,
    capture_output=True,
    text=True,
    timeout=120,
)

if completed.returncode != 0:
    print(completed.stderr[-5000:])
    raise RuntimeError("V7 subprocess failed")

matching = [
    line.split("AUDIT_RESULT=", 1)[1]
    for line in completed.stdout.splitlines()
    if line.startswith("AUDIT_RESULT=")
]

if not matching:
    raise RuntimeError("V7 audit result missing")

audit = json.loads(matching[-1])

results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

check(1, "V7 bundle loaded", audit["version"] == "v7")
check(2, "V7 target selected",
      audit["table"] == "playoff_shap_values_v7")
check(3, "121 model features loaded",
      audit["feature_count"] == 121)
check(4, "V7 output schema includes 121 SHAP columns",
      len([c for c in audit["output_columns"]
           if c.startswith("shap_") and c != "shap_base_value"]) == 121)
check(5, "Writer supports table creation", audit["has_create"])
check(6, "Writer creates unique index", audit["has_index"])
check(7, "Index name is version-specific", audit["index_template"])
check(8, "Writer uses transaction", audit["has_transaction"])
check(9, "Writer supports rollback", audit["has_rollback"])
check(10, "Writer validates schema", audit["has_schema_check"])

tables = {
    row[0]
    for row in con.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'main'
    """).fetchall()
}

check(11, "V6 SHAP table still exists",
      "playoff_shap_values_v2" in tables)
check(12, "V7 SHAP table not yet created",
      "playoff_shap_values_v7" not in tables)
check(13, "V6 remains active",
      config.MODEL_VERSION == "v6")

print("\n=== V7 TARGET SCHEMA ===")
print("Table:", audit["table"])
print("Total columns:", len(audit["output_columns"]))
print("SHAP feature columns:", audit["feature_count"])
print("Unique index: shap_v7_unique_snapshot")

print("\n" + "=" * 60)
print(f"BLOCK 71A: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 71B — V7 SHAP ISOLATED WRITE TEST
# IN-MEMORY DATABASE ONLY — NO PRODUCTION DB WRITES
# ============================================================

import subprocess
import sys
import json
from pathlib import Path
import config

print("\n=== BLOCK 71B: V7 SHAP ISOLATED WRITE TEST ===\n")

root = Path(config.__file__).resolve().parent

probe = r'''
import json
from pathlib import Path

import duckdb
import pandas as pd
import config

config.MODEL_PATH = (
    Path(config.MODEL_PATH).parent
    / "playoff_models_v7.joblib"
)

import prediction_model.explainability.daily_feature_predictions as m

# Isolated in-memory database
test_con = duckdb.connect(":memory:")

# Construct one valid SHAP output row using the v7 schema.
row = {}

for col in m.OUTPUT_COLS:
    if col == "season_id":
        row[col] = 20252026
    elif col in ("snapshot_date", "game_date"):
        row[col] = pd.Timestamp("2026-01-15").date()
    elif col == "team_id":
        row[col] = 6
    elif col == "tri_code":
        row[col] = "BOS"
    elif col == "model_version":
        row[col] = "v7"
    elif col == "created_at":
        row[col] = pd.Timestamp.now(tz="UTC")
    else:
        row[col] = 0.0

frame = pd.DataFrame([row], columns=m.OUTPUT_COLS)

m._write_shap_dataframe(test_con, frame)

table = m.FEATURE_TARGET_TABLE

stored = test_con.execute(
    f'SELECT COUNT(*) FROM "{table}"'
).fetchone()[0]

schema = test_con.execute(
    f'DESCRIBE "{table}"'
).df()["column_name"].tolist()

indexes = test_con.execute("""
    SELECT index_name
    FROM duckdb_indexes()
    WHERE table_name = ?
""", [table]).df()["index_name"].tolist()

# Repeat same write: replacement should preserve one row.
m._write_shap_dataframe(test_con, frame)

stored_after_repeat = test_con.execute(
    f'SELECT COUNT(*) FROM "{table}"'
).fetchone()[0]

print("TEST_RESULT=" + json.dumps({
    "version": m.MODEL_VERSION_USED,
    "table": table,
    "initial_rows": stored,
    "rows_after_repeat": stored_after_repeat,
    "schema_matches": schema == m.OUTPUT_COLS,
    "index_exists": "shap_v7_unique_snapshot" in indexes,
    "column_count": len(schema),
}))

test_con.close()
'''

completed = subprocess.run(
    [sys.executable, "-c", probe],
    cwd=root,
    capture_output=True,
    text=True,
    timeout=120,
)

if completed.returncode != 0:
    print(completed.stderr[-6000:])
    raise RuntimeError("Isolated v7 SHAP write test failed")

matching = [
    line.split("TEST_RESULT=", 1)[1]
    for line in completed.stdout.splitlines()
    if line.startswith("TEST_RESULT=")
]

if not matching:
    raise RuntimeError("Test result missing")

audit = json.loads(matching[-1])

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
    )

check(1, "V7 model loaded", audit["version"] == "v7")
check(2, "Correct v7 table created",
      audit["table"] == "playoff_shap_values_v7")
check(3, "One row inserted", audit["initial_rows"] == 1)
check(4, "All 131 columns created",
      audit["column_count"] == 131)
check(5, "Schema matches v7 model",
      audit["schema_matches"])
check(6, "Version-specific unique index created",
      audit["index_exists"])
check(7, "Repeated write replaces existing snapshot",
      audit["rows_after_repeat"] == 1)

print("\n" + "=" * 60)
print(f"BLOCK 71B: {sum(results)}/7 PASS")
print("=" * 60)

print("\nProduction database unchanged.")
print("Active model:", config.MODEL_VERSION)


In [ ]:

# ============================================================
# BLOCK 71C — REAL V7 SHAP CALCULATION
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

import subprocess
import sys
import json
import tempfile
from pathlib import Path

import pandas as pd
import numpy as np
import config

print("\n=== BLOCK 71C: REAL V7 SHAP CALCULATION ===\n")

root = Path(config.__file__).resolve().parent
season_id = 20252026
snapshot_date = pd.Timestamp("2026-01-15").date()

# Read a real historical snapshot using the existing connection.
# Do not open or close another connection to the production DB.
raw = con.execute("""
    SELECT *
    FROM team_daily_stats
    WHERE season_id = ?
      AND CAST(game_date AS DATE) <= ?
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY team_id
        ORDER BY game_date DESC, game_id DESC
    ) = 1
    ORDER BY team_id
""", [season_id, snapshot_date]).df()

print("Snapshot date:", snapshot_date)
print("Teams:", len(raw))

probe = r'''
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import config

config.MODEL_PATH = (
    Path(config.MODEL_PATH).parent
    / "playoff_models_v7.joblib"
)

import prediction_model.explainability.daily_feature_predictions as m

raw = pd.read_pickle(sys.argv[1])

out = m._build_shap_dataframe(
    raw,
    season_id=20252026,
    snapshot_date="2026-01-15",
)

shap_values = out[m.SHAP_COLS].to_numpy(dtype=float)

print("TEST_RESULT=" + json.dumps({
    "version": m.MODEL_VERSION_USED,
    "table": m.FEATURE_TARGET_TABLE,
    "rows": len(out),
    "columns": len(out.columns),
    "shap_columns": len(m.SHAP_COLS),
    "unique_teams": out["team_id"].nunique(),
    "finite_shap": bool(np.isfinite(shap_values).all()),
    "finite_probability": bool(
        np.isfinite(out["xgb_probability"]).all()
    ),
    "probability_in_range": bool(
        out["xgb_probability"].between(0, 1).all()
    ),
    "unique_keys": bool(
        ~out.duplicated(
            ["season_id", "snapshot_date", "team_id", "model_version"]
        ).any()
    ),
}))
'''

with tempfile.TemporaryDirectory() as temp_dir:
    snapshot_path = Path(temp_dir) / "snapshot.pkl"
    raw.to_pickle(snapshot_path)

    completed = subprocess.run(
        [sys.executable, "-c", probe, str(snapshot_path)],
        cwd=root,
        capture_output=True,
        text=True,
        timeout=180,
    )

if completed.returncode != 0:
    print(completed.stderr[-6000:])
    raise RuntimeError("V7 SHAP calculation failed")

matching = [
    line.split("TEST_RESULT=", 1)[1]
    for line in completed.stdout.splitlines()
    if line.startswith("TEST_RESULT=")
]

if not matching:
    raise RuntimeError("V7 calculation result missing")

audit = json.loads(matching[-1])

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
    )

check(1, "32-team historical snapshot loaded", len(raw) == 32)
check(2, "V7 model used", audit["version"] == "v7")
check(3, "Correct SHAP target selected",
      audit["table"] == "playoff_shap_values_v7")
check(4, "All 32 teams explained", audit["rows"] == 32)
check(5, "All team IDs unique", audit["unique_teams"] == 32)
check(6, "121 SHAP feature columns", audit["shap_columns"] == 121)
check(7, "131 output columns", audit["columns"] == 131)
check(8, "All SHAP values finite", audit["finite_shap"])
check(9, "All XGBoost probabilities finite",
      audit["finite_probability"])
check(10, "All probabilities between 0 and 1",
      audit["probability_in_range"])
check(11, "No duplicate SHAP keys", audit["unique_keys"])
check(12, "Active configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 71C: {sum(results)}/12 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 72A — FULL-SEASON SHAP SNAPSHOT ALIGNMENT
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

import pandas as pd
import config

print("\n=== BLOCK 72A: FULL-SEASON SHAP ALIGNMENT ===\n")

season_id = 20252026
results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# Expected snapshots from previously validated probability runner
expected = runner_raw.copy()
expected["game_date"] = pd.to_datetime(
    expected["game_date"]
).dt.date

expected_counts = (
    expected.groupby("game_date")["team_id"]
    .nunique()
    .sort_index()
)

# SHAP runner's expected count logic:
# teams become eligible after their first recorded game.
shap_counts = con.execute("""
    WITH first_games AS (
        SELECT
            team_id,
            MIN(CAST(game_date AS DATE)) AS first_date
        FROM team_daily_stats
        WHERE season_id = ?
        GROUP BY team_id
    ),
    dates AS (
        SELECT DISTINCT
            CAST(game_date AS DATE) AS snapshot_date
        FROM team_daily_stats
        WHERE season_id = ?
    )
    SELECT
        d.snapshot_date,
        COUNT(f.team_id) AS expected_teams
    FROM dates d
    LEFT JOIN first_games f
        ON f.first_date <= d.snapshot_date
    GROUP BY d.snapshot_date
    ORDER BY d.snapshot_date
""", [season_id, season_id]).df()

shap_counts["snapshot_date"] = pd.to_datetime(
    shap_counts["snapshot_date"]
).dt.date

shap_counts = shap_counts.set_index(
    "snapshot_date"
)["expected_teams"].sort_index()

check(
    1,
    "Probability runner has 167 dates",
    len(expected_counts) == 167
)

check(
    2,
    "SHAP runner has 167 dates",
    len(shap_counts) == 167
)

check(
    3,
    "Both runners use identical dates",
    expected_counts.index.equals(shap_counts.index)
)

aligned = expected_counts.reindex(
    shap_counts.index
)

check(
    4,
    "Team counts match on every date",
    aligned.equals(shap_counts.astype(aligned.dtype))
)

check(
    5,
    "Probability runner expects 5,299 rows",
    int(expected_counts.sum()) == 5299
)

check(
    6,
    "SHAP runner expects 5,299 rows",
    int(shap_counts.sum()) == 5299
)

check(
    7,
    "All team counts are between 1 and 32",
    shap_counts.between(1, 32).all()
)

check(
    8,
    "Active model remains v6",
    config.MODEL_VERSION == "v6"
)

print("\n=== SNAPSHOT SUMMARY ===")
print("Dates:", len(shap_counts))
print("Expected team-day rows:", int(shap_counts.sum()))
print("Minimum teams per date:", int(shap_counts.min()))
print("Maximum teams per date:", int(shap_counts.max()))

mismatches = pd.DataFrame({
    "probability_teams": aligned,
    "shap_teams": shap_counts
})

mismatches = mismatches[
    mismatches["probability_teams"]
    != mismatches["shap_teams"]
]

print("\nMismatched dates:", len(mismatches))

if not mismatches.empty:
    print(mismatches.head(20).to_string())

print("\n" + "=" * 60)
print(f"BLOCK 72A: {sum(results)}/8 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 72B — FULL-SEASON TEAM-KEY ALIGNMENT
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

import pandas as pd
import config

print("\n=== BLOCK 72B: TEAM-KEY ALIGNMENT ===\n")

season_id = 20252026
results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# Previously validated probability-runner snapshots
prob = runner_raw[["game_date", "team_id"]].copy()
prob["game_date"] = pd.to_datetime(prob["game_date"]).dt.date

# Reproduce the SHAP runner's per-date as-of selection in one SQL query.
shap = con.execute("""
    WITH dates AS (
        SELECT DISTINCT CAST(game_date AS DATE) AS snapshot_date
        FROM team_daily_stats
        WHERE season_id = ?
    ),
    ranked AS (
        SELECT
            d.snapshot_date AS game_date,
            s.team_id,
            ROW_NUMBER() OVER (
                PARTITION BY d.snapshot_date, s.team_id
                ORDER BY s.game_date DESC, s.game_id DESC
            ) AS rn
        FROM dates d
        JOIN team_daily_stats s
          ON s.season_id = ?
         AND CAST(s.game_date AS DATE) <= d.snapshot_date
    )
    SELECT game_date, team_id
    FROM ranked
    WHERE rn = 1
    ORDER BY game_date, team_id
""", [season_id, season_id]).df()

shap["game_date"] = pd.to_datetime(shap["game_date"]).dt.date

prob_keys = set(map(tuple, prob[["game_date", "team_id"]].to_numpy()))
shap_keys = set(map(tuple, shap[["game_date", "team_id"]].to_numpy()))

missing_from_shap = prob_keys - shap_keys
extra_in_shap = shap_keys - prob_keys

check(1, "Probability snapshot keys unique",
      len(prob) == len(prob_keys))
check(2, "SHAP snapshot keys unique",
      len(shap) == len(shap_keys))
check(3, "Probability snapshot has 5,299 rows",
      len(prob) == 5299)
check(4, "SHAP snapshot has 5,299 rows",
      len(shap) == 5299)
check(5, "No probability keys missing from SHAP",
      len(missing_from_shap) == 0)
check(6, "No extra SHAP keys",
      len(extra_in_shap) == 0)
check(7, "Exact team-day key sets match",
      prob_keys == shap_keys)
check(8, "Active model remains v6",
      config.MODEL_VERSION == "v6")

print("\n=== ALIGNMENT SUMMARY ===")
print("Probability keys:", len(prob_keys))
print("SHAP keys:", len(shap_keys))
print("Missing from SHAP:", len(missing_from_shap))
print("Extra in SHAP:", len(extra_in_shap))

if missing_from_shap:
    print("\nSample missing keys:")
    print(sorted(missing_from_shap)[:10])

if extra_in_shap:
    print("\nSample extra keys:")
    print(sorted(extra_in_shap)[:10])

print("\n" + "=" * 60)
print(f"BLOCK 72B: {sum(results)}/8 PASS")
print("=" * 60)


In [ ]:
# ============================================================
# BLOCK 72C-0 — NOTEBOOK STATE DIAGNOSTIC
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

import config

print("\n=== BLOCK 72C-0: NOTEBOOK STATE ===\n")

objects = [
    "con",
    "runner_raw",
    "df_canonical_all",
    "df_canonical_features_all",
    "df_training_canonical",
]

for name in objects:
    obj = globals().get(name)

    if obj is None:
        print(f"MISSING | {name}")
    elif hasattr(obj, "shape"):
        print(f"FOUND   | {name} | shape={obj.shape}")
    else:
        print(f"FOUND   | {name} | type={type(obj).__name__}")

print("\n=== MODEL CONFIGURATION ===")
print("MODEL_VERSION:", getattr(config, "MODEL_VERSION", "NOT DEFINED"))
print("MODEL_PATH:", getattr(config, "MODEL_PATH", "NOT DEFINED"))

print("\n" + "=" * 60)
print("BLOCK 72C-0 COMPLETE")
print("=" * 60)

In [ ]:

# ============================================================
# BLOCK 72C — SOURCE-RECORD ALIGNMENT
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

import pandas as pd
import config

print("\n=== BLOCK 72C: SOURCE-RECORD ALIGNMENT ===\n")

season_id = 20252026
results = []

def check(number, description, condition, detail=""):
    passed = bool(condition)
    results.append(passed)
    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{number:02d} {description}"
        + (f" | {detail}" if detail else "")
    )

# The previously validated probability-runner output.
prob = runner_raw[
    ["game_date", "team_id", "game_id"]
].copy()

prob["game_date"] = pd.to_datetime(
    prob["game_date"]
).dt.date

# Reconstruct the SHAP runner's as-of source selection.
shap = con.execute("""
    WITH dates AS (
        SELECT DISTINCT
            CAST(game_date AS DATE) AS snapshot_date
        FROM team_daily_stats
        WHERE season_id = ?
    ),
    ranked AS (
        SELECT
            d.snapshot_date AS game_date,
            s.team_id,
            s.game_id AS source_game_id,
            ROW_NUMBER() OVER (
                PARTITION BY d.snapshot_date, s.team_id
                ORDER BY s.game_date DESC, s.game_id DESC
            ) AS rn
        FROM dates d
        JOIN team_daily_stats s
          ON s.season_id = ?
         AND CAST(s.game_date AS DATE) <= d.snapshot_date
    )
    SELECT game_date, team_id, source_game_id
    FROM ranked
    WHERE rn = 1
""", [season_id, season_id]).df()

shap["game_date"] = pd.to_datetime(
    shap["game_date"]
).dt.date

merged = prob.merge(
    shap,
    on=["game_date", "team_id"],
    how="outer",
    indicator=True
)

both = merged["_merge"] == "both"

game_id_match = (
    merged.loc[both, "game_id"].astype("string")
    == merged.loc[both, "source_game_id"].astype("string")
)

mismatches = merged.loc[
    both & ~game_id_match.reindex(
        merged.index, fill_value=False
    )
]

check(1, "Probability source has 5,299 rows", len(prob) == 5299)
check(2, "SHAP source has 5,299 rows", len(shap) == 5299)
check(3, "All team-day keys matched",
      (merged["_merge"] == "both").all())
check(4, "All source game IDs match",
      game_id_match.all(),
      f"mismatches={len(mismatches)}")
check(5, "No duplicate merged keys",
      not merged.duplicated(["game_date", "team_id"]).any())
check(6, "Active model remains v6",
      config.MODEL_VERSION == "v6")

print("\n=== SOURCE-RECORD SUMMARY ===")
print("Compared rows:", int(both.sum()))
print("Mismatched source game IDs:", len(mismatches))

if not mismatches.empty:
    print("\nSample mismatches:")
    print(
        mismatches[
            ["game_date", "team_id", "game_id", "source_game_id"]
        ].head(20).to_string(index=False)
    )

print("\n" + "=" * 60)
print(f"BLOCK 72C: {sum(results)}/6 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 73A — PRE-ACTIVATION CONFIGURATION AUDIT
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

from pathlib import Path
import re
import config
import joblib

print("\n=== BLOCK 73A: PRE-ACTIVATION AUDIT ===\n")

root = Path(config.__file__).resolve().parent
v6_path = root / "models" / "playoff_models_v6.joblib"
v7_path = root / "models" / "playoff_models_v7.joblib"

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Active model remains v6",
      config.MODEL_VERSION == "v6")

check(2, "Active model path points to v6",
      Path(config.MODEL_PATH).resolve() == v6_path.resolve())

check(3, "V6 model file exists", v6_path.exists())
check(4, "V7 model file exists", v7_path.exists())

if v7_path.exists():
    bundle = joblib.load(v7_path)
    check(5, "V7 bundle identifies as v7",
          str(bundle.get("version")) == "v7")
    print("V7 bundle keys:", sorted(bundle.keys()))
else:
    check(5, "V7 bundle identifies as v7", False)

paths = {
    "Probability runner":
        root / "prediction_model" / "inference" / "daily_playoff_probability.py",
    "SHAP runner":
        root / "prediction_model" / "explainability" / "daily_feature_predictions.py",
    "SHAP summary":
        root / "prediction_model" / "explainability" / "compute_shap.py",
    "Pipeline":
        root / "pipeline.py",
}

for name, path in paths.items():
    print(f"\n{name}: {path.relative_to(root)}")
    print("Exists:", path.exists())

    if path.exists():
        source = path.read_text(encoding="utf-8")
        print("References MODEL_PATH:", "MODEL_PATH" in source)
        print("References MODEL_VERSION:", "MODEL_VERSION" in source)

check(6, "Probability runner exists",
      paths["Probability runner"].exists())
check(7, "SHAP runner exists",
      paths["SHAP runner"].exists())
check(8, "SHAP summary exists",
      paths["SHAP summary"].exists())
check(9, "Pipeline exists",
      paths["Pipeline"].exists())

print("\n=== CONFIGURATION ===")
print("MODEL_VERSION:", config.MODEL_VERSION)
print("MODEL_PATH:", config.MODEL_PATH)

print("\n" + "=" * 60)
print(f"BLOCK 73A: {sum(results)}/9 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 73B — MODEL-LOADING DEPENDENCY AUDIT
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

from pathlib import Path
import ast
import config

print("\n=== BLOCK 73B: MODEL-LOADING DEPENDENCIES ===\n")

root = Path(config.__file__).resolve().parent

files = {
    "Probability runner":
        root / "prediction_model" / "inference" / "daily_playoff_probability.py",
    "SHAP runner":
        root / "prediction_model" / "explainability" / "daily_feature_predictions.py",
    "SHAP summary":
        root / "prediction_model" / "explainability" / "compute_shap.py",
    "Pipeline":
        root / "pipeline.py",
}

keywords = (
    "model",
    "bundle",
    "joblib",
    "predict_proba",
    "load(",
    "config",
    "run_probability",
    "run_shap",
)

for label, path in files.items():
    print("\n" + "=" * 65)
    print(label)
    print(path.relative_to(root))
    print("=" * 65)

    if not path.exists():
        print("FILE MISSING")
        continue

    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()

    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        print("SYNTAX ERROR:", exc)
        continue

    print("\nIMPORTS:")
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            statement = ast.get_source_segment(source, node)
            if statement:
                print(" ", statement.replace("\n", " "))

    print("\nMODEL / RUNNER REFERENCES:")

    matches = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    selected = set()
    for i in matches:
        selected.update(
            range(max(0, i - 2), min(len(lines), i + 4))
        )

    previous = -2
    for i in sorted(selected):
        if i > previous + 1:
            print("    ...")
        print(f"{i + 1:4d} | {lines[i]}")
        previous = i

print("\n" + "=" * 65)
print("BLOCK 73B COMPLETE — READ ONLY")
print("=" * 65)


In [ ]:

# ============================================================
# BLOCK 73C — PROBABILITY MODEL / METADATA ALIGNMENT
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

from pathlib import Path
import ast
import config

print("\n=== BLOCK 73C: MODEL AND METADATA ALIGNMENT ===\n")

root = Path(config.__file__).resolve().parent

paths = {
    "Inference package": root / "prediction_model" / "inference" / "__init__.py",
    "Metadata module": root / "prediction_model" / "metadata.py",
    "Configuration": root / "config.py",
}

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

for label, path in paths.items():
    print(f"\n{'=' * 65}")
    print(label, "|", path.relative_to(root))
    print("=" * 65)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()

    try:
        ast.parse(source)
        print("Syntax: VALID")
    except SyntaxError as exc:
        print("Syntax: INVALID", exc)
        continue

    keywords = [
        "MODEL_PATH", "MODEL_VERSION", "MODEL_METADATA",
        "score_dataframe", "joblib", "load(",
        "version", "model_name", "features",
        "predict_proba", "playoff_models",
    ]

    matches = [
        i for i, line in enumerate(lines)
        if any(k.lower() in line.lower() for k in keywords)
    ]

    selected = set()
    for i in matches:
        selected.update(range(max(0, i - 3), min(len(lines), i + 7)))

    previous = -2
    for i in sorted(selected):
        if i > previous + 1:
            print("    ...")
        print(f"{i + 1:4d} | {lines[i]}")
        previous = i

print("\n=== CONFIGURATION CHECKS ===")

check(1, "Inference package exists", paths["Inference package"].exists())
check(2, "Metadata module exists", paths["Metadata module"].exists())
check(3, "Configuration file exists", paths["Configuration"].exists())
check(4, "Active model remains v6", config.MODEL_VERSION == "v6")
check(5, "Active model path points to v6",
      Path(config.MODEL_PATH).name == "playoff_models_v6.joblib")

print("\n" + "=" * 65)
print(f"BLOCK 73C: {sum(results)}/5 PASS")
print("=" * 65)



In [ ]:

# ============================================================
# BLOCK 73D — PROBABILITY SCORING ENGINE AUDIT
# READ ONLY — NO DATABASE OR MODEL CHANGES
# ============================================================

from pathlib import Path
import ast
import config

print("\n=== BLOCK 73D: SCORING ENGINE AUDIT ===\n")

root = Path(config.__file__).resolve().parent
path = root / "prediction_model" / "inference" / "inference.py"

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Scoring engine exists", path.exists())

if path.exists():
    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()

    try:
        tree = ast.parse(source)
        syntax_valid = True
    except SyntaxError as exc:
        syntax_valid = False
        print("Syntax error:", exc)

    check(2, "Scoring engine syntax valid", syntax_valid)

    print("\n=== SCORING ENGINE SOURCE ===")

    for i, line in enumerate(lines, start=1):
        print(f"{i:4d} | {line}")

    print("\n=== MODEL CONFIGURATION REFERENCES ===")

    check(3, "References MODEL_PATH",
          "MODEL_PATH" in source)

    check(4, "No hardcoded v6 model filename",
          "playoff_models_v6.joblib" not in source)

    check(5, "Defines score_dataframe",
          "def score_dataframe" in source)

else:
    for number, description in [
        (2, "Scoring engine syntax valid"),
        (3, "References MODEL_PATH"),
        (4, "No hardcoded v6 model filename"),
        (5, "Defines score_dataframe"),
    ]:
        check(number, description, False)

check(6, "Active model remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 73D: {sum(results)}/6 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 73E — V7 ACTIVATION REHEARSAL
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import subprocess
import sys
import json
from pathlib import Path
import config

print("\n=== BLOCK 73E: V7 ACTIVATION REHEARSAL ===\n")

root = Path(config.__file__).resolve().parent

probe = r'''
import json
from pathlib import Path

import config

# Simulate v7 activation only inside this subprocess.
config.MODEL_VERSION = "v7"
config.MODEL_FILENAME = "playoff_models_v7.joblib"
config.MODEL_PATH = (
    Path(config.MODEL_PATH).parent
    / config.MODEL_FILENAME
)

from prediction_model.inference import inference as scoring
import prediction_model.metadata as metadata
import prediction_model.explainability.daily_feature_predictions as shap_writer
import prediction_model.explainability.compute_shap as shap_summary

print("TEST_RESULT=" + json.dumps({
    "config_version": config.MODEL_VERSION,
    "config_filename": Path(config.MODEL_PATH).name,
    "scoring_version": scoring._bundle["version"],
    "scoring_features": len(scoring.FEATURE_COLS),
    "metadata_version": metadata.MODEL_METADATA["version"],
    "shap_writer_version": shap_writer.MODEL_VERSION_USED,
    "shap_writer_table": shap_writer.FEATURE_TARGET_TABLE,
    "shap_writer_features": len(shap_writer.FEATURE_COLS),
    "shap_summary_version": shap_summary.MODEL_VERSION_USED,
    "shap_summary_table": shap_summary.FEATURE_TARGET_TABLE,
    "shap_summary_features": len(shap_summary.FEATURE_COLS),
    "feature_lists_match": (
        list(scoring.FEATURE_COLS)
        == list(shap_writer.FEATURE_COLS)
        == list(shap_summary.FEATURE_COLS)
    ),
}))
'''

completed = subprocess.run(
    [sys.executable, "-c", probe],
    cwd=root,
    capture_output=True,
    text=True,
    timeout=180,
)

if completed.returncode != 0:
    print(completed.stderr[-6000:])
    raise RuntimeError("V7 activation rehearsal failed")

matching = [
    line.split("TEST_RESULT=", 1)[1]
    for line in completed.stdout.splitlines()
    if line.startswith("TEST_RESULT=")
]

if not matching:
    print(completed.stdout[-4000:])
    raise RuntimeError("Activation rehearsal result missing")

audit = json.loads(matching[-1])
results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Simulated configuration uses v7",
      audit["config_version"] == "v7")
check(2, "Simulated model filename is v7",
      audit["config_filename"] == "playoff_models_v7.joblib")
check(3, "Probability engine loads v7",
      audit["scoring_version"] == "v7")
check(4, "Metadata identifies v7",
      audit["metadata_version"] == "v7")
check(5, "SHAP writer loads v7",
      audit["shap_writer_version"] == "v7")
check(6, "SHAP writer selects v7 table",
      audit["shap_writer_table"] == "playoff_shap_values_v7")
check(7, "SHAP summary loads v7",
      audit["shap_summary_version"] == "v7")
check(8, "SHAP summary selects v7 table",
      audit["shap_summary_table"] == "playoff_shap_values_v7")
check(9, "All components use 121 features",
      all(audit[k] == 121 for k in [
          "scoring_features",
          "shap_writer_features",
          "shap_summary_features",
      ]))
check(10, "Exact feature lists match",
      audit["feature_lists_match"])
check(11, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n=== REHEARSAL SUMMARY ===")
print("Probability model:", audit["scoring_version"])
print("Metadata version:", audit["metadata_version"])
print("SHAP writer:", audit["shap_writer_table"])
print("SHAP summary:", audit["shap_summary_table"])

print("\n" + "=" * 60)
print(f"BLOCK 73E: {sum(results)}/11 PASS")
print("=" * 60)

print("\nProduction configuration unchanged.")


In [ ]:

# ============================================================
# BLOCK 74A — V7 PROBABILITY SCORING SMOKE TEST
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import subprocess
import sys
import json
from pathlib import Path
import config

print("\n=== BLOCK 74A: V7 SCORING SMOKE TEST ===\n")

root = Path(config.__file__).resolve().parent

probe = r'''
import json
import numpy as np
import pandas as pd
from pathlib import Path
import config

config.MODEL_VERSION = "v7"
config.MODEL_FILENAME = "playoff_models_v7.joblib"
config.MODEL_PATH = Path(config.MODEL_PATH).parent / config.MODEL_FILENAME

from prediction_model.inference.inference import (
    score_dataframe, FEATURE_COLS
)

# Neutral, finite inputs to test the production scoring interface.
# This tests execution and output contracts, not prediction accuracy.
sample = pd.DataFrame(
    np.zeros((3, len(FEATURE_COLS))),
    columns=FEATURE_COLS,
)

result = score_dataframe(sample)

columns = [
    "playoff_prob_xgb",
    "playoff_prob_rf",
    "playoff_prob",
]

values = result[columns].to_numpy(dtype=float)

print("TEST_RESULT=" + json.dumps({
    "rows": len(result),
    "feature_count": len(FEATURE_COLS),
    "output_columns_present": all(c in result for c in columns),
    "finite": bool(np.isfinite(values).all()),
    "bounded": bool(((values >= 0) & (values <= 1)).all()),
    "ensemble_matches": bool(np.allclose(
        result["playoff_prob"],
        (result["playoff_prob_xgb"] + result["playoff_prob_rf"]) / 2,
        atol=1e-12,
    )),
    "probabilities": result[columns].round(6).to_dict("records"),
}))
'''

completed = subprocess.run(
    [sys.executable, "-c", probe],
    cwd=root,
    capture_output=True,
    text=True,
    timeout=180,
)

if completed.returncode != 0:
    print(completed.stderr[-6000:])
    raise RuntimeError("V7 scoring smoke test failed")

matching = [
    line.split("TEST_RESULT=", 1)[1]
    for line in completed.stdout.splitlines()
    if line.startswith("TEST_RESULT=")
]

if not matching:
    print(completed.stdout[-4000:])
    raise RuntimeError("Scoring result missing")

audit = json.loads(matching[-1])
results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Scored all 3 sample rows", audit["rows"] == 3)
check(2, "V7 uses 121 features", audit["feature_count"] == 121)
check(3, "All probability columns present", audit["output_columns_present"])
check(4, "All probabilities finite", audit["finite"])
check(5, "All probabilities between 0 and 1", audit["bounded"])
check(6, "Ensemble equals XGB/RF average", audit["ensemble_matches"])
check(7, "Production configuration remains v6", config.MODEL_VERSION == "v6")

print("\n=== SAMPLE PROBABILITIES ===")
for i, row in enumerate(audit["probabilities"], 1):
    print(f"Sample {i}: {row}")

print("\n" + "=" * 60)
print(f"BLOCK 74A: {sum(results)}/7 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 74B — REAL-DATA V6/V7 SCORING COMPARISON
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import numpy as np
import pandas as pd
import joblib
import config
from pathlib import Path

print("\n=== BLOCK 74B: REAL-DATA MODEL COMPARISON ===\n")

root = Path(config.__file__).resolve().parent
v6 = joblib.load(root / "models" / "playoff_models_v6.joblib")
v7 = joblib.load(root / "models" / "playoff_models_v7.joblib")

# Reuse the real source records from Block 72C.
# Build features with the same function used by production.
from feature_engineering import build_features

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Real source snapshot available",
      "runner_raw" in globals() and isinstance(globals().get("runner_raw"), pd.DataFrame))

if results[-1]:
    raw = runner_raw.copy()
    features = build_features(raw)

    check(2, "Feature engineering returned rows", len(features) > 0)

    predictions = {}

    for version, bundle in [("v6", v6), ("v7", v7)]:
        cols = list(bundle["features"])
        missing = sorted(set(cols) - set(features.columns))

        check(3 if version == "v6" else 4,
              f"{version} feature contract satisfied", not missing)

        if missing:
            print(f"{version} missing features:", missing[:20])
            continue

        X = features[cols].fillna(0)

        xgb = bundle["xgb_pipeline"].predict_proba(X)[:, 1]
        rf = bundle["rf_pipeline"].predict_proba(X)[:, 1]
        predictions[version] = (xgb + rf) / 2

    if len(predictions) == 2:
        p6 = predictions["v6"]
        p7 = predictions["v7"]
        delta = p7 - p6

        check(5, "Both models return one probability per row",
              len(p6) == len(features) == len(p7))
        check(6, "All predictions finite",
              np.isfinite(p6).all() and np.isfinite(p7).all())
        check(7, "All predictions bounded [0, 1]",
              ((p6 >= 0) & (p6 <= 1)).all()
              and ((p7 >= 0) & (p7 <= 1)).all())

        print("\n=== REAL-DATA COMPARISON ===")
        print("Rows:", len(features))
        print("V6 mean probability:", round(float(p6.mean()), 6))
        print("V7 mean probability:", round(float(p7.mean()), 6))
        print("Mean absolute change:", round(float(np.abs(delta).mean()), 6))
        print("Maximum absolute change:", round(float(np.abs(delta).max()), 6))
        print("Correlation:", round(float(np.corrcoef(p6, p7)[0, 1]), 6))

        print("\n=== CHANGE DISTRIBUTION ===")
        print(pd.Series(delta).describe().to_string())

check(8, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 74B: {sum(results)}/{len(results)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 74C — V6/V7 PREDICTION SHIFT ANALYSIS
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import config
from feature_engineering import build_features

print("\n=== BLOCK 74C: PREDICTION SHIFT ANALYSIS ===\n")

root = Path(config.__file__).resolve().parent
v6 = joblib.load(root / "models" / "playoff_models_v6.joblib")
v7 = joblib.load(root / "models" / "playoff_models_v7.joblib")

raw = runner_raw.copy()
features = build_features(raw)

comparison = raw[[
    c for c in ["season_id", "game_date", "team_id", "tri_code", "game_id"]
    if c in raw.columns
]].reset_index(drop=True).copy()

for version, bundle in [("v6", v6), ("v7", v7)]:
    X = features[list(bundle["features"])].fillna(0)
    xgb = bundle["xgb_pipeline"].predict_proba(X)[:, 1]
    rf = bundle["rf_pipeline"].predict_proba(X)[:, 1]
    comparison[f"prob_{version}"] = (xgb + rf) / 2

comparison["change_pp"] = (
    comparison["prob_v7"] - comparison["prob_v6"]
) * 100

comparison["abs_change_pp"] = comparison["change_pp"].abs()
comparison["game_date"] = pd.to_datetime(comparison["game_date"])

print("=== SHIFT THRESHOLDS ===")
for threshold in [1, 2, 5, 10, 15, 20]:
    count = int((comparison["abs_change_pp"] >= threshold).sum())
    print(f"Change >= {threshold:2d} pp: {count:5d} "
          f"({count / len(comparison):.2%})")

print("\n=== TOP 20 LARGEST CHANGES ===")
cols = [
    c for c in [
        "game_date", "tri_code", "team_id", "prob_v6",
        "prob_v7", "change_pp", "game_id"
    ] if c in comparison.columns
]

print(
    comparison.nlargest(20, "abs_change_pp")[cols]
    .round(4)
    .to_string(index=False)
)

print("\n=== MONTHLY SHIFT SUMMARY ===")
comparison["month"] = comparison["game_date"].dt.to_period("M").astype(str)

monthly = (
    comparison.groupby("month")
    .agg(
        rows=("abs_change_pp", "size"),
        mean_abs_change_pp=("abs_change_pp", "mean"),
        max_abs_change_pp=("abs_change_pp", "max"),
        count_over_5pp=("abs_change_pp", lambda x: (x >= 5).sum()),
        count_over_10pp=("abs_change_pp", lambda x: (x >= 10).sum()),
    )
    .round(3)
)

print(monthly.to_string())

print("\n=== VALIDATION ===")
results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "All 5,299 observations compared", len(comparison) == 5299)
check(2, "All changes finite", np.isfinite(comparison["change_pp"]).all())
check(3, "V6 and V7 predictions differ",
      (comparison["abs_change_pp"] > 0).any())
check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 74C: {sum(results)}/4 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 74D — SOURCE / FEATURE ROW ALIGNMENT
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import pandas as pd
import numpy as np
import config
from feature_engineering import build_features

print("\n=== BLOCK 74D: SOURCE / FEATURE ALIGNMENT ===\n")

raw = runner_raw.copy().reset_index(drop=True)
features = build_features(raw.copy()).reset_index(drop=True)

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Source and feature row counts match",
      len(raw) == len(features))

keys = ["season_id", "game_date", "team_id", "game_id"]
available = [c for c in keys if c in raw.columns and c in features.columns]

print("\nAvailable shared identity columns:", available)

check(2, "Team ID available for alignment",
      "team_id" in available)

if available and len(raw) == len(features):
    raw_keys = raw[available].copy()
    feature_keys = features[available].copy()

    for col in available:
        if "date" in col:
            raw_keys[col] = pd.to_datetime(raw_keys[col], errors="coerce")
            feature_keys[col] = pd.to_datetime(feature_keys[col], errors="coerce")
        else:
            raw_keys[col] = raw_keys[col].astype("string")
            feature_keys[col] = feature_keys[col].astype("string")

    row_match = (
        raw_keys.eq(feature_keys)
        | (raw_keys.isna() & feature_keys.isna())
    ).all(axis=1)

    mismatches = int((~row_match).sum())

    check(3, "Source and feature rows align positionally",
          mismatches == 0)

    print("\nRows with mismatched identity:", mismatches)

    if mismatches:
        print("\n=== FIRST 10 MISMATCHES ===")
        idx = np.flatnonzero(~row_match.to_numpy())[:10]

        for i in idx:
            print(f"\nRow {i}")
            print("Source: ", raw_keys.iloc[i].to_dict())
            print("Feature:", feature_keys.iloc[i].to_dict())
else:
    check(3, "Source and feature rows align positionally", False)

check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 74D: {sum(results)}/4 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 74E — OUTLIER MODEL COMPONENT ANALYSIS
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import config
from feature_engineering import build_features

print("\n=== BLOCK 74E: OUTLIER COMPONENT ANALYSIS ===\n")

root = Path(config.__file__).resolve().parent
bundles = {
    version: joblib.load(root / "models" / f"playoff_models_{version}.joblib")
    for version in ["v6", "v7"]
}

raw = runner_raw.copy().reset_index(drop=True)
features = build_features(raw.copy()).reset_index(drop=True)

identity = [
    c for c in ["game_date", "tri_code", "team_id", "game_id"]
    if c in raw.columns
]

comparison = raw[identity].copy()
comparison["game_date"] = pd.to_datetime(comparison["game_date"])

for version, bundle in bundles.items():
    X = features[list(bundle["features"])].fillna(0)

    comparison[f"xgb_{version}"] = (
        bundle["xgb_pipeline"].predict_proba(X)[:, 1]
    )
    comparison[f"rf_{version}"] = (
        bundle["rf_pipeline"].predict_proba(X)[:, 1]
    )
    comparison[f"ensemble_{version}"] = (
        comparison[f"xgb_{version}"] + comparison[f"rf_{version}"]
    ) / 2

for component in ["xgb", "rf", "ensemble"]:
    comparison[f"{component}_change_pp"] = (
        comparison[f"{component}_v7"] -
        comparison[f"{component}_v6"]
    ) * 100

comparison["abs_change_pp"] = comparison["ensemble_change_pp"].abs()

print("=== TOP 20 ENSEMBLE SHIFTS ===")

display_cols = [
    "game_date", "tri_code",
    "ensemble_v6", "ensemble_v7",
    "xgb_change_pp", "rf_change_pp",
    "ensemble_change_pp"
]

print(
    comparison.nlargest(20, "abs_change_pp")[display_cols]
    .round(3)
    .to_string(index=False)
)

print("\n=== COMPONENT SHIFT SUMMARY ===")

for component in ["xgb", "rf", "ensemble"]:
    values = comparison[f"{component}_change_pp"]
    print(
        f"{component.upper():8s} | "
        f"Mean abs: {values.abs().mean():.3f} pp | "
        f"Max abs: {values.abs().max():.3f} pp | "
        f">= 10 pp: {(values.abs() >= 10).sum()}"
    )

print("\n=== ISLANDERS: FEB 20 – MAR 5 ===")

nyi = comparison[
    (comparison["team_id"].astype(str) == "2") &
    comparison["game_date"].between("2026-02-20", "2026-03-05")
]

print(nyi[display_cols].round(3).to_string(index=False))

print("\n=== WASHINGTON: DEC 5 – DEC 12 ===")

wsh = comparison[
    (comparison["team_id"].astype(str) == "15") &
    comparison["game_date"].between("2025-12-05", "2025-12-12")
]

print(wsh[display_cols].round(3).to_string(index=False))

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "All 5,299 records scored", len(comparison) == 5299)
check(2, "All component probabilities finite",
      np.isfinite(comparison[
          [f"{c}_{v}" for c in ["xgb", "rf", "ensemble"]
           for v in ["v6", "v7"]]
      ].to_numpy()).all())
check(3, "Both model components analyzed",
      "xgb_change_pp" in comparison and "rf_change_pp" in comparison)
check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 74E: {sum(results)}/4 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 74F — V6/V7 DAILY PREDICTION STABILITY
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import numpy as np
import pandas as pd
import config

print("\n=== BLOCK 74F: DAILY PREDICTION STABILITY ===\n")

# Reuse the scored comparison from Block 74E.
stability = comparison.copy()

stability["game_date"] = pd.to_datetime(stability["game_date"])
stability = stability.sort_values(["team_id", "game_date"]).reset_index(drop=True)

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "All 5,299 snapshots available", len(stability) == 5299)
check(2, "Unique team-day records",
      not stability.duplicated(["team_id", "game_date"]).any())

for version in ["v6", "v7"]:
    stability[f"daily_change_{version}_pp"] = (
        stability.groupby("team_id")[f"ensemble_{version}"]
        .diff() * 100
    )
    stability[f"abs_daily_change_{version}_pp"] = (
        stability[f"daily_change_{version}_pp"].abs()
    )

valid = stability.dropna(subset=[
    "daily_change_v6_pp", "daily_change_v7_pp"
]).copy()

print("\n=== OVERALL DAILY STABILITY ===")

for version in ["v6", "v7"]:
    values = valid[f"abs_daily_change_{version}_pp"]

    print(
        f"{version.upper()} | "
        f"Mean: {values.mean():.3f} pp | "
        f"Median: {values.median():.3f} pp | "
        f"95th percentile: {values.quantile(.95):.3f} pp | "
        f"Max: {values.max():.3f} pp"
    )

print("\n=== LARGE DAILY CHANGES ===")

for threshold in [5, 10, 15, 20]:
    counts = {
        version: int(
            (valid[f"abs_daily_change_{version}_pp"] >= threshold).sum()
        )
        for version in ["v6", "v7"]
    }
    print(
        f">= {threshold:2d} pp | "
        f"V6: {counts['v6']:4d} | "
        f"V7: {counts['v7']:4d}"
    )

valid["volatility_difference_pp"] = (
    valid["abs_daily_change_v7_pp"] -
    valid["abs_daily_change_v6_pp"]
)

print("\n=== TOP 15 INCREASES IN DAILY VOLATILITY ===")

cols = [
    "game_date", "tri_code",
    "daily_change_v6_pp", "daily_change_v7_pp",
    "volatility_difference_pp"
]

print(
    valid.nlargest(15, "volatility_difference_pp")[cols]
    .round(3)
    .to_string(index=False)
)

print("\n=== ISLANDERS: FEB 25 – MAR 5 ===")

nyi = stability[
    (stability["team_id"].astype(str) == "2") &
    stability["game_date"].between("2026-02-25", "2026-03-05")
]

print(nyi[cols].round(3).to_string(index=False))

print("\n=== WASHINGTON: DEC 5 – DEC 12 ===")

wsh = stability[
    (stability["team_id"].astype(str) == "15") &
    stability["game_date"].between("2025-12-05", "2025-12-12")
]

print(wsh[cols].round(3).to_string(index=False))

print("\n=== VALIDATION ===")

check(3, "Daily changes calculated for both models",
      len(valid) > 0 and all(
          c in valid.columns
          for c in ["daily_change_v6_pp", "daily_change_v7_pp"]
      ))

check(4, "All daily changes finite",
      np.isfinite(valid[[
          "daily_change_v6_pp", "daily_change_v7_pp"
      ]].to_numpy()).all())

check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 74F: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 74F — FIX AND COMPLETE
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import numpy as np
import config

print("\n=== BLOCK 74F: DAILY STABILITY — CONTINUED ===\n")

# Add the missing column to the full stability DataFrame
stability["volatility_difference_pp"] = (
    stability["abs_daily_change_v7_pp"] -
    stability["abs_daily_change_v6_pp"]
)

cols = [
    "game_date", "tri_code",
    "daily_change_v6_pp", "daily_change_v7_pp",
    "volatility_difference_pp"
]

print("=== ISLANDERS: FEB 25 – MAR 5 ===")
nyi = stability[
    (stability["team_id"].astype(str) == "2") &
    stability["game_date"].between("2026-02-25", "2026-03-05")
]
print(nyi[cols].round(3).to_string(index=False))

print("\n=== WASHINGTON: DEC 5 – DEC 12 ===")
wsh = stability[
    (stability["team_id"].astype(str) == "15") &
    stability["game_date"].between("2025-12-05", "2025-12-12")
]
print(wsh[cols].round(3).to_string(index=False))

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "All 5,299 snapshots available", len(stability) == 5299)
check(2, "Unique team-day records",
      not stability.duplicated(["team_id", "game_date"]).any())
check(3, "Daily changes calculated for both models", len(valid) > 0)
check(4, "All daily changes finite",
      np.isfinite(valid[[
          "daily_change_v6_pp", "daily_change_v7_pp"
      ]].to_numpy()).all())
check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 74F: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 74G — OVERALL VOLATILITY ASSESSMENT
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import numpy as np
import pandas as pd
import config

print("\n=== BLOCK 74G: OVERALL VOLATILITY ASSESSMENT ===\n")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Paired daily observations available", len(valid) > 0)

summary = []

for version in ["v6", "v7"]:
    values = valid[f"abs_daily_change_{version}_pp"]

    summary.append({
        "model": version,
        "observations": len(values),
        "mean_pp": values.mean(),
        "median_pp": values.median(),
        "p90_pp": values.quantile(.90),
        "p95_pp": values.quantile(.95),
        "p99_pp": values.quantile(.99),
        "max_pp": values.max(),
        "over_5pp": int((values >= 5).sum()),
        "over_10pp": int((values >= 10).sum()),
        "over_20pp": int((values >= 20).sum()),
    })

summary_df = pd.DataFrame(summary).set_index("model")

print("\n=== V6 VS V7 DAILY VOLATILITY ===")
print(summary_df.round(3).to_string())

v6_mean = summary_df.loc["v6", "mean_pp"]
v7_mean = summary_df.loc["v7", "mean_pp"]

print("\n=== RELATIVE VOLATILITY ===")
print(f"V6 mean absolute daily change: {v6_mean:.3f} pp")
print(f"V7 mean absolute daily change: {v7_mean:.3f} pp")
print(f"V7/V6 volatility ratio: {v7_mean / v6_mean:.3f}x")

print("\n=== TEAM-LEVEL VOLATILITY ===")

team_summary = (
    valid.groupby(["team_id", "tri_code"])
    .agg(
        observations=("abs_daily_change_v6_pp", "size"),
        v6_mean=("abs_daily_change_v6_pp", "mean"),
        v7_mean=("abs_daily_change_v7_pp", "mean"),
        v6_max=("abs_daily_change_v6_pp", "max"),
        v7_max=("abs_daily_change_v7_pp", "max"),
    )
    .reset_index()
)

team_summary["increase_pp"] = (
    team_summary["v7_mean"] - team_summary["v6_mean"]
)

print(
    team_summary.sort_values("increase_pp", ascending=False)
    .head(15)
    .round(3)
    .to_string(index=False)
)

print("\n=== VALIDATION ===")

check(2, "Same observations compared",
      summary_df.loc["v6", "observations"] ==
      summary_df.loc["v7", "observations"])

check(3, "All summary metrics finite",
      np.isfinite(summary_df.select_dtypes(include="number").to_numpy()).all())

check(4, "All 32 teams represented",
      team_summary["team_id"].nunique() == 32)

check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 74G: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75A — MODEL EVALUATION METADATA AUDIT
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import joblib
import config
from pathlib import Path

print("\n=== BLOCK 75A: EVALUATION METADATA AUDIT ===\n")

root = Path(config.__file__).resolve().parent

bundles = {
    version: joblib.load(root / "models" / f"playoff_models_{version}.joblib")
    for version in ["v6", "v7"]
}

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

for version, bundle in bundles.items():
    print(f"\n{'=' * 60}")
    print(f"{version.upper()} MODEL METADATA")
    print("=" * 60)

    for key in [
        "version",
        "trained_on",
        "source_table",
        "target",
        "model_type",
        "snapshot",
        "metrics",
    ]:
        value = bundle.get(key)

        print(f"\n{key}:")

        if isinstance(value, dict):
            for subkey, subvalue in value.items():
                print(f"  {subkey}: {subvalue}")
        else:
            print(f"  {value}")

    print("\nFeature count:", len(bundle.get("features", [])))

print("\n=== VALIDATION ===")

check(1, "V6 bundle identifies as v6",
      bundles["v6"].get("version") == "v6")
check(2, "V7 bundle identifies as v7",
      bundles["v7"].get("version") == "v7")
check(3, "V6 has evaluation metrics",
      bool(bundles["v6"].get("metrics")))
check(4, "V7 has evaluation metrics",
      bool(bundles["v7"].get("metrics")))
check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75A: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75B — HOLDOUT EVALUATION CODE AUDIT
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

from pathlib import Path
import re
import config

print("\n=== BLOCK 75B: HOLDOUT EVALUATION CODE AUDIT ===\n")

root = Path(config.__file__).resolve().parent

patterns = {
    "Brier calculation": r"brier_score_loss|brier_score",
    "Holdout split": r"test_season_id|evaluation_test_rows|train_test_split",
    "Final retraining": r"final_training_rows|final.*fit|fit.*full",
    "Metric storage": r"holdout_xgb_brier|holdout_rf_brier|xgb_brier|rf_brier",
    "Ensemble evaluation": r"ensemble.*brier|brier.*ensemble",
}

excluded = {
    ".git", ".venv", "venv", "new_hockey_env",
    "__pycache__", "site-packages", ".ipynb_checkpoints"
}

files = [
    p for p in root.rglob("*.py")
    if not any(part in excluded for part in p.relative_to(root).parts)
]

matches = {name: [] for name in patterns}

for path in files:
    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except (UnicodeError, OSError):
        continue

    for number, line in enumerate(lines, 1):
        for name, pattern in patterns.items():
            if re.search(pattern, line, flags=re.IGNORECASE):
                matches[name].append((
                    str(path.relative_to(root)), number, line.strip()
                ))

for name, hits in matches.items():
    print(f"\n=== {name.upper()} ({len(hits)} matches) ===")

    for filename, number, line in hits[:20]:
        print(f"{filename}:{number} | {line[:180]}")

    if len(hits) > 20:
        print(f"... {len(hits) - 20} additional matches")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Python source files found", len(files) > 0)
check(2, "Brier calculation code located",
      len(matches["Brier calculation"]) > 0)
check(3, "Holdout split code located",
      len(matches["Holdout split"]) > 0)
check(4, "Final retraining code located",
      len(matches["Final retraining"]) > 0)
check(5, "Metric storage code located",
      len(matches["Metric storage"]) > 0)
check(6, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\nNote: Missing code matches indicate items to inspect,")
print("not necessarily a model validation failure.")

print("\n" + "=" * 60)
print(f"BLOCK 75B: {sum(results)}/6 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75C — TRAINING / HOLDOUT SEQUENCE INSPECTION
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

from pathlib import Path
import config

print("\n=== BLOCK 75C: TRAINING SEQUENCE INSPECTION ===\n")

root = Path(config.__file__).resolve().parent
path = root / "prediction_model" / "training" / "train_new_model.py"

lines = path.read_text(encoding="utf-8").splitlines()

sections = [
    ("HOLDOUT SPLIT", 385, 420),
    ("TRAINING AND HOLDOUT PREDICTIONS", 470, 575),
    ("ENSEMBLE EVALUATION", 605, 660),
    ("FINAL TRAINING AND MODEL SAVE", 1390, 1475),
]

for title, start, end in sections:
    print(f"\n{'=' * 65}")
    print(f"{title} — LINES {start}-{end}")
    print("=" * 65)

    for number in range(start, min(end, len(lines)) + 1):
        print(f"{number:4d} | {lines[number - 1]}")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Training script exists", path.is_file())
check(2, "Holdout split section available", len(lines) >= 420)
check(3, "Holdout model evaluation available", len(lines) >= 575)
check(4, "Ensemble evaluation section available", len(lines) >= 660)
check(5, "Final training section available", len(lines) >= 1475)
check(6, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75C: {sum(results)}/6 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75D — ENSEMBLE EVALUATION OUTPUT INSPECTION
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

from pathlib import Path
import config

print("\n=== BLOCK 75D: ENSEMBLE EVALUATION OUTPUTS ===\n")

root = Path(config.__file__).resolve().parent
path = root / "prediction_model" / "training" / "train_new_model.py"
lines = path.read_text(encoding="utf-8").splitlines()

sections = [
    ("HOLDOUT ENSEMBLE CONSTRUCTION", 570, 605),
    ("ENSEMBLE METRICS AND OUTPUT", 625, 720),
]

for title, start, end in sections:
    print(f"\n{'=' * 65}")
    print(f"{title} — LINES {start}-{end}")
    print("=" * 65)

    for number in range(start, min(end, len(lines)) + 1):
        print(f"{number:4d} | {lines[number - 1]}")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Training script exists", path.is_file())
check(2, "Ensemble construction section available", len(lines) >= 605)
check(3, "Ensemble metrics section available", len(lines) >= 720)
check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75D: {sum(results)}/4 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75E — LOCATE HISTORICAL HOLDOUT RESULTS
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

from pathlib import Path
import config

print("\n=== BLOCK 75E: HISTORICAL EVALUATION ARTIFACTS ===\n")

root = Path(config.__file__).resolve().parent

extensions = {
    ".csv", ".parquet", ".json", ".jsonl",
    ".txt", ".log", ".xlsx", ".pkl", ".joblib"
}

keywords = [
    "holdout", "evaluation", "eval", "brier",
    "calibration", "metrics", "validation",
    "backtest", "training", "v6", "v7"
]

excluded = {
    ".git", ".venv", "venv", "new_hockey_env",
    "__pycache__", "site-packages", ".ipynb_checkpoints"
}

candidates = []

for path in root.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue

    if any(part in excluded for part in path.relative_to(root).parts):
        continue

    relative = str(path.relative_to(root))

    if any(word in relative.lower() for word in keywords):
        candidates.append({
            "path": relative,
            "size_kb": round(path.stat().st_size / 1024, 1),
            "extension": path.suffix.lower()
        })

candidates.sort(key=lambda x: x["path"])

print(f"\nMatching artifacts: {len(candidates)}\n")

for item in candidates[:100]:
    print(
        f"{item['path']} | "
        f"{item['size_kb']:,} KB"
    )

if len(candidates) > 100:
    print(f"\n... {len(candidates) - 100} additional artifacts")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Project directory exists", root.is_dir())
check(2, "Artifact search completed", isinstance(candidates, list))
check(3, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75E: {sum(results)}/3 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75F — EXISTING VALIDATION ARTIFACT INSPECTION
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import json
import joblib
import config
from pathlib import Path

print("\n=== BLOCK 75F: VALIDATION ARTIFACT INSPECTION ===\n")

root = Path(config.__file__).resolve().parent

validation_path = root / "models" / "playoff_model_v6_validation.joblib"
manifest_path = root / "models" / "playoff_models_v6_manifest.json"

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

print("=== V6 VALIDATION ARTIFACT ===")

if validation_path.exists():
    try:
        validation = joblib.load(validation_path)

        print("Object type:", type(validation).__name__)

        if isinstance(validation, dict):
            print("\nTop-level keys:", list(validation.keys()))

            for key, value in validation.items():
                print(f"\n{key}:")
                if isinstance(value, (dict, list, tuple)):
                    print(str(value)[:1500])
                elif hasattr(value, "shape"):
                    print(f"Type: {type(value).__name__}, shape: {value.shape}")
                else:
                    print(str(value)[:500])
        else:
            print("Contents:", str(validation)[:3000])

        validation_loaded = True

    except Exception as e:
        print("Unable to load:", repr(e))
        validation_loaded = False
else:
    print("Artifact not found")
    validation_loaded = False

print("\n=== V6 MODEL MANIFEST ===")

if manifest_path.exists():
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

        print("Top-level keys:", list(manifest.keys()))

        for key, value in manifest.items():
            if any(term in key.lower() for term in [
                "metric", "brier", "evaluation",
                "holdout", "test", "validation"
            ]):
                print(f"\n{key}: {str(value)[:2000]}")

        manifest_loaded = True

    except Exception as e:
        print("Unable to load:", repr(e))
        manifest_loaded = False
else:
    print("Manifest not found")
    manifest_loaded = False

print("\n=== VALIDATION ===")

check(1, "V6 validation artifact exists", validation_path.exists())
check(2, "V6 validation artifact loaded", validation_loaded)
check(3, "V6 manifest exists", manifest_path.exists())
check(4, "V6 manifest loaded", manifest_loaded)
check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75F: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75G — HISTORICAL VALIDATION FRAMEWORK DISCOVERY
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

from pathlib import Path
import re
import config

print("\n=== BLOCK 75G: VALIDATION FRAMEWORK DISCOVERY ===\n")

root = Path(config.__file__).resolve().parent

patterns = {
    "Validation artifact": r"playoff_model_v6_validation|validation\.joblib",
    "Historical evaluation": r"walk.forward|expanding.window|rolling.origin|out.of.sample|oos_rows",
    "Validation structure": r"season_performance|stage_performance|component_comparison",
    "Calibration metrics": r"expected_calibration_error|calibration_error|roc_auc_score",
    "Validation conclusions": r"core_validation_status|temporal_integrity|target_integrity",
}

excluded = {
    ".git", ".venv", "venv", "new_hockey_env",
    "__pycache__", "site-packages", ".ipynb_checkpoints"
}

matches = {name: [] for name in patterns}
files_scanned = 0

for path in root.rglob("*.py"):
    if any(part in excluded for part in path.relative_to(root).parts):
        continue

    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except (UnicodeError, OSError):
        continue

    files_scanned += 1

    for number, line in enumerate(lines, 1):
        for name, pattern in patterns.items():
            if re.search(pattern, line, flags=re.IGNORECASE):
                matches[name].append((
                    str(path.relative_to(root)), number, line.strip()
                ))

for name, hits in matches.items():
    print(f"\n=== {name.upper()} ({len(hits)} matches) ===")

    for filename, number, line in hits[:25]:
        print(f"{filename}:{number} | {line[:180]}")

    if len(hits) > 25:
        print(f"... {len(hits) - 25} additional matches")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Python source files scanned", files_scanned > 0)
check(2, "Validation framework references found",
      any(len(hits) > 0 for hits in matches.values()))
check(3, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75G: {sum(results)}/3 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75H — NOTEBOOK VALIDATION FRAMEWORK SEARCH
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import json
import re
from pathlib import Path
import config

print("\n=== BLOCK 75H: NOTEBOOK VALIDATION SEARCH ===\n")

root = Path(config.__file__).resolve().parent

patterns = {
    "Validation artifact": r"playoff_model_v6_validation|validation\.joblib",
    "Walk-forward evaluation": r"walk.forward|expanding.window|rolling.origin",
    "Historical evaluation": r"oos_rows|holdout_seasons|season_performance",
    "Validation structure": r"component_comparison|stage_performance|core_validation_status",
    "Evaluation metrics": r"roc_auc_score|brier_score_loss|expected_calibration_error",
}

excluded = {
    ".git", ".venv", "venv", "new_hockey_env",
    "__pycache__", "site-packages", ".ipynb_checkpoints"
}

matches = {name: [] for name in patterns}
notebooks_scanned = 0
notebook_errors = []

for path in root.rglob("*.ipynb"):
    if any(part in excluded for part in path.relative_to(root).parts):
        continue

    try:
        notebook = json.loads(path.read_text(encoding="utf-8"))
        notebooks_scanned += 1

        for cell_number, cell in enumerate(notebook.get("cells", []), 1):
            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))

            for name, pattern in patterns.items():
                if re.search(pattern, source, flags=re.IGNORECASE):
                    matching_lines = [
                        line.strip()
                        for line in source.splitlines()
                        if re.search(pattern, line, flags=re.IGNORECASE)
                    ]

                    matches[name].append({
                        "notebook": str(path.relative_to(root)),
                        "cell": cell_number,
                        "examples": matching_lines[:3]
                    })

    except (OSError, UnicodeError, json.JSONDecodeError) as e:
        notebook_errors.append((str(path), str(e)))

print(f"Notebooks scanned: {notebooks_scanned}")
print(f"Notebook read errors: {len(notebook_errors)}")

for name, hits in matches.items():
    print(f"\n=== {name.upper()} ({len(hits)} matching cells) ===")

    for hit in hits[:15]:
        print(f"\n{hit['notebook']} | Cell {hit['cell']}")
        for line in hit["examples"]:
            print(f"  {line[:180]}")

    if len(hits) > 15:
        print(f"... {len(hits) - 15} additional matches")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Notebook search completed", len(notebook_errors) == 0)
check(2, "Notebooks available to inspect", notebooks_scanned > 0)
check(3, "Historical validation references located",
      any(matches[name] for name in [
          "Validation artifact",
          "Walk-forward evaluation",
          "Historical evaluation",
          "Validation structure"
      ]))
check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75H: {sum(results)}/4 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75I — HISTORICAL VALIDATION METHODOLOGY
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import json
from pathlib import Path
import config

print("\n=== BLOCK 75I: VALIDATION METHODOLOGY INSPECTION ===\n")

root = Path(config.__file__).resolve().parent
path = root / "validation_check.ipynb"

notebook = json.loads(path.read_text(encoding="utf-8"))
cells = notebook["cells"]

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

print(f"Notebook: {path.name}")
print(f"Total cells: {len(cells)}")

# Cell numbers match the 1-based numbering used in Block 75H.
for cell_number in [37, 38, 39, 40]:
    print(f"\n{'=' * 70}")
    print(f"CELL {cell_number}")
    print("=" * 70)

    if cell_number > len(cells):
        print("Cell not found")
        continue

    cell = cells[cell_number - 1]
    source = "".join(cell.get("source", []))

    print("Cell type:", cell.get("cell_type"))
    print("Source lines:", len(source.splitlines()))

    for line_number, line in enumerate(source.splitlines(), 1):
        print(f"{line_number:4d} | {line}")

print("\n=== VALIDATION ===")

check(1, "Validation notebook exists", path.is_file())
check(2, "Historical validation cell available", len(cells) >= 39)
check(3, "Historical validation cell contains artifact reference",
      "validation.joblib" in "".join(cells[38].get("source", []))
      if len(cells) >= 39 else False)
check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75I: {sum(results)}/4 PASS")
print("=" * 60)



In [ ]:

# ============================================================
# BLOCK 75J — TRACE ROLLING PREDICTION GENERATION
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import json
import re
from pathlib import Path
import config

print("\n=== BLOCK 75J: ROLLING PREDICTION LINEAGE ===\n")

root = Path(config.__file__).resolve().parent
path = root / "validation_check.ipynb"

notebook = json.loads(path.read_text(encoding="utf-8"))
cells = notebook["cells"]

patterns = {
    "Rolling predictions": r"\brolling_predictions\b",
    "Season training filters": r"train_mask|test_mask|train_seasons|test_season|holdout_season",
    "Model training": r"\.fit\(|clone\(|XGBClassifier|RandomForestClassifier",
    "Prediction generation": r"predict_proba\(|playoff_prob_xgb|playoff_prob_rf",
}

matches = {name: [] for name in patterns}

for cell_number, cell in enumerate(cells, 1):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    for name, pattern in patterns.items():
        matching_lines = [
            (line_number, line.strip())
            for line_number, line in enumerate(source.splitlines(), 1)
            if re.search(pattern, line, flags=re.IGNORECASE)
        ]

        if matching_lines:
            matches[name].append({
                "cell": cell_number,
                "lines": matching_lines
            })

for name, hits in matches.items():
    print(f"\n{'=' * 65}")
    print(f"{name.upper()} — {len(hits)} matching cells")
    print("=" * 65)

    for hit in hits:
        print(f"\nCELL {hit['cell']}")

        for line_number, line in hit["lines"][:15]:
            print(f"  {line_number:4d} | {line[:160]}")

        if len(hit["lines"]) > 15:
            print(f"  ... {len(hit['lines']) - 15} more matching lines")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Validation notebook loaded", len(cells) > 0)
check(2, "Rolling prediction references found",
      len(matches["Rolling predictions"]) > 0)
check(3, "Training or prediction logic references found",
      bool(matches["Model training"] or matches["Prediction generation"]))
check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75J: {sum(results)}/4 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75K — EXPANDING-WINDOW TRAINING INTEGRITY
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import json
from pathlib import Path
import config

print("\n=== BLOCK 75K: EXPANDING-WINDOW TRAINING INTEGRITY ===\n")

root = Path(config.__file__).resolve().parent
path = root / "validation_check.ipynb"

notebook = json.loads(path.read_text(encoding="utf-8"))
cells = notebook["cells"]

# Cell numbering follows the 1-based numbering from Block 75J.
source = "".join(cells[25].get("source", []))
lines = source.splitlines()

sections = {
    "Season setup and training masks": (1, 145),
    "Feature selection and preprocessing": (145, 285),
    "Model fitting and predictions": (285, 350),
    "Holdout prediction construction": (440, 525),
    "Rolling prediction finalization": (620, 665),
}

for name, (start, end) in sections.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()} — LINES {start}-{end}")
    print("=" * 70)

    for number in range(start, min(end, len(lines)) + 1):
        print(f"{number:4d} | {lines[number - 1]}")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Validation notebook loaded", len(cells) >= 26)
check(2, "Expanding-window training filter present",
      'df_features["season_id"] < test_season' in source)
check(3, "Holdout-season filter present",
      'df_features["season_id"] == test_season' in source)
check(4, "Model training calls present",
      "xgb_pipeline.fit(" in source and "rf_pipeline.fit(" in source)
check(5, "Rolling predictions constructed",
      "rolling_predictions.append(" in source)
check(6, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75K: {sum(results)}/6 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75L — V6/V7 TRAINING CONFIGURATION COMPARISON
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

from pathlib import Path
from joblib import load
import config

print("\n=== BLOCK 75L: MODEL CONFIGURATION COMPARISON ===\n")

root = Path(config.__file__).resolve().parent

v6 = load(root / "models" / "playoff_models_v6.joblib")
v7 = load(root / "models" / "playoff_models_v7.joblib")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

def extract_params(bundle, pipeline_key):
    pipeline = bundle[pipeline_key]

    return {
        "pipeline_steps": list(pipeline.named_steps.keys()),
        "parameters": {
            key: value
            for key, value in pipeline.get_params(deep=True).items()
            if "__" in key and not hasattr(value, "fit")
        }
    }

for pipeline_key in ["xgb_pipeline", "rf_pipeline"]:
    print(f"\n{'=' * 65}")
    print(pipeline_key.upper())
    print("=" * 65)

    p6 = extract_params(v6, pipeline_key)
    p7 = extract_params(v7, pipeline_key)

    print("V6 steps:", p6["pipeline_steps"])
    print("V7 steps:", p7["pipeline_steps"])

    keys = sorted(set(p6["parameters"]) | set(p7["parameters"]))

    differences = []

    for key in keys:
        a = p6["parameters"].get(key)
        b = p7["parameters"].get(key)

        if str(a) != str(b):
            differences.append((key, a, b))

    print(f"\nParameter differences: {len(differences)}")

    for key, a, b in differences[:50]:
        print(f"\n{key}")
        print(f"  V6: {a}")
        print(f"  V7: {b}")

    if not differences:
        print("No parameter differences detected.")

features6 = set(v6["features"])
features7 = set(v7["features"])

print("\n=== FEATURE COMPARISON ===")

print("V6 feature count:", len(v6["features"]))
print("V7 feature count:", len(v7["features"]))
print("Shared features:", len(features6 & features7))

print("\nOnly in V6:", sorted(features6 - features7))
print("Only in V7:", sorted(features7 - features6))

print("\n=== VALIDATION ===")

check(1, "Both saved bundles loaded", bool(v6) and bool(v7))
check(2, "Both use XGBoost and Random Forest",
      all(k in v6 and k in v7 for k in ["xgb_pipeline", "rf_pipeline"]))
check(3, "Both have 121 features",
      len(v6["features"]) == 121 and len(v7["features"]) == 121)
check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75L: {sum(results)}/4 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75M — FEATURE-SELECTION COMPATIBILITY
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import inspect
from pathlib import Path
from joblib import load
import config

from prediction_model.training import train_new_model as training

print("\n=== BLOCK 75M: FEATURE-SELECTION COMPATIBILITY ===\n")

root = Path(config.__file__).resolve().parent

v6 = load(root / "models" / "playoff_models_v6.joblib")
v7 = load(root / "models" / "playoff_models_v7.joblib")

features6 = set(v6["features"])
features7 = set(v7["features"])

new_features = sorted(features7 - features6)
removed_features = sorted(features6 - features7)

print("=== FEATURE CHANGES ===")

print("\nNew in V7:")
for feature in new_features:
    print(" +", feature)

print("\nRemoved from V6:")
for feature in removed_features:
    print(" -", feature)

print("\n=== TRAINING FUNCTION SIGNATURES ===")

functions = [
    "_build_feature_matrix",
    "_make_xgb_pipeline",
    "_make_rf_pipeline",
    "select_features_from_shap",
]

for name in functions:
    func = getattr(training, name, None)

    print(f"\n{name}:")

    if callable(func):
        print("Signature:", inspect.signature(func))
        print("Defined at:", inspect.getsourcefile(func))
    else:
        print("NOT FOUND")

print("\n=== CURRENT FEATURE MATRIX COVERAGE ===")

if "df_features" in globals():
    available = set(df_features.columns)

    print("df_features rows:", len(df_features))
    print("df_features columns:", len(df_features.columns))

    for feature in new_features:
        print(
            f"{'AVAILABLE' if feature in available else 'MISSING'} "
            f"| {feature}"
        )

    missing = sorted(features7 - available)

    print("\nV7 selected features missing:", missing)
else:
    print("df_features not present in notebook globals.")
    missing = None

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "V6 and V7 bundles loaded", bool(v6) and bool(v7))
check(2, "Five features added in V7", len(new_features) == 5)
check(3, "Five features removed from V6", len(removed_features) == 5)
check(4, "Training helper functions available",
      all(callable(getattr(training, name, None)) for name in functions))
check(5, "V7 feature coverage verified",
      missing is not None and len(missing) == 0)
check(6, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75M: {sum(results)}/6 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75N — HISTORICAL FEATURE DATA AVAILABILITY
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import pandas as pd
from joblib import load
from pathlib import Path
import config

print("\n=== BLOCK 75N: HISTORICAL FEATURE DATA ===\n")

root = Path(config.__file__).resolve().parent
v6 = load(root / "models" / "playoff_models_v6.joblib")
v7 = load(root / "models" / "playoff_models_v7.joblib")

required = {"season_id", "made_playoffs"}
candidates = []

for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame):
        continue

    columns = set(obj.columns)

    if "season_id" not in columns:
        continue

    seasons = pd.to_numeric(
        obj["season_id"], errors="coerce"
    ).dropna()

    candidates.append({
        "name": name,
        "rows": len(obj),
        "columns": len(obj.columns),
        "seasons": int(seasons.nunique()),
        "first": int(seasons.min()) if len(seasons) else None,
        "last": int(seasons.max()) if len(seasons) else None,
        "has_target": "made_playoffs" in columns,
        "v6_features": len(set(v6["features"]) & columns),
        "v7_features": len(set(v7["features"]) & columns),
    })

print("=== AVAILABLE DATAFRAMES ===")

if candidates:
    summary = pd.DataFrame(candidates).sort_values(
        ["seasons", "rows"], ascending=False
    )
    print(summary.to_string(index=False))
else:
    print("No DataFrames with season_id found.")

print("\n=== V7-SPECIFIC FEATURES ===")

new_features = sorted(set(v7["features"]) - set(v6["features"]))

for item in candidates:
    df = globals()[item["name"]]
    present = [f for f in new_features if f in df.columns]

    print(
        f"{item['name']}: "
        f"{len(present)}/{len(new_features)} available"
    )

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Saved bundles available", bool(v6) and bool(v7))
check(2, "Historical DataFrame candidates found", len(candidates) > 0)
check(3, "Multi-season historical data available",
      any(x["seasons"] >= 10 for x in candidates))
check(4, "Historical target available",
      any(x["seasons"] >= 10 and x["has_target"] for x in candidates))
check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75N: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75O — HISTORICAL DATA SOURCE DISCOVERY
# READ ONLY — NO DATABASE OR CONFIGURATION CHANGES
# ============================================================

import json
import re
from pathlib import Path
import config

print("\n=== BLOCK 75O: HISTORICAL DATA SOURCE DISCOVERY ===\n")

root = Path(config.__file__).resolve().parent
path = root / "validation_check.ipynb"

notebook = json.loads(path.read_text(encoding="utf-8"))
cells = notebook["cells"]

patterns = {
    "Historical feature assignment": r"\bdf_features\s*=",
    "Raw data assignment": r"\bdf_raw\s*=|\bdf\s*=\s*(?:con|conn|duckdb)",
    "Feature engineering": r"build_features\(|build_canonical_training_features\(",
    "Database reads": r"read_sql|\.execute\(|\.sql\(|duckdb\.connect|fetchdf\(|\.df\(\)",
    "Training table references": r"team_daily_stats|DB_PATH|season_id",
}

matches = {name: [] for name in patterns}

for cell_number, cell in enumerate(cells, 1):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    for name, pattern in patterns.items():
        found = [
            (line_number, line.strip())
            for line_number, line in enumerate(source.splitlines(), 1)
            if re.search(pattern, line, flags=re.IGNORECASE)
        ]

        if found:
            matches[name].append((cell_number, found))

for name, hits in matches.items():
    print(f"\n=== {name.upper()} ({len(hits)} cells) ===")

    for cell_number, lines in hits:
        # Focus on setup cells preceding the rolling backtest.
        if cell_number > 26:
            continue

        print(f"\nCELL {cell_number}")

        for line_number, line in lines[:12]:
            print(f"  {line_number:4d} | {line[:180]}")

        if len(lines) > 12:
            print(f"  ... {len(lines) - 12} additional matches")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Validation notebook loaded", len(cells) > 0)
check(2, "Historical feature assignment located",
      bool(matches["Historical feature assignment"]))
check(3, "Feature engineering references located",
      bool(matches["Feature engineering"]))
check(4, "Database access references located",
      bool(matches["Database reads"]))
check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75O: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75P — HISTORICAL DATA AND FEATURE COVERAGE
# READ ONLY — NO NEW CONNECTIONS OR DATABASE WRITES
# ============================================================

print("\n=== BLOCK 75P: HISTORICAL FEATURE COVERAGE ===\n")

# ------------------------------------------------------------
# 1. Verify existing dependencies
# ------------------------------------------------------------

build_fn = globals().get("build_features_func", globals().get("build_features"))
db = globals().get("con")

if not callable(build_fn):
    raise RuntimeError("build_features is not loaded. Run its existing import first.")

if db is None:
    raise RuntimeError("Existing DuckDB connection 'con' is not available.")

# ------------------------------------------------------------
# 2. Load historical data using existing connection
# ------------------------------------------------------------

historical_raw = db.execute("""
    SELECT *
    FROM team_daily_stats
    WHERE season_id < 20252026
    ORDER BY season_id, game_date, team_id
""").df()

print("=== RAW HISTORICAL COVERAGE ===")
print("Rows:", len(historical_raw))
print("Seasons:", historical_raw["season_id"].nunique())
print("First season:", historical_raw["season_id"].min())
print("Last season:", historical_raw["season_id"].max())

print("\nRows by season:")
print(historical_raw.groupby("season_id").size().to_string())

# ------------------------------------------------------------
# 3. Build historical features
# ------------------------------------------------------------

print("\n=== FEATURE ENGINEERING ===")

historical_features = build_fn(historical_raw.copy())

features6 = set(v6["features"])
features7 = set(v7["features"])
available = set(historical_features.columns)

missing6 = sorted(features6 - available)
missing7 = sorted(features7 - available)

print("Raw rows:", len(historical_raw))
print("Engineered rows:", len(historical_features))
print("Engineered columns:", len(historical_features.columns))

print("V6 features available:", len(features6 - set(missing6)), "/ 121")
print("V7 features available:", len(features7 - set(missing7)), "/ 121")

print("\nMissing V6 features:", missing6)
print("Missing V7 features:", missing7)

# ------------------------------------------------------------
# 4. V7-specific feature coverage
# ------------------------------------------------------------

print("\n=== V7-ONLY FEATURE COVERAGE ===")

new_features = sorted(features7 - features6)

for feature in new_features:
    if feature not in available:
        print(f"MISSING | {feature}")
        continue

    values = pd.to_numeric(historical_features[feature], errors="coerce")

    print(
        f"{feature:25s} | "
        f"non-null={values.notna().sum():,} | "
        f"null={values.isna().sum():,} | "
        f"unique={values.nunique():,}"
    )

# ------------------------------------------------------------
# 5. Coverage by season
# ------------------------------------------------------------

print("\n=== V7 FEATURE COVERAGE BY SEASON ===")

coverage = []

for season, frame in historical_features.groupby("season_id"):
    row = {"season_id": season, "rows": len(frame)}

    for feature in new_features:
        if feature in frame.columns:
            row[feature] = round(
                pd.to_numeric(frame[feature], errors="coerce").notna().mean() * 100,
                2
            )
        else:
            row[feature] = 0.0

    coverage.append(row)

print(pd.DataFrame(coverage).to_string(index=False))

# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Historical source contains multiple seasons",
      historical_raw["season_id"].nunique() >= 10)

check(2, "Historical data precedes 2025-26",
      historical_raw["season_id"].max() < 20252026)

check(3, "Feature engineering preserves row count",
      len(historical_features) == len(historical_raw))

check(4, "All V6 features available", len(missing6) == 0)
check(5, "All V7 features available", len(missing7) == 0)

check(6, "Historical playoff outcomes available",
      "made_playoffs" in historical_features.columns)

check(7, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75P: {sum(results)}/7 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75Q — HISTORICAL FEATURE LOOK-AHEAD VALIDATION
# READ ONLY — NO MODEL TRAINING OR DATABASE WRITES
# ============================================================

print("\n=== BLOCK 75Q: FEATURE LOOK-AHEAD VALIDATION ===\n")

# Reuse historical_raw, historical_features, build_fn, v6, v7
key_cols = ["season_id", "game_date", "team_id", "game_id"]
model_features = sorted(set(v6["features"]) | set(v7["features"]))

# Select representative dates across historical seasons.
sample_seasons = [20142015, 20182019, 20222023, 20242025]
cutoffs = []

for season in sample_seasons:
    frame = historical_raw[historical_raw["season_id"] == season]
    dates = sorted(frame["game_date"].dropna().unique())

    for fraction in [0.15, 0.50, 0.85]:
        if dates:
            cutoffs.append((season, dates[int((len(dates) - 1) * fraction)]))

print("Historical cutoff tests:", len(cutoffs))

results = []
details = []

for season, cutoff in cutoffs:
    # Include all earlier seasons and only games through cutoff.
    truncated_raw = historical_raw[
        (historical_raw["season_id"] < season) |
        (
            (historical_raw["season_id"] == season) &
            (historical_raw["game_date"] <= cutoff)
        )
    ].copy()

    truncated_features = build_fn(truncated_raw)

    original = historical_features[
        (historical_features["season_id"] == season) &
        (historical_features["game_date"] == cutoff)
    ][key_cols + model_features].copy()

    truncated = truncated_features[
        (truncated_features["season_id"] == season) &
        (truncated_features["game_date"] == cutoff)
    ][key_cols + model_features].copy()

    merged = original.merge(
        truncated,
        on=key_cols,
        suffixes=("_full", "_cutoff"),
        how="outer",
        indicator=True,
        validate="one_to_one"
    )

    mismatched = []

    for feature in model_features:
        a = pd.to_numeric(merged[f"{feature}_full"], errors="coerce")
        b = pd.to_numeric(merged[f"{feature}_cutoff"], errors="coerce")

        equal = (a - b).abs().le(1e-8) | (a.isna() & b.isna())

        if not equal.all():
            mismatched.append(feature)

    passed = (
        merged["_merge"].eq("both").all()
        and not mismatched
    )

    results.append(passed)

    details.append({
        "season": season,
        "cutoff": str(cutoff)[:10],
        "rows": len(merged),
        "status": "PASS" if passed else "FAIL",
        "mismatched_features": len(mismatched),
        "examples": ", ".join(mismatched[:8])
    })

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{season} | {str(cutoff)[:10]} | "
        f"{len(mismatched)} mismatched features"
    )

print("\n=== DETAILED RESULTS ===")
print(pd.DataFrame(details).to_string(index=False))

print("\n=== VALIDATION ===")

checks = []

def check(number, description, condition):
    passed = bool(condition)
    checks.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "All historical cutoff tests completed",
      len(results) == len(cutoffs) and len(cutoffs) > 0)

check(2, "No look-ahead differences detected",
      all(results))

check(3, "Both model feature sets tested",
      len(model_features) == len(set(v6["features"]) | set(v7["features"])))

check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75Q: {sum(checks)}/4 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75R — PAIRED BACKTEST PREPARATION
# READ ONLY — NO TRAINING, DATABASE QUERIES OR WRITES
# ============================================================

print("\n=== BLOCK 75R: PAIRED BACKTEST PREPARATION ===\n")

# Include the latest completed season as the final holdout.
# Earlier seasons are already available in historical_features.
full_features = pd.concat(
    [historical_features, features],
    ignore_index=True
).sort_values(["season_id", "game_date", "team_id"]).reset_index(drop=True)

seasons = sorted(full_features["season_id"].dropna().astype(int).unique())
holdout_seasons = seasons[1:]

print("=== EVALUATION POPULATION ===")
print("Total rows:", len(full_features))
print("Seasons:", len(seasons))
print("First season:", seasons[0])
print("Last season:", seasons[-1])
print("Holdout seasons:", len(holdout_seasons))

print("\n=== MODEL FEATURE SETS ===")
print("V6 features:", len(v6["features"]))
print("V7 features:", len(v7["features"]))
print("Shared features:", len(set(v6["features"]) & set(v7["features"])))

print("\n=== HOLDOUT PLAN ===")

plan = []

for season in holdout_seasons:
    train = full_features["season_id"] < season
    test = full_features["season_id"] == season

    plan.append({
        "holdout_season": season,
        "train_rows": int(train.sum()),
        "test_rows": int(test.sum()),
        "train_last_season": int(full_features.loc[train, "season_id"].max())
    })

print(pd.DataFrame(plan).to_string(index=False))

print("\n=== DATA INTEGRITY ===")

key_cols = ["season_id", "game_date", "team_id", "game_id"]
duplicate_keys = full_features.duplicated(key_cols).sum()
missing_targets = full_features["made_playoffs"].isna().sum()

print("Duplicate team-game keys:", duplicate_keys)
print("Missing playoff outcomes:", missing_targets)

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "13 completed seasons available", len(seasons) == 13)
check(2, "12 historical holdout seasons", len(holdout_seasons) == 12)
check(3, "Expected full training population",
      len(full_features) == 31944)
check(4, "No duplicate team-game keys", duplicate_keys == 0)
check(5, "No missing playoff outcomes", missing_targets == 0)
check(6, "All V6 features present",
      set(v6["features"]).issubset(full_features.columns))
check(7, "All V7 features present",
      set(v7["features"]).issubset(full_features.columns))
check(8, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75R: {sum(results)}/8 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75S — 2025-26 OBSERVATION GRAIN DIAGNOSTIC
# READ ONLY — NO DATABASE ACCESS OR MODEL TRAINING
# ============================================================

print("\n=== BLOCK 75S: OBSERVATION GRAIN DIAGNOSTIC ===\n")

prior = historical_features[
    historical_features["season_id"] == 20242025
].copy()

current = features[
    features["season_id"] == 20252026
].copy()

print("=== SEASON COUNTS ===")
print("2024-25 rows:", len(prior))
print("2025-26 rows:", len(current))
print("Difference:", len(current) - len(prior))

print("\n=== OBSERVATION GRAIN ===")

for name, frame in [("2024-25", prior), ("2025-26", current)]:
    print(f"\n{name}")

    print("Rows:", len(frame))
    print("Teams:", frame["team_id"].nunique())
    print("Game dates:", frame["game_date"].nunique())
    print("Unique game IDs:", frame["game_id"].nunique())

    for keys in [
        ["team_id", "game_date"],
        ["team_id", "game_id"],
        ["team_id", "games_played"],
    ]:
        duplicates = frame.duplicated(keys).sum()
        print(f"Duplicate {keys}: {duplicates:,}")

    print("\nRows per team:")
    print(frame.groupby("team_id").size().describe().to_string())

    print("\nGames played:")
    print(pd.to_numeric(frame["games_played"], errors="coerce").describe().to_string())

print("\n=== 2025-26 ROW FREQUENCY ===")

print("\nRows per team-game date:")
print(
    current.groupby(["team_id", "game_date"])
    .size()
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nRows per team-games played:")
print(
    current.groupby(["team_id", "games_played"])
    .size()
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n=== 2025-26 DATE RANGE ===")
print("First:", current["game_date"].min())
print("Last:", current["game_date"].max())

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "2024-25 has expected 2,624 rows", len(prior) == 2624)
check(2, "2025-26 has 5,299 prediction observations", len(current) == 5299)
check(3, "Both seasons contain 32 teams",
      prior["team_id"].nunique() == 32 and current["team_id"].nunique() == 32)
check(4, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75S: {sum(results)}/4 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75T — GAME-LEVEL TRAINING GRAIN VALIDATION
# READ ONLY — NO DATABASE ACCESS OR MODEL TRAINING
# ============================================================

print("\n=== BLOCK 75T: GAME-LEVEL TRAINING GRAIN ===\n")

current = features[features["season_id"] == 20252026].copy()
current["game_date"] = pd.to_datetime(current["game_date"])

# ------------------------------------------------------------
# 1. Examine repeated team-game observations
# ------------------------------------------------------------

grouped = current.groupby(["team_id", "game_id"], dropna=False)

summary = grouped.agg(
    snapshots=("game_date", "size"),
    first_date=("game_date", "min"),
    last_date=("game_date", "max"),
    games_played_min=("games_played", "min"),
    games_played_max=("games_played", "max")
).reset_index()

print("=== TEAM-GAME SUMMARY ===")
print("Unique team-game pairs:", len(summary))
print("Expected:", 2624)

print("\nSnapshots per team-game:")
print(summary["snapshots"].value_counts().sort_index().to_string())

print("\nRepeated pairs:", (summary["snapshots"] > 1).sum())

print("\n=== DATE SPAN FOR REPEATED PAIRS ===")

repeated = summary[summary["snapshots"] > 1].copy()
repeated["days_between"] = (
    repeated["last_date"] - repeated["first_date"]
).dt.days

print(repeated["days_between"].describe().to_string())

print("\n=== EXAMPLE REPEATED PAIRS ===")
print(repeated.head(10).to_string(index=False))

# ------------------------------------------------------------
# 2. Candidate game-level extraction
# ------------------------------------------------------------

# The last snapshot for each team-game is a candidate,
# not yet confirmed as the correct training observation.
candidate = (
    current.sort_values(["team_id", "game_id", "game_date"])
    .drop_duplicates(["team_id", "game_id"], keep="last")
    .copy()
)

print("\n=== CANDIDATE GAME-LEVEL DATASET ===")
print("Candidate rows:", len(candidate))
print("Unique teams:", candidate["team_id"].nunique())
print("Unique game IDs:", candidate["game_id"].nunique())

print("\nGames played per team:")
print(
    candidate.groupby("team_id")["games_played"]
    .agg(["min", "max", "nunique"])
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# 3. Validation
# ------------------------------------------------------------

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Exactly 2,624 unique team-game pairs", len(summary) == 2624)
check(2, "Candidate contains 2,624 rows", len(candidate) == 2624)
check(3, "Candidate contains 32 teams", candidate["team_id"].nunique() == 32)
check(4, "Candidate contains 1,312 games", candidate["game_id"].nunique() == 1312)
check(5, "Each team has 82 game observations",
      candidate.groupby("team_id").size().eq(82).all())
check(6, "Each team has 82 distinct games-played values",
      candidate.groupby("team_id")["games_played"].nunique().eq(82).all())
check(7, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75T: {sum(results)}/7 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75U — GAME-LEVEL SNAPSHOT SELECTION VALIDATION
# READ ONLY — NO DATABASE ACCESS OR MODEL TRAINING
# ============================================================

print("\n=== BLOCK 75U: SNAPSHOT SELECTION VALIDATION ===\n")

keys = ["team_id", "game_id"]
model_features = sorted(set(v6["features"]) | set(v7["features"]))

ordered = current.sort_values(keys + ["game_date"])

first = ordered.drop_duplicates(keys, keep="first").copy()
last = ordered.drop_duplicates(keys, keep="last").copy()

paired = first[keys + ["game_date", "games_played"] + model_features].merge(
    last[keys + ["game_date", "games_played"] + model_features],
    on=keys,
    suffixes=("_first", "_last"),
    validate="one_to_one"
)

print("=== SNAPSHOT DATE COMPARISON ===")

date_gap = (
    pd.to_datetime(paired["game_date_last"]) -
    pd.to_datetime(paired["game_date_first"])
).dt.days

print("Team-game pairs:", len(paired))
print("Same-day snapshots:", (date_gap == 0).sum())
print("Different-day snapshots:", (date_gap > 0).sum())
print("Maximum date gap:", date_gap.max())

print("\n=== FEATURE DIFFERENCES ===")

feature_changes = []

for feature in model_features:
    a = pd.to_numeric(paired[f"{feature}_first"], errors="coerce")
    b = pd.to_numeric(paired[f"{feature}_last"], errors="coerce")

    changed = ~((a - b).abs().le(1e-8) | (a.isna() & b.isna()))

    if changed.any():
        feature_changes.append({
            "feature": feature,
            "changed_pairs": int(changed.sum()),
            "changed_pct": round(changed.mean() * 100, 2)
        })

changes_df = pd.DataFrame(feature_changes)

print("Features changing between snapshots:", len(feature_changes))

if not changes_df.empty:
    print(
        changes_df.sort_values("changed_pairs", ascending=False)
        .head(25)
        .to_string(index=False)
    )
else:
    print("No feature differences detected.")

print("\n=== V7-SPECIFIC FEATURES ===")

new_features = sorted(set(v7["features"]) - set(v6["features"]))

for feature in new_features:
    a = pd.to_numeric(paired[f"{feature}_first"], errors="coerce")
    b = pd.to_numeric(paired[f"{feature}_last"], errors="coerce")

    changed = ~((a - b).abs().le(1e-8) | (a.isna() & b.isna()))

    print(f"{feature:25s} | changed pairs: {changed.sum():,}")

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "First snapshot has 2,624 team-game pairs", len(first) == 2624)
check(2, "Last snapshot has 2,624 team-game pairs", len(last) == 2624)
check(3, "Paired snapshots align exactly", len(paired) == 2624)
check(4, "All model features compared", len(model_features) == 126)
check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75U: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75V — GAME-DATE AND SNAPSHOT ALIGNMENT
# READ ONLY — NO DATABASE ACCESS OR MODEL TRAINING
# ============================================================

print("\n=== BLOCK 75V: GAME-DATE ALIGNMENT ===\n")

# ------------------------------------------------------------
# 1. Compare first and last snapshot dates
# ------------------------------------------------------------

keys = ["team_id", "game_id"]

first = current.sort_values("game_date").drop_duplicates(keys, keep="first")
last = current.sort_values("game_date").drop_duplicates(keys, keep="last")

comparison = first[keys + ["game_date", "games_played"]].merge(
    last[keys + ["game_date", "games_played"]],
    on=keys,
    suffixes=("_first", "_last"),
    validate="one_to_one"
)

comparison["date_gap"] = (
    pd.to_datetime(comparison["game_date_last"]) -
    pd.to_datetime(comparison["game_date_first"])
).dt.days

print("=== SNAPSHOT TIMING ===")
print("Team-game pairs:", len(comparison))
print("First/last same date:", comparison["date_gap"].eq(0).sum())
print("First/last different dates:", comparison["date_gap"].gt(0).sum())

print("\nDate gap distribution:")
print(comparison["date_gap"].value_counts().sort_index().to_string())

# ------------------------------------------------------------
# 2. Check game ID consistency across both teams
# ------------------------------------------------------------

print("\n=== GAME ID DATE CONSISTENCY ===")

for label, frame in [("First", first), ("Last", last)]:
    game_dates = frame.groupby("game_id")["game_date"].nunique()

    print(f"\n{label} snapshot:")
    print("Games:", len(game_dates))
    print("Games with one shared date:", game_dates.eq(1).sum())
    print("Games with different team dates:", game_dates.gt(1).sum())

# ------------------------------------------------------------
# 3. Compare snapshot selection with historical game grain
# ------------------------------------------------------------

print("\n=== HISTORICAL GAME-GRAIN REFERENCE ===")

historical_game_dates = historical_raw.groupby(
    ["season_id", "game_id"]
)["game_date"].nunique()

print("Historical games:", len(historical_game_dates))
print("Historical games with one shared date:",
      historical_game_dates.eq(1).sum())
print("Historical games with different team dates:",
      historical_game_dates.gt(1).sum())

# ------------------------------------------------------------
# 4. Identify examples of date discrepancies
# ------------------------------------------------------------

print("\n=== FIRST SNAPSHOT DATE DISCREPANCIES ===")

first_dates = first.groupby("game_id")["game_date"].nunique()
different_games = first_dates[first_dates > 1].index

examples = first[first["game_id"].isin(different_games)].sort_values(
    ["game_id", "team_id"]
)

print(examples[keys + ["game_date", "games_played"]].head(20).to_string(index=False))

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Exactly 2,624 team-game pairs", len(comparison) == 2624)
check(2, "First snapshot has 1,312 games", first["game_id"].nunique() == 1312)
check(3, "Last snapshot has 1,312 games", last["game_id"].nunique() == 1312)
check(4, "Historical games share consistent dates",
      historical_game_dates.eq(1).all())
check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75V: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75W — RECONSTRUCT TRAINING POPULATION
# READ ONLY — NO DATABASE ACCESS OR MODEL TRAINING
# ============================================================

print("\n=== BLOCK 75W: RECONSTRUCT TRAINING POPULATION ===\n")

# ------------------------------------------------------------
# 1. Select first snapshot per team-game
# ------------------------------------------------------------

game_level_2026 = (
    features[features["season_id"] == 20252026]
    .sort_values(["team_id", "game_id", "game_date"])
    .drop_duplicates(["team_id", "game_id"], keep="first")
    .copy()
)

# ------------------------------------------------------------
# 2. Combine historical and 2025-26 game-level observations
# ------------------------------------------------------------

backtest_features = pd.concat(
    [historical_features, game_level_2026],
    ignore_index=True
).sort_values(
    ["season_id", "game_date", "team_id"]
).reset_index(drop=True)

print("=== POPULATION ===")
print("Historical rows:", len(historical_features))
print("2025-26 game-level rows:", len(game_level_2026))
print("Combined rows:", len(backtest_features))
print("Seasons:", backtest_features["season_id"].nunique())

print("\nRows by season:")
print(backtest_features.groupby("season_id").size().to_string())

# ------------------------------------------------------------
# 3. Validate game-level uniqueness
# ------------------------------------------------------------

keys = ["season_id", "team_id", "game_id"]

duplicates = backtest_features.duplicated(keys).sum()
missing_targets = backtest_features["made_playoffs"].isna().sum()

print("\n=== INTEGRITY ===")
print("Duplicate team-game keys:", duplicates)
print("Missing playoff outcomes:", missing_targets)

print("\n=== 2025-26 GAME-DATE CONSISTENCY ===")

game_dates = game_level_2026.groupby("game_id")["game_date"].nunique()

print("Games:", len(game_dates))
print("Games with matching team dates:", game_dates.eq(1).sum())

# ------------------------------------------------------------
# 4. Validate model feature availability
# ------------------------------------------------------------

features6 = set(v6["features"])
features7 = set(v7["features"])
available = set(backtest_features.columns)

missing6 = sorted(features6 - available)
missing7 = sorted(features7 - available)

print("\n=== FEATURE AVAILABILITY ===")
print("Missing V6 features:", missing6)
print("Missing V7 features:", missing7)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Exactly 2,624 game-level observations for 2025-26",
      len(game_level_2026) == 2624)

check(2, "Expected 31,944 total observations",
      len(backtest_features) == 31944)

check(3, "13 completed seasons available",
      backtest_features["season_id"].nunique() == 13)

check(4, "No duplicate team-game keys", duplicates == 0)

check(5, "No missing playoff outcomes", missing_targets == 0)

check(6, "Both teams share game dates",
      game_dates.eq(1).all())

check(7, "All V6 and V7 features available",
      not missing6 and not missing7)

check(8, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75W: {sum(results)}/8 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75X — CANONICAL TRAINING SOURCE ALIGNMENT
# READ ONLY — NO MODEL TRAINING OR DATABASE WRITES
# ============================================================

print("\n=== BLOCK 75X: CANONICAL TRAINING ALIGNMENT ===\n")

# Use the same raw source and season exclusion as training.
source_raw = con.execute("""
    SELECT *
    FROM team_daily_stats
    WHERE season_id = 20252026
    ORDER BY season_id, game_date, team_id
""").df()

print("=== CANONICAL SOURCE ===")
print("Raw 2025-26 rows:", len(source_raw))
print("Unique team-game pairs:",
      source_raw[["team_id", "game_id"]].drop_duplicates().shape[0])
print("Unique game IDs:", source_raw["game_id"].nunique())

# Inspect the source's grain before feature engineering.
print("\n=== SOURCE GRAIN ===")
print("Duplicate team-game keys:",
      source_raw.duplicated(["team_id", "game_id"]).sum())
print("Duplicate team-date keys:",
      source_raw.duplicated(["team_id", "game_date"]).sum())

print("\n=== RECONSTRUCTED GRAIN ===")
print("Reconstructed rows:", len(game_level_2026))
print("Reconstructed unique games:", game_level_2026["game_id"].nunique())

# Compare identities without assuming feature equivalence yet.
keys = ["season_id", "team_id", "game_id"]

source_keys = source_raw[keys].drop_duplicates()
reconstructed_keys = game_level_2026[keys].drop_duplicates()

alignment = source_keys.merge(
    reconstructed_keys,
    on=keys,
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print("\n=== IDENTITY ALIGNMENT ===")
print(alignment["_merge"].value_counts().to_string())

# Compare source dates against reconstructed first-snapshot dates.
date_comparison = source_raw[keys + ["game_date"]].merge(
    game_level_2026[keys + ["game_date"]],
    on=keys,
    suffixes=("_source", "_reconstructed"),
    validate="one_to_one"
)

date_matches = (
    pd.to_datetime(date_comparison["game_date_source"]) ==
    pd.to_datetime(date_comparison["game_date_reconstructed"])
)

print("\n=== DATE ALIGNMENT ===")
print("Compared:", len(date_comparison))
print("Matching dates:", date_matches.sum())
print("Different dates:", (~date_matches).sum())

if (~date_matches).any():
    print("\nExamples:")
    print(date_comparison.loc[~date_matches].head(10).to_string(index=False))

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Canonical source contains 2,624 observations",
      len(source_raw) == 2624)

check(2, "Canonical source has unique team-game keys",
      not source_raw.duplicated(keys).any())

check(3, "Reconstructed keys match canonical source",
      alignment["_merge"].eq("both").all() and len(alignment) == 2624)

check(4, "Reconstructed dates match canonical source",
      len(date_comparison) == 2624 and date_matches.all())

check(5, "Reconstructed training population remains 31,944",
      len(backtest_features) == 31944)

check(6, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75X: {sum(results)}/6 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75Y — CANONICAL FEATURE EQUIVALENCE
# READ ONLY — NO MODEL TRAINING OR DATABASE WRITES
# ============================================================

print("\n=== BLOCK 75Y: CANONICAL FEATURE EQUIVALENCE ===\n")

from prediction_model.training.train_new_model import build_canonical_training_features

canonical_2026 = build_canonical_training_features(source_raw.copy())

keys = ["season_id", "team_id", "game_id"]
model_features = sorted(set(v6["features"]) | set(v7["features"]))

print("=== POPULATION ===")
print("Canonical rows:", len(canonical_2026))
print("Reconstructed rows:", len(game_level_2026))
print("Features to compare:", len(model_features))

missing_canonical = sorted(set(model_features) - set(canonical_2026.columns))
missing_reconstructed = sorted(set(model_features) - set(game_level_2026.columns))

print("\nMissing canonical features:", missing_canonical)
print("Missing reconstructed features:", missing_reconstructed)

# ------------------------------------------------------------
# Compare identical team-game observations
# ------------------------------------------------------------

common_features = [
    f for f in model_features
    if f in canonical_2026.columns and f in game_level_2026.columns
]

paired = canonical_2026[keys + common_features].merge(
    game_level_2026[keys + common_features],
    on=keys,
    how="outer",
    suffixes=("_canonical", "_reconstructed"),
    indicator=True,
    validate="one_to_one"
)

differences = []

for feature in common_features:
    a = pd.to_numeric(paired[f"{feature}_canonical"], errors="coerce")
    b = pd.to_numeric(paired[f"{feature}_reconstructed"], errors="coerce")

    equal = (a - b).abs().le(1e-8) | (a.isna() & b.isna())
    changed = ~equal

    if changed.any():
        differences.append({
            "feature": feature,
            "different_rows": int(changed.sum()),
            "different_pct": round(changed.mean() * 100, 2),
            "max_abs_difference": (a - b).abs().max()
        })

differences_df = pd.DataFrame(differences)

print("\n=== IDENTITY ALIGNMENT ===")
print(paired["_merge"].value_counts().to_string())

print("\n=== FEATURE DIFFERENCES ===")
print("Features with differences:", len(differences))

if not differences_df.empty:
    print(
        differences_df.sort_values("different_rows", ascending=False)
        .head(30)
        .to_string(index=False)
    )
else:
    print("All compared features match.")

print("\n=== V7-SPECIFIC FEATURES ===")

v7_only = sorted(set(v7["features"]) - set(v6["features"]))

for feature in v7_only:
    match = next(
        (d for d in differences if d["feature"] == feature),
        None
    )

    print(
        f"{feature:25s} | "
        f"different rows: {match['different_rows'] if match else 0}"
    )

print("\n=== VALIDATION ===")

results = []

def check(number, description, condition):
    passed = bool(condition)
    results.append(passed)
    print(f"{'PASS' if passed else 'FAIL'} | {number:02d} {description}")

check(1, "Canonical feature engineering preserves 2,624 rows",
      len(canonical_2026) == 2624)

check(2, "All 2,624 team-game keys align",
      len(paired) == 2624 and paired["_merge"].eq("both").all())

check(3, "All 126 model features available",
      len(common_features) == 126)

check(4, "No canonical feature differences",
      len(differences) == 0)

check(5, "Production configuration remains v6",
      config.MODEL_VERSION == "v6")

print("\n" + "=" * 60)
print(f"BLOCK 75Y: {sum(results)}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 75Z — PAIRED V6/V7 EXPANDING-WINDOW BACKTEST
# NO PRODUCTION WRITES
# ============================================================

print("\n=== BLOCK 75Z: PAIRED EXPANDING-WINDOW BACKTEST ===\n")

from sklearn.base import clone
from sklearn.metrics import brier_score_loss, roc_auc_score

data = backtest_features.copy()
data["season_id"] = data["season_id"].astype(int)

seasons = sorted(data["season_id"].unique())
holdout_seasons = seasons[1:]

backtest_results = []
backtest_predictions = []

for season in holdout_seasons:
    train_mask = data["season_id"] < season
    test_mask = data["season_id"] == season

    y_train = data.loc[train_mask, "made_playoffs"].astype(int)
    y_test = data.loc[test_mask, "made_playoffs"].astype(int)

    for version, bundle in [("v6", v6), ("v7", v7)]:
        feature_cols = bundle["features"]

        X_train = (
            data.loc[train_mask, feature_cols]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0)
        )

        X_test = (
            data.loc[test_mask, feature_cols]
            .apply(pd.to_numeric, errors="coerce")
            .fillna(0)
        )

        xgb = clone(bundle["xgb_pipeline"])
        rf = clone(bundle["rf_pipeline"])

        xgb.fit(X_train, y_train)
        rf.fit(X_train, y_train)

        p_xgb = xgb.predict_proba(X_test)[:, 1]
        p_rf = rf.predict_proba(X_test)[:, 1]
        p_ensemble = (p_xgb + p_rf) / 2

        backtest_results.append({
            "season": season,
            "version": version,
            "train_rows": len(X_train),
            "test_rows": len(X_test),
            "brier": brier_score_loss(y_test, p_ensemble),
            "auc": roc_auc_score(y_test, p_ensemble)
        })

        predictions = data.loc[
            test_mask,
            ["season_id", "game_date", "team_id", "game_id", "made_playoffs"]
        ].copy()

        predictions["version"] = version
        predictions["probability"] = p_ensemble
        backtest_predictions.append(predictions)

        print(
            f"{season} | {version} | "
            f"Brier={backtest_results[-1]['brier']:.6f} | "
            f"AUC={backtest_results[-1]['auc']:.6f}"
        )

backtest_results = pd.DataFrame(backtest_results)
backtest_predictions = pd.concat(backtest_predictions, ignore_index=True)

print("\n=== SEASONAL BRIER COMPARISON ===")

seasonal = backtest_results.pivot(
    index="season", columns="version", values="brier"
)

seasonal["v7_minus_v6"] = seasonal["v7"] - seasonal["v6"]
print(seasonal.round(6).to_string())

print("\n=== VALIDATION ===")

checks = {
    "24 model-season evaluations": len(backtest_results) == 24,
    "Both versions evaluated in every season":
        backtest_results.groupby("season")["version"].nunique().eq(2).all(),
    "All probabilities within [0,1]":
        backtest_predictions["probability"].between(0, 1).all(),
    "Expected holdout prediction rows":
        len(backtest_predictions) == 2 * (len(data) - 2460),
    "Production remains v6": config.MODEL_VERSION == "v6"
}

for i, (name, passed) in enumerate(checks.items(), 1):
    print(f"{'PASS' if passed else 'FAIL'} | {i:02d} {name}")

print("\n" + "=" * 60)
print(f"BLOCK 75Z: {sum(checks.values())}/{len(checks)} PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 76A — AGGREGATE PERFORMANCE AND CALIBRATION
# READ ONLY — NO MODEL TRAINING OR DATABASE WRITES
# ============================================================

print("\n=== BLOCK 76A: AGGREGATE PERFORMANCE ===\n")

p = backtest_predictions.copy()
p["made_playoffs"] = p["made_playoffs"].astype(int)

# Add game-progress information for season-stage analysis.
p = p.merge(
    backtest_features[
        ["season_id", "team_id", "game_id", "games_played"]
    ],
    on=["season_id", "team_id", "game_id"],
    how="left",
    validate="many_to_one"
)

p["stage"] = pd.cut(
    p["games_played"],
    bins=[0, 25, 55, 82],
    labels=["early", "mid", "late"],
    include_lowest=True
)

def metrics(frame):
    y = frame["made_playoffs"]
    prob = frame["probability"]
    return pd.Series({
        "rows": len(frame),
        "brier": brier_score_loss(y, prob),
        "auc": roc_auc_score(y, prob),
        "mean_predicted": prob.mean(),
        "actual_rate": y.mean(),
        "calibration_bias": prob.mean() - y.mean()
    })

overall = p.groupby("version").apply(
    metrics, include_groups=False
)

print("=== OVERALL OUT-OF-SAMPLE PERFORMANCE ===")
print(overall.round(6).to_string())

print("\n=== PERFORMANCE BY SEASON STAGE ===")

stage_metrics = p.groupby(
    ["stage", "version"], observed=True
).apply(metrics, include_groups=False)

print(stage_metrics.round(6).to_string())

print("\n=== CALIBRATION BY PROBABILITY BIN ===")

p["probability_bin"] = pd.cut(
    p["probability"],
    bins=[0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1],
    include_lowest=True
)

calibration = p.groupby(
    ["version", "probability_bin"], observed=True
).agg(
    rows=("made_playoffs", "size"),
    mean_probability=("probability", "mean"),
    actual_rate=("made_playoffs", "mean")
)

calibration["error"] = (
    calibration["mean_probability"] - calibration["actual_rate"]
).abs()

print(calibration.round(5).to_string())

print("\n=== VALIDATION ===")

checks = {
    "Both versions have equal prediction counts":
        p.groupby("version").size().nunique() == 1,
    "All game-progress values available":
        p["games_played"].notna().all(),
    "All observations assigned a season stage":
        p["stage"].notna().all(),
    "Both versions have overall metrics":
        len(overall) == 2,
    "Production remains v6":
        config.MODEL_VERSION == "v6"
}

for i, (name, passed) in enumerate(checks.items(), 1):
    print(f"{'PASS' if passed else 'FAIL'} | {i:02d} {name}")

print("\n" + "=" * 60)
print(f"BLOCK 76A: {sum(checks.values())}/5 PASS")
print("=" * 60)


In [ ]:

# ============================================================
# BLOCK 76B — PAIRED ACCURACY AND VOLATILITY
# READ ONLY — NO TRAINING OR DATABASE WRITES
# ============================================================

print("\n=== BLOCK 76B: PAIRED ACCURACY AND VOLATILITY ===\n")

keys = ["season_id", "game_date", "team_id", "game_id"]

paired = p.pivot(
    index=keys,
    columns="version",
    values="probability"
).reset_index()

outcomes = p.drop_duplicates(keys)[keys + ["made_playoffs", "games_played"]]
paired = paired.merge(outcomes, on=keys, validate="one_to_one")

paired["error_v6"] = (paired["v6"] - paired["made_playoffs"]) ** 2
paired["error_v7"] = (paired["v7"] - paired["made_playoffs"]) ** 2
paired["brier_difference"] = paired["error_v7"] - paired["error_v6"]
paired["probability_difference_pp"] = (paired["v7"] - paired["v6"]) * 100

print("=== PAIRED BRIER PERFORMANCE ===")
print("Observations:", len(paired))
print("V6 Brier:", round(paired["error_v6"].mean(), 6))
print("V7 Brier:", round(paired["error_v7"].mean(), 6))
print("Difference:", round(paired["brier_difference"].mean(), 6))

season_diff = paired.groupby("season_id")["brier_difference"].mean()
print("\nSeasons favoring v7:", int((season_diff < 0).sum()))
print("Seasons favoring v6:", int((season_diff > 0).sum()))

# Season-level bootstrap: preserves observations within seasons.
rng = np.random.default_rng(42)
season_stats = paired.groupby("season_id").agg(
    total_difference=("brier_difference", "sum"),
    rows=("brier_difference", "size")
)

boot = []
for _ in range(5000):
    sample = season_stats.iloc[
        rng.integers(0, len(season_stats), len(season_stats))
    ]
    boot.append(sample["total_difference"].sum() / sample["rows"].sum())

ci = np.percentile(boot, [2.5, 97.5])
print("\n=== SEASON-LEVEL BOOTSTRAP ===")
print(f"95% interval for V7 - V6 Brier: [{ci[0]:.6f}, {ci[1]:.6f}]")
print("Interval excludes zero:", bool(ci[0] > 0 or ci[1] < 0))

# Compare consecutive team-game predictions, not calendar-day snapshots.
ordered = paired.sort_values(["season_id", "team_id", "games_played"]).copy()

for version in ["v6", "v7"]:
    ordered[f"change_{version}_pp"] = (
        ordered.groupby(["season_id", "team_id"])[version]
        .diff().abs() * 100
    )

movement = ordered.dropna(subset=["change_v6_pp", "change_v7_pp"])

print("\n=== OUT-OF-SAMPLE GAME-TO-GAME MOVEMENT ===")

for version in ["v6", "v7"]:
    values = movement[f"change_{version}_pp"]
    print(
        f"{version} | mean={values.mean():.3f}pp | "
        f"median={values.median():.3f}pp | "
        f"p95={values.quantile(.95):.3f}pp | "
        f">=10pp={(values >= 10).sum():,}"
    )

increase = (
    movement["change_v7_pp"].mean() /
    movement["change_v6_pp"].mean() - 1
) * 100

print(f"\nV7 relative movement change: {increase:+.2f}%")

print("\n=== VALIDATION ===")

checks = {
    "29,484 paired predictions": len(paired) == 29484,
    "12 holdout seasons": paired["season_id"].nunique() == 12,
    "All probabilities available": paired[["v6", "v7"]].notna().all().all(),
    "Bootstrap completed": len(boot) == 5000,
    "Game-to-game movement available": len(movement) > 0,
    "Production remains v6": config.MODEL_VERSION == "v6"
}

for i, (name, passed) in enumerate(checks.items(), 1):
    print(f"{'PASS' if passed else 'FAIL'} | {i:02d} {name}")

print("\n" + "=" * 60)
print(f"BLOCK 76B: {sum(checks.values())}/6 PASS")
print("=" * 60)


In [ ]:
import requests
import numpy as np
import pandas as pd
import time
import random
import warnings

# ============================================================
# SAFE REQUEST WITH BACKOFF
# ============================================================
def safe_request(url, params=None, max_retries=5):
    delay = 1.0
    for _ in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=10)
            if r.status_code == 200:
                return r
            if r.status_code == 429:
                time.sleep(delay)
                delay = min(delay * 2, 5)
                continue
            return None
        except Exception:
            time.sleep(delay)
            delay = min(delay * 2, 5)
    return None


In [ ]:

# ============================================================
# BLOCK 77B — DIAGNOSE THREE FAILED PBP GAMES
# READ ONLY
# ============================================================

failed_ids = [2026020186, 2026020187, 2026020188]

games = con.execute("""
    SELECT game_id, link, gameState
    FROM all_game_summary
    WHERE game_id IN (2026020186, 2026020187, 2026020188)
""").df()

for game_id in failed_ids:
    match = games[games["game_id"] == game_id]

    print(f"\n=== GAME {game_id} ===")

    if match.empty:
        print("FAIL | Game missing from all_game_summary")
        continue

    row = match.iloc[0]
    print("Game state:", row["gameState"])
    print("URL:", row["link"])

    try:
        response = requests.get(str(row["link"]).strip(), timeout=20)

        print("HTTP status:", response.status_code)
        print("Content type:", response.headers.get("Content-Type"))
        print("Retry-After:", response.headers.get("Retry-After"))

        if response.ok:
            data = response.json()
            print("JSON keys:", list(data.keys())[:15])
            print("Plays count:", len(data.get("plays", [])))
        else:
            print("Response preview:", response.text[:250])

    except requests.RequestException as e:
        print("Request error:", repr(e))

    time.sleep(2)

print("\n=== DIAGNOSTIC COMPLETE ===")


In [ ]:

# ============================================================
# BLOCK 77C — TARGETED PBP BACKFILL
# 2026-27 | THREE FAILED GAMES
# ============================================================

failed_ids = [2026020186, 2026020187, 2026020188]
season_id = 20262027

summary_df = load_game_summary_df(con, season_id)
target_games = summary_df[summary_df["game_id"].isin(failed_ids)].copy()

print(f"Target games found: {len(target_games)}/3")

if len(target_games) != 3:
    print("STOP | Not all target games found in game summary.")
else:
    # Fetch only these games
    pbp_updates = play_by_play_events(target_games, season_id)

    if pbp_updates.empty:
        print("FAIL | No PBP data retrieved.")
    else:
        fetched_ids = set(pbp_updates["game_id"].unique())
        print("Successfully fetched:", sorted(fetched_ids))
        print("Still missing:", sorted(set(failed_ids) - fetched_ids))

        pbp_updates = normalize_play_by_play(pbp_updates)
        pbp_updates = enforce_pbp_schema(pbp_updates)

        # Only write if all three games were successfully fetched.
        if fetched_ids == set(failed_ids):
            con.register("pbp_updates", pbp_updates)

            try:
                con.execute("BEGIN TRANSACTION")

                con.execute("""
                    DELETE FROM all_play_by_play
                    WHERE season_id = ?
                      AND game_id IN (SELECT DISTINCT game_id FROM pbp_updates)
                """, [season_id])

                con.execute("""
                    INSERT INTO all_play_by_play
                    SELECT * FROM pbp_updates
                """)

                con.execute("COMMIT")
                print(f"SUCCESS | Inserted {len(pbp_updates):,} PBP events.")
            except Exception:
                con.execute("ROLLBACK")
                raise
            finally:
                con.unregister("pbp_updates")
        else:
            print("STOP | No database changes; not all games succeeded.")


In [ ]:
season_id = 20262027

prob = con.execute("""
    SELECT game_date AS snapshot_date, team_id, model_version
    FROM playoff_daily_predictions
    WHERE season_id = ?
""", [season_id]).df()

shap = con.execute("""
    SELECT snapshot_date, team_id, model_version
    FROM playoff_shap_values_v2
    WHERE season_id = ?
""", [season_id]).df()

keys = ["snapshot_date", "team_id", "model_version"]

prob["snapshot_date"] = pd.to_datetime(prob["snapshot_date"]).dt.date
shap["snapshot_date"] = pd.to_datetime(shap["snapshot_date"]).dt.date

prob_keys = set(map(tuple, prob[keys].to_numpy()))
shap_keys = set(map(tuple, shap[keys].to_numpy()))

print("=== BLOCK 72B: TEAM-KEY ALIGNMENT ===")
print(f"Probability rows: {len(prob):,}")
print(f"SHAP rows: {len(shap):,}")
print(f"Probability duplicates: {prob.duplicated(keys).sum()}")
print(f"SHAP duplicates: {shap.duplicated(keys).sum()}")
print(f"Missing from SHAP: {len(prob_keys - shap_keys)}")
print(f"Extra in SHAP: {len(shap_keys - prob_keys)}")
print(f"Exact alignment: {prob_keys == shap_keys}")

In [ ]:
season_id = 20262027

# Inspect available probability columns
print(con.execute("""
    DESCRIBE playoff_daily_predictions
""").df()[["column_name", "column_type"]].to_string(index=False))

# Validate SHAP reconstruction values
shap_checks = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE xgb_probability IS NULL
               OR NOT isfinite(xgb_probability)
               OR xgb_probability < 0
               OR xgb_probability > 1
        ) AS invalid_xgb_probabilities,
        COUNT(*) FILTER (
            WHERE shap_base_value IS NULL
               OR NOT isfinite(shap_base_value)
               OR xgb_raw_margin IS NULL
               OR NOT isfinite(xgb_raw_margin)
        ) AS invalid_shap_metrics,
        MIN(snapshot_date) AS first_snapshot,
        MAX(snapshot_date) AS latest_snapshot,
        COUNT(DISTINCT snapshot_date) AS snapshot_dates,
        COUNT(DISTINCT team_id) AS unique_teams,
        COUNT(DISTINCT model_version) AS model_versions
    FROM playoff_shap_values_v2
    WHERE season_id = ?
""", [season_id]).df()

print("\n=== BLOCK 72C: SHAP INTEGRITY ===")
print(shap_checks.to_string(index=False))

In [ ]:
season_id = 20262027

comparison = con.execute("""
    SELECT
        p.game_date,
        p.team_id,
        p.tri_code,
        p.model_version,
        p.playoff_prob_xgb,
        s.xgb_probability AS shap_xgb_probability,
        ABS(p.playoff_prob_xgb - s.xgb_probability) AS probability_diff
    FROM playoff_daily_predictions p
    INNER JOIN playoff_shap_values_v2 s
        ON p.season_id = s.season_id
        AND p.game_date = s.snapshot_date
        AND p.team_id = s.team_id
        AND p.model_version = s.model_version
    WHERE p.season_id = ?
""", [season_id]).df()

tolerance = 1e-4

print("=== BLOCK 72D: PROBABILITY-SHAP CONSISTENCY ===")
print(f"Compared rows: {len(comparison):,}")
print(f"Null XGB predictions: {comparison['playoff_prob_xgb'].isna().sum()}")
print(f"Null SHAP probabilities: {comparison['shap_xgb_probability'].isna().sum()}")
print(f"Maximum difference: {comparison['probability_diff'].max():.10f}")
print(f"Mean difference: {comparison['probability_diff'].mean():.10f}")
print(f"Rows exceeding tolerance: {(comparison['probability_diff'] > tolerance).sum()}")
print(f"Exact model version: {comparison['model_version'].unique().tolist()}")

print("\nLargest differences:")
print(
    comparison.nlargest(10, "probability_diff")[
        ["game_date", "tri_code", "playoff_prob_xgb",
         "shap_xgb_probability", "probability_diff"]
    ].to_string(index=False)
)

In [ ]:
season_id = 20262027

diagnostics = con.execute("""
    WITH same_day_games AS (
        SELECT
            game_date,
            team_id,
            COUNT(*) AS game_records,
            COUNT(DISTINCT game_id) AS distinct_games
        FROM team_daily_stats
        WHERE season_id = ?
        GROUP BY game_date, team_id
        HAVING COUNT(*) > 1
    )
    SELECT *
    FROM same_day_games
    ORDER BY game_date, team_id
""", [season_id]).df()

print("=== BLOCK 72E: SOURCE SNAPSHOT DIAGNOSTICS ===")
print(f"Duplicate team-date combinations: {len(diagnostics)}")
print(f"Teams affected: {diagnostics['team_id'].nunique()}")
print(f"Dates affected: {diagnostics['game_date'].nunique()}")

if not diagnostics.empty:
    print(diagnostics.head(20).to_string(index=False))

In [ ]:
season_id = 20262027

source_check = con.execute("""
    WITH expected AS (
        SELECT
            p.game_date AS snapshot_date,
            p.team_id,
            MAX(t.game_date) AS expected_source_date
        FROM playoff_daily_predictions p
        JOIN team_daily_stats t
            ON p.season_id = t.season_id
            AND p.team_id = t.team_id
            AND t.game_date <= p.game_date
        WHERE p.season_id = ?
        GROUP BY p.game_date, p.team_id
    )
    SELECT
        p.game_date AS snapshot_date,
        p.tri_code,
        p.team_id,
        s.game_date AS shap_source_date,
        e.expected_source_date,
        ABS(p.playoff_prob_xgb - s.xgb_probability) AS probability_diff
    FROM playoff_daily_predictions p
    JOIN playoff_shap_values_v2 s
        ON p.season_id = s.season_id
        AND p.game_date = s.snapshot_date
        AND p.team_id = s.team_id
        AND p.model_version = s.model_version
    JOIN expected e
        ON p.game_date = e.snapshot_date
        AND p.team_id = e.team_id
    WHERE p.season_id = ?
""", [season_id, season_id]).df()

source_mismatches = source_check[
    source_check["shap_source_date"] != source_check["expected_source_date"]
]

print("=== BLOCK 72F: HISTORICAL SOURCE-DATE ALIGNMENT ===")
print(f"Rows checked: {len(source_check):,}")
print(f"Source-date mismatches: {len(source_mismatches)}")
print(f"Probability mismatches: {(source_check['probability_diff'] > 1e-4).sum()}")

if not source_mismatches.empty:
    print(source_mismatches.head(20).to_string(index=False))

In [ ]:
# ============================================================
# BLOCK 72G: SNAPSHOT PROBABILITY ALIGNMENT TEST
# ============================================================

import pandas as pd
import numpy as np

from prediction_model.explainability.daily_feature_predictions import (
    _load_asof_snapshot,
    _build_shap_dataframe,
    MODEL_VERSION_USED
)

season_id = 20262027
snapshot_date = pd.Timestamp("2026-11-02").date()

# Load historical team snapshot
raw = _load_asof_snapshot(con, season_id, snapshot_date)

# Apply the same date alignment as the updated script
raw["source_game_date"] = raw["game_date"]
raw["game_date"] = pd.Timestamp(snapshot_date)

# Recalculate without writing to DuckDB
shap_test = _build_shap_dataframe(
    raw,
    season_id=season_id,
    snapshot_date=snapshot_date
)

# Existing production predictions
pred = con.execute("""
    SELECT team_id, tri_code, playoff_prob_xgb
    FROM playoff_daily_predictions
    WHERE season_id = ?
      AND game_date = ?
      AND model_version = ?
""", [season_id, snapshot_date, MODEL_VERSION_USED]).df()

# Compare
comparison = pred.merge(
    shap_test[["team_id", "xgb_probability"]],
    on="team_id",
    how="outer",
    validate="one_to_one",
    indicator=True
)

comparison["difference"] = (
    comparison["playoff_prob_xgb"] - comparison["xgb_probability"]
).abs()

tolerance = 1e-4
mismatches = comparison[
    comparison["difference"].isna() |
    (comparison["difference"] > tolerance)
]

print("=== BLOCK 72G: SNAPSHOT PROBABILITY ALIGNMENT ===")
print(f"Production rows: {len(pred)}")
print(f"SHAP rows: {len(shap_test)}")
print(f"Unmatched teams: {(comparison['_merge'] != 'both').sum()}")
print(f"Maximum difference: {comparison['difference'].max():.10f}")
print(f"Mean difference: {comparison['difference'].mean():.10f}")
print(f"Rows exceeding tolerance: {len(mismatches)}")
print(f"RESULT: {'PASS' if mismatches.empty else 'FAIL'}")

if not mismatches.empty:
    display(
        mismatches.sort_values(
            "difference", ascending=False, na_position="first"
        ).head(15)
    )

In [ ]:
# ============================================================
# BLOCK 72H: FULL HISTORICAL PROBABILITY ALIGNMENT
# ============================================================

import pandas as pd
import numpy as np

from prediction_model.explainability.daily_feature_predictions import (
    _load_asof_snapshot, _build_shap_dataframe, MODEL_VERSION_USED
)

season_id = 20262027
tolerance = 1e-4

pred = con.execute("""
    SELECT game_date AS snapshot_date, team_id, playoff_prob_xgb
    FROM playoff_daily_predictions
    WHERE season_id = ? AND model_version = ?
""", [season_id, MODEL_VERSION_USED]).df()

pred["snapshot_date"] = pd.to_datetime(pred["snapshot_date"]).dt.date
dates = sorted(pred["snapshot_date"].unique())

results = []

for date in dates:
    raw = _load_asof_snapshot(con, season_id, date)
    raw["source_game_date"] = raw["game_date"]
    raw["game_date"] = pd.Timestamp(date)

    shap_test = _build_shap_dataframe(
        raw, season_id=season_id, snapshot_date=date
    )

    expected = pred.loc[
        pred["snapshot_date"] == date,
        ["team_id", "playoff_prob_xgb"]
    ]

    comparison = expected.merge(
        shap_test[["team_id", "xgb_probability"]],
        on="team_id",
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    comparison["difference"] = (
        comparison["playoff_prob_xgb"] - comparison["xgb_probability"]
    ).abs()

    unmatched = (comparison["_merge"] != "both").sum()
    mismatches = (
        comparison["difference"].isna() |
        (comparison["difference"] > tolerance)
    ).sum()

    results.append({
        "snapshot_date": date,
        "production_rows": len(expected),
        "shap_rows": len(shap_test),
        "unmatched": int(unmatched),
        "mismatches": int(mismatches),
        "max_difference": comparison["difference"].max()
    })

results_df = pd.DataFrame(results)

print("=== BLOCK 72H: FULL HISTORICAL ALIGNMENT ===")
print(f"Dates checked: {len(results_df)}")
print(f"Production rows: {results_df['production_rows'].sum()}")
print(f"SHAP rows: {results_df['shap_rows'].sum()}")
print(f"Unmatched teams: {results_df['unmatched'].sum()}")
print(f"Probability mismatches: {results_df['mismatches'].sum()}")
print(f"Maximum difference: {results_df['max_difference'].max():.10f}")

passed = (
    results_df["unmatched"].sum() == 0
    and results_df["mismatches"].sum() == 0
)

print(f"\nRESULT: {'PASS' if passed else 'FAIL'}")

if not passed:
    display(results_df[
        (results_df["unmatched"] > 0) |
        (results_df["mismatches"] > 0)
    ])

In [ ]:
# ============================================================
# BLOCK 72J: STORED PROBABILITY + SHAP INTEGRITY
# ============================================================

import pandas as pd
import numpy as np

season_id = 20262027
model_version = "v6"
tolerance = 1e-4

# ------------------------------------------------------------
# 1. Load stored production predictions
# ------------------------------------------------------------
pred = con.execute("""
    SELECT
        season_id,
        CAST(game_date AS DATE) AS snapshot_date,
        team_id,
        model_version,
        playoff_prob_xgb
    FROM playoff_daily_predictions
    WHERE season_id = ?
      AND model_version = ?
""", [season_id, model_version]).df()

# ------------------------------------------------------------
# 2. Load stored SHAP predictions
# ------------------------------------------------------------
shap = con.execute("""
    SELECT
        season_id,
        CAST(snapshot_date AS DATE) AS snapshot_date,
        team_id,
        model_version,
        xgb_probability
    FROM playoff_shap_values_v2
    WHERE season_id = ?
      AND model_version = ?
""", [season_id, model_version]).df()

keys = ["season_id", "snapshot_date", "team_id", "model_version"]

# ------------------------------------------------------------
# 3. Validate unique keys
# ------------------------------------------------------------
pred_duplicates = pred.duplicated(keys).sum()
shap_duplicates = shap.duplicated(keys).sum()

# ------------------------------------------------------------
# 4. Compare stored team-date records
# ------------------------------------------------------------
comparison = pred.merge(
    shap,
    on=keys,
    how="outer",
    validate="one_to_one" if not (pred_duplicates or shap_duplicates) else "many_to_many",
    indicator=True
)

comparison["difference"] = (
    comparison["playoff_prob_xgb"] -
    comparison["xgb_probability"]
).abs()

missing_shap = (comparison["_merge"] == "left_only").sum()
extra_shap = (comparison["_merge"] == "right_only").sum()

probability_mismatches = (
    (comparison["_merge"] == "both") &
    (
        comparison["difference"].isna() |
        (comparison["difference"] > tolerance)
    )
).sum()

# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------
checks = {
    "01 Production keys unique": pred_duplicates == 0,
    "02 SHAP keys unique": shap_duplicates == 0,
    "03 Production has 1,073 rows": len(pred) == 1073,
    "04 SHAP has 1,073 rows": len(shap) == 1073,
    "05 No production keys missing from SHAP": missing_shap == 0,
    "06 No extra SHAP keys": extra_shap == 0,
    "07 Stored probabilities match": probability_mismatches == 0,
    "08 Maximum difference within tolerance": (
        comparison["difference"].notna().all()
        and comparison["difference"].le(tolerance).all()
    ),
    "09 Active model remains v6": (
        pred["model_version"].eq("v6").all()
        and shap["model_version"].eq("v6").all()
        and not pred.empty
        and not shap.empty
    )
}

print("=== BLOCK 72J: STORED PROBABILITY + SHAP INTEGRITY ===\n")

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

print("\n=== ALIGNMENT SUMMARY ===")
print(f"Production rows: {len(pred)}")
print(f"SHAP rows: {len(shap)}")
print(f"Production duplicates: {pred_duplicates}")
print(f"SHAP duplicates: {shap_duplicates}")
print(f"Missing from SHAP: {missing_shap}")
print(f"Extra in SHAP: {extra_shap}")
print(f"Probability mismatches: {probability_mismatches}")
print(f"Maximum difference: {comparison['difference'].max():.10f}")
print(f"Mean difference: {comparison['difference'].mean():.10f}")

passed_count = sum(checks.values())

print("\n" + "=" * 60)
print(f"BLOCK 72J: {passed_count}/{len(checks)} PASS")
print("=" * 60)

if not all(checks.values()):
    display(
        comparison[
            (comparison["_merge"] != "both") |
            (comparison["difference"].isna()) |
            (comparison["difference"] > tolerance)
        ].head(20)
    )

In [ ]:
# ============================================================
# BLOCK 73A: SHAP DATA INTEGRITY
# ============================================================

import pandas as pd

season_id = 20262027
model_version = "v6"

# ------------------------------------------------------------
# 1. Inspect SHAP table structure
# ------------------------------------------------------------
schema = con.execute("""
    DESCRIBE playoff_shap_values_v2
""").df()

columns = schema["column_name"].tolist()

# ------------------------------------------------------------
# 2. Inspect stored records
# ------------------------------------------------------------
shap = con.execute("""
    SELECT *
    FROM playoff_shap_values_v2
    WHERE season_id = ?
      AND model_version = ?
""", [season_id, model_version]).df()

# ------------------------------------------------------------
# 3. Identify explainability columns
# ------------------------------------------------------------
shap_columns = [
    col for col in columns
    if "shap" in col.lower()
]

metadata_columns = [
    "season_id", "snapshot_date", "team_id",
    "model_version", "xgb_probability"
]

feature_columns = [
    col for col in columns
    if col not in metadata_columns
]

# ------------------------------------------------------------
# 4. Validate data completeness
# ------------------------------------------------------------
null_counts = shap.isna().sum()
columns_with_nulls = null_counts[null_counts > 0]

duplicate_keys = shap.duplicated(
    ["season_id", "snapshot_date", "team_id", "model_version"]
).sum()

print("=== BLOCK 73A: SHAP DATA INTEGRITY ===\n")

print(f"Total rows: {len(shap):,}")
print(f"Total columns: {len(columns)}")
print(f"SHAP-named columns: {len(shap_columns)}")
print(f"Other non-key columns: {len(feature_columns)}")
print(f"Duplicate keys: {duplicate_keys}")
print(f"Columns containing nulls: {len(columns_with_nulls)}")

print("\n=== SHAP COLUMNS ===")
print(shap_columns)

print("\n=== NULL COUNTS ===")
if columns_with_nulls.empty:
    print("No null values detected")
else:
    display(columns_with_nulls.to_frame("null_count"))

print("\n=== TABLE SCHEMA ===")
display(schema)

In [ ]:
# ============================================================
# BLOCK 73B: SHAP MATHEMATICAL RECONSTRUCTION
# ============================================================

import numpy as np
import pandas as pd
from scipy.special import expit

season_id = 20262027
model_version = "v6"
tolerance = 1e-5

# ------------------------------------------------------------
# 1. Load stored SHAP records
# ------------------------------------------------------------
shap = con.execute("""
    SELECT *
    FROM playoff_shap_values_v2
    WHERE season_id = ?
      AND model_version = ?
""", [season_id, model_version]).df()

# ------------------------------------------------------------
# 2. Identify feature contribution columns
# ------------------------------------------------------------
feature_shap_cols = [
    col for col in shap.columns
    if col.startswith("shap_") and col != "shap_base_value"
]

# ------------------------------------------------------------
# 3. Reconstruct probabilities
# ------------------------------------------------------------
contributions = shap[feature_shap_cols].to_numpy(dtype=float)
base_values = shap["shap_base_value"].to_numpy(dtype=float)
stored_probabilities = shap["xgb_probability"].to_numpy(dtype=float)

reconstructed_log_odds = base_values + contributions.sum(axis=1)
reconstructed_probabilities = expit(reconstructed_log_odds)

differences = np.abs(
    reconstructed_probabilities - stored_probabilities
)

# ------------------------------------------------------------
# 4. Validate
# ------------------------------------------------------------
finite_values = (
    np.isfinite(contributions).all()
    and np.isfinite(base_values).all()
    and np.isfinite(stored_probabilities).all()
    and np.isfinite(reconstructed_probabilities).all()
)

checks = {
    "01 SHAP records present": len(shap) == 1073,
    "02 Feature contributions present": len(feature_shap_cols) > 0,
    "03 All SHAP values finite": finite_values,
    "04 Stored probabilities within [0, 1]": (
        ((stored_probabilities >= 0) & (stored_probabilities <= 1)).all()
    ),
    "05 Reconstructed probabilities match": (
        finite_values and (differences <= tolerance).all()
    ),
    "06 No reconstruction mismatches": (
        np.count_nonzero(differences > tolerance) == 0
        and np.isfinite(differences).all()
    )
}

print("=== BLOCK 73B: SHAP MATHEMATICAL RECONSTRUCTION ===\n")

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

print("\n=== RECONSTRUCTION SUMMARY ===")
print(f"Records checked: {len(shap):,}")
print(f"SHAP feature columns: {len(feature_shap_cols)}")
print(f"Maximum difference: {differences.max():.12f}")
print(f"Mean difference: {differences.mean():.12f}")
print(f"Rows exceeding tolerance: {(differences > tolerance).sum()}")

print("\n" + "=" * 60)
print(f"BLOCK 73B: {sum(checks.values())}/{len(checks)} PASS")
print("=" * 60)

if not all(checks.values()):
    diagnostics = shap[
        ["snapshot_date", "team_id", "xgb_probability"]
    ].copy()

    diagnostics["reconstructed_probability"] = reconstructed_probabilities
    diagnostics["difference"] = differences

    display(
        diagnostics.sort_values(
            "difference", ascending=False
        ).head(20)
    )

In [ ]:
# ============================================================
# BLOCK 73C: GLOBAL SHAP FEATURE IMPORTANCE
# ============================================================

import numpy as np
import pandas as pd

season_id = 20262027
model_version = "v6"

shap = con.execute("""
    SELECT *
    FROM playoff_shap_values_v2
    WHERE season_id = ?
      AND model_version = ?
""", [season_id, model_version]).df()

feature_cols = [
    col for col in shap.columns
    if col.startswith("shap_") and col != "shap_base_value"
]

# Global feature importance
importance = pd.DataFrame({
    "feature": [col.removeprefix("shap_") for col in feature_cols],
    "mean_abs_shap": shap[feature_cols].abs().mean().to_numpy(),
    "mean_signed_shap": shap[feature_cols].mean().to_numpy(),
    "std_shap": shap[feature_cols].std().to_numpy(),
    "pct_positive": (
        shap[feature_cols].gt(0).mean().to_numpy() * 100
    )
})

importance = (
    importance.sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

importance.insert(0, "rank", np.arange(1, len(importance) + 1))

# Calculate daily importance rankings
daily_importance = (
    shap.groupby("snapshot_date")[feature_cols]
    .agg(lambda x: x.abs().mean())
)

daily_ranks = daily_importance.rank(
    axis=1, ascending=False, method="min"
)

importance["median_daily_rank"] = [
    daily_ranks[col].median() for col in feature_cols
]

importance["pct_days_top_10"] = [
    daily_ranks[col].le(10).mean() * 100 for col in feature_cols
]

# Re-sort to ensure ranks and rows stay aligned
importance = importance.sort_values(
    "mean_abs_shap", ascending=False
).reset_index(drop=True)

print("=== BLOCK 73C: GLOBAL SHAP FEATURE IMPORTANCE ===\n")
print(f"Records analyzed: {len(shap):,}")
print(f"Historical snapshots: {shap['snapshot_date'].nunique()}")
print(f"Features analyzed: {len(feature_cols)}")

print("\n=== TOP 25 FEATURES ===")
display(importance.head(25).round(4))

print("\n=== BOTTOM 15 FEATURES ===")
display(importance.tail(15).round(4))

In [ ]:
# ============================================================
# BLOCK 73D: CORRECTED SHAP FEATURE IMPORTANCE + STABILITY
# ============================================================

import numpy as np
import pandas as pd

season_id = 20262027
model_version = "v6"

shap = con.execute("""
    SELECT *
    FROM playoff_shap_values_v2
    WHERE season_id = ?
      AND model_version = ?
""", [season_id, model_version]).df()

feature_cols = [
    col for col in shap.columns
    if col.startswith("shap_") and col != "shap_base_value"
]

# ------------------------------------------------------------
# 1. Global importance
# ------------------------------------------------------------
importance = pd.DataFrame({
    "feature_col": feature_cols,
    "mean_abs_shap": shap[feature_cols].abs().mean().to_numpy(),
    "mean_signed_shap": shap[feature_cols].mean().to_numpy(),
    "std_shap": shap[feature_cols].std().to_numpy(),
    "pct_positive": shap[feature_cols].gt(0).mean().to_numpy() * 100
})

# ------------------------------------------------------------
# 2. Daily importance and rankings
# ------------------------------------------------------------
daily_importance = (
    shap.groupby("snapshot_date")[feature_cols]
    .agg(lambda x: x.abs().mean())
)

daily_ranks = daily_importance.rank(
    axis=1, ascending=False, method="min"
)

daily_metrics = pd.DataFrame({
    "feature_col": feature_cols,
    "median_daily_rank": daily_ranks.median().reindex(feature_cols).to_numpy(),
    "pct_days_top_10": (
        daily_ranks.le(10).mean().reindex(feature_cols).to_numpy() * 100
    )
})

# ------------------------------------------------------------
# 3. Join metrics using feature names
# ------------------------------------------------------------
importance = importance.merge(
    daily_metrics,
    on="feature_col",
    how="left",
    validate="one_to_one"
)

importance["feature"] = importance["feature_col"].str.removeprefix("shap_")

importance = (
    importance.sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

importance.insert(0, "rank", np.arange(1, len(importance) + 1))

importance = importance.drop(columns="feature_col")

# ------------------------------------------------------------
# 4. Validate alignment
# ------------------------------------------------------------
checks = {
    "01 All 121 features present": len(importance) == 121,
    "02 Feature names unique": importance["feature"].is_unique,
    "03 No missing daily metrics": importance[
        ["median_daily_rank", "pct_days_top_10"]
    ].notna().all().all(),
    "04 Global importance correctly sorted": importance[
        "mean_abs_shap"
    ].is_monotonic_decreasing
}

print("=== BLOCK 73D: CORRECTED SHAP FEATURE STABILITY ===\n")

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

print(f"\nRecords: {len(shap):,}")
print(f"Snapshot dates: {shap['snapshot_date'].nunique()}")
print(f"Features: {len(feature_cols)}")

print("\n=== TOP 25 FEATURES ===")
display(importance.head(25).round(4))

print("\n=== BOTTOM 15 FEATURES ===")
display(importance.tail(15).round(4))

print("\n" + "=" * 60)
print(f"BLOCK 73D: {sum(checks.values())}/{len(checks)} PASS")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 73E: FEATURE DOMINANCE ACROSS TEAMS
# ============================================================

import pandas as pd
import numpy as np

season_id = 20262027
model_version = "v6"

shap = con.execute("""
    SELECT *
    FROM playoff_shap_values_v2
    WHERE season_id = ?
      AND model_version = ?
""", [season_id, model_version]).df()

features = [
    "sc_net",
    "division_rank",
    "points_pct",
    "league_rank",
    "goal_diff"
]

cols = [f"shap_{feature}" for feature in features]

# ------------------------------------------------------------
# 1. Calculate team-level importance
# ------------------------------------------------------------
team_importance = (
    shap.groupby("team_id")[cols]
    .agg(lambda x: x.abs().mean())
    .reset_index()
)

team_importance = team_importance.rename(
    columns={f"shap_{f}": f for f in features}
)

team_importance["dominant_feature"] = (
    team_importance[features].idxmax(axis=1)
)

# ------------------------------------------------------------
# 2. Compare league-wide dominance
# ------------------------------------------------------------
dominance_counts = (
    team_importance["dominant_feature"]
    .value_counts()
    .rename_axis("feature")
    .reset_index(name="teams")
)

# ------------------------------------------------------------
# 3. Compare sc_net to standings features
# ------------------------------------------------------------
team_importance["sc_vs_division"] = (
    team_importance["sc_net"] /
    team_importance["division_rank"].replace(0, np.nan)
)

team_importance["sc_vs_points_pct"] = (
    team_importance["sc_net"] /
    team_importance["points_pct"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------
print("=== BLOCK 73E: FEATURE DOMINANCE ACROSS TEAMS ===\n")

print(f"Teams analyzed: {len(team_importance)}")
print(f"Historical snapshots: {shap['snapshot_date'].nunique()}")

print("\n=== DOMINANT FEATURE COUNTS ===")
display(dominance_counts)

print("\n=== TOP 15 TEAMS BY SC_NET IMPORTANCE ===")
display(
    team_importance.sort_values(
        "sc_net", ascending=False
    ).head(15).round(4)
)

print("\n=== BOTTOM 10 TEAMS BY SC_NET IMPORTANCE ===")
display(
    team_importance.sort_values(
        "sc_net", ascending=True
    ).head(10).round(4)
)

print("\n=== LEAGUE-WIDE COMPARISON ===")
display(
    team_importance[features].describe().round(4)
)

In [ ]:
# ============================================================
# BLOCK 73F: SC_NET DIRECTION + CONCENTRATION
# ============================================================

import pandas as pd
import numpy as np

season_id = 20262027
model_version = "v6"

# ------------------------------------------------------------
# 1. Load stored SHAP contributions
# ------------------------------------------------------------
shap = con.execute("""
    SELECT snapshot_date, team_id, shap_sc_net,
           shap_division_rank, shap_points_pct
    FROM playoff_shap_values_v2
    WHERE season_id = ? AND model_version = ?
""", [season_id, model_version]).df()

# ------------------------------------------------------------
# 2. Inspect sc_net availability in source data
# ------------------------------------------------------------
source_cols = con.execute("""
    DESCRIBE team_daily_stats
""").df()["column_name"].tolist()

print("=== BLOCK 73F: SC_NET DIRECTION + CONCENTRATION ===\n")
print(f"Source sc_net column available: {'sc_net' in source_cols}")

# ------------------------------------------------------------
# 3. Team-level concentration
# ------------------------------------------------------------
team = (
    shap.groupby("team_id")["shap_sc_net"]
    .agg(
        mean_abs_shap=lambda x: x.abs().mean(),
        mean_signed_shap="mean",
        min_shap="min",
        max_shap="max",
        observations="count"
    )
    .reset_index()
    .sort_values("mean_abs_shap", ascending=False)
)

team["share_of_total_abs"] = (
    team["mean_abs_shap"] * team["observations"] /
    (team["mean_abs_shap"] * team["observations"]).sum()
)

print("\n=== SC_NET CONCENTRATION ===")
print(f"Top 3 teams' share: {team['share_of_total_abs'].head(3).sum():.2%}")
print(f"Top 5 teams' share: {team['share_of_total_abs'].head(5).sum():.2%}")
print(f"Top 10 teams' share: {team['share_of_total_abs'].head(10).sum():.2%}")

print("\n=== TOP 10 TEAMS ===")
display(team.head(10).round(4))

# ------------------------------------------------------------
# 4. Inspect contribution direction
# ------------------------------------------------------------
print("\n=== SC_NET CONTRIBUTION DIRECTION ===")
print(f"Positive contributions: {(shap['shap_sc_net'] > 0).mean():.2%}")
print(f"Negative contributions: {(shap['shap_sc_net'] < 0).mean():.2%}")
print(f"Mean signed contribution: {shap['shap_sc_net'].mean():.6f}")
print(f"Mean absolute contribution: {shap['shap_sc_net'].abs().mean():.6f}")

# ------------------------------------------------------------
# 5. Source feature alignment (if available)
# ------------------------------------------------------------
if "sc_net" in source_cols:
    source = con.execute("""
        SELECT
            s.snapshot_date,
            s.team_id,
            t.sc_net
        FROM (
            SELECT DISTINCT snapshot_date, team_id
            FROM playoff_shap_values_v2
            WHERE season_id = ? AND model_version = ?
        ) s
        ASOF LEFT JOIN (
            SELECT team_id, CAST(game_date AS DATE) AS game_date, sc_net
            FROM team_daily_stats
            WHERE season_id = ?
        ) t
        ON s.team_id = t.team_id
           AND s.snapshot_date >= t.game_date
    """, [season_id, model_version, season_id]).df()

    aligned = shap.merge(
        source,
        on=["snapshot_date", "team_id"],
        how="left",
        validate="one_to_one"
    )

    valid = aligned.dropna(subset=["sc_net", "shap_sc_net"]).copy()

    print("\n=== SOURCE FEATURE VS SHAP CONTRIBUTION ===")
    print(f"Matched observations: {len(valid)} / {len(shap)}")

    if len(valid) > 1 and valid["sc_net"].nunique() > 1:
        print(f"Pearson correlation: {valid['sc_net'].corr(valid['shap_sc_net']):.4f}")
        print(f"Spearman correlation: {valid['sc_net'].corr(valid['shap_sc_net'], method='spearman'):.4f}")

        valid["sc_net_quintile"] = pd.qcut(
            valid["sc_net"].rank(method="first"),
            q=5,
            labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"]
        )

        display(
            valid.groupby("sc_net_quintile", observed=True)
            .agg(
                observations=("team_id", "size"),
                mean_sc_net=("sc_net", "mean"),
                mean_shap=("shap_sc_net", "mean"),
                mean_abs_shap=("shap_sc_net", lambda x: x.abs().mean())
            ).round(4)
        )

In [ ]:
# ============================================================
# BLOCK 73G: INSPECT SC_NET FEATURE ENGINEERING
# ============================================================

import inspect
import feature_engineering

source = inspect.getsource(feature_engineering)

lines = source.splitlines()

matches = [
    i for i, line in enumerate(lines)
    if "sc_net" in line.lower()
]

print("=== BLOCK 73G: SC_NET FEATURE DEFINITION ===\n")

print(f"References found: {len(matches)}")

for i in matches:
    start = max(0, i - 6)
    end = min(len(lines), i + 7)

    print(f"\n--- Reference at line {i + 1} ---")

    for j in range(start, end):
        print(f"{j + 1:4d} | {lines[j]}")

In [ ]:
# ============================================================
# BLOCK 73H: SC_NET FEATURE VALUE VS SHAP CONTRIBUTION
# ============================================================

import pandas as pd
import numpy as np

from prediction_model.explainability.daily_feature_predictions import (
    _load_asof_snapshot
)

season_id = 20262027
model_version = "v6"

# ------------------------------------------------------------
# 1. Load stored SHAP contributions
# ------------------------------------------------------------
shap = con.execute("""
    SELECT snapshot_date, team_id, shap_sc_net
    FROM playoff_shap_values_v2
    WHERE season_id = ? AND model_version = ?
""", [season_id, model_version]).df()

shap["snapshot_date"] = pd.to_datetime(shap["snapshot_date"]).dt.date

# ------------------------------------------------------------
# 2. Reconstruct scoring-chance differential
# ------------------------------------------------------------
frames = []

for date in sorted(shap["snapshot_date"].unique()):
    raw = _load_asof_snapshot(con, season_id, date)

    raw["snapshot_date"] = date
    raw["sc_net"] = raw["SCF_pg"] - raw["SCA_pg"]

    frames.append(raw[["snapshot_date", "team_id", "sc_net"]])

features = pd.concat(frames, ignore_index=True)

# ------------------------------------------------------------
# 3. Align source features with stored SHAP
# ------------------------------------------------------------
aligned = shap.merge(
    features,
    on=["snapshot_date", "team_id"],
    how="left",
    validate="one_to_one"
)

valid = aligned.dropna(subset=["sc_net", "shap_sc_net"]).copy()

# ------------------------------------------------------------
# 4. Correlation and direction
# ------------------------------------------------------------
pearson = valid["sc_net"].corr(valid["shap_sc_net"])
spearman = valid["sc_net"].corr(
    valid["shap_sc_net"], method="spearman"
)

valid["sc_net_quintile"] = pd.qcut(
    valid["sc_net"].rank(method="first"),
    q=5,
    labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"]
)

quintiles = (
    valid.groupby("sc_net_quintile", observed=True)
    .agg(
        observations=("team_id", "size"),
        mean_sc_net=("sc_net", "mean"),
        mean_shap=("shap_sc_net", "mean"),
        mean_abs_shap=("shap_sc_net", lambda x: x.abs().mean()),
        pct_positive_shap=("shap_sc_net", lambda x: (x > 0).mean() * 100)
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------
print("=== BLOCK 73H: SC_NET DIRECTION VALIDATION ===\n")

print(f"Stored SHAP rows: {len(shap)}")
print(f"Matched feature rows: {len(valid)}")
print(f"Missing feature rows: {len(shap) - len(valid)}")

print(f"\nPearson correlation: {pearson:.4f}")
print(f"Spearman correlation: {spearman:.4f}")

print("\n=== SC_NET QUINTILES ===")
display(quintiles.round(4))

print("\n=== FEATURE DISTRIBUTION ===")
display(
    valid[["sc_net", "shap_sc_net"]].describe().round(4)
)

print("\n=== EXTREME SC_NET OBSERVATIONS ===")
display(
    valid.sort_values("sc_net")[
        ["snapshot_date", "team_id", "sc_net", "shap_sc_net"]
    ].iloc[np.r_[0:5, -5:0]].round(4)
)

In [ ]:
# ============================================================
# BLOCK 73I: EARLY-SEASON SC_NET STABILITY
# ============================================================

import pandas as pd
import numpy as np

from prediction_model.explainability.daily_feature_predictions import _load_asof_snapshot

season_id = 20262027
model_version = "v6"

shap = con.execute("""
    SELECT snapshot_date, team_id, shap_sc_net,
           shap_points_pct, shap_division_rank
    FROM playoff_shap_values_v2
    WHERE season_id = ? AND model_version = ?
""", [season_id, model_version]).df()

shap["snapshot_date"] = pd.to_datetime(shap["snapshot_date"]).dt.date

frames = []

for date in sorted(shap["snapshot_date"].unique()):
    raw = _load_asof_snapshot(con, season_id, date)

    raw["snapshot_date"] = date
    raw["sc_net"] = raw["SCF_pg"] - raw["SCA_pg"]

    frames.append(raw[["snapshot_date", "team_id", "sc_net"]])

features = pd.concat(frames, ignore_index=True)

aligned = shap.merge(
    features,
    on=["snapshot_date", "team_id"],
    how="left",
    validate="one_to_one"
)

daily = (
    aligned.groupby("snapshot_date")
    .agg(
        teams=("team_id", "nunique"),
        sc_net_std=("sc_net", "std"),
        sc_net_range=("sc_net", lambda x: x.max() - x.min()),
        sc_net_mean_abs=("sc_net", lambda x: x.abs().mean()),
        sc_shap_mean_abs=("shap_sc_net", lambda x: x.abs().mean()),
        points_shap_mean_abs=("shap_points_pct", lambda x: x.abs().mean()),
        division_shap_mean_abs=("shap_division_rank", lambda x: x.abs().mean())
    )
    .reset_index()
    .sort_values("snapshot_date")
)

daily["snapshot_number"] = np.arange(1, len(daily) + 1)

daily["sc_vs_points"] = (
    daily["sc_shap_mean_abs"] /
    daily["points_shap_mean_abs"].replace(0, np.nan)
)

daily["sc_vs_division"] = (
    daily["sc_shap_mean_abs"] /
    daily["division_shap_mean_abs"].replace(0, np.nan)
)

print("=== BLOCK 73I: EARLY-SEASON SC_NET STABILITY ===\n")

print(f"Snapshots analyzed: {len(daily)}")
print(f"Average SC_NET SHAP importance: {daily['sc_shap_mean_abs'].mean():.4f}")

print("\n=== FIRST 10 SNAPSHOTS ===")
display(daily.head(10).round(4))

print("\n=== LAST 10 SNAPSHOTS ===")
display(daily.tail(10).round(4))

print("\n=== EARLY VS LATE COMPARISON ===")

midpoint = len(daily) // 2

early = daily.iloc[:midpoint]
late = daily.iloc[midpoint:]

comparison = pd.DataFrame({
    "metric": [
        "SC_NET standard deviation",
        "SC_NET mean absolute value",
        "SC_NET SHAP importance",
        "Points % SHAP importance",
        "Division rank SHAP importance",
        "SC_NET / Points % importance"
    ],
    "early": [
        early["sc_net_std"].mean(),
        early["sc_net_mean_abs"].mean(),
        early["sc_shap_mean_abs"].mean(),
        early["points_shap_mean_abs"].mean(),
        early["division_shap_mean_abs"].mean(),
        early["sc_vs_points"].mean()
    ],
    "late": [
        late["sc_net_std"].mean(),
        late["sc_net_mean_abs"].mean(),
        late["sc_shap_mean_abs"].mean(),
        late["points_shap_mean_abs"].mean(),
        late["division_shap_mean_abs"].mean(),
        late["sc_vs_points"].mean()
    ]
})

comparison["pct_change"] = (
    (comparison["late"] - comparison["early"]) /
    comparison["early"].replace(0, np.nan) * 100
)

display(comparison.round(4))

In [ ]:
# ============================================================
# BLOCK 73J: PLAYOFF PROBABILITY DISTRIBUTION + EXTREMES
# ============================================================

import pandas as pd
import numpy as np

season_id = 20262027
model_version = "v6"

pred = con.execute("""
    SELECT
        game_date AS snapshot_date,
        team_id,
        playoff_prob_xgb
    FROM playoff_daily_predictions
    WHERE season_id = ? AND model_version = ?
""", [season_id, model_version]).df()

pred["snapshot_date"] = pd.to_datetime(pred["snapshot_date"]).dt.date
pred = pred.sort_values(["snapshot_date", "team_id"])

# ------------------------------------------------------------
# 1. Probability distribution by snapshot
# ------------------------------------------------------------
daily = (
    pred.groupby("snapshot_date")
    .agg(
        teams=("team_id", "nunique"),
        mean_prob=("playoff_prob_xgb", "mean"),
        median_prob=("playoff_prob_xgb", "median"),
        std_prob=("playoff_prob_xgb", "std"),
        min_prob=("playoff_prob_xgb", "min"),
        max_prob=("playoff_prob_xgb", "max"),
        pct_below_10=("playoff_prob_xgb", lambda x: (x < 0.10).mean() * 100),
        pct_above_90=("playoff_prob_xgb", lambda x: (x > 0.90).mean() * 100)
    )
    .reset_index()
)

# ------------------------------------------------------------
# 2. Latest snapshot
# ------------------------------------------------------------
latest_date = pred["snapshot_date"].max()
latest = pred[pred["snapshot_date"] == latest_date].copy()

latest["probability_band"] = pd.cut(
    latest["playoff_prob_xgb"],
    bins=[-0.001, 0.10, 0.25, 0.40, 0.60, 0.75, 0.90, 1.001],
    labels=[
        "0-10%", "10-25%", "25-40%", "40-60%",
        "60-75%", "75-90%", "90-100%"
    ]
)

bands = (
    latest["probability_band"]
    .value_counts(sort=False)
    .rename_axis("probability_band")
    .reset_index(name="teams")
)

# ------------------------------------------------------------
# 3. Sanity checks
# ------------------------------------------------------------
checks = {
    "01 No duplicate team-day keys": not pred.duplicated(["snapshot_date", "team_id"]).any(),
    "02 All probabilities between 0 and 1": pred["playoff_prob_xgb"].between(0, 1).all(),
    "03 No null probabilities": pred["playoff_prob_xgb"].notna().all(),
    "04 Latest snapshot contains 32 teams": len(latest) == 32,
    "05 Latest snapshot probabilities sum to a finite value": np.isfinite(latest["playoff_prob_xgb"].sum())
}

print("=== BLOCK 73J: PROBABILITY DISTRIBUTION ===\n")

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

print(f"\nSnapshots: {len(daily)}")
print(f"Latest snapshot: {latest_date}")
print(f"Latest expected playoff teams (sum of probabilities): {latest['playoff_prob_xgb'].sum():.4f}")

print("\n=== FIRST 5 SNAPSHOTS ===")
display(daily.head(5).round(4))

print("\n=== LAST 10 SNAPSHOTS ===")
display(daily.tail(10).round(4))

print("\n=== LATEST PROBABILITY BANDS ===")
display(bands)

print("\n=== LATEST 32 TEAM PROBABILITIES ===")
display(
    latest[["team_id", "playoff_prob_xgb"]]
    .sort_values("playoff_prob_xgb", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

print("\n" + "=" * 60)
print(f"BLOCK 73J: {sum(checks.values())}/{len(checks)} PASS")
print("=" * 60)

In [ ]:
# ============================================================
# BLOCK 73K: HISTORICAL CALIBRATION DATA INVENTORY
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Historical prediction coverage
# ------------------------------------------------------------
coverage = con.execute("""
    SELECT
        season_id,
        model_version,
        COUNT(*) AS prediction_rows,
        COUNT(DISTINCT team_id) AS teams,
        COUNT(DISTINCT game_date) AS snapshot_dates,
        MIN(game_date) AS first_snapshot,
        MAX(game_date) AS last_snapshot,
        MIN(playoff_prob_xgb) AS min_probability,
        MAX(playoff_prob_xgb) AS max_probability
    FROM playoff_daily_predictions
    GROUP BY season_id, model_version
    ORDER BY season_id, model_version
""").df()

# ------------------------------------------------------------
# 2. Discover potentially relevant tables
# ------------------------------------------------------------
tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

keywords = [
    "playoff", "season", "standing",
    "result", "outcome", "label", "target"
]

candidate_tables = tables[
    tables["table_name"].str.lower().apply(
        lambda name: any(word in name for word in keywords)
    )
].copy()

# ------------------------------------------------------------
# 3. Inspect potentially relevant columns
# ------------------------------------------------------------
columns = con.execute("""
    SELECT
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'main'
    ORDER BY table_name, ordinal_position
""").df()

outcome_keywords = [
    "playoff", "made_", "qualified",
    "target", "label", "outcome", "postseason"
]

candidate_columns = columns[
    columns["column_name"].str.lower().apply(
        lambda name: any(word in name for word in outcome_keywords)
    )
].copy()

# ------------------------------------------------------------
# 4. Report
# ------------------------------------------------------------
print("=== BLOCK 73K: HISTORICAL CALIBRATION INVENTORY ===\n")

print("=== PREDICTION COVERAGE BY SEASON ===")
display(coverage)

print("\n=== POTENTIAL OUTCOME TABLES ===")
display(candidate_tables)

print("\n=== POTENTIAL OUTCOME COLUMNS ===")
display(candidate_columns)

print("\n=== INVENTORY SUMMARY ===")
print(f"Seasons with predictions: {coverage['season_id'].nunique()}")
print(f"Model versions found: {coverage['model_version'].nunique()}")
print(f"Potential outcome tables: {len(candidate_tables)}")
print(f"Potential outcome columns: {len(candidate_columns)}")

In [ ]:
# ============================================================
# BLOCK 73L: VERIFY HISTORICAL PLAYOFF OUTCOME LABELS
# ============================================================

import pandas as pd

# 1. Inspect outcome coverage by season
coverage = con.execute("""
    SELECT
        season_id,
        COUNT(*) AS rows,
        COUNT(DISTINCT team_id) AS teams,
        COUNT(made_playoffs) AS labeled_rows,
        COUNT(DISTINCT CASE
            WHEN made_playoffs IS NOT NULL THEN team_id
        END) AS labeled_teams,
        MIN(made_playoffs) AS min_label,
        MAX(made_playoffs) AS max_label
    FROM team_daily_stats
    GROUP BY season_id
    ORDER BY season_id
""").df()

print("=== BLOCK 73L: HISTORICAL OUTCOME VALIDATION ===\n")

print("=== OUTCOME COVERAGE ===")
display(coverage)

# 2. Check whether labels are consistent within each season/team
consistency = con.execute("""
    SELECT
        season_id,
        team_id,
        COUNT(DISTINCT made_playoffs) AS distinct_labels,
        MIN(made_playoffs) AS min_label,
        MAX(made_playoffs) AS max_label
    FROM team_daily_stats
    WHERE made_playoffs IS NOT NULL
    GROUP BY season_id, team_id
    ORDER BY season_id, team_id
""").df()

inconsistent = consistency[
    consistency["distinct_labels"] > 1
]

print("\n=== LABEL CONSISTENCY ===")
print(f"Teams with inconsistent labels: {len(inconsistent)}")
display(inconsistent)

# 3. Count playoff teams by season
outcomes = con.execute("""
    SELECT
        season_id,
        COUNT(*) AS teams,
        SUM(made_playoffs) AS playoff_teams
    FROM (
        SELECT
            season_id,
            team_id,
            MAX(made_playoffs) AS made_playoffs
        FROM team_daily_stats
        WHERE made_playoffs IS NOT NULL
        GROUP BY season_id, team_id
    )
    GROUP BY season_id
    ORDER BY season_id
""").df()

print("\n=== PLAYOFF TEAM COUNTS ===")
display(outcomes)

print("\n=== VALIDATION SUMMARY ===")
print(f"Seasons inspected: {len(coverage)}")
print(f"Inconsistent team labels: {len(inconsistent)}")

In [ ]:
# ============================================================
# BLOCK 73M: HISTORICAL PROBABILITY CALIBRATION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

# ------------------------------------------------------------
# 1. Load historical predictions
# ------------------------------------------------------------
pred = con.execute("""
    SELECT
        season_id,
        model_version,
        game_date AS snapshot_date,
        team_id,
        playoff_prob_xgb AS probability
    FROM playoff_daily_predictions
    WHERE season_id = 20252026
      AND model_version IN ('v5', 'wc_field_pressure_v4')
""").df()

# ------------------------------------------------------------
# 2. Load final playoff outcomes
# ------------------------------------------------------------
outcomes = con.execute("""
    SELECT
        season_id,
        team_id,
        MAX(made_playoffs) AS made_playoffs
    FROM team_daily_stats
    WHERE season_id = 20252026
      AND made_playoffs IS NOT NULL
    GROUP BY season_id, team_id
""").df()

df = pred.merge(
    outcomes,
    on=["season_id", "team_id"],
    how="left",
    validate="many_to_one"
)

df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
df["probability"] = pd.to_numeric(df["probability"], errors="coerce")
df["made_playoffs"] = pd.to_numeric(df["made_playoffs"], errors="coerce")

# ------------------------------------------------------------
# 3. Evaluate each model and time period
# ------------------------------------------------------------
def evaluate_predictions(frame):
    valid = frame.dropna(subset=["probability", "made_playoffs"])

    if valid.empty:
        return None

    y = valid["made_playoffs"].astype(int)
    p = valid["probability"].clip(1e-12, 1 - 1e-12)

    return {
        "observations": len(valid),
        "teams": valid["team_id"].nunique(),
        "brier_score": brier_score_loss(y, p),
        "log_loss": log_loss(y, p, labels=[0, 1]),
        "roc_auc": roc_auc_score(y, p) if y.nunique() == 2 else np.nan,
        "mean_probability": p.mean(),
        "actual_playoff_rate": y.mean(),
        "pct_extreme": ((p < 0.10) | (p > 0.90)).mean() * 100
    }

results = []

for model, group in df.groupby("model_version"):
    periods = {
        "All available dates": group,
        "October": group[group["snapshot_date"].dt.month == 10],
        "November": group[group["snapshot_date"].dt.month == 11],
        "Oct-Nov combined": group[group["snapshot_date"].dt.month.isin([10, 11])]
    }

    for period, subset in periods.items():
        metrics = evaluate_predictions(subset)

        if metrics is not None:
            results.append({
                "model_version": model,
                "period": period,
                **metrics
            })

metrics_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 4. Historical probability calibration bands
# ------------------------------------------------------------
early = df[df["snapshot_date"].dt.month.isin([10, 11])].copy()

early["probability_band"] = pd.cut(
    early["probability"],
    bins=[0, 0.10, 0.25, 0.40, 0.60, 0.75, 0.90, 1],
    labels=[
        "0-10%", "10-25%", "25-40%", "40-60%",
        "60-75%", "75-90%", "90-100%"
    ],
    include_lowest=True
)

calibration = (
    early.groupby(["model_version", "probability_band"], observed=True)
    .agg(
        observations=("made_playoffs", "count"),
        teams=("team_id", "nunique"),
        mean_predicted=("probability", "mean"),
        actual_playoff_rate=("made_playoffs", "mean")
    )
    .reset_index()
)

calibration["calibration_gap"] = (
    calibration["actual_playoff_rate"] -
    calibration["mean_predicted"]
)

# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------
print("=== BLOCK 73M: HISTORICAL CALIBRATION ===\n")

print("=== MODEL ACCURACY BY PERIOD ===")
display(metrics_df.round(4))

print("\n=== OCT-NOV CALIBRATION BANDS ===")
display(calibration.round(4))

print("\n=== DATA INTEGRITY ===")
print(f"Prediction rows: {len(pred):,}")
print(f"Matched outcomes: {df['made_playoffs'].notna().sum():,}")
print(f"Missing outcomes: {df['made_playoffs'].isna().sum():,}")
print(f"Historical teams: {df['team_id'].nunique()}")

In [ ]:
# ============================================================
# BLOCK 73N: FINAL V6 MODEL VALIDATION SUMMARY
# ============================================================

import pandas as pd

audit = pd.DataFrame([
    ["72H", "Prediction/SHAP alignment", "PASS",
     "1,073 records; zero probability mismatches"],

    ["72J", "Persisted prediction consistency", "PASS",
     "9/9 checks passed"],

    ["73A", "SHAP data integrity", "PASS",
     "121 feature contributions; no missing values"],

    ["73B", "SHAP mathematical reconstruction", "PASS",
     "All 1,073 probabilities reconstructed"],

    ["73D", "Feature importance stability", "PASS",
     "121 features correctly ranked"],

    ["73E", "Team-level feature dominance", "PASS",
     "SC_NET leads among five selected features for 13 teams"],

    ["73H", "SC_NET directional behavior", "PASS",
     "Spearman correlation = 0.9278"],

    ["73I", "Early-season feature stability", "PASS",
     "SC_NET influence declines as standings influence rises"],

    ["73J", "Probability distribution", "PASS",
     "16.4752 expected playoff teams on November 2"],

    ["73L", "Historical outcome integrity", "PASS",
     "13 completed seasons; 16 playoff teams each"],

    ["73M", "Historical calibration", "WARNING",
     "v5/v4 show poor early-season calibration"],

    ["FINAL", "v6 historical calibration", "NOT TESTED",
     "Requires leakage-safe historical v6 predictions"]
], columns=["block", "validation", "status", "finding"])

print("=== BLOCK 73N: FINAL MODEL VALIDATION AUDIT ===\n")

display(audit)

print("\n=== AUDIT SUMMARY ===")
print(audit["status"].value_counts().to_string())

print("\n=== FINAL ASSESSMENT ===")
print("V6 prediction pipeline: VALIDATED")
print("V6 SHAP explainability: VALIDATED")
print("V6 feature behavior: PLAUSIBLE")
print("V6 historical predictive accuracy: NOT YET VALIDATED")
print("Production code changes required by this audit: NONE IDENTIFIED")

In [ ]:
# ============================================================
# BLOCK 74A: V6 TRAINING AND HOLDOUT INVENTORY
# ============================================================

from pathlib import Path
import re

# ------------------------------------------------------------
# 1. Find model training scripts
# ------------------------------------------------------------
root = Path.cwd()

keywords = [
    "train_test_split", "train_seasons", "test_seasons",
    "validation_seasons", "holdout", "TimeSeriesSplit",
    "GroupKFold", "xgb.train", "XGBClassifier",
    "model_version", "20252026", "20262027"
]

files = [
    p for p in root.rglob("*.py")
    if not any(x in p.parts for x in [".venv", "venv", "__pycache__", ".git"])
]

print("=== BLOCK 74A: V6 TRAINING INVENTORY ===\n")
print(f"Project root: {root}")
print(f"Python files scanned: {len(files)}")

# ------------------------------------------------------------
# 2. Identify relevant training configuration
# ------------------------------------------------------------
matches = []

for path in files:
    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except (UnicodeError, OSError):
        continue

    for line_no, line in enumerate(lines, start=1):
        if any(re.search(re.escape(k), line, re.I) for k in keywords):
            matches.append({
                "file": str(path.relative_to(root)),
                "line": line_no,
                "code": line.strip()[:180]
            })

# ------------------------------------------------------------
# 3. Report
# ------------------------------------------------------------
import pandas as pd

matches_df = pd.DataFrame(matches, columns=["file", "line", "code"])

print("\n=== FILES WITH TRAINING REFERENCES ===")
if matches_df.empty:
    print("No matching references found.")
else:
    summary = (
        matches_df.groupby("file")
        .size()
        .reset_index(name="matching_lines")
        .sort_values("matching_lines", ascending=False)
    )
    display(summary.head(30))

print("\n=== TRAINING / SPLIT REFERENCES ===")

split_terms = [
    "train_test_split", "train_seasons", "test_seasons",
    "validation_seasons", "holdout", "TimeSeriesSplit",
    "GroupKFold", "xgb.train", "XGBClassifier"
]

split_matches = matches_df[
    matches_df["code"].str.contains(
        "|".join(re.escape(x) for x in split_terms),
        case=False, regex=True, na=False
    )
]

display(split_matches.head(100))

print("\n=== V6 REFERENCES ===")
display(
    matches_df[
        matches_df["code"].str.contains(
            r"\bv6\b|20252026|20262027",
            case=False, regex=True, na=False
        )
    ].head(100)
)

In [ ]:
# ============================================================
# BLOCK 74B: VERIFY V6 TRAINING AND HOLDOUT LOGIC
# ============================================================

from pathlib import Path

path = Path("prediction_model/training/train_new_model.py")
lines = path.read_text(encoding="utf-8").splitlines()

sections = [
    (350, 470, "Training / test season selection"),
    (530, 590, "Holdout prediction generation"),
    (1360, 1460, "Final model training and saving")
]

print("=== BLOCK 74B: TRAINING SPLIT VERIFICATION ===")

for start, end, title in sections:
    print(f"\n{'=' * 65}")
    print(f"{title} — Lines {start}-{end}")
    print("=" * 65)

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")

In [ ]:
# ============================================================
# BLOCK 74C: INSPECT SAVED V6 HOLDOUT METRICS
# ============================================================

from pathlib import Path
from joblib import load
import pandas as pd

# Locate model bundles
models = [
    p for p in Path.cwd().rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".joblib", ".pkl", ".pickle"}
    and not any(x in p.parts for x in [".venv", "venv", "__pycache__"])
]

print("=== BLOCK 74C: SAVED MODEL INVENTORY ===\n")

print("=== MODEL FILES ===")
display(pd.DataFrame({
    "path": [str(p) for p in models],
    "size_mb": [round(p.stat().st_size / 1048576, 2) for p in models]
}))

print("\n=== V6 MODEL METADATA ===")

for path in models:
    try:
        bundle = load(path)

        if not isinstance(bundle, dict):
            continue

        if bundle.get("version") != "v6":
            continue

        print(f"\nModel: {path}")
        print(f"Version: {bundle.get('version')}")
        print(f"Trained on: {bundle.get('trained_on')}")
        print(f"Features: {len(bundle.get('features', []))}")

        print("\n=== HOLDOUT METRICS ===")
        for key, value in bundle.get("metrics", {}).items():
            print(f"{key}: {value}")

    except Exception as e:
        print(f"Could not inspect {path.name}: {e}")

In [ ]:
# ============================================================
# BLOCK 74D: HOLDOUT ISOLATION AND FEATURE SELECTION
# ============================================================

from pathlib import Path

path = Path("prediction_model/training/train_new_model.py")
lines = path.read_text(encoding="utf-8").splitlines()

print("=== BLOCK 74D: HOLDOUT ISOLATION ===\n")

# Inspect code between split definition and final feature selection
for start, end, title in [
    (469, 535, "Training/test split and feature selection")
]:
    print(f"\n=== {title} ===")

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")

# Locate other feature-selection and preprocessing references
keywords = [
    "FEATURE_COLS",
    "X_train",
    "X_test",
    "y_train",
    "y_test",
    "feature_importances_",
    "select_features",
    "fit_transform",
    "cross_val",
    "shap"
]

print("\n=== RELEVANT TRAINING REFERENCES ===")

for i, line in enumerate(lines[:535], start=1):
    if any(k.lower() in line.lower() for k in keywords):
        print(f"{i:4}: {line.strip()[:160]}")

In [ ]:
# ============================================================
# BLOCK 74E: HISTORICAL FEATURE LEAKAGE INSPECTION
# ============================================================

from pathlib import Path
import re

path = Path("prediction_model/training/train_new_model.py")
lines = path.read_text(encoding="utf-8").splitlines()

print("=== BLOCK 74E: HISTORICAL FEATURE CONSTRUCTION ===\n")

# 1. Find the canonical training feature function
matches = [
    i for i, line in enumerate(lines)
    if re.match(r"^def build_canonical_training_features\s*\(", line)
]

if not matches:
    print("Function definition not found.")
else:
    start = matches[0]
    end = next(
        (i for i in range(start + 1, len(lines))
         if re.match(r"^(def |class |if __name__)", lines[i])),
        len(lines)
    )

    print("=== CANONICAL FEATURE CONSTRUCTION ===")
    for i in range(start, end):
        print(f"{i + 1:4}: {lines[i]}")

# 2. Identify potential leakage-related operations
patterns = [
    r"\bshift\s*\(",
    r"\brolling\s*\(",
    r"\bexpanding\s*\(",
    r"\bmerge\s*\(",
    r"\bgroupby\s*\(",
    r"\btransform\s*\(",
    r"\bbfill\s*\(",
    r"\bffill\s*\(",
    r"\bmade_playoffs\b",
    r"\bTARGET_COL\b",
    r"\bgame_date\b",
    r"\bseason_id\b"
]

print("\n=== POTENTIAL LEAKAGE-RELATED REFERENCES ===")

for i, line in enumerate(lines[:360], start=1):
    if any(re.search(p, line, re.I) for p in patterns):
        print(f"{i:4}: {line.strip()[:160]}")

In [ ]:
# ============================================================
# BLOCK 74F: FEATURE ENGINEERING LEAKAGE INVENTORY
# ============================================================

import inspect
import re
import pandas as pd

# Use the exact build_features function imported by training
source_file = inspect.getsourcefile(build_features)
source = inspect.getsource(build_features)

print("=== BLOCK 74F: FEATURE ENGINEERING LEAKAGE INVENTORY ===\n")
print(f"Source file: {source_file}")

patterns = {
    "Target references": r"made_playoffs|TARGET_COL",
    "Future shifts": r"\.shift\s*\(\s*-\d+",
    "Backward filling": r"\.bfill\s*\(|method\s*=\s*['\"]bfill",
    "Rolling calculations": r"\.rolling\s*\(",
    "Cumulative calculations": r"\.expanding\s*\(|\.cumsum\s*\(",
    "Group operations": r"\.groupby\s*\(",
    "Sorting": r"\.sort_values\s*\(",
    "Merging": r"\.merge\s*\(|merge_asof\s*\(",
    "Full-season aggregates": r"\.transform\s*\(|\.mean\s*\(|\.max\s*\("
}

rows = []

for category, pattern in patterns.items():
    matches = [
        (i, line.strip())
        for i, line in enumerate(source.splitlines(), start=1)
        if re.search(pattern, line, re.I)
    ]

    rows.append({
        "category": category,
        "matches": len(matches),
        "examples": " | ".join(
            f"{line}: {code[:90]}"
            for line, code in matches[:5]
        )
    })

display(pd.DataFrame(rows))

print("\n=== BUILD_FEATURES SOURCE ===")
print(source)

In [ ]:
# ============================================================
# BLOCK 74G: FEATURE ENGINEERING DEPENDENCY AUDIT (FIXED)
# ============================================================

import inspect
import re
import pandas as pd

# Get the module where build_features was defined
module = inspect.getmodule(build_features)

function_names = [
    "normalize_made_playoffs",
    "add_conference_baseline",
    "add_streak_features",
    "add_rank_bucket",
    "add_wc_field_pressure_features",
    "add_wc_density_features",
    "add_special_teams_features",
    "add_season_context",
    "add_interaction_features",
    "add_rush_features",
    "add_rush_interactions",
    "add_corsi_features",
    "add_corsi_relative_features",
    "add_corsi_interactions",
    "add_shot_quality_features",
    "add_shot_relative_features",
    "add_shot_interactions",
    "add_standings_features",
    "add_standings_interactions"
]

patterns = {
    "Target usage": r"made_playoffs|TARGET_COL",
    "Future shift": r"\.shift\s*\(\s*-\d+",
    "Backward fill": r"\.bfill\s*\(|backfill",
    "Full-group aggregation": r"\.groupby\s*\(|\.transform\s*\(",
    "Rolling/expanding": r"\.rolling\s*\(|\.expanding\s*\(",
    "Merge operations": r"\.merge\s*\(|merge_asof",
    "Cross-date aggregation": r"\.pivot\s*\(|\.pivot_table\s*\(",
    "Date sorting": r"\.sort_values\s*\("
}

results = []
flagged_sources = {}

for name in function_names:
    func = getattr(module, name, None)

    if func is None:
        results.append({
            "function": name,
            "flags": "FUNCTION NOT FOUND",
            "lines": 0
        })
        continue

    try:
        source = inspect.getsource(func)
    except (OSError, TypeError):
        results.append({
            "function": name,
            "flags": "SOURCE UNAVAILABLE",
            "lines": 0
        })
        continue

    flags = [
        category
        for category, pattern in patterns.items()
        if re.search(pattern, source, re.I)
    ]

    results.append({
        "function": name,
        "flags": ", ".join(flags) if flags else "None",
        "lines": len(source.splitlines())
    })

    if flags:
        flagged_sources[name] = source

print("=== BLOCK 74G: FEATURE DEPENDENCY AUDIT ===\n")
print(f"Module: {module.__name__}")

display(pd.DataFrame(results))

print("\n=== FLAGGED FUNCTION SOURCES ===")

for name, source in flagged_sources.items():
    print(f"\n{'=' * 60}")
    print(name)
    print("=" * 60)
    print(source)

In [ ]:
# ============================================================
# BLOCK 75A: LOCATE HISTORICAL HOLDOUT PREDICTIONS
# ============================================================

from pathlib import Path

path = Path("prediction_model/training/train_new_model.py")
lines = path.read_text(encoding="utf-8").splitlines()

print("=== BLOCK 75A: HISTORICAL HOLDOUT PREDICTIONS ===\n")

keywords = [
    "holdout_eval",
    "xgb_prob",
    "rf_prob",
    "to_csv",
    "to_parquet",
    "to_sql",
    "COPY",
    "INSERT INTO",
    "brier_score_loss"
]

for i, line in enumerate(lines, start=1):
    if any(k.lower() in line.lower() for k in keywords):
        start = max(0, i - 3)
        end = min(len(lines), i + 4)

        print(f"\n--- Line {i} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")

In [ ]:
# ============================================================
# BLOCK 75B: FIND SAVED TRAINING OUTPUT
# ============================================================

from pathlib import Path

root = Path.cwd()

extensions = {".log", ".txt", ".csv", ".json", ".parquet"}
keywords = ["train", "holdout", "evaluation", "metrics", "brier", "validation"]

matches = []

for path in root.rglob("*"):
    if not path.is_file():
        continue

    if any(part in {".git", ".venv", "venv", "__pycache__"} for part in path.parts):
        continue

    if path.suffix.lower() not in extensions:
        continue

    if any(word in path.name.lower() for word in keywords):
        matches.append({
            "file": str(path.relative_to(root)),
            "size_kb": round(path.stat().st_size / 1024, 1),
            "modified": path.stat().st_mtime
        })

matches = sorted(matches, key=lambda x: x["modified"], reverse=True)

print("=== BLOCK 75B: SAVED TRAINING OUTPUT ===\n")

if matches:
    import pandas as pd

    results = pd.DataFrame(matches)
    results["modified"] = pd.to_datetime(results["modified"], unit="s")
    display(results.head(50))
else:
    print("No likely training logs or evaluation files found.")

In [ ]:
# ============================================================
# BLOCK 76A: EARLY-SEASON CALIBRATION & CONFIDENCE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import brier_score_loss

path = Path("evaluation/v7/holdout_predictions.parquet")

if not path.exists():
    matches = list(Path.cwd().rglob("evaluation/v7/holdout_predictions.parquet"))
    if not matches:
        raise FileNotFoundError("Cannot find holdout_predictions.parquet")
    path = matches[0]

df = pd.read_parquet(path)
early = df[df["games_played"].between(1, 20)].copy()

early["display_prob"] = 0.45 * early["playoff_prob"] + 0.275
early["game_window"] = pd.cut(
    early["games_played"],
    bins=[0, 5, 10, 15, 20],
    labels=["1-5", "6-10", "11-15", "16-20"]
)

early["confidence_band"] = pd.cut(
    early["playoff_prob"],
    bins=[0, .2, .4, .6, .8, 1.000001],
    labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"],
    include_lowest=True
)

print("=== BLOCK 76A: EARLY-SEASON CALIBRATION ===")
print(f"Rows: {len(early):,}")
print(f"Teams: {early['team_id'].nunique()}")
print(f"Season: {early['season_id'].unique().tolist()}")

results = []

for (window, band), group in early.groupby(
    ["game_window", "confidence_band"], observed=True
):
    actual = group["made_playoffs"].astype(int)
    raw = group["playoff_prob"]
    adjusted = group["display_prob"]

    results.append({
        "games": str(window),
        "confidence_band": str(band),
        "rows": len(group),
        "raw_mean": raw.mean(),
        "adjusted_mean": adjusted.mean(),
        "actual_rate": actual.mean(),
        "raw_brier": brier_score_loss(actual, raw),
        "adjusted_brier": brier_score_loss(actual, adjusted),
        "brier_change": brier_score_loss(actual, adjusted) - brier_score_loss(actual, raw)
    })

results = pd.DataFrame(results)

print("\n=== CALIBRATION BY GAME WINDOW & CONFIDENCE ===")
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== ADJUSTMENT IMPROVEMENT BY GAME WINDOW ===")
summary = results.groupby("games", sort=False).apply(
    lambda g: pd.Series({
        "rows": g["rows"].sum(),
        "raw_brier": np.average(g["raw_brier"], weights=g["rows"]),
        "adjusted_brier": np.average(g["adjusted_brier"], weights=g["rows"]),
        "brier_change": np.average(g["brier_change"], weights=g["rows"])
    }),
    include_groups=False
).reset_index()

print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== BLOCK 76A COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 76B: TEAM-LEVEL EARLY-SEASON VALIDATION
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_parquet(path)
early = df[df["games_played"].between(1, 20)].copy()

early["adjusted_prob"] = 0.45 * early["playoff_prob"] + 0.275

early["raw_error"] = (
    early["playoff_prob"] - early["made_playoffs"]
) ** 2

early["adjusted_error"] = (
    early["adjusted_prob"] - early["made_playoffs"]
) ** 2

team_results = (
    early.groupby("team_id", as_index=False)
    .agg(
        games=("games_played", "count"),
        made_playoffs=("made_playoffs", "max"),
        raw_brier=("raw_error", "mean"),
        adjusted_brier=("adjusted_error", "mean")
    )
)

team_results["brier_change"] = (
    team_results["adjusted_brier"] - team_results["raw_brier"]
)

team_results["adjustment_helps"] = team_results["brier_change"] < 0

print("=== BLOCK 76B: TEAM-LEVEL ADJUSTMENT ===\n")

print(team_results.sort_values("brier_change").to_string(
    index=False, float_format=lambda x: f"{x:.4f}"
))

print("\n=== SUMMARY ===")
print(f"Teams improved: {team_results['adjustment_helps'].sum()} / {len(team_results)}")
print(f"Teams worsened: {(team_results['brier_change'] > 0).sum()} / {len(team_results)}")
print(f"Average Brier change: {team_results['brier_change'].mean():.4f}")

print("\n=== BY PLAYOFF OUTCOME ===")
print(
    team_results.groupby("made_playoffs")
    .agg(
        teams=("team_id", "count"),
        improved=("adjustment_helps", "sum"),
        mean_change=("brier_change", "mean")
    ).to_string(float_format=lambda x: f"{x:.4f}")
)

print("\n=== BLOCK 76B COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 76C: EARLY-SEASON ADJUSTMENT STRATEGY COMPARISON
# READ-ONLY — NO MODEL OR DATABASE CHANGES
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_parquet(path)
early = df[df["games_played"].between(1, 20)].copy()

p = early["playoff_prob"].astype(float)
g = early["games_played"].astype(int)
y = early["made_playoffs"].astype(int)

# Adjustment formula:
# adjusted = weight * raw + (1 - weight) * 0.50

strategies = {
    "raw": p,
    "current_1_20": 0.45 * p + 0.275,
    "adjust_1_15": p.where(g > 15, 0.45 * p + 0.275),
    "taper_10_20": (
        p * np.where(g <= 10, 0.45, 0.45 + 0.55 * (g - 10) / 10)
        + 0.5 * (1 - np.where(g <= 10, 0.45, 0.45 + 0.55 * (g - 10) / 10))
    ),
    "weaker_1_20": 0.70 * p + 0.15
}

for name, values in strategies.items():
    early[name] = values

early["game_window"] = pd.cut(
    g,
    bins=[0, 5, 10, 15, 20],
    labels=["1-5", "6-10", "11-15", "16-20"]
)

# ------------------------------------------------------------
# Overall Brier score
# ------------------------------------------------------------

overall = []

for name in strategies:
    overall.append({
        "strategy": name,
        "brier": np.mean((early[name] - y) ** 2)
    })

overall = pd.DataFrame(overall)
raw_brier = overall.loc[overall["strategy"] == "raw", "brier"].iloc[0]
overall["vs_raw"] = overall["brier"] - raw_brier

print("=== BLOCK 76C: OVERALL BRIER ===")
print(overall.sort_values("brier").to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

# ------------------------------------------------------------
# Brier by game window
# ------------------------------------------------------------

window_results = []

for window, group in early.groupby("game_window", observed=True):
    for name in strategies:
        window_results.append({
            "games": str(window),
            "strategy": name,
            "brier": np.mean(
                (group[name] - group["made_playoffs"]) ** 2
            )
        })

window_results = pd.DataFrame(window_results)

print("\n=== BRIER BY GAME WINDOW ===")
print(window_results.pivot(
    index="games", columns="strategy", values="brier"
).to_string(float_format=lambda x: f"{x:.5f}"))

# ------------------------------------------------------------
# Team-level comparison against raw probabilities
# ------------------------------------------------------------

team_results = []

for team_id, group in early.groupby("team_id"):
    raw_error = np.mean(
        (group["raw"] - group["made_playoffs"]) ** 2
    )

    for name in strategies:
        strategy_error = np.mean(
            (group[name] - group["made_playoffs"]) ** 2
        )

        team_results.append({
            "team_id": team_id,
            "strategy": name,
            "brier_change": strategy_error - raw_error
        })

team_results = pd.DataFrame(team_results)

team_summary = (
    team_results.groupby("strategy")
    .agg(
        teams_improved=("brier_change", lambda x: (x < -1e-10).sum()),
        teams_worsened=("brier_change", lambda x: (x > 1e-10).sum()),
        mean_change=("brier_change", "mean"),
        median_change=("brier_change", "median")
    )
    .reset_index()
)

print("\n=== TEAM-LEVEL IMPACT VS RAW ===")
print(team_summary.sort_values("mean_change").to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

print("\n=== BLOCK 76C COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77A: HISTORICAL WALK-FORWARD ELIGIBILITY
# READ-ONLY — NO MODEL OR DATABASE CHANGES
# ============================================================

import duckdb
import pandas as pd
from config import DB_PATH, EXCLUDE_SEASON_ID

con = duckdb.connect(DB_PATH, read_only=True)

df = con.execute("""
    SELECT
        season_id,
        team_id,
        game_id,
        game_date,
        games_played,
        made_playoffs
    FROM team_daily_stats
""").df()

con.close()

# Exclude the active season
excluded = (
    [EXCLUDE_SEASON_ID]
    if isinstance(EXCLUDE_SEASON_ID, int)
    else list(EXCLUDE_SEASON_ID)
)

df = df[~df["season_id"].isin(excluded)].copy()

# Check historical season coverage
season_summary = (
    df.groupby("season_id")
    .agg(
        teams=("team_id", "nunique"),
        rows=("game_id", "size"),
        first_game=("game_date", "min"),
        last_game=("game_date", "max"),
        min_games=("games_played", "min"),
        max_games=("games_played", "max"),
        missing_targets=("made_playoffs", lambda x: x.isna().sum()),
        playoff_teams=("team_id", lambda x: 0)
    )
    .reset_index()
    .sort_values("season_id")
)

# Count teams that made playoffs
playoff_counts = (
    df.groupby(["season_id", "team_id"])["made_playoffs"]
    .max()
    .groupby("season_id")
    .sum()
)

season_summary["playoff_teams"] = (
    season_summary["season_id"].map(playoff_counts)
)

# Check for inconsistent team-season targets
target_consistency = (
    df.groupby(["season_id", "team_id"])["made_playoffs"]
    .nunique(dropna=False)
    .groupby("season_id")
    .max()
)

season_summary["targets_consistent"] = (
    season_summary["season_id"].map(target_consistency).eq(1)
)

# Count earlier seasons available for training
season_summary["prior_seasons"] = range(len(season_summary))

# Eligibility: at least 3 prior seasons, 16 playoff teams,
# complete 82-game season, no missing or inconsistent targets.
season_summary["eligible"] = (
    (season_summary["prior_seasons"] >= 3)
    & (season_summary["playoff_teams"] == 16)
    & (season_summary["max_games"] == 82)
    & (season_summary["missing_targets"] == 0)
    & season_summary["targets_consistent"]
)

print("=== BLOCK 77A: HISTORICAL SEASON COVERAGE ===\n")

print(season_summary.to_string(index=False))

print("\n=== ELIGIBLE WALK-FORWARD TEST SEASONS ===")

eligible = season_summary.loc[
    season_summary["eligible"], "season_id"
].tolist()

print(eligible)
print(f"\nEligible seasons: {len(eligible)}")

print("\n=== BLOCK 77A COMPLETE ===")


In [ ]:
# ============================================================
# BLOCK 77B: WALK-FORWARD TRAINING WINDOW VALIDATION
# READ-ONLY — NO TRAINING OR DATABASE MODIFICATIONS
# ============================================================

import pandas as pd

eligible_seasons = season_summary.loc[
    season_summary["eligible"], "season_id"
].tolist()

results = []

for test_season in eligible_seasons:
    train = df[df["season_id"] < test_season]
    test = df[df["season_id"] == test_season]

    train_keys = set(zip(train["season_id"], train["team_id"], train["game_id"]))
    test_keys = set(zip(test["season_id"], test["team_id"], test["game_id"]))

    results.append({
        "test_season": test_season,
        "train_seasons": train["season_id"].nunique(),
        "train_rows": len(train),
        "test_rows": len(test),
        "train_end": train["game_date"].max(),
        "test_start": test["game_date"].min(),
        "chronological": train["game_date"].max() < test["game_date"].min(),
        "key_overlap": len(train_keys & test_keys),
        "train_classes": train["made_playoffs"].nunique(),
        "test_classes": test["made_playoffs"].nunique()
    })

validation = pd.DataFrame(results)

validation["pass"] = (
    validation["chronological"]
    & validation["key_overlap"].eq(0)
    & validation["train_classes"].eq(2)
    & validation["test_classes"].eq(2)
)

print("=== BLOCK 77B: WALK-FORWARD TRAINING WINDOWS ===\n")
print(validation.to_string(index=False))

print("\n=== VALIDATION SUMMARY ===")
print(f"Passed: {validation['pass'].sum()} / {len(validation)}")
print(f"Failed: {(~validation['pass']).sum()}")

assert validation["pass"].all(), "Walk-forward validation failed"

print("\n=== BLOCK 77B COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77C: PREPARE WALK-FORWARD FEATURE DATASET
# READ-ONLY — NO MODEL TRAINING OR PRODUCTION CHANGES
# ============================================================

import duckdb
import pandas as pd
import numpy as np

from config import DB_PATH, EXCLUDE_SEASON_ID
from prediction_model.training.train_new_model import (
    build_canonical_training_features,
    _build_feature_matrix
)

# ------------------------------------------------------------
# Load completed historical seasons
# ------------------------------------------------------------

con = duckdb.connect(DB_PATH, read_only=True)

historical_raw = con.execute(
    "SELECT * FROM team_daily_stats"
).df()

con.close()

excluded = (
    [EXCLUDE_SEASON_ID]
    if isinstance(EXCLUDE_SEASON_ID, int)
    else list(EXCLUDE_SEASON_ID)
)

historical_raw = (
    historical_raw[
        ~historical_raw["season_id"].isin(excluded)
    ]
    .sort_values("game_id")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Build canonical features using the existing v7 function
# ------------------------------------------------------------

print("=== BLOCK 77C: CANONICAL HISTORICAL FEATURES ===")

historical_features = build_canonical_training_features(
    historical_raw
)

X_historical, historical_feature_cols = _build_feature_matrix(
    historical_features
)

# ------------------------------------------------------------
# Validate feature alignment and season coverage
# ------------------------------------------------------------

assert len(historical_features) == len(historical_raw)
assert len(X_historical) == len(historical_features)
assert X_historical.index.equals(historical_features.index)
assert not X_historical.columns.duplicated().any()
assert np.isfinite(X_historical.to_numpy(dtype=float)).all()

assert set(eligible_seasons).issubset(
    set(historical_features["season_id"].unique())
)

print("\n=== FEATURE DATASET SUMMARY ===")
print(f"Historical rows: {len(historical_features):,}")
print(f"Feature columns: {len(historical_feature_cols)}")
print(f"Seasons: {historical_features['season_id'].nunique()}")
print(f"Eligible test seasons: {len(eligible_seasons)}")
print(f"Missing feature values: {X_historical.isna().sum().sum():,}")

print("\n=== ROWS BY SEASON ===")
print(
    historical_features.groupby("season_id")
    .size()
    .to_string()
)

print("\n=== BLOCK 77C COMPLETE ===")


In [ ]:
# ============================================================
# BLOCK 77D: HISTORICAL WALK-FORWARD MODEL TRAINING
# NO PRODUCTION MODEL OR DATABASE MODIFICATIONS
# ============================================================

import pandas as pd
import numpy as np
import shap

from sklearn.metrics import brier_score_loss
from prediction_model.training.train_new_model  import (
    _make_xgb_pipeline,
    _make_rf_pipeline,
    select_features_from_shap,
    LEAK_COLS,
    ID_COLS
)

walkforward_predictions = []
walkforward_metrics = []

for test_season in eligible_seasons:

    print(f"\n{'=' * 60}")
    print(f"WALK-FORWARD TEST SEASON: {test_season}")
    print(f"{'=' * 60}")

    train_mask = historical_features["season_id"] < test_season
    test_mask = historical_features["season_id"] == test_season

    train_df = historical_features.loc[train_mask]
    test_df = historical_features.loc[test_mask]

    # Build features using training data only to determine
    # feature inclusion and missing-value treatment.
    X_train = (
        train_df
        .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
        .select_dtypes(include="number")
    )

    feature_cols = X_train.columns[
        X_train.isna().mean() < 0.20
    ].tolist()

    X_train = X_train[feature_cols].fillna(0)

    X_test = (
        test_df
        .reindex(columns=feature_cols, fill_value=0)
        .fillna(0)
    )

    y_train = train_df["made_playoffs"].astype(int)
    y_test = test_df["made_playoffs"].astype(int)

    assert X_train.columns.tolist() == X_test.columns.tolist()
    assert train_df["season_id"].max() < test_season
    assert np.isfinite(X_train.to_numpy(dtype=float)).all()
    assert np.isfinite(X_test.to_numpy(dtype=float)).all()

    print(f"Train rows: {len(X_train):,}")
    print(f"Test rows: {len(X_test):,}")

    # Initial XGBoost for train-only SHAP selection
    initial_xgb = _make_xgb_pipeline()
    initial_xgb.fit(X_train, y_train)

    X_scaled = initial_xgb.named_steps["scaler"].transform(X_train)
    explainer = shap.TreeExplainer(initial_xgb.named_steps["model"])
    shap_values = explainer(X_scaled).values

    selected_features = select_features_from_shap(
        X_train,
        shap_values,
        min_usage=0.03,
        min_impact=0.0005
    )

    if not selected_features:
        raise RuntimeError(f"No SHAP features selected for {test_season}")

    X_train = X_train[selected_features]
    X_test = X_test[selected_features]

    print(f"Selected features: {len(selected_features)}")

    # Final holdout models
    xgb_model = _make_xgb_pipeline()
    xgb_model.fit(X_train, y_train)

    rf_model = _make_rf_pipeline()
    rf_model.fit(X_train, y_train)

    xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
    rf_prob = rf_model.predict_proba(X_test)[:, 1]
    ensemble_prob = (xgb_prob + rf_prob) / 2

    # Preserve original test-season identifiers
    predictions = test_df[
        ["season_id", "team_id", "game_id",
         "game_date", "games_played", "made_playoffs"]
    ].copy()

    predictions["playoff_prob_xgb"] = xgb_prob
    predictions["playoff_prob_rf"] = rf_prob
    predictions["playoff_prob"] = ensemble_prob

    walkforward_predictions.append(predictions)

    metrics = {
        "test_season": test_season,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "selected_features": len(selected_features),
        "xgb_brier": brier_score_loss(y_test, xgb_prob),
        "rf_brier": brier_score_loss(y_test, rf_prob),
        "ensemble_brier": brier_score_loss(y_test, ensemble_prob)
    }

    walkforward_metrics.append(metrics)

    print(
        f"Brier — XGB: {metrics['xgb_brier']:.4f} | "
        f"RF: {metrics['rf_brier']:.4f} | "
        f"Ensemble: {metrics['ensemble_brier']:.4f}"
    )

# Combine all historical predictions
walkforward_predictions = pd.concat(
    walkforward_predictions, ignore_index=True
)

walkforward_metrics = pd.DataFrame(walkforward_metrics)

print("\n=== BLOCK 77D: WALK-FORWARD MODEL PERFORMANCE ===")
print(walkforward_metrics.to_string(
    index=False, float_format=lambda x: f"{x:.4f}"
))

print("\n=== VALIDATION ===")
assert walkforward_predictions.duplicated(
    ["season_id", "team_id", "game_id"]
).sum() == 0

assert len(walkforward_metrics) == len(eligible_seasons)

print(f"Test seasons: {len(walkforward_metrics)}")
print(f"Prediction rows: {len(walkforward_predictions):,}")
print("Duplicate prediction keys: 0")
print("\n=== BLOCK 77D COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77E: HISTORICAL EARLY-SEASON ADJUSTMENT COMPARISON
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

early = walkforward_predictions.loc[
    walkforward_predictions["games_played"].between(1, 20)
].copy()

early["raw"] = early["playoff_prob"]

g = early["games_played"].to_numpy()
p = early["raw"].to_numpy()

# ------------------------------------------------------------
# Five adjustment strategies
# ------------------------------------------------------------

early["current_1_20"] = 0.45 * p + 0.275

early["weaker_1_20"] = 0.70 * p + 0.15

early["adjust_1_15"] = np.where(
    g <= 15,
    0.45 * p + 0.275,
    p
)

# Weight rises from 0.45 at game 10 to 1.0 at game 20
taper_weight = np.where(
    g <= 10,
    0.45,
    np.minimum(1.0, 0.45 + 0.055 * (g - 10))
)

early["taper_10_20"] = (
    taper_weight * p + (1 - taper_weight) * 0.5
)

strategies = [
    "raw",
    "current_1_20",
    "weaker_1_20",
    "adjust_1_15",
    "taper_10_20"
]

early["window"] = pd.cut(
    early["games_played"],
    bins=[0, 5, 10, 15, 20],
    labels=["1-5", "6-10", "11-15", "16-20"]
)

# ------------------------------------------------------------
# Overall pooled Brier scores
# ------------------------------------------------------------

overall = pd.DataFrame([
    {
        "strategy": strategy,
        "brier": brier_score_loss(
            early["made_playoffs"], early[strategy]
        )
    }
    for strategy in strategies
])

raw_brier = overall.loc[
    overall["strategy"].eq("raw"), "brier"
].iloc[0]

overall["vs_raw"] = overall["brier"] - raw_brier
overall = overall.sort_values("brier")

print("=== BLOCK 77E: POOLED EARLY-SEASON BRIER ===")
print(overall.to_string(index=False, float_format=lambda x: f"{x:.5f}"))

# ------------------------------------------------------------
# Brier by historical season
# ------------------------------------------------------------

season_results = []

for season, group in early.groupby("season_id"):
    result = {"season_id": season}

    for strategy in strategies:
        result[strategy] = brier_score_loss(
            group["made_playoffs"], group[strategy]
        )

    season_results.append(result)

season_results = pd.DataFrame(season_results)

print("\n=== BRIER BY SEASON ===")
print(season_results.to_string(index=False, float_format=lambda x: f"{x:.5f}"))

# ------------------------------------------------------------
# Brier by five-game window
# ------------------------------------------------------------

window_results = []

for window, group in early.groupby("window", observed=True):
    result = {"window": str(window)}

    for strategy in strategies:
        result[strategy] = brier_score_loss(
            group["made_playoffs"], group[strategy]
        )

    window_results.append(result)

window_results = pd.DataFrame(window_results)

print("\n=== BRIER BY GAME WINDOW ===")
print(window_results.to_string(index=False, float_format=lambda x: f"{x:.5f}"))

# ------------------------------------------------------------
# Team-season improvement counts
# ------------------------------------------------------------

team_results = []

for (season, team), group in early.groupby(["season_id", "team_id"]):
    result = {
        "season_id": season,
        "team_id": team,
        "raw": brier_score_loss(group["made_playoffs"], group["raw"])
    }

    for strategy in strategies[1:]:
        result[strategy] = brier_score_loss(
            group["made_playoffs"], group[strategy]
        )

    team_results.append(result)

team_results = pd.DataFrame(team_results)

improvement = pd.DataFrame([
    {
        "strategy": strategy,
        "improved": (team_results[strategy] < team_results["raw"]).sum(),
        "worsened": (team_results[strategy] > team_results["raw"]).sum(),
        "unchanged": (team_results[strategy] == team_results["raw"]).sum()
    }
    for strategy in strategies[1:]
])

print("\n=== TEAM-SEASON IMPROVEMENT COUNTS ===")
print(improvement.to_string(index=False))

print("\n=== VALIDATION ===")
print(f"Early-season predictions: {len(early):,}")
print(f"Historical seasons: {early['season_id'].nunique()}")
print(f"Team-seasons: {len(team_results)}")

assert early[strategies].notna().all().all()
assert early[strategies].apply(lambda s: s.between(0, 1).all()).all()
assert not early.duplicated(["season_id", "team_id", "game_id"]).any()

print("\n=== BLOCK 77E COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77F: GAMES 1–5 CALIBRATION DIAGNOSTIC
# READ-ONLY — NO PRODUCTION CHANGES
# ============================================================

import pandas as pd
import numpy as np

first5 = early.loc[early["games_played"].between(1, 5)].copy()

first5["prob_band"] = pd.cut(
    first5["raw"],
    bins=[0, .2, .4, .6, .8, 1],
    labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"],
    include_lowest=True
)

# Calibration by probability band
calibration = (
    first5.groupby("prob_band", observed=True)
    .agg(
        observations=("made_playoffs", "size"),
        avg_prediction=("raw", "mean"),
        actual_rate=("made_playoffs", "mean")
    )
    .reset_index()
)

calibration["calibration_error"] = (
    calibration["avg_prediction"] - calibration["actual_rate"]
)

print("=== BLOCK 77F: GAMES 1–5 CALIBRATION ===")
print(calibration.to_string(
    index=False, float_format=lambda x: f"{x:.4f}"
))

# Season-by-season comparison
season_comparison = (
    first5.groupby("season_id")
    .apply(
        lambda x: pd.Series({
            "raw_brier": np.mean(
                (x["raw"] - x["made_playoffs"]) ** 2
            ),
            "weaker_brier": np.mean(
                (x["weaker_1_20"] - x["made_playoffs"]) ** 2
            ),
            "mean_raw_prob": x["raw"].mean(),
            "actual_playoff_rate": x["made_playoffs"].mean()
        }),
        include_groups=False
    )
    .reset_index()
)

season_comparison["improvement"] = (
    season_comparison["raw_brier"]
    - season_comparison["weaker_brier"]
)

print("\n=== GAMES 1–5: SEASON COMPARISON ===")
print(season_comparison.to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

print("\n=== SUMMARY ===")
print(f"Seasons improved: {(season_comparison['improvement'] > 0).sum()} / 8")
print(f"Seasons worsened: {(season_comparison['improvement'] < 0).sum()} / 8")
print(f"Observations: {len(first5):,}")

print("\n=== BLOCK 77F COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77G: TEAM-LEVEL EARLY-SEASON VALIDATION
# READ-ONLY — NO PRODUCTION CHANGES
# ============================================================

import pandas as pd
import numpy as np

first5 = early.loc[early["games_played"].between(1, 5)].copy()

# One observation per team-season
team_first5 = (
    first5.groupby(["season_id", "team_id"])
    .agg(
        games=("games_played", "nunique"),
        made_playoffs=("made_playoffs", "first"),
        raw=("raw", "mean"),
        weaker=("weaker_1_20", "mean")
    )
    .reset_index()
)

assert team_first5["games"].eq(5).all()
assert team_first5["made_playoffs"].isin([0, 1]).all()

team_first5["raw_error"] = (
    team_first5["raw"] - team_first5["made_playoffs"]
) ** 2

team_first5["weaker_error"] = (
    team_first5["weaker"] - team_first5["made_playoffs"]
) ** 2

team_first5["improvement"] = (
    team_first5["raw_error"] - team_first5["weaker_error"]
)

print("=== BLOCK 77G: TEAM-LEVEL VALIDATION ===")

print(f"Team-seasons: {len(team_first5)}")
print(f"Raw Brier: {team_first5['raw_error'].mean():.5f}")
print(f"Weaker Brier: {team_first5['weaker_error'].mean():.5f}")

print("\n=== PLAYOFF VS NON-PLAYOFF TEAMS ===")

outcome_results = (
    team_first5.groupby("made_playoffs")
    .agg(
        teams=("team_id", "size"),
        raw_brier=("raw_error", "mean"),
        weaker_brier=("weaker_error", "mean"),
        avg_improvement=("improvement", "mean")
    )
    .reset_index()
)

print(outcome_results.to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

print("\n=== TEAM-SEASON IMPROVEMENT COUNTS ===")

print(f"Improved: {(team_first5['improvement'] > 0).sum()}")
print(f"Worsened: {(team_first5['improvement'] < 0).sum()}")
print(f"Unchanged: {(team_first5['improvement'] == 0).sum()}")

print("\n=== SEASON-LEVEL CONSISTENCY ===")

season_results = (
    team_first5.groupby("season_id")
    .agg(
        teams=("team_id", "size"),
        raw_brier=("raw_error", "mean"),
        weaker_brier=("weaker_error", "mean"),
        avg_improvement=("improvement", "mean")
    )
    .reset_index()
)

print(season_results.to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

print("\n=== BLOCK 77G COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77H: FEATURE DEFINITION AUDIT
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import inspect
import feature_engineering as fe

source = inspect.getsource(fe)

target = "wc_pressure_x_performance"
lines = source.splitlines()

matches = [
    i for i, line in enumerate(lines)
    if target in line
]

print("=== BLOCK 77H: FEATURE DEFINITION AUDIT ===")
print(f"Occurrences of '{target}': {len(matches)}")

for i in matches:
    start = max(0, i - 12)
    end = min(len(lines), i + 13)

    print(f"\n{'=' * 60}")
    print(f"Occurrence at line {i + 1}")
    print(f"{'=' * 60}")

    for j in range(start, end):
        print(f"{j + 1:4d} | {lines[j]}")

print("\n=== BLOCK 77H COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77I: TRACE WC PRESSURE FEATURE DEPENDENCIES
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import inspect
import feature_engineering as fe

source = inspect.getsource(fe)
lines = source.splitlines()

targets = ["wc_points_pct_margin", "season_pct"]

print("=== BLOCK 77I: FEATURE DEPENDENCY AUDIT ===")

for target in targets:
    matches = [
        i for i, line in enumerate(lines)
        if target in line
    ]

    print(f"\n{'=' * 60}")
    print(f"FEATURE: {target}")
    print(f"Occurrences: {len(matches)}")
    print(f"{'=' * 60}")

    for i in matches:
        start = max(0, i - 10)
        end = min(len(lines), i + 11)

        print(f"\nOccurrence at line {i + 1}")

        for j in range(start, end):
            print(f"{j + 1:4d} | {lines[j]}")

print("\n=== BLOCK 77I COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77J: WILDCARD CUTOFF LEAKAGE AUDIT
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import inspect
import feature_engineering as fe

source = inspect.getsource(fe)
lines = source.splitlines()

print("=== BLOCK 77J: WILDCARD CUTOFF CONSTRUCTION ===")

# Display the beginning of the feature engineering module
# containing the wildcard cutoff calculation.

for i in range(min(145, len(lines))):
    print(f"{i + 1:4d} | {lines[i]}")

print("\n=== BLOCK 77J COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77K: CANONICAL SNAPSHOT LEAKAGE AUDIT
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import inspect

from prediction_model.training.train_new_model import (
    build_canonical_training_features
)

print("=== BLOCK 77K: CANONICAL SNAPSHOT CONSTRUCTION ===")

source = inspect.getsource(build_canonical_training_features)

for i, line in enumerate(source.splitlines(), start=1):
    print(f"{i:4d} | {line}")

print("\n=== BLOCK 77K COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77L: FULL-SEASON PROBABILITY CALIBRATION
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

cal = walkforward_predictions.copy()

cal["season_stage"] = pd.cut(
    cal["games_played"],
    bins=[0, 5, 10, 20, 40, 60, 82],
    labels=["1-5", "6-10", "11-20", "21-40", "41-60", "61-82"]
)

cal["prob_band"] = pd.cut(
    cal["playoff_prob"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

# ------------------------------------------------------------
# Overall calibration by season stage
# ------------------------------------------------------------

stage_results = (
    cal.groupby("season_stage", observed=True)
    .agg(
        observations=("made_playoffs", "size"),
        avg_prediction=("playoff_prob", "mean"),
        actual_rate=("made_playoffs", "mean")
    )
    .reset_index()
)

stage_results["brier"] = [
    brier_score_loss(
        group["made_playoffs"], group["playoff_prob"]
    )
    for _, group in cal.groupby("season_stage", observed=True)
]

stage_results["mean_bias"] = (
    stage_results["avg_prediction"] - stage_results["actual_rate"]
)

print("=== BLOCK 77L: CALIBRATION BY SEASON STAGE ===")
print(stage_results.to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

# ------------------------------------------------------------
# Calibration by probability band and stage
# ------------------------------------------------------------

band_results = (
    cal.groupby(["season_stage", "prob_band"], observed=True)
    .agg(
        observations=("made_playoffs", "size"),
        avg_prediction=("playoff_prob", "mean"),
        actual_rate=("made_playoffs", "mean")
    )
    .reset_index()
)

band_results["calibration_error"] = (
    band_results["avg_prediction"] - band_results["actual_rate"]
)

print("\n=== CALIBRATION BY PROBABILITY BAND ===")
print(band_results.to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

# ------------------------------------------------------------
# Season-by-season performance
# ------------------------------------------------------------

season_results = (
    cal.groupby("season_id")
    .apply(
        lambda x: pd.Series({
            "observations": len(x),
            "brier": brier_score_loss(
                x["made_playoffs"], x["playoff_prob"]
            ),
            "avg_prediction": x["playoff_prob"].mean(),
            "actual_rate": x["made_playoffs"].mean()
        }),
        include_groups=False
    )
    .reset_index()
)

print("\n=== FULL-SEASON PERFORMANCE BY YEAR ===")
print(season_results.to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

print("\n=== VALIDATION ===")
print(f"Total predictions: {len(cal):,}")
print(f"Historical seasons: {cal['season_id'].nunique()}")
print(f"Overall Brier: {brier_score_loss(cal['made_playoffs'], cal['playoff_prob']):.5f}")

assert cal["playoff_prob"].between(0, 1).all()
assert cal["made_playoffs"].isin([0, 1]).all()

print("\n=== BLOCK 77L COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77M: SEASON-LEVEL CALIBRATION CONSISTENCY
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import numpy as np
import pandas as pd

calibration_consistency = []

for (season, stage), group in cal.groupby(
    ["season_id", "season_stage"], observed=True
):
    bands = pd.cut(
        group["playoff_prob"],
        bins=np.linspace(0, 1, 11),
        include_lowest=True
    )

    band_stats = (
        group.assign(band=bands)
        .groupby("band", observed=True)
        .agg(
            n=("made_playoffs", "size"),
            predicted=("playoff_prob", "mean"),
            actual=("made_playoffs", "mean")
        )
    )

    # Expected calibration error, weighted by observations
    ece = (
        (
            band_stats["predicted"] - band_stats["actual"]
        ).abs() * band_stats["n"]
    ).sum() / len(group)

    calibration_consistency.append({
        "season_id": season,
        "season_stage": str(stage),
        "observations": len(group),
        "ece": ece,
        "mean_bias": (
            group["playoff_prob"].mean()
            - group["made_playoffs"].mean()
        )
    })

calibration_consistency = pd.DataFrame(calibration_consistency)

print("=== BLOCK 77M: CALIBRATION ERROR BY SEASON ===")

print(
    calibration_consistency.pivot(
        index="season_id",
        columns="season_stage",
        values="ece"
    ).round(4).to_string()
)

print("\n=== CALIBRATION ERROR SUMMARY BY STAGE ===")

stage_summary = (
    calibration_consistency.groupby("season_stage")
    .agg(
        mean_ece=("ece", "mean"),
        median_ece=("ece", "median"),
        min_ece=("ece", "min"),
        max_ece=("ece", "max"),
        seasons=("season_id", "nunique")
    )
    .reset_index()
)

stage_order = ["1-5", "6-10", "11-20", "21-40", "41-60", "61-82"]

stage_summary["season_stage"] = pd.Categorical(
    stage_summary["season_stage"],
    categories=stage_order,
    ordered=True
)

stage_summary = stage_summary.sort_values("season_stage")

print(stage_summary.to_string(
    index=False, float_format=lambda x: f"{x:.5f}"
))

print("\n=== VALIDATION ===")
print(f"Season-stage combinations: {len(calibration_consistency)}")
print(f"Expected combinations: {8 * 6}")

assert len(calibration_consistency) == 48

print("\n=== BLOCK 77M COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77N: CALIBRATION DIRECTION BY PROBABILITY RANGE
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import pandas as pd

direction = cal.copy()

direction["probability_range"] = pd.cut(
    direction["playoff_prob"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"],
    include_lowest=True
)

results = (
    direction.groupby(
        ["season_stage", "probability_range"],
        observed=True
    )
    .agg(
        observations=("made_playoffs", "size"),
        predicted=("playoff_prob", "mean"),
        actual=("made_playoffs", "mean")
    )
    .reset_index()
)

results["bias"] = results["predicted"] - results["actual"]

results["direction"] = results["bias"].apply(
    lambda x: "Overconfident" if x > 0.02
    else "Underconfident" if x < -0.02
    else "Near calibrated"
)

print("=== BLOCK 77N: CALIBRATION DIRECTION ===")
print(results.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

print("\n=== BLOCK 77N COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77O: CALIBRATION EXPERIMENT BASELINE
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss

baseline = cal.copy()

results = []

for (season, stage), group in baseline.groupby(
    ["season_id", "season_stage"], observed=True
):
    y = group["made_playoffs"].astype(int)
    p = group["playoff_prob"].clip(1e-6, 1 - 1e-6)

    results.append({
        "season_id": season,
        "season_stage": str(stage),
        "observations": len(group),
        "brier_raw": brier_score_loss(y, p),
        "logloss_raw": log_loss(y, p, labels=[0, 1])
    })

calibration_baseline = pd.DataFrame(results)

stage_order = ["1-5", "6-10", "11-20", "21-40", "41-60", "61-82"]

calibration_baseline["season_stage"] = pd.Categorical(
    calibration_baseline["season_stage"],
    categories=stage_order,
    ordered=True
)

calibration_baseline = calibration_baseline.sort_values(
    ["season_id", "season_stage"]
).reset_index(drop=True)

print("=== BLOCK 77O: RAW ENSEMBLE BASELINE ===")

print(
    calibration_baseline.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

print("\n=== OVERALL BASELINE ===")

print(f"Predictions: {len(baseline):,}")
print(f"Brier: {brier_score_loss(baseline['made_playoffs'], baseline['playoff_prob']):.5f}")
print(f"Log loss: {log_loss(baseline['made_playoffs'], baseline['playoff_prob'], labels=[0, 1]):.5f}")

assert len(calibration_baseline) == 48

print("\n=== BLOCK 77O COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77P: CHRONOLOGICAL PLATT CALIBRATION
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from scipy.special import logit

platt_data = walkforward_predictions.copy()
platt_data = platt_data.sort_values(
    ["season_id", "game_date", "team_id"]
).reset_index(drop=True)

seasons = sorted(platt_data["season_id"].unique())

platt_results = []
platt_predictions = []

for test_season in seasons[1:]:

    train = platt_data[
        platt_data["season_id"] < test_season
    ].copy()

    test = platt_data[
        platt_data["season_id"] == test_season
    ].copy()

    # Log-odds transformation of raw ensemble probabilities
    X_train = logit(
        train["playoff_prob"].clip(1e-6, 1 - 1e-6)
    ).to_numpy().reshape(-1, 1)

    X_test = logit(
        test["playoff_prob"].clip(1e-6, 1 - 1e-6)
    ).to_numpy().reshape(-1, 1)

    y_train = train["made_playoffs"].astype(int)
    y_test = test["made_playoffs"].astype(int)

    calibrator = LogisticRegression(
        solver="lbfgs",
        max_iter=1000
    )

    calibrator.fit(X_train, y_train)

    test["platt_prob"] = calibrator.predict_proba(X_test)[:, 1]

    raw_brier = brier_score_loss(
        y_test, test["playoff_prob"]
    )

    platt_brier = brier_score_loss(
        y_test, test["platt_prob"]
    )

    raw_logloss = log_loss(
        y_test, test["playoff_prob"], labels=[0, 1]
    )

    platt_logloss = log_loss(
        y_test, test["platt_prob"], labels=[0, 1]
    )

    platt_results.append({
        "season_id": test_season,
        "calibration_seasons": train["season_id"].nunique(),
        "raw_brier": raw_brier,
        "platt_brier": platt_brier,
        "brier_change": platt_brier - raw_brier,
        "raw_logloss": raw_logloss,
        "platt_logloss": platt_logloss,
        "logloss_change": platt_logloss - raw_logloss,
        "coefficient": calibrator.coef_[0, 0],
        "intercept": calibrator.intercept_[0]
    })

    platt_predictions.append(test)

platt_results = pd.DataFrame(platt_results)
platt_predictions = pd.concat(
    platt_predictions, ignore_index=True
)

print("=== BLOCK 77P: WALK-FORWARD PLATT CALIBRATION ===")

print(platt_results.to_string(
    index=False,
    float_format=lambda x: f"{x:.5f}"
))

print("\n=== POOLED TEST PERFORMANCE ===")

y = platt_predictions["made_playoffs"].astype(int)

for col in ["playoff_prob", "platt_prob"]:
    p = platt_predictions[col].clip(1e-6, 1 - 1e-6)

    print(
        f"{col:15s} | "
        f"Brier: {brier_score_loss(y, p):.5f} | "
        f"Log loss: {log_loss(y, p, labels=[0, 1]):.5f}"
    )

print("\n=== SEASON-LEVEL RESULTS ===")

print(
    f"Platt improved Brier in "
    f"{(platt_results['brier_change'] < 0).sum()}"
    f"/{len(platt_results)} seasons"
)

print(
    f"Platt improved log loss in "
    f"{(platt_results['logloss_change'] < 0).sum()}"
    f"/{len(platt_results)} seasons"
)

print("\n=== VALIDATION ===")

assert platt_predictions["platt_prob"].between(0, 1).all()
assert platt_predictions["season_id"].nunique() == len(seasons) - 1

print("Chronological calibration validated")
print("\n=== BLOCK 77P COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77Q: CHRONOLOGICAL ISOTONIC CALIBRATION
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, log_loss

isotonic_results = []
isotonic_predictions = []

for test_season in seasons[1:]:

    train = platt_data[
        platt_data["season_id"] < test_season
    ].copy()

    test = platt_data[
        platt_data["season_id"] == test_season
    ].copy()

    calibrator = IsotonicRegression(
        y_min=0,
        y_max=1,
        out_of_bounds="clip"
    )

    calibrator.fit(
        train["playoff_prob"].to_numpy(),
        train["made_playoffs"].astype(int).to_numpy()
    )

    test["isotonic_prob"] = calibrator.predict(
        test["playoff_prob"].to_numpy()
    )

    y = test["made_playoffs"].astype(int)

    raw = test["playoff_prob"].clip(1e-6, 1 - 1e-6)
    iso = test["isotonic_prob"].clip(1e-6, 1 - 1e-6)

    isotonic_results.append({
        "season_id": test_season,
        "calibration_seasons": train["season_id"].nunique(),
        "raw_brier": brier_score_loss(y, raw),
        "isotonic_brier": brier_score_loss(y, iso),
        "brier_change": (
            brier_score_loss(y, iso)
            - brier_score_loss(y, raw)
        ),
        "raw_logloss": log_loss(y, raw, labels=[0, 1]),
        "isotonic_logloss": log_loss(y, iso, labels=[0, 1]),
        "logloss_change": (
            log_loss(y, iso, labels=[0, 1])
            - log_loss(y, raw, labels=[0, 1])
        )
    })

    isotonic_predictions.append(test)

isotonic_results = pd.DataFrame(isotonic_results)

isotonic_predictions = pd.concat(
    isotonic_predictions, ignore_index=True
)

print("=== BLOCK 77Q: WALK-FORWARD ISOTONIC CALIBRATION ===")

print(isotonic_results.to_string(
    index=False,
    float_format=lambda x: f"{x:.5f}"
))

print("\n=== POOLED TEST PERFORMANCE ===")

y = isotonic_predictions["made_playoffs"].astype(int)

for col in ["playoff_prob", "isotonic_prob"]:
    p = isotonic_predictions[col].clip(1e-6, 1 - 1e-6)

    print(
        f"{col:15s} | "
        f"Brier: {brier_score_loss(y, p):.5f} | "
        f"Log loss: {log_loss(y, p, labels=[0, 1]):.5f}"
    )

print("\n=== SEASON-LEVEL RESULTS ===")

print(
    f"Isotonic improved Brier in "
    f"{(isotonic_results['brier_change'] < 0).sum()}"
    f"/{len(isotonic_results)} seasons"
)

print(
    f"Isotonic improved log loss in "
    f"{(isotonic_results['logloss_change'] < 0).sum()}"
    f"/{len(isotonic_results)} seasons"
)

print("\n=== VALIDATION ===")

assert isotonic_predictions["isotonic_prob"].between(0, 1).all()
assert isotonic_predictions["season_id"].nunique() == len(seasons) - 1

print("Chronological isotonic calibration validated")
print("\n=== BLOCK 77Q COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77R: LOCATE EARLY-SEASON PROBABILITY ADJUSTMENTS
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import inspect
import feature_engineering as fe
import prediction_model.training.train_new_model as tm

patterns = [
    "0.45",
    "0.275",
    "early",
    "regression",
    "shrink",
    "adjust",
    "playoff_prob"
]

modules = {
    "feature_engineering": fe,
    "train_new_model": tm
}

print("=== BLOCK 77R: PROBABILITY ADJUSTMENT SEARCH ===")

for module_name, module in modules.items():
    source = inspect.getsource(module)
    lines = source.splitlines()

    print(f"\n=== MODULE: {module_name} ===")

    matches = [
        i for i, line in enumerate(lines)
        if any(pattern in line.lower() for pattern in patterns)
    ]

    if not matches:
        print("No matching lines")
        continue

    shown = set()

    for i in matches:
        for j in range(max(0, i - 2), min(len(lines), i + 3)):
            if j not in shown:
                print(f"{j + 1:4d} | {lines[j]}")
                shown.add(j)

        print("-----")

print("\n=== BLOCK 77R COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77S: LOCATE ACTIVE DISPLAY ADJUSTMENT
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

from pathlib import Path
import prediction_model.training.train_new_model as tm

project_root = Path(tm.__file__).resolve().parents[2]

patterns = [
    "0.45",
    "0.275",
    "playoff_prob_display",
    "display_prob",
    "adjusted_prob",
]

exclude_dirs = {
    ".git", ".venv", "venv", "__pycache__",
    "node_modules", ".ipynb_checkpoints"
}

print("=== BLOCK 77S: ACTIVE DISPLAY ADJUSTMENT SEARCH ===")
print(f"Project root: {project_root}")

matches = []

for path in project_root.rglob("*.py"):
    if any(part in exclude_dirs for part in path.parts):
        continue

    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except (UnicodeError, OSError):
        continue

    for i, line in enumerate(lines, start=1):
        if any(pattern in line.lower() for pattern in patterns):
            matches.append((path.relative_to(project_root), i, line.strip()))

for path, line_num, content in matches:
    print(f"{path}:{line_num} | {content}")

print(f"\nTotal matches: {len(matches)}")
print("=== BLOCK 77S COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77T: INFERENCE VERSION & ADJUSTMENT AUDIT
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

from pathlib import Path
import prediction_model.training.train_new_model as tm

root = Path(tm.__file__).resolve().parents[2]

files = {
    "DAILY INFERENCE": (
        root / "prediction_model/inference/daily_playoff_probability.py",
        [(1, 110), (320, 440)]
    ),
    "HISTORICAL BACKFILL": (
        root / "scripts/backfill/backfill_probability_scores.py",
        [(1, 100), (190, 260)]
    )
}

for name, (path, ranges) in files.items():
    print(f"\n=== {name} ===")

    lines = path.read_text(encoding="utf-8").splitlines()

    for start, end in ranges:
        print(f"\n--- Lines {start}-{end} ---")

        for i in range(start - 1, min(end, len(lines))):
            print(f"{i + 1:4d} | {lines[i]}")

    print("\n--- Model/version references ---")

    for i, line in enumerate(lines, start=1):
        if any(term in line.lower() for term in [
            "model_version", "active_model", "manifest",
            "model_dir", "v6", "v7"
        ]):
            print(f"{i:4d} | {line.strip()}")

print("\n=== BLOCK 77T COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77U: MODEL SELECTION & VERSION METADATA AUDIT
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import inspect
import prediction_model.metadata as metadata
import prediction_model.inference as inference

print("=== BLOCK 77U: MODEL METADATA ===")

print("MODEL_METADATA:")
print(metadata.MODEL_METADATA)

print("\n=== INFERENCE MODULE ===")

print("Inference module:", inference.__file__)

print("\n=== score_dataframe ===")

print(inspect.getsource(inference.score_dataframe))

print("\n=== VERSION-RELATED METADATA ===")

for name in [
    "EARLY_SEASON_NOTE",
    "PROBABILITY_TIERS"
]:
    if hasattr(metadata, name):
        print(f"\n{name}:")
        print(getattr(metadata, name))

print("\n=== BLOCK 77U COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77V: MODEL ARTIFACT LOADING AUDIT
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import inspect
import prediction_model.inference as inference
import prediction_model.metadata as metadata

print("=== BLOCK 77V: INFERENCE INITIALIZATION ===")

source = inspect.getsource(inference)

for i, line in enumerate(source.splitlines(), start=1):
    print(f"{i:4d} | {line}")

print("\n=== LOADED MODEL OBJECTS ===")

for name in ["XGB_PIPELINE", "RF_PIPELINE", "FEATURE_COLS"]:
    obj = getattr(inference, name, None)

    if name == "FEATURE_COLS":
        print(f"{name}: {len(obj) if obj is not None else 'MISSING'} features")
    else:
        print(f"{name}: {type(obj).__name__ if obj is not None else 'MISSING'}")

print("\n=== ACTIVE VERSION ===")
print(metadata.MODEL_METADATA["version"])

print("\n=== BLOCK 77V COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77W: ACTUAL MODEL ARTIFACT LOADING AUDIT
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

import inspect
import importlib
from pathlib import Path

inference_impl = importlib.import_module(
    "prediction_model.inference.inference"
)

print("=== BLOCK 77W: MODEL LOADER ===")
print(f"Module: {inference_impl.__file__}")

source = inspect.getsource(inference_impl)

for i, line in enumerate(source.splitlines(), start=1):
    print(f"{i:4d} | {line}")

print("\n=== LOADED MODEL OBJECTS ===")

for name in ["XGB_PIPELINE", "RF_PIPELINE", "FEATURE_COLS"]:
    obj = getattr(inference_impl, name, None)

    if name == "FEATURE_COLS":
        print(f"{name}: {len(obj) if obj is not None else 'MISSING'} features")
    else:
        print(f"{name}: {type(obj).__name__ if obj is not None else 'MISSING'}")

print("\n=== BLOCK 77W COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77X: MODEL PATH & ARTIFACT INVENTORY
# READ-ONLY — NO MODEL OR DATABASE MODIFICATIONS
# ============================================================

from pathlib import Path
import config
import prediction_model.metadata as metadata
import prediction_model.inference.inference as inference_impl
import importlib

inference_impl = importlib.import_module(
    "prediction_model.inference.inference"
)

print("=== BLOCK 77X: MODEL CONFIGURATION ===")

for name in [
    "MODEL_PATH",
    "DB_PATH",
    "CURRENT_SEASON_ID",
    "EXCLUDE_SEASON_ID"
]:
    print(f"{name}: {getattr(config, name, 'NOT DEFINED')}")

print("\n=== ACTIVE MODEL ===")

active_path = Path(config.MODEL_PATH)

print("Active path:", active_path)
print("Exists:", active_path.exists())
print("Metadata version:", metadata.MODEL_METADATA["version"])
print("Loaded features:", len(inference_impl.FEATURE_COLS))

print("\n=== AVAILABLE MODEL ARTIFACTS ===")

root = Path(config.__file__).resolve().parent

artifacts = sorted(
    [
        p for p in root.rglob("*.joblib")
        if ".venv" not in p.parts
        and "venv" not in p.parts
    ]
)

for path in artifacts:
    print(path.relative_to(root))

print(f"\nTotal joblib artifacts: {len(artifacts)}")

print("\n=== BLOCK 77X COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77Y: V7 MODEL ARTIFACT VALIDATION
# READ-ONLY — PRODUCTION V6 UNCHANGED
# ============================================================

from pathlib import Path
from joblib import load
import config
import prediction_model.metadata as metadata
import importlib

inference_impl = importlib.import_module(
    "prediction_model.inference.inference"
)

v6_path = Path(config.MODEL_PATH)
v7_path = v6_path.with_name("playoff_models_v7.joblib")

print("=== BLOCK 77Y: V7 MODEL ARTIFACT VALIDATION ===")

assert v6_path.exists(), "Production v6 artifact missing"
assert v7_path.exists(), "Candidate v7 artifact missing"
assert v6_path != v7_path, "v6 and v7 paths must differ"

v7_bundle = load(v7_path)

print("\n=== V7 BUNDLE CONTENTS ===")
print("Keys:", list(v7_bundle.keys()))

required_keys = {"xgb_pipeline", "rf_pipeline", "features"}
assert required_keys.issubset(v7_bundle), "Missing required model components"

v7_xgb = v7_bundle["xgb_pipeline"]
v7_rf = v7_bundle["rf_pipeline"]
v7_features = list(v7_bundle["features"])
v6_features = list(inference_impl.FEATURE_COLS)

print("\n=== FEATURE COMPARISON ===")
print("v6 features:", len(v6_features))
print("v7 features:", len(v7_features))
print("Shared features:", len(set(v6_features) & set(v7_features)))
print("v7-only features:", len(set(v7_features) - set(v6_features)))
print("v6-only features:", len(set(v6_features) - set(v7_features)))

print("\n=== V7 MODEL TYPES ===")
print("XGBoost:", type(v7_xgb).__name__)
print("Random Forest:", type(v7_rf).__name__)

print("\n=== PRODUCTION SAFETY CHECK ===")
print("Configured model:", Path(config.MODEL_PATH).name)
print("Metadata version:", metadata.MODEL_METADATA["version"])

assert Path(config.MODEL_PATH).resolve() == v6_path.resolve()
assert metadata.MODEL_METADATA["version"] == "v6"

print("\nPASS: v7 artifact loaded independently")
print("PASS: production v6 configuration unchanged")
print("\n=== BLOCK 77Y COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 77Z: INDEPENDENT V7 INFERENCE VALIDATION
# READ-ONLY — PRODUCTION V6 UNCHANGED
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
import config
import prediction_model.metadata as metadata

print("=== BLOCK 77Z: V7 INFERENCE VALIDATION ===")

# Use existing historical feature matrix
assert "X_historical" in globals(), "Historical feature matrix missing"
assert "v7_features" in globals(), "V7 feature list missing"

# Confirm every required feature exists
missing = [c for c in v7_features if c not in X_historical.columns]

print("\n=== FEATURE ALIGNMENT ===")
print("Required v7 features:", len(v7_features))
print("Missing features:", len(missing))

assert not missing, f"Missing v7 features: {missing}"

# Use a small, deterministic sample
X_test_v7 = (
    X_historical
    .reindex(columns=v7_features)
    .head(100)
    .fillna(0)
    .copy()
)

assert list(X_test_v7.columns) == v7_features
assert len(X_test_v7) == 100

print("Sample rows:", len(X_test_v7))

# Independent v7 predictions
p_xgb = v7_xgb.predict_proba(X_test_v7)[:, 1]
p_rf = v7_rf.predict_proba(X_test_v7)[:, 1]
p_raw = (p_xgb + p_rf) / 2

# v7 display policy: raw probability
p_display = p_raw.copy()

print("\n=== PROBABILITY VALIDATION ===")

for name, values in {
    "XGBoost": p_xgb,
    "Random Forest": p_rf,
    "Raw ensemble": p_raw,
    "V7 display": p_display
}.items():
    assert np.isfinite(values).all(), f"{name} contains non-finite values"
    assert ((values >= 0) & (values <= 1)).all(), f"{name} out of range"

    print(
        f"{name:15s} | "
        f"Min: {values.min():.4f} | "
        f"Max: {values.max():.4f} | "
        f"Mean: {values.mean():.4f}"
    )

assert np.allclose(p_display, p_raw)

print("\n=== PRODUCTION SAFETY ===")

assert Path(config.MODEL_PATH).name == "playoff_models_v6.joblib"
assert metadata.MODEL_METADATA["version"] == "v6"

print("PASS: v7 predictions generated independently")
print("PASS: v7 display probabilities equal raw ensemble")
print("PASS: production v6 remains active")

print("\n=== BLOCK 77Z COMPLETE ===")


In [ ]:
# ============================================================
# BLOCK 77AA: CREATE VERSION-AWARE SCORING MODULE
# V6 DEFAULT — V7 EXPLICIT OPT-IN
# ============================================================

from pathlib import Path
import textwrap
import importlib

import config

root = Path(config.__file__).resolve().parent
target = root / "prediction_model" / "inference" / "versioned_scoring.py"

assert not target.exists(), f"File already exists: {target}"

source = textwrap.dedent('''
    from pathlib import Path
    import pandas as pd
    from joblib import load
    from config import MODEL_PATH

    LEAK_COLS = [
        "game_id", "game_date", "date", "season_id",
        "team_name", "opponent", "clinch_type", "made_playoffs"
    ]

    ID_COLS = ["team_id", "opponent_id", "tri_code"]

    def score_version(df_features: pd.DataFrame, version: str = "v6") -> pd.DataFrame:
        if version not in {"v6", "v7"}:
            raise ValueError(f"Unsupported model version: {version}")

        path = (
            Path(MODEL_PATH) if version == "v6"
            else Path(MODEL_PATH).with_name("playoff_models_v7.joblib")
        )

        bundle = load(path)
        features = list(bundle["features"])

        df = df_features.copy()
        X = (
            df.drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
              .select_dtypes(include="number")
              .copy()
        )

        missing = [c for c in features if c not in X.columns]
        if missing:
            raise ValueError(f"Missing model features: {missing}")

        X = X.reindex(columns=features).fillna(0)

        df["playoff_prob_xgb"] = bundle["xgb_pipeline"].predict_proba(X)[:, 1]
        df["playoff_prob_rf"] = bundle["rf_pipeline"].predict_proba(X)[:, 1]
        df["playoff_prob"] = (
            df["playoff_prob_xgb"] + df["playoff_prob_rf"]
        ) / 2

        # Preserve v6 behavior; v7 displays unadjusted probabilities.
        if version == "v6":
            early = df["games_played"].isna() | df["games_played"].le(20)
            df["playoff_prob_display"] = df["playoff_prob"].where(
                ~early, 0.45 * df["playoff_prob"] + 0.275
            )
        else:
            df["playoff_prob_display"] = df["playoff_prob"]

        df["model_version"] = version
        return df
''').lstrip()

target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(source, encoding="utf-8")

# Validate syntax without importing or executing the new module.
compile(source, str(target), "exec")

# Verify the active configuration remains untouched.
metadata = importlib.import_module("prediction_model.metadata")

assert Path(config.MODEL_PATH).name == "playoff_models_v6.joblib"
assert metadata.MODEL_METADATA["version"] == "v6"

print("=== BLOCK 77AA: VERSION-AWARE SCORING ===")
print("Created:", target.relative_to(root))
print("PASS: Module syntax validated")
print("PASS: v6 remains the default")
print("PASS: v7 requires explicit selection")
print("PASS: v7 uses raw display probabilities")
print("PASS: Production configuration unchanged")
print("=== BLOCK 77AA COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 78A: VERSION-AWARE SCORING INTEGRATION TEST
# READ-ONLY — PRODUCTION V6 UNCHANGED
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
import config
import prediction_model.metadata as metadata

from prediction_model.inference import score_dataframe
from prediction_model.inference.versioned_scoring import score_version

print("=== BLOCK 78A: VERSION-AWARE SCORING TEST ===")

# ------------------------------------------------------------
# 1. Prepare historical test data
# ------------------------------------------------------------

assert "historical_features" in globals(), "Historical features missing"

test_df = historical_features.copy()

assert "games_played" in test_df.columns

# Include both early- and late-season snapshots
early_df = test_df[test_df["games_played"].between(1, 20)].head(50)
late_df = test_df[test_df["games_played"] > 20].head(50)

sample = pd.concat([early_df, late_df], ignore_index=True)

assert len(early_df) == 50
assert len(late_df) == 50

print("Sample rows:", len(sample))
print("Early-season rows:", len(early_df))
print("Later-season rows:", len(late_df))

# ------------------------------------------------------------
# 2. Score with existing v6 and versioned v6
# ------------------------------------------------------------

existing_v6 = score_dataframe(sample)
versioned_v6 = score_version(sample, version="v6")

v6_diff = np.max(
    np.abs(
        existing_v6["playoff_prob"].to_numpy()
        - versioned_v6["playoff_prob"].to_numpy()
    )
)

assert np.allclose(
    existing_v6["playoff_prob"],
    versioned_v6["playoff_prob"],
    rtol=0,
    atol=1e-12
)

print("\n=== V6 COMPATIBILITY ===")
print(f"Maximum probability difference: {v6_diff:.12f}")
print("PASS: Versioned v6 matches existing production scoring")

# ------------------------------------------------------------
# 3. Validate v6 display adjustment
# ------------------------------------------------------------

early_mask = versioned_v6["games_played"].le(20)

expected_v6_display = np.where(
    early_mask,
    0.45 * versioned_v6["playoff_prob"] + 0.275,
    versioned_v6["playoff_prob"]
)

assert np.allclose(
    versioned_v6["playoff_prob_display"],
    expected_v6_display
)

print("PASS: v6 display adjustment preserved")

# ------------------------------------------------------------
# 4. Score with independent v7
# ------------------------------------------------------------

versioned_v7 = score_version(sample, version="v7")

assert np.allclose(
    versioned_v7["playoff_prob_display"],
    versioned_v7["playoff_prob"],
    rtol=0,
    atol=1e-12
)

assert (versioned_v7["model_version"] == "v7").all()
assert (versioned_v6["model_version"] == "v6").all()

print("\n=== V7 SCORING ===")
print("PASS: v7 loaded and scored independently")
print("PASS: v7 display equals raw ensemble")
print("PASS: Model versions correctly identified")

# ------------------------------------------------------------
# 5. Validate probability outputs
# ------------------------------------------------------------

for version, frame in [
    ("v6", versioned_v6),
    ("v7", versioned_v7)
]:
    for col in [
        "playoff_prob_xgb",
        "playoff_prob_rf",
        "playoff_prob",
        "playoff_prob_display"
    ]:
        values = frame[col].to_numpy()

        assert np.isfinite(values).all(), f"{version}: {col} non-finite"
        assert ((values >= 0) & (values <= 1)).all(), (
            f"{version}: {col} outside [0, 1]"
        )

    print(
        f"{version} | "
        f"Min: {frame['playoff_prob'].min():.4f} | "
        f"Max: {frame['playoff_prob'].max():.4f} | "
        f"Mean: {frame['playoff_prob'].mean():.4f}"
    )

# ------------------------------------------------------------
# 6. Confirm production configuration unchanged
# ------------------------------------------------------------

assert Path(config.MODEL_PATH).name == "playoff_models_v6.joblib"
assert metadata.MODEL_METADATA["version"] == "v6"

print("\n=== PRODUCTION SAFETY ===")
print("PASS: Production model remains v6")
print("PASS: Production metadata remains v6")
print("PASS: No database writes performed")

print("\n=== BLOCK 78A COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 78B: DAILY INFERENCE INTEGRATION AUDIT
# READ-ONLY — NO DATABASE OR MODEL MODIFICATIONS
# ============================================================

from pathlib import Path
import inspect
import prediction_model.inference.daily_playoff_probability as daily

print("=== BLOCK 78B: DAILY INFERENCE INTEGRATION AUDIT ===")

source = inspect.getsource(daily)
lines = source.splitlines()

sections = [
    ("SCORING & FEATURE ENGINEERING", 170, 270),
    ("EXISTING RECORD FILTERING", 260, 325),
    ("METADATA & DISPLAY", 320, 410),
    ("DATABASE WRITES", 435, len(lines))
]

for title, start, end in sections:
    print(f"\n=== {title} ===")

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:4d} | {lines[i]}")

print("\n=== BLOCK 78B COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 78C: VERSION-AWARE DAILY INFERENCE
# SOURCE EDIT ONLY — NO DATABASE OR MODEL MODIFICATIONS
# ============================================================

from pathlib import Path
import shutil
import importlib
import ast

daily = importlib.import_module(
    "prediction_model.inference.daily_playoff_probability"
)

path = Path(daily.__file__)
source = path.read_text(encoding="utf-8")
original = source

print("=== BLOCK 78C: VERSION-AWARE DAILY INFERENCE ===")
print("Target:", path)

# ------------------------------------------------------------
# 1. Add version-aware scoring import
# ------------------------------------------------------------

old = "from prediction_model.inference import score_dataframe"
new = (
    "from prediction_model.inference import score_dataframe\n"
    "from prediction_model.inference.versioned_scoring import score_version"
)

assert source.count(old) == 1
source = source.replace(old, new, 1)

# ------------------------------------------------------------
# 2. Add explicit model_version parameter
# ------------------------------------------------------------

old = "    refresh_existing: bool = False,\n) -> None:"
new = (
    '    refresh_existing: bool = False,\n'
    '    model_version: str = "v6",\n'
    ') -> None:'
)

assert source.count(old) == 1
source = source.replace(old, new, 1)

# ------------------------------------------------------------
# 3. Validate selected version before database operations
# ------------------------------------------------------------

old = '    print(f"▶ Running playoff probabilities for season {season_id}")'

new = (
    '    if model_version not in {"v6", "v7"}:\n'
    '        raise ValueError(f"Unsupported model version: {model_version}")\n\n'
    '    print(f"▶ Running playoff probabilities for season {season_id} ({model_version})")'
)

assert source.count(old) == 1
source = source.replace(old, new, 1)

# ------------------------------------------------------------
# 4. Select the correct scoring implementation
# ------------------------------------------------------------

old = (
    "    df_scored = score_dataframe(\n"
    "        df_features\n"
    "    )"
)

new = (
    '    if model_version == "v6":\n'
    '        df_scored = score_dataframe(df_features)\n'
    '    else:\n'
    '        df_scored = score_version(df_features, version=model_version)'
)

assert source.count(old) == 1
source = source.replace(old, new, 1)

# ------------------------------------------------------------
# 5. Replace version metadata references
# ------------------------------------------------------------

old = 'MODEL_METADATA["version"]'

# Three occurrences expected:
# existing-record lookup, metadata, DELETE
assert source.count(old) == 3

source = source.replace(old, "model_version")

# ------------------------------------------------------------
# 6. Make display adjustment version-aware
# ------------------------------------------------------------

old = 'if bucket == "early":'
new = 'if model_version == "v6" and bucket == "early":'

assert source.count(old) == 1
source = source.replace(old, new, 1)

# ------------------------------------------------------------
# 7. Validate before writing
# ------------------------------------------------------------

ast.parse(source)
assert source != original

backup = path.with_suffix(".py.bak_78C")
assert not backup.exists(), f"Backup already exists: {backup}"

shutil.copy2(path, backup)
path.write_text(source, encoding="utf-8")

# ------------------------------------------------------------
# 8. Verify saved source
# ------------------------------------------------------------

saved = path.read_text(encoding="utf-8")

assert saved == source
assert 'model_version: str = "v6"' in saved
assert 'score_version(df_features, version=model_version)' in saved
assert 'if model_version == "v6" and bucket == "early":' in saved
assert 'MODEL_METADATA["version"]' not in saved

print("\n=== VALIDATION ===")
print("PASS: Original source backed up")
print("PASS: Version-aware scoring implemented")
print("PASS: v6 remains the default")
print("PASS: v7 uses its own model artifact")
print("PASS: Database operations use selected version")
print("PASS: v7 display adjustment disabled")
print("PASS: Python syntax validated")
print("PASS: No database operations executed")

print("\nBackup:", backup.name)
print("=== BLOCK 78C COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 78D FIX: RELOAD UPDATED DAILY INFERENCE MODULE
# ============================================================

import importlib
import inspect

daily = importlib.import_module(
    "prediction_model.inference.daily_playoff_probability"
)

daily = importlib.reload(daily)

print("=== DAILY INFERENCE RELOAD ===")
print("Function signature:")
print(inspect.signature(daily.run_probability_scores))

assert "model_version" in inspect.signature(
    daily.run_probability_scores
).parameters

print("\nPASS: Updated function loaded")
print("PASS: model_version parameter available")
print("PASS: No database modifications")

In [ ]:
# ============================================================
# BLOCK 78D: FINAL V7 INTEGRATION VALIDATION
# TEMPORARY DATABASE — PRODUCTION UNCHANGED
# ============================================================

import duckdb
import pandas as pd
import numpy as np
import tempfile
import shutil
import hashlib
import importlib
from pathlib import Path

from config import DB_PATH, EXCLUDE_SEASON_ID

daily = importlib.import_module(
    "prediction_model.inference.daily_playoff_probability"
)

print("=== BLOCK 78D: FINAL INTEGRATION VALIDATION ===")

source_db = Path(DB_PATH).resolve()
assert source_db.exists(), f"Database not found: {source_db}"

# ------------------------------------------------------------
# 1. Create isolated temporary database
# ------------------------------------------------------------

def table_fingerprint(con, table, version):
    df = con.execute(
        f"""
        SELECT *
        FROM {table}
        WHERE model_version = ?
        ORDER BY season_id, game_date, team_id
        """,
        [version]
    ).df()

    digest = hashlib.sha256(
        pd.util.hash_pandas_object(
            df, index=False
        ).values.tobytes()
    ).hexdigest()

    return len(df), digest


with tempfile.TemporaryDirectory(prefix="hockey_v7_test_") as temp_dir:

    temp_db = Path(temp_dir) / "validation.duckdb"

    # Use DuckDB's consistent database export rather than copying
    # a potentially active database file.
    source_con = duckdb.connect(str(source_db), read_only=True)

    try:
        export_dir = Path(temp_dir) / "export"
        source_con.execute(
            f"EXPORT DATABASE '{export_dir.as_posix()}'"
        )
    finally:
        source_con.close()

    con = duckdb.connect(str(temp_db))

    try:
        con.execute(
            f"IMPORT DATABASE '{export_dir.as_posix()}'"
        )

        table = daily.TARGET_TABLE

        # ----------------------------------------------------
        # 2. Snapshot existing v6 predictions
        # ----------------------------------------------------

        v6_before = table_fingerprint(con, table, "v6")

        print("\n=== BEFORE V7 ===")
        print("Existing v6 rows:", v6_before[0])

        # ----------------------------------------------------
        # 3. Execute v7 on isolated database
        # ----------------------------------------------------

        daily.run_probability_scores(
            con=con,
            season_id=EXCLUDE_SEASON_ID,
            refresh_existing=True,
            model_version="v7"
        )

        # ----------------------------------------------------
        # 4. Verify v6 was not modified
        # ----------------------------------------------------

        v6_after = table_fingerprint(con, table, "v6")

        assert v6_before == v6_after, (
            "FAIL: v6 records changed during v7 execution"
        )

        print("\n=== PRODUCTION COMPATIBILITY ===")
        print("PASS: Existing v6 rows unchanged")
        print("PASS: Existing v6 data fingerprint unchanged")

        # ----------------------------------------------------
        # 5. Validate v7 records
        # ----------------------------------------------------

        v7 = con.execute(
            f"""
            SELECT *
            FROM {table}
            WHERE season_id = ?
              AND model_version = 'v7'
            """,
            [EXCLUDE_SEASON_ID]
        ).df()

        assert not v7.empty, "No v7 predictions written"

        assert v7.duplicated(
            ["season_id", "game_date", "team_id", "model_version"]
        ).sum() == 0

        assert np.allclose(
            v7["playoff_prob"],
            v7["playoff_prob_display"],
            rtol=0,
            atol=1e-12
        )

        for col in [
            "playoff_prob",
            "playoff_prob_xgb",
            "playoff_prob_rf",
            "playoff_prob_display"
        ]:
            assert np.isfinite(v7[col]).all()
            assert v7[col].between(0, 1).all()

        print("\n=== V7 DATABASE VALIDATION ===")
        print("PASS: v7 predictions written successfully")
        print("PASS: No duplicate team-day-version keys")
        print("PASS: v7 display equals raw probability")
        print("PASS: All probabilities valid")
        print("v7 rows:", len(v7))
        print("v7 teams:", v7["team_id"].nunique())
        print("v7 dates:", v7["game_date"].nunique())

    finally:
        con.close()

print("\n=== SAFETY CHECK ===")
print("PASS: Temporary database removed")
print("PASS: Production database not written to")

print("\n=== BLOCK 78D COMPLETE ===")

In [ ]:
# ============================================================
# BLOCK 79A: DEPLOY V7 ALONGSIDE V6 — CORRECTED
# PRODUCTION DATABASE — V6 PRESERVED
# ============================================================

import duckdb
import importlib
import pandas as pd
from config import DB_PATH, EXCLUDE_SEASON_ID

daily = importlib.reload(
    importlib.import_module(
        "prediction_model.inference.daily_playoff_probability"
    )
)

con = duckdb.connect(DB_PATH)

try:
    table = daily.TARGET_TABLE
    season = EXCLUDE_SEASON_ID

    print("=== V7 PARALLEL DEPLOYMENT ===")
    print("Target table:", table)
    print("Season:", season)

    # Snapshot v6 records before deployment
    v6_before = con.execute(f"""
        SELECT *
        FROM {table}
        WHERE season_id = ? AND model_version = 'v6'
        ORDER BY game_date, team_id
    """, [season]).df()

    print("Existing v6 rows:", len(v6_before))

    # Deploy v7 independently
    daily.run_probability_scores(
        con=con,
        season_id=season,
        refresh_existing=True,
        model_version="v7"
    )

    # Verify v6 records remain identical
    v6_after = con.execute(f"""
        SELECT *
        FROM {table}
        WHERE season_id = ? AND model_version = 'v6'
        ORDER BY game_date, team_id
    """, [season]).df()

    pd.testing.assert_frame_equal(v6_before, v6_after)

    print("\nPASS: All existing v6 records unchanged")

    # Confirm both models are stored
    results = con.execute(f"""
        SELECT
            model_version,
            COUNT(*) AS predictions,
            COUNT(DISTINCT team_id) AS teams,
            COUNT(DISTINCT game_date) AS dates
        FROM {table}
        WHERE season_id = ?
          AND model_version IN ('v6', 'v7')
        GROUP BY model_version
        ORDER BY model_version
    """, [season]).df()

    print("\n=== DEPLOYMENT RESULTS ===")
    print(results.to_string(index=False))

    assert {"v6", "v7"}.issubset(set(results["model_version"]))

    print("\nPASS: Both model versions present")
    print("PASS: v6 remains the production default")
    print("=== BLOCK 79A COMPLETE ===")

finally:
    con.close()

In [ ]:
# ============================================================
# BLOCK 79B: V6 VS V7 — CURRENT TEAM COMPARISON
# READ-ONLY
# ============================================================

import duckdb
import pandas as pd
from config import DB_PATH, EXCLUDE_SEASON_ID

con = duckdb.connect(DB_PATH, read_only=True)

try:
    table = daily.TARGET_TABLE

    comparison = con.execute(f"""
        WITH latest_date AS (
            SELECT MAX(game_date) AS game_date
            FROM {table}
            WHERE season_id = ?
              AND model_version IN ('v6', 'v7')
        )
        SELECT
            a.game_date,
            a.team_id,
            a.playoff_prob AS v6_raw,
            a.playoff_prob_display AS v6_display,
            b.playoff_prob AS v7_raw,
            b.playoff_prob_display AS v7_display
        FROM {table} a
        JOIN {table} b
          ON a.season_id = b.season_id
         AND a.game_date = b.game_date
         AND a.team_id = b.team_id
        WHERE a.season_id = ?
          AND a.model_version = 'v6'
          AND b.model_version = 'v7'
          AND a.game_date = (SELECT game_date FROM latest_date)
        ORDER BY a.team_id
    """, [EXCLUDE_SEASON_ID, EXCLUDE_SEASON_ID]).df()

    assert len(comparison) == 32, (
        f"Expected 32 teams, found {len(comparison)}"
    )

    comparison["raw_change_pp"] = (
        comparison["v7_raw"] - comparison["v6_raw"]
    ) * 100

    comparison["display_change_pp"] = (
        comparison["v7_display"] - comparison["v6_display"]
    ) * 100

    print("=== BLOCK 79B: V6 VS V7 ===")
    print("Date:", comparison["game_date"].iloc[0])
    print("Teams compared:", len(comparison))

    print("\n=== RAW PROBABILITY DIFFERENCES ===")
    print(
        comparison["raw_change_pp"]
        .describe()
        .round(2)
        .to_string()
    )

    print("\n=== LARGEST MODEL DISAGREEMENTS ===")
    print(
        comparison.reindex(
            comparison["raw_change_pp"].abs()
            .sort_values(ascending=False).index
        )[
            ["team_id", "v6_raw", "v7_raw", "raw_change_pp"]
        ]
        .head(15)
        .round(3)
        .to_string(index=False)
    )

    print("\n=== DISPLAY PROBABILITY DIFFERENCES ===")
    print(
        comparison["display_change_pp"]
        .describe()
        .round(2)
        .to_string()
    )

    print("\n=== BLOCK 79B COMPLETE ===")

finally:
    con.close()

In [ ]:
# ============================================================
# BLOCK 80A: HARDCODED SEASON AUDIT
# READ-ONLY — NO FILE MODIFICATIONS
# ============================================================

from pathlib import Path
import re

root = Path.cwd()

patterns = [
    r"20252026",
    r"20262027",
    r"EXCLUDE_SEASON_ID",
    r"CURRENT_SEASON_ID",
    r"exclude.*season",
    r"season.*exclude",
]

skip_dirs = {
    ".git", ".venv", "venv", "new_hockey_env",
    "__pycache__", ".ipynb_checkpoints", "site-packages"
}

print("=== BLOCK 80A: HARDCODED SEASON AUDIT ===")

matches = []

for path in root.rglob("*.py"):
    if any(part in skip_dirs for part in path.relative_to(root).parts):
        continue

    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except (UnicodeError, OSError):
        continue

    for line_num, line in enumerate(lines, 1):
        if any(re.search(p, line, re.IGNORECASE) for p in patterns):
            matches.append((str(path.relative_to(root)), line_num, line.strip()))

for filename, line_num, line in matches:
    print(f"{filename}:{line_num} | {line[:180]}")

print(f"\nTotal matches: {len(matches)}")
print("=== BLOCK 80A COMPLETE ===")

In [ ]:
con.execute("""
    CREATE TABLE IF NOT EXISTS pipeline_run_log (
        run_id UUID PRIMARY KEY,
        season_id BIGINT,
        started_at TIMESTAMPTZ,
        finished_at TIMESTAMPTZ,
        status VARCHAR,
        error_message VARCHAR
    )
""")

In [ ]:
con.execute("""
    SELECT
        season_id,
        started_at,
        finished_at,
        status,
        error_message
    FROM pipeline_run_log
    ORDER BY started_at DESC
    LIMIT 10
""").df()

In [ ]:
con.execute("DESCRIBE ").fetchdf()

In [ ]:
con.close()
print("DuckDB connection closed.")
